In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
from time import perf_counter

import hashlib
import json
import os
import pickle
import re
import tempfile
import traceback

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


# =============================================================================
# USER SETTINGS
# =============================================================================

EXPECTED_M03_VERSION = "2026-10-01-M03-seam-safe-cprime-20m-v1"

PICKRELL_SOURCE_DIR = Path(
    r"C:\Users\sruddy1\Documents\satuRnPaper\Pickrell_null_benchmark"
)
COUNTS_FILE = PICKRELL_SOURCE_DIR / "pickrell_29_male_counts.csv"
SPLITS_FILE = PICKRELL_SOURCE_DIR / "pickrell_30_random_5v5_splits.csv"

RUN_NAMESPACE = "pickrell_m03_seam_safe_20M_kappa_coverage_v1"
N_COVERAGE_BINS = 12
RAW_P_THRESHOLD = 0.01
FDR_CUTOFF = 0.05

# Fresh once because RUN_NAMESPACE is new; resume automatically thereafter.
FRESH_START = False
RESUME = True
RETRY_FAILED_ON_RESUME = True

N_JOBS_EXACT = 6
M03_VERBOSE = 1
M03_EXACT_MODE = "default"
REPLICATES = None  # None = all 30 frozen 5v5 splits

M03_NUMERICAL_SETTINGS = {
    "use_tmm": True,
    "alpha_floor": 1e-8,
    "alpha_cap": 20.0,
    "deb_dp_maxiter": 2000,
    "deb_dp_flex_gamma_bounds": (0.05, 3.0),
    "theta_floor": 1e-6,
    "theta_cap": 1e6,
    "mean_fit_maxiter": 50,
    "mean_fit_tol": 1e-10,
    "web_maxiter": 2000,
    "mixture_maxiter": 2000,
    "ash_n_scales": 30,
    "ash_em_maxiter": 2000,
    "ash_em_tol": 1e-10,
    "ash_prior_pseudocount": 1e-8,
    "cprime_grid_n_mu": 90,
    "cprime_grid_n_r": 80,
    "cprime_grid_max_refinements": 2,
    "cprime_grid_emergency_refinements": 2,
    "cprime_audit_n": 120,
    "cprime_audit_tol": 0.001,
    "theta_cprime_order": 3,
    "theta_cprime_audit_n": 120,
    "theta_cprime_audit_tol": 0.001,
    "theta_cprime_max_abs_delta": 0.25,
    "cprime_tail_probability": 1e-12,
    "cprime_support_max": 20_000_000,
    "cprime_expect_maxcount": 1_000_000,
    "cprime_chunk_size": 200_000,
    "cprime_grid_exact_width_max": 250_000,
    "cprime_gamma_quad_n": 128,
    "cprime_gamma_min_mu_over_r": 100.0,
    "cprime_continuum_audit_quad_n": 1024,
    "cprime_continuum_audit_quad_n_max": 8192,
    "cprime_continuum_audit_min_mu": 50_000.0,
    "cprime_continuum_audit_min_mu_over_r": 100.0,
    "cprime_continuum_audit_selfcheck_tol": 1e-5,
    "corrected_score_tolerance": 1e-10,
    "deb_precision_maxiter": 2000,
    "exact_theta_search_min": 1e-5,
    "exact_theta_search_max": 1e16,
    "exact_score_root_xtol": 1e-10,
    "exact_score_root_rtol": 1e-10,
    "exact_score_zero_tol": 1e-8,
    "exact_profile_local_step": 0.01,
    "support_floor_tolerance": 1e-12,
}


# =============================================================================
# PATHS
# =============================================================================

CWD = Path.cwd().resolve()
if CWD.name.lower() in {"notebook", "notebooks"}:
    PROJECT_ROOT = CWD.parent
elif (CWD / "data").is_dir():
    PROJECT_ROOT = CWD
else:
    raise RuntimeError(
        "Run this from <repo>/notebook, <repo>/notebooks, or the repo root. "
        f"Current working directory: {CWD}"
    )

OUTPUT_DIR = PROJECT_ROOT / "data" / RUN_NAMESPACE
CELL_DIR = OUTPUT_DIR / "cells"
BIN_DIR = OUTPUT_DIR / "coverage_bins"
FAILURE_DIR = OUTPUT_DIR / "failures"
M03_CHECKPOINT_ROOT = OUTPUT_DIR / "m03_checkpoints"
SENTINEL = OUTPUT_DIR / "_initialized.json"


# =============================================================================
# UTILITIES
# =============================================================================

def _now() -> str:
    return datetime.now().astimezone().strftime("%Y-%m-%d %I:%M:%S %p %Z")


def _safe(x: object) -> str:
    x = re.sub(r"[^A-Za-z0-9._-]+", "_", str(x))
    return x.strip("._") or "unnamed"


def _atomic_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp_name = tempfile.mkstemp(prefix=f".{path.name}.", dir=str(path.parent))
    os.close(fd)
    tmp = Path(tmp_name)
    try:
        df.to_csv(tmp, index=False)
        os.replace(tmp, path)
    finally:
        if tmp.exists():
            try:
                tmp.unlink()
            except OSError:
                pass


def _atomic_text(text: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp_name = tempfile.mkstemp(prefix=f".{path.name}.", dir=str(path.parent))
    os.close(fd)
    tmp = Path(tmp_name)
    try:
        tmp.write_text(str(text), encoding="utf-8")
        os.replace(tmp, path)
    finally:
        if tmp.exists():
            try:
                tmp.unlink()
            except OSError:
                pass


def _load_pickle(path: Path):
    with open(path, "rb") as f:
        return pickle.load(f)


def _load_success_stage(checkpoint_dir: Path, stage_name: str):
    path = checkpoint_dir / "stages" / f"{stage_name}.pkl"
    if not path.exists():
        raise FileNotFoundError(path)
    payload = _load_pickle(path)
    if not isinstance(payload, dict) or payload.get("status") != "success":
        raise RuntimeError(f"Stage is not successful: {path}")
    return payload["data"]


def _weighted_mean(x, w):
    x = np.asarray(x, float)
    w = np.asarray(w, float)
    ok = np.isfinite(x) & np.isfinite(w) & (w >= 0)
    if not np.any(ok):
        return np.nan
    sw = float(w[ok].sum())
    if sw <= 0:
        return np.nan
    return float(np.sum(w[ok] * x[ok]) / sw)


def _seed(rep: int) -> int:
    h = hashlib.sha256(f"PICKRELL-M03-20M|{int(rep)}".encode()).digest()
    return int.from_bytes(h[:4], "little") % 2_147_483_647 or 1


def _paths(rep: int):
    stem = f"pickrell_rep{int(rep):02d}"
    return (
        CELL_DIR / f"{stem}.csv",
        BIN_DIR / f"{stem}__coverage_bins.csv",
        FAILURE_DIR / f"{stem}__FAILED.txt",
        M03_CHECKPOINT_ROOT / stem,
    )


def _existing(rep: int):
    summary_path, bins_path, _, _ = _paths(rep)
    if not RESUME or not summary_path.exists() or not bins_path.exists():
        return None
    try:
        summary = pd.read_csv(summary_path)
        bins = pd.read_csv(bins_path)
    except Exception:
        return None
    if len(summary) == 1 and str(summary.loc[0, "status"]) == "success" and not bins.empty:
        return summary, bins
    return None


# =============================================================================
# INPUTS
# =============================================================================

def _load_inputs():
    if not COUNTS_FILE.exists():
        raise FileNotFoundError(COUNTS_FILE)
    if not SPLITS_FILE.exists():
        raise FileNotFoundError(SPLITS_FILE)

    counts_df = pd.read_csv(COUNTS_FILE)
    splits = pd.read_csv(SPLITS_FILE)

    if "gene_id" not in counts_df.columns:
        raise RuntimeError("Counts file is missing gene_id.")

    required = {"replicate", "sample_id", "group"}
    missing = required - set(splits.columns)
    if missing:
        raise RuntimeError(f"Splits file missing columns: {sorted(missing)}")

    gene_ids = counts_df["gene_id"].astype(str).to_numpy()
    sample_names = counts_df.columns.drop("gene_id").astype(str).to_numpy()
    counts = counts_df.drop(columns=["gene_id"]).to_numpy(float)

    if len(set(gene_ids)) != len(gene_ids):
        raise RuntimeError("Duplicate gene IDs in Pickrell counts.")
    if len(set(sample_names)) != len(sample_names):
        raise RuntimeError("Duplicate sample IDs in Pickrell counts.")
    if np.any(~np.isfinite(counts)) or np.any(counts < 0):
        raise RuntimeError("Counts must be finite and nonnegative.")

    sample_to_col = {s: i for i, s in enumerate(sample_names)}

    splits["replicate"] = splits["replicate"].astype(int)
    splits["sample_id"] = splits["sample_id"].astype(str)
    splits["group"] = splits["group"].astype(str)

    available = sorted(pd.unique(splits["replicate"]).astype(int).tolist())
    selected = available if REPLICATES is None else [int(x) for x in REPLICATES]

    unknown = sorted(set(selected) - set(available))
    if unknown:
        raise RuntimeError(f"Unknown Pickrell replicate IDs: {unknown}")

    for rep in selected:
        x = splits.loc[splits["replicate"] == rep]
        if len(x) != 10 or x["sample_id"].nunique() != 10:
            raise RuntimeError(f"Replicate {rep} is not a 10-sample frozen split.")
        vc = x["group"].value_counts().to_numpy()
        if sorted(vc.tolist()) != [5, 5]:
            raise RuntimeError(f"Replicate {rep} is not 5-vs-5.")
        missing_samples = [s for s in x["sample_id"] if s not in sample_to_col]
        if missing_samples:
            raise RuntimeError(f"Replicate {rep} missing samples: {missing_samples}")

    return {
        "counts": counts,
        "gene_ids": gene_ids,
        "sample_names": sample_names,
        "sample_to_col": sample_to_col,
        "splits": splits,
        "selected": selected,
    }


# =============================================================================
# kappa_c(coverage) AUDIT FROM CURRENT M03 CHECKPOINTS
# =============================================================================

def _audit_fit(rep: int, result: dict, checkpoint_dir: Path, n_genes_fixed: int):
    core = _load_success_stage(checkpoint_dir, "01_core")
    full = _load_success_stage(checkpoint_dir, "02_full_firstorder")
    geom = _load_success_stage(checkpoint_dir, "03_geometry_support")

    gene_ids = np.asarray(core["gene_ids"]).astype(str)
    y = np.asarray(core["y"], float)
    exposure = np.asarray(core["exposure"], float)
    theta_F = np.asarray(full["theta_F_first"], float)
    delta = np.asarray(full["delta"], float)
    weights = np.asarray(geom["weights"], float)
    geom_valid = np.asarray(geom["geom_valid"], bool)
    global_kappa = float(geom["kappa_c"])

    total_cov = np.sum(y / exposure[None, :], axis=1)
    log_cov = np.log1p(total_cov)
    Q_c = 2.0 * theta_F * delta

    valid = geom_valid & np.isfinite(Q_c) & np.isfinite(weights) & (weights >= 0)
    reconstructed = _weighted_mean(Q_c[valid], weights[valid])
    relerr = abs(reconstructed - global_kappa) / max(abs(global_kappa), 1e-15)
    if not np.isfinite(reconstructed) or relerr > 1e-9:
        raise RuntimeError(
            f"rep {rep}: global kappa mismatch: saved={global_kappa}, "
            f"reconstructed={reconstructed}, relerr={relerr:.3e}"
        )

    # Align p/q to Stage-1 gene order.
    gr = result["gene_results"].copy()
    lookup = gr.assign(gene_id=gr["gene_id"].astype(str)).set_index("gene_id")
    aligned = pd.DataFrame(index=pd.Index(gene_ids, name="gene_id")).join(
        lookup[["p_value", "q_value"]], how="left"
    )
    p = aligned["p_value"].to_numpy(float)
    q = aligned["q_value"].to_numpy(float)

    bin_id = np.asarray(
        pd.qcut(pd.Series(log_cov), q=N_COVERAGE_BINS, labels=False, duplicates="drop"),
        dtype=int,
    )
    n_bins = int(np.max(bin_id)) + 1

    rows = []
    for b in range(n_bins):
        in_bin = bin_id == b
        use = in_bin & valid
        kappa_bin = _weighted_mean(Q_c[use], weights[use])
        if not np.isfinite(kappa_bin) or kappa_bin <= 0:
            raise RuntimeError(f"rep {rep}, bin {b+1}: invalid kappa_c={kappa_bin}")
        n_bin = int(in_bin.sum())
        rows.append({
            "replicate": rep,
            "coverage_bin": b + 1,
            "n_genes_bin": n_bin,
            "n_geometry_valid_bin": int(use.sum()),
            "median_log1p_total_normalized_coverage": float(np.median(log_cov[in_bin])),
            "median_total_normalized_coverage": float(np.median(total_cov[in_bin])),
            "kappa_c_global": global_kappa,
            "kappa_c_bin": float(kappa_bin),
            "kappa_bin_over_global": float(kappa_bin / global_kappa),
            "raw_p_lt_0_01_fraction_bin": float(
                np.sum(in_bin & np.isfinite(p) & (p < RAW_P_THRESHOLD)) / max(n_bin, 1)
            ),
            "bh_q_le_0_05_fraction_bin": float(
                np.sum(in_bin & np.isfinite(q) & (q <= FDR_CUTOFF)) / max(n_bin, 1)
            ),
        })

    bins = pd.DataFrame(rows)
    d = result["diagnostics"]
    summary = pd.DataFrame([{
        "replicate": rep,
        "status": "success",
        "n_genes_fixed": int(n_genes_fixed),
        "n_genes_analyzed": int(len(gene_ids)),
        "global_kappa_c": global_kappa,
        "lambda_c": float(d.get("lambda_c", np.nan)),
        "ASH_pi0": float(d.get("ASH_pi0", np.nan)),
        "fraction_raw_p_lt_0_01_fixed_denominator": float(
            np.sum(np.isfinite(p) & (p < RAW_P_THRESHOLD)) / max(n_genes_fixed, 1)
        ),
        "n_bh_q_le_0_05": int(np.sum(np.isfinite(q) & (q <= FDR_CUTOFF))),
        "any_bh_q_le_0_05": bool(np.any(np.isfinite(q) & (q <= FDR_CUTOFF))),
        "n_R_F_nonpositive": int(d.get("n_R_F_nonpositive", -1)),
        "n_m03_mandatory_exact_success": int(d.get("n_m03_mandatory_exact_success", -1)),
        "cprime_build_seconds": float(d.get("cprime_build_seconds", np.nan)),
        "m03_code_version": str(d.get("code_version", "")),
    }])

    return summary, bins


# =============================================================================
# ONE SPLIT
# =============================================================================

def _run_one(rep: int, inputs: dict):
    summary_path, bins_path, failure_path, checkpoint_dir = _paths(rep)
    old = _existing(rep)
    if old is not None:
        summary, bins = old
        print(
            f"[rep reload] {_now()} | rep={rep:02d} | "
            f"kappa={float(summary.loc[0, 'global_kappa_c']):.6f} | "
            f"raw p<.01={float(summary.loc[0, 'fraction_raw_p_lt_0_01_fixed_denominator']):.6f}",
            flush=True,
        )
        return summary, bins, True

    split = inputs["splits"].loc[inputs["splits"]["replicate"] == rep].copy()
    sample_ids = split["sample_id"].astype(str).to_numpy()
    group = split["group"].astype(str).to_numpy()
    cols = np.asarray([inputs["sample_to_col"][s] for s in sample_ids], int)
    y = np.asarray(inputs["counts"][:, cols], float)

    print(
        f"[rep start] {_now()} | rep={rep:02d} | genes={y.shape[0]:,} | samples=10 | seed={_seed(rep)}",
        flush=True,
    )
    t0 = perf_counter()

    try:
        result = fit_webdnb_m03(
            mode="observed",
            y=y,
            group=group,
            gene_ids=inputs["gene_ids"],
            sample_names=sample_ids,
            kappa_F=None,
            random_seed=_seed(rep),
            numerical_settings=dict(M03_NUMERICAL_SETTINGS),
            checkpoint_dir=str(checkpoint_dir),
            fresh_start=FRESH_START,
            resume=RESUME,
            retry_failed_on_resume=RETRY_FAILED_ON_RESUME,
            n_jobs_exact=N_JOBS_EXACT,
            verbose=M03_VERBOSE,
            exact_mode=M03_EXACT_MODE,
            p_cutoff=None,
            fdr_cutoff=None,
            exact_top_n=None,
            exact_top_fraction=None,
            exact_gene_ids=None,
            refine_from=None,
        )

        summary, bins = _audit_fit(rep, result, checkpoint_dir, len(inputs["gene_ids"]))
        summary["elapsed_seconds"] = float(perf_counter() - t0)
        _atomic_csv(bins, bins_path)
        _atomic_csv(summary, summary_path)
        if failure_path.exists():
            try:
                failure_path.unlink()
            except OSError:
                pass

        print(
            f"[rep end]   {_now()} | rep={rep:02d} | {float(summary.loc[0, 'elapsed_seconds']):.1f}s | "
            f"kappa={float(summary.loc[0, 'global_kappa_c']):.6f} | "
            f"raw p<.01={float(summary.loc[0, 'fraction_raw_p_lt_0_01_fixed_denominator']):.6f} | "
            f"BH={int(summary.loc[0, 'n_bh_q_le_0_05'])}",
            flush=True,
        )
        return summary, bins, False

    except Exception as exc:
        _atomic_text(
            f"replicate: {rep}\nerror_type: {type(exc).__name__}\nerror_message: {exc}\n\n{traceback.format_exc()}",
            failure_path,
        )
        print(f"[rep FAILED] {_now()} | rep={rep:02d} | {type(exc).__name__}: {exc}", flush=True)
        raise


# =============================================================================
# AGGREGATION + PLOTS
# =============================================================================

def _collect(reps):
    ss, bb, unresolved = [], [], []
    for rep in reps:
        summary_path, bins_path, failure_path, _ = _paths(rep)
        if summary_path.exists() and bins_path.exists():
            try:
                s = pd.read_csv(summary_path)
                b = pd.read_csv(bins_path)
                if len(s) == 1 and str(s.loc[0, "status"]) == "success" and not b.empty:
                    ss.append(s)
                    bb.append(b)
                    continue
            except Exception as exc:
                unresolved.append({"replicate": rep, "state": "corrupt", "detail": repr(exc)})
                continue
        unresolved.append({
            "replicate": rep,
            "state": "failed" if failure_path.exists() else "missing",
            "detail": "",
        })
    return (
        pd.concat(ss, ignore_index=True) if ss else pd.DataFrame(),
        pd.concat(bb, ignore_index=True) if bb else pd.DataFrame(),
        pd.DataFrame(unresolved),
    )


def _aggregate_bins(bins):
    rows = []
    for b, x in bins.groupby("coverage_bin", sort=True):
        def q(col, p):
            a = x[col].astype(float).to_numpy()
            a = a[np.isfinite(a)]
            return float(np.quantile(a, p)) if len(a) else np.nan
        rows.append({
            "coverage_bin": int(b),
            "n_splits": int(x["replicate"].nunique()),
            "median_log1p_total_normalized_coverage": q("median_log1p_total_normalized_coverage", .5),
            "kappa_c_bin_median": q("kappa_c_bin", .5),
            "kappa_c_bin_q25": q("kappa_c_bin", .25),
            "kappa_c_bin_q75": q("kappa_c_bin", .75),
            "kappa_ratio_median": q("kappa_bin_over_global", .5),
            "kappa_ratio_q25": q("kappa_bin_over_global", .25),
            "kappa_ratio_q75": q("kappa_bin_over_global", .75),
            "raw_p_lt_0_01_fraction_median": q("raw_p_lt_0_01_fraction_bin", .5),
        })
    return pd.DataFrame(rows)


def _show_plots(split_summary, bins, agg):
    x = agg.sort_values("coverage_bin")
    global_kappa = split_summary["global_kappa_c"].astype(float)
    gmed = float(global_kappa.quantile(.50))
    gq25 = float(global_kappa.quantile(.25))
    gq75 = float(global_kappa.quantile(.75))

    # Absolute curve + global median/IQR.
    fig, ax = plt.subplots(figsize=(16, 9))
    for _, z in bins.groupby("replicate", sort=True):
        z = z.sort_values("coverage_bin")
        ax.plot(z["median_log1p_total_normalized_coverage"], z["kappa_c_bin"], linewidth=1, alpha=.18)
    ax.plot(
        x["median_log1p_total_normalized_coverage"], x["kappa_c_bin_median"],
        marker="o", linewidth=3, markersize=8, label="Median kappa_c(coverage)"
    )
    ax.fill_between(
        x["median_log1p_total_normalized_coverage"].to_numpy(float),
        x["kappa_c_bin_q25"].to_numpy(float), x["kappa_c_bin_q75"].to_numpy(float), alpha=.15
    )
    xmin = float(x["median_log1p_total_normalized_coverage"].min())
    xmax = float(x["median_log1p_total_normalized_coverage"].max())
    ax.plot([xmin, xmax], [gmed, gmed], linestyle="--", linewidth=2.2, label="Global kappa_c median")
    ax.fill_between([xmin, xmax], [gq25, gq25], [gq75, gq75], alpha=.10)
    ax.set_xlabel("log1p(total normalized gene coverage)", fontsize=14)
    ax.set_ylabel(r"$\kappa_c$", fontsize=14)
    ax.set_title(r"Pickrell Complete Null: $\kappa_c$(coverage) vs Global $\kappa_c$ — 30 Frozen Random 5v5 Splits", fontsize=18, pad=16)
    ax.tick_params(axis="both", labelsize=12)
    ax.grid(alpha=.20)
    ax.legend(fontsize=12)
    plt.tight_layout()
    plt.show()

    # Relative curve.
    fig, ax = plt.subplots(figsize=(16, 9))
    for _, z in bins.groupby("replicate", sort=True):
        z = z.sort_values("coverage_bin")
        ax.plot(z["median_log1p_total_normalized_coverage"], z["kappa_bin_over_global"], linewidth=1, alpha=.18)
    ax.plot(
        x["median_log1p_total_normalized_coverage"], x["kappa_ratio_median"],
        marker="o", linewidth=3, markersize=8, label="Median local/global kappa_c"
    )
    ax.fill_between(
        x["median_log1p_total_normalized_coverage"].to_numpy(float),
        x["kappa_ratio_q25"].to_numpy(float), x["kappa_ratio_q75"].to_numpy(float), alpha=.15
    )
    ax.axhline(1.0, linestyle="--", linewidth=2, label="Flat global-kappa assumption")
    ax.set_xlabel("log1p(total normalized gene coverage)", fontsize=14)
    ax.set_ylabel(r"$\kappa_c(coverage) / \kappa_c(global)$", fontsize=14)
    ax.set_title(r"Pickrell Complete Null: Relative $\kappa_c$(coverage) — 30 Frozen Random 5v5 Splits", fontsize=18, pad=16)
    ax.tick_params(axis="both", labelsize=12)
    ax.grid(alpha=.20)
    ax.legend(fontsize=12)
    plt.tight_layout()
    plt.show()

    # Raw-p calibration by coverage.
    fig, ax = plt.subplots(figsize=(16, 9))
    ax.plot(
        x["median_log1p_total_normalized_coverage"], x["raw_p_lt_0_01_fraction_median"],
        marker="o", linewidth=3, markersize=8, label="Median raw p<.01 fraction"
    )
    ax.axhline(.01, linestyle="--", linewidth=2, label="Nominal 1%")
    ax.set_xlabel("log1p(total normalized gene coverage)", fontsize=14)
    ax.set_ylabel("Fraction of genes with raw p < .01", fontsize=14)
    ax.set_title("Pickrell Complete Null: Raw-p Calibration by Coverage — 30 Frozen Random 5v5 Splits", fontsize=18, pad=16)
    ax.tick_params(axis="both", labelsize=12)
    ax.grid(alpha=.20)
    ax.legend(fontsize=12)
    plt.tight_layout()
    plt.show()


# =============================================================================
# TOP LEVEL
# =============================================================================

def run_pickrell_m03_kappa_coverage_audit():
    started = datetime.now().astimezone()
    print(f"[start] {_now()}", flush=True)

    try:
        if "fit_webdnb_m03" not in globals():
            raise RuntimeError("fit_webdnb_m03 is not defined. Run the current M03 function first.")

        loaded_version = globals().get("WEB_DNB_M03_VERSION")
        if str(loaded_version) != EXPECTED_M03_VERSION:
            raise RuntimeError(
                f"Wrong M03 implementation. Expected {EXPECTED_M03_VERSION!r}; found {loaded_version!r}."
            )

        print(
            "\nPICKRELL M03 kappa_c(COVERAGE) AUDIT\n"
            f"external source: {PICKRELL_SOURCE_DIR}\n"
            f"repo output:     {OUTPUT_DIR}\n"
            f"coverage bins:   {N_COVERAGE_BINS}\n"
            f"resume:          {RESUME}\n",
            flush=True,
        )

        namespace_preexisted = OUTPUT_DIR.exists()
        if namespace_preexisted and not SENTINEL.exists():
            raise RuntimeError(
                "Output namespace already exists without this runner's sentinel. "
                "Refusing to mix unrelated checkpoints."
            )

        for d in (OUTPUT_DIR, CELL_DIR, BIN_DIR, FAILURE_DIR, M03_CHECKPOINT_ROOT):
            d.mkdir(parents=True, exist_ok=True)

        if not SENTINEL.exists():
            SENTINEL.write_text(
                json.dumps({
                    "initialized_local_time": _now(),
                    "run_namespace": RUN_NAMESPACE,
                    "m03_version": EXPECTED_M03_VERSION,
                    "resume_enabled": True,
                    "fresh_first_launch_by_new_namespace": True,
                }, indent=2, sort_keys=True),
                encoding="utf-8",
            )
            print(f"[new namespace] {_now()} | {OUTPUT_DIR}", flush=True)
        else:
            print(f"[resume namespace] {_now()} | {OUTPUT_DIR}", flush=True)

        inputs = _load_inputs()
        reps = inputs["selected"]
        print(
            f"[input] genes={len(inputs['gene_ids']):,} | donor columns={len(inputs['sample_names'])} | splits={len(reps)}",
            flush=True,
        )

        new = reused = failed = 0
        for j, rep in enumerate(reps, 1):
            try:
                _, _, was_reused = _run_one(rep, inputs)
                reused += int(was_reused)
                new += int(not was_reused)
            except Exception:
                failed += 1
            print(f"[progress] {j:02d}/{len(reps):02d} | new={new} reused={reused} failed={failed}", flush=True)

        split_summary, bins, unresolved = _collect(reps)
        if split_summary.empty or bins.empty:
            raise RuntimeError("No successful Pickrell kappa-coverage outputs are available.")

        agg = _aggregate_bins(bins)
        _atomic_csv(split_summary, OUTPUT_DIR / "split_summary.csv")
        _atomic_csv(bins, OUTPUT_DIR / "kappa_coverage_bins.csv")
        _atomic_csv(agg, OUTPUT_DIR / "kappa_coverage_aggregate.csv")
        _atomic_csv(unresolved, OUTPUT_DIR / "unresolved.csv")

        g = split_summary["global_kappa_c"].astype(float)
        global_summary = pd.DataFrame([{
            "n_splits": len(g),
            "median_global_kappa_c": float(g.quantile(.50)),
            "q25_global_kappa_c": float(g.quantile(.25)),
            "q75_global_kappa_c": float(g.quantile(.75)),
            "mean_raw_p_lt_0_01": float(split_summary["fraction_raw_p_lt_0_01_fixed_denominator"].mean()),
            "median_raw_p_lt_0_01": float(split_summary["fraction_raw_p_lt_0_01_fixed_denominator"].median()),
            "mean_BH_discoveries": float(split_summary["n_bh_q_le_0_05"].mean()),
            "fraction_splits_with_any_BH": float(split_summary["any_bh_q_le_0_05"].astype(bool).mean()),
        }])

        print("\nGLOBAL kappa_c ACROSS PICKRELL SPLITS\n", flush=True)
        display(global_summary)
        print("\nMEDIAN kappa_c(COVERAGE) CURVE\n", flush=True)
        display(agg)

        # Plot windows only; no PNGs are written.
        _show_plots(split_summary, bins, agg)

        return {
            "split_summary": split_summary,
            "coverage_bins": bins,
            "aggregate": agg,
            "unresolved": unresolved,
            "output_dir": OUTPUT_DIR,
        }

    finally:
        ended = datetime.now().astimezone()
        print(f"[end]   {_now()} | elapsed={ended-started}", flush=True)


pickrell_m03_kappa_coverage = run_pickrell_m03_kappa_coverage_audit()


In [ ]:
from __future__ import annotations

# =============================================================================
# PICKRELL FULL NULL-CALIBRATION COMPARISON
# =============================================================================
#
# Compare, on the SAME 30 frozen Pickrell 5v5 random-label null splits:
#
#   1. edgeR exact                  [saved original baseline p-values]
#   2. edgeR QL                     [saved original baseline p-values]
#   3. DESeq2 LRT                   [saved original baseline p-values]
#   4. DESeq2 Wald                  [saved original baseline p-values]
#   5. WEB-DNB M03 global kappa_c   [current seam-safe 20M M03 checkpoints]
#   6. WEB-DNB M03 kappa_c(coverage)
#      [12-bin same-split diagnostic counterfactual]
#
# IMPORTANT
# ---------
# The edgeR / DESeq2 methods are NOT rerun.  This deliberately reuses the
# original saved p-values from pickrell_null_baseline_pvalues.csv so the
# comparison is on the identical frozen splits and the same historical
# competitor implementations used in the original Pickrell comparison.
#
# The global-kappa M03 method is also NOT rerun.  It is read from the completed
# current Pickrell M03 checkpoint namespace.
#
# Only the coverage-conditioned M03 counterfactual is computed here.  It:
#   - estimates 12 equal-count coverage-bin kappa_c values using the SAME
#     ASH-weighted estimator used by production M03;
#   - recomputes S0dag, the support-floor correction, null DEB precision,
#     the first-order normalized statistic, BH, and the mandatory exact NULL
#     profiles for R_F <= 0 genes.
#
# The 12-bin method remains a DIAGNOSTIC, not the proposed final smooth method.
#
# Output policy
# -------------
# Numerical comparison results/checkpoints are written under <repo>/data.
# Figures are DISPLAYED ONLY and are not saved.
#
# Original Pickrell comparison metrics are reproduced:
#   - fraction of finite p-values < .01
#   - BH(.05) discoveries
#   - P(any BH(.05)) = complete-null BH FDR
#   - median p
#   - calibration across a threshold grid
#
# Additional distributional summaries are included (IQR / q97.5).
#
# =============================================================================

from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime
from pathlib import Path
from scipy.optimize import minimize, brentq
from scipy.special import betaln, gammaln, logsumexp, roots_legendre, xlogy
from scipy.stats import chi2, gamma as gamma_dist, nbinom

import json
import os
import pickle
import re
import tempfile
import time
import traceback

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


# =============================================================================
# SETTINGS / PATHS
# =============================================================================

PICKRELL_SOURCE_DIR = Path(
    r"C:\Users\sruddy1\Documents\satuRnPaper\Pickrell_null_benchmark"
)

BASELINE_PVALUES_FILE = (
    PICKRELL_SOURCE_DIR / "pickrell_null_baseline_pvalues.csv"
)

COUNTS_FILE = (
    PICKRELL_SOURCE_DIR / "pickrell_29_male_counts.csv"
)

CURRENT_M03_NAMESPACE = "pickrell_m03_seam_safe_20M_kappa_coverage_v1"
COMPARISON_NAMESPACE = "pickrell_full_comparison_global_vs_coverage_kappa_v1"

N_COVERAGE_BINS = 12
ALPHA = 0.05
RAW_P_THRESHOLD = 0.01

THRESHOLD_GRID = np.asarray(
    [0.001, 0.005, 0.01, 0.025, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99],
    dtype=float,
)

# The Pickrell global M03 run is already complete.  The only potentially
# expensive work here is recomputing the few mandatory exact NULL profiles.
N_JOBS_EXACT = 6

RESUME = True
SHOW_PLOTS = True

METHOD_GLOBAL = "WEB-DNB M03 global kappa_c"
METHOD_LOCAL = "WEB-DNB M03 kappa_c(coverage), 12-bin diagnostic"


CWD = Path.cwd().resolve()

if CWD.name.lower() in {"notebook", "notebooks"}:
    PROJECT_ROOT = CWD.parent
elif (CWD / "data").is_dir():
    PROJECT_ROOT = CWD
else:
    raise RuntimeError(
        "Could not resolve repository root. Run from <repo>/notebook, "
        "<repo>/notebooks, or the repository root containing ./data. "
        f"Current working directory: {CWD}"
    )

PROJECT_DATA_DIR = PROJECT_ROOT / "data"

GLOBAL_RUN_DIR = PROJECT_DATA_DIR / CURRENT_M03_NAMESPACE
GLOBAL_CELL_DIR = GLOBAL_RUN_DIR / "cells"
M03_CHECKPOINT_ROOT = GLOBAL_RUN_DIR / "m03_checkpoints"

OUTPUT_DIR = PROJECT_DATA_DIR / COMPARISON_NAMESPACE
LOCAL_CACHE_DIR = OUTPUT_DIR / "local_kappa_split_checkpoints"

SENTINEL = OUTPUT_DIR / "_pickrell_full_comparison_v1_initialized.json"


# =============================================================================
# FILE / CHECKPOINT UTILITIES
# =============================================================================

def _now() -> str:
    return datetime.now().astimezone().strftime(
        "%Y-%m-%d %I:%M:%S %p %Z"
    )


def _safe(x: object) -> str:
    x = re.sub(r"[^A-Za-z0-9._-]+", "_", str(x))
    return x.strip("._") or "unnamed"


def _atomic_csv(df: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp_name = tempfile.mkstemp(
        prefix=f".{path.name}.",
        dir=str(path.parent),
    )
    os.close(fd)
    tmp = Path(tmp_name)

    try:
        df.to_csv(tmp, index=False)
        os.replace(tmp, path)
    finally:
        if tmp.exists():
            try:
                tmp.unlink()
            except OSError:
                pass


def _atomic_pickle(obj, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, tmp_name = tempfile.mkstemp(
        prefix=f".{path.name}.",
        dir=str(path.parent),
    )
    os.close(fd)
    tmp = Path(tmp_name)

    try:
        with open(tmp, "wb") as f:
            pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
            f.flush()
            try:
                os.fsync(f.fileno())
            except OSError:
                pass
        os.replace(tmp, path)
    finally:
        if tmp.exists():
            try:
                tmp.unlink()
            except OSError:
                pass


def _load_pickle(path: Path):
    with open(path, "rb") as f:
        return pickle.load(f)


def _global_checkpoint_dir(replicate: int) -> Path:
    return M03_CHECKPOINT_ROOT / f"pickrell_rep{int(replicate):02d}"


def _load_success_stage(replicate: int, stage_name: str):
    path = (
        _global_checkpoint_dir(replicate)
        / "stages"
        / f"{stage_name}.pkl"
    )

    if not path.exists():
        raise FileNotFoundError(
            f"Missing current M03 stage checkpoint: {path}"
        )

    payload = _load_pickle(path)

    if (
        not isinstance(payload, dict)
        or payload.get("status") != "success"
    ):
        raise RuntimeError(
            f"Pickrell replicate {replicate}: {stage_name} is not a successful checkpoint."
        )

    return payload["data"]


def _load_manifest(replicate: int) -> dict:
    path = _global_checkpoint_dir(replicate) / "m03_manifest.pkl"

    if not path.exists():
        raise FileNotFoundError(
            f"Missing current M03 manifest: {path}"
        )

    return _load_pickle(path)


def _load_existing_exact_payload(split_id: str, gene_index: int):
    # split_id here is the checkpoint stem, e.g. "pickrell_rep01".
    path = (
        M03_CHECKPOINT_ROOT
        / _safe(split_id)
        / "exact_gene_checkpoints"
        / f"gene_{int(gene_index):07d}.pkl"
    )

    if not path.exists():
        return None

    try:
        payload = _load_pickle(path)
    except Exception:
        return None

    return payload if isinstance(payload, dict) else None


def _local_cache_path(replicate: int) -> Path:
    return (
        LOCAL_CACHE_DIR
        / f"pickrell_rep{int(replicate):02d}__local_kappa.pkl"
    )


# =============================================================================
# METHOD LABELS
# =============================================================================

def _canonical_competitor_label(value: object) -> str:
    raw = str(value).strip()
    low = raw.lower()

    if "edger" in low:
        if "ql" in low or "quasi" in low:
            return "edgeR QL"
        return "edgeR exact"

    if "deseq" in low:
        if "lrt" in low or "likelihood" in low:
            return "DESeq2 LRT"
        if "wald" in low:
            return "DESeq2 Wald"

    return raw


# =============================================================================
# DISCOVER THE COMPLETED CURRENT PICKRELL M03 RUN
# =============================================================================

def _discover_completed_global_replicates() -> pd.DataFrame:
    rows = []

    if not GLOBAL_CELL_DIR.exists():
        raise RuntimeError(
            f"Current Pickrell M03 cell directory does not exist: {GLOBAL_CELL_DIR}"
        )

    for path in sorted(GLOBAL_CELL_DIR.glob("pickrell_rep*.csv")):
        try:
            x = pd.read_csv(path)
        except Exception:
            continue

        if (
            len(x) != 1
            or str(x.loc[0, "status"]) != "success"
        ):
            continue

        rep = int(x.loc[0, "replicate"])

        rows.append(
            {
                "replicate": rep,
                "cell_path": str(path),
            }
        )

    if not rows:
        raise RuntimeError(
            "No completed current Pickrell M03 replicates were found."
        )

    out = (
        pd.DataFrame(rows)
        .drop_duplicates("replicate")
        .sort_values("replicate", kind="stable")
        .reset_index(drop=True)
    )

    return out


# =============================================================================
# LOAD GLOBAL M03 P-VALUES FROM THE CURRENT STAGE-5 CHECKPOINTS
# =============================================================================

def _global_m03_pvalues_for_replicate(replicate: int) -> pd.DataFrame:
    core = _load_success_stage(replicate, "01_core")
    base = _load_success_stage(replicate, "05_base_m03_result")

    gene_ids = np.asarray(core["gene_ids"]).astype(str)
    p = np.asarray(base["p_value_m03"], dtype=float)

    if len(gene_ids) != len(p):
        raise RuntimeError(
            f"Pickrell replicate {replicate}: Stage-1 genes and Stage-5 p-values differ in length."
        )

    return pd.DataFrame(
        {
            "replicate": int(replicate),
            "method": METHOD_GLOBAL,
            "gene_id": gene_ids,
            "p_value": p,
        }
    )


# =============================================================================
# COVERAGE-BIN kappa_c FOR ONE SPLIT
# =============================================================================

def _coverage_bin_kappa(
    core: dict,
    full: dict,
    geom: dict,
):
    y = np.asarray(core["y"], dtype=float)
    exposure = np.asarray(core["exposure"], dtype=float)

    theta_F = np.asarray(
        full["theta_F_first"],
        dtype=float,
    )

    delta = np.asarray(
        full["delta"],
        dtype=float,
    )

    weights = np.asarray(
        geom["weights"],
        dtype=float,
    )

    geom_valid = np.asarray(
        geom["geom_valid"],
        dtype=bool,
    )

    global_kappa = float(
        geom["kappa_c"]
    )

    total_coverage = np.sum(
        y / exposure[None, :],
        axis=1,
    )

    log_coverage = np.log1p(
        total_coverage
    )

    bin_id = np.asarray(
        pd.qcut(
            pd.Series(log_coverage),
            q=N_COVERAGE_BINS,
            labels=False,
            duplicates="drop",
        ),
        dtype=int,
    )

    n_bins_actual = int(
        np.max(bin_id)
    ) + 1

    Q_c = (
        2.0
        * theta_F
        * delta
    )

    valid = (
        geom_valid
        & np.isfinite(Q_c)
        & np.isfinite(weights)
        & (weights >= 0)
    )

    reconstructed = _weighted_mean(
        Q_c[valid],
        weights[valid],
    )

    relerr = abs(
        reconstructed - global_kappa
    ) / max(
        abs(global_kappa),
        1e-15,
    )

    if (
        not np.isfinite(reconstructed)
        or relerr > 1e-9
    ):
        raise RuntimeError(
            "Global kappa_c reconstruction mismatch: "
            f"saved={global_kappa}, reconstructed={reconstructed}, "
            f"relative error={relerr:.3e}"
        )

    kappa_by_bin = {}
    bin_rows = []

    for b in range(n_bins_actual):
        in_bin = (
            bin_id == b
        )

        use = (
            in_bin
            & valid
        )

        local = _weighted_mean(
            Q_c[use],
            weights[use],
        )

        if (
            not np.isfinite(local)
            or local <= 0
        ):
            raise RuntimeError(
                f"Coverage bin {b+1}: invalid local kappa_c={local}."
            )

        kappa_by_bin[b] = float(local)

        bin_rows.append(
            {
                "coverage_bin": int(b + 1),
                "n_genes": int(np.sum(in_bin)),
                "median_log1p_total_normalized_coverage": float(
                    np.median(log_coverage[in_bin])
                ),
                "global_kappa_c": global_kappa,
                "local_kappa_c": float(local),
                "local_over_global": float(
                    local / global_kappa
                ),
                "null_weight_sum": float(
                    np.sum(weights[use])
                ),
            }
        )

    lambda_g = np.asarray(
        [
            kappa_by_bin[int(b)]
            for b in bin_id
        ],
        dtype=float,
    )

    return {
        "bin_id": bin_id,
        "log_coverage": log_coverage,
        "lambda_g": lambda_g,
        "global_kappa": global_kappa,
        "bin_table": pd.DataFrame(bin_rows),
    }

def _bh_adjust(p):
    p = np.asarray(p, dtype=float)
    q = np.full_like(p, np.nan)

    ok = np.isfinite(p) & (p >= 0) & (p <= 1)
    if not np.any(ok):
        return q

    p0 = p[ok]
    order = np.argsort(p0, kind="stable")
    ranked = p0[order]
    n = len(ranked)

    adjusted = ranked * n / np.arange(1, n + 1)
    adjusted = np.minimum.accumulate(adjusted[::-1])[::-1]
    adjusted = np.minimum(adjusted, 1.0)

    restored = np.empty_like(adjusted)
    restored[order] = adjusted
    q[ok] = restored

    return q


def _half_nb_deviance(y, mu, r):
    y = np.asarray(y, dtype=float)
    mu = np.maximum(np.asarray(mu, dtype=float), 1e-12)
    r = np.maximum(np.asarray(r, dtype=float), 1e-12)

    return np.maximum(
        xlogy(y, y / mu)
        - (y + r) * np.log((y + r) / (mu + r)),
        0.0,
    )


def _weighted_mean(x, w):
    x = np.asarray(x, dtype=float)
    w = np.asarray(w, dtype=float)

    ok = (
        np.isfinite(x)
        & np.isfinite(w)
        & (w >= 0)
    )

    if not np.any(ok):
        return np.nan

    sw = float(np.sum(w[ok]))
    if sw <= 0:
        return np.nan

    return float(
        np.sum(w[ok] * x[ok]) / sw
    )

# =============================================================================
# DEB / FIRST-ORDER TEST — MATCH CURRENT M03
# =============================================================================

def _fit_deb_masked(S, a, settings, label):
    S = np.asarray(S, dtype=float)

    corrected_score_tolerance = float(
        settings["corrected_score_tolerance"]
    )
    theta_floor = float(settings["theta_floor"])
    theta_cap = float(settings["theta_cap"])
    deb_precision_maxiter = int(
        settings["deb_precision_maxiter"]
    )

    valid = (
        np.isfinite(S)
        & (S > corrected_score_tolerance)
    )

    if valid.sum() < 100:
        raise RuntimeError(
            f"{label}: too few positive corrected scores for DEB."
        )

    x = S[valid]
    a0 = float(a)

    theta_pool = float(
        np.clip(
            a0 * len(x) / np.sum(x),
            theta_floor,
            theta_cap,
        )
    )

    def objective(par):
        alpha0 = float(np.exp(par[0]))
        theta_bar = float(np.exp(par[1]))
        beta0 = alpha0 / theta_bar

        ll = (
            (a0 - 1.0) * np.log(x)
            - a0 * np.log(beta0)
            - (a0 + alpha0) * np.log1p(x / beta0)
            - betaln(a0, alpha0)
        )

        return (
            -float(np.mean(ll))
            if np.all(np.isfinite(ll))
            else 1e100
        )

    fits = []

    theta_median = float(
        np.clip(
            np.median(a0 / x),
            theta_floor,
            theta_cap,
        )
    )

    for alpha_start in (
        0.05,
        0.2,
        1.0,
        5.0,
        25.0,
        100.0,
        1000.0,
        1e4,
    ):
        for theta_start in (
            theta_pool,
            theta_median,
        ):
            fit = minimize(
                objective,
                x0=np.log(
                    [alpha_start, theta_start]
                ),
                method="L-BFGS-B",
                bounds=[
                    (np.log(1e-8), np.log(1e9)),
                    (
                        np.log(theta_floor),
                        np.log(theta_cap),
                    ),
                ],
                options={
                    "maxiter": deb_precision_maxiter,
                    "ftol": 1e-12,
                },
            )

            if np.isfinite(fit.fun):
                fits.append(fit)

    if not fits:
        raise RuntimeError(
            f"{label}: DEB fit failed."
        )

    best = min(
        fits,
        key=lambda x: x.fun,
    )

    alpha0 = float(np.exp(best.x[0]))
    theta_bar = float(np.exp(best.x[1]))
    beta0 = alpha0 / theta_bar

    theta_valid = (
        a0 + alpha0
    ) / (
        x + beta0
    )

    theta = np.full(
        len(S),
        np.nan,
    )

    theta[valid] = np.clip(
        theta_valid,
        theta_floor,
        theta_cap,
    )

    return {
        "theta": theta,
        "valid_mask": valid,
        "alpha0": alpha0,
        "beta0": beta0,
        "prior_mean_theta": theta_bar,
        "objective": float(best.fun),
        "n_valid": int(valid.sum()),
        "n_invalid": int((~valid).sum()),
    }


def _normalized_statistic(
    S_F,
    S_0_used,
    C_F,
    C_0,
    theta_F,
    theta_0,
    n,
    settings,
):
    corrected_score_tolerance = float(
        settings["corrected_score_tolerance"]
    )

    R_F = S_F - C_F
    R_0 = S_0_used - C_0

    ok = (
        np.isfinite(R_F)
        & np.isfinite(R_0)
        & np.isfinite(theta_F)
        & np.isfinite(theta_0)
        & (R_F > corrected_score_tolerance)
        & (R_0 > corrected_score_tolerance)
        & (theta_F > 0)
        & (theta_0 > 0)
    )

    T_raw = np.full(
        len(S_F),
        np.nan,
    )

    T_raw[ok] = (
        2.0
        * (
            theta_0[ok] * R_0[ok]
            - theta_F[ok] * R_F[ok]
        )
        + float(n)
        * np.log(
            theta_F[ok]
            / theta_0[ok]
        )
        + 2.0
        * (
            C_0[ok]
            - C_F[ok]
        )
    )

    T = np.full(
        len(S_F),
        np.nan,
    )

    T[ok] = np.maximum(
        T_raw[ok],
        0.0,
    )

    p = np.full(
        len(S_F),
        np.nan,
    )

    p[ok] = chi2.sf(
        T[ok],
        1,
    )

    return {
        "R_F": R_F,
        "R_0": R_0,
        "valid_mask": ok,
        "T_raw": T_raw,
        "T": T,
        "p": p,
    }


# =============================================================================
# EXACT NORMALIZED-DNB BACKEND FOR THE MANDATORY R_F <= 0 BRANCH
# =============================================================================

class _ExactBackend:

    def __init__(self, settings):
        self.settings = dict(settings)

        self.cprime_tail_probability = float(
            settings["cprime_tail_probability"]
        )
        self.cprime_support_max = int(
            settings["cprime_support_max"]
        )
        self.cprime_chunk_size = int(
            settings["cprime_chunk_size"]
        )
        self.cprime_grid_exact_width_max = int(
            settings["cprime_grid_exact_width_max"]
        )

        self.cprime_continuum_audit_quad_n = int(
            settings["cprime_continuum_audit_quad_n"]
        )
        self.cprime_continuum_audit_quad_n_max = int(
            settings["cprime_continuum_audit_quad_n_max"]
        )
        self.cprime_continuum_audit_min_mu = float(
            settings["cprime_continuum_audit_min_mu"]
        )
        self.cprime_continuum_audit_min_mu_over_r = float(
            settings["cprime_continuum_audit_min_mu_over_r"]
        )
        self.cprime_continuum_audit_selfcheck_tol = float(
            settings["cprime_continuum_audit_selfcheck_tol"]
        )

        self.exact_theta_search_min = float(
            settings["exact_theta_search_min"]
        )
        self.exact_theta_search_max = float(
            settings["exact_theta_search_max"]
        )
        self.exact_score_root_xtol = float(
            settings["exact_score_root_xtol"]
        )
        self.exact_score_root_rtol = float(
            settings["exact_score_root_rtol"]
        )
        self.exact_score_zero_tol = float(
            settings["exact_score_zero_tol"]
        )
        self.exact_profile_local_step = float(
            settings["exact_profile_local_step"]
        )

        self._gamma_quad_cache = {}

    def log_nb_pmf(self, yv, mu, r):
        yv = np.asarray(
            yv,
            dtype=float,
        )

        mu = float(mu)
        r = float(r)

        p = r / (r + mu)

        return (
            gammaln(yv + r)
            - gammaln(r)
            - gammaln(yv + 1.0)
            + r * np.log(p)
            + xlogy(yv, 1.0 - p)
        )

    def nb_quantile_bounds(
        self,
        mu,
        r,
        tail_probability=None,
    ):
        if tail_probability is None:
            tail_probability = (
                self.cprime_tail_probability
            )

        tail_probability = float(
            np.clip(
                tail_probability,
                1e-15,
                1e-4,
            )
        )

        side = 0.5 * tail_probability
        p = float(r) / (
            float(r)
            + float(mu)
        )

        lo_q = nbinom.ppf(
            side,
            float(r),
            p,
        )

        hi_q = nbinom.ppf(
            1.0 - side,
            float(r),
            p,
        )

        if not np.isfinite(lo_q):
            lo_q = 0.0

        if not np.isfinite(hi_q):
            sd = np.sqrt(
                float(mu)
                + float(mu)
                * float(mu)
                / float(r)
            )

            hi_q = (
                float(mu)
                + 14.0 * sd
                + 50.0
            )

        lo = max(
            0,
            int(np.floor(lo_q)),
        )

        hi = max(
            lo,
            int(np.ceil(hi_q)),
        )

        return lo, hi

    def _get_gamma_quad(self, n_quad):
        n_quad = int(n_quad)

        if n_quad not in self._gamma_quad_cache:
            nodes, weights = roots_legendre(
                n_quad
            )

            eps = max(
                min(
                    self.cprime_tail_probability,
                    1e-10,
                ),
                1e-14,
            )

            u = (
                0.5
                * (nodes + 1.0)
                * (1.0 - 2.0 * eps)
                + eps
            )

            w = (
                0.5
                * (1.0 - 2.0 * eps)
                * weights
            )

            w = w / np.sum(w)

            self._gamma_quad_cache[
                n_quad
            ] = (
                u,
                w,
            )

        return self._gamma_quad_cache[
            n_quad
        ]

    def continuum_allowed(
        self,
        mu,
        r,
    ):
        return bool(
            np.isfinite(mu)
            and np.isfinite(r)
            and mu
            >= self.cprime_continuum_audit_min_mu
            and r > 0
            and (
                mu / r
            )
            >= self.cprime_continuum_audit_min_mu_over_r
        )

    def make_continuum_arrays(
        self,
        mu,
        r,
        n_quad,
    ):
        u, w = self._get_gamma_quad(
            n_quad
        )

        x = gamma_dist.ppf(
            u,
            a=float(r),
            scale=1.0 / float(r),
        )

        yv = float(mu) * x

        s = _half_nb_deviance(
            yv,
            float(mu),
            float(r),
        )

        return (
            np.asarray(
                s,
                dtype=float,
            ),
            np.log(
                np.asarray(
                    w,
                    dtype=float,
                )
            ),
        )

    def prepare_point_state(
        self,
        mu,
        r,
    ):
        mu = float(mu)
        r = float(r)

        lo, hi = self.nb_quantile_bounds(
            mu,
            r,
        )

        width = (
            hi - lo + 1
        )

        if (
            width
            <= self.cprime_grid_exact_width_max
        ):
            yv = np.arange(
                lo,
                hi + 1,
                dtype=float,
            )

            logp = self.log_nb_pmf(
                yv,
                mu,
                r,
            )

            s = _half_nb_deviance(
                yv,
                mu,
                r,
            )

            return {
                "kind": "discrete_cached",
                "mu": mu,
                "r": r,
                "lo": lo,
                "hi": hi,
                "logp": logp,
                "s": s,
                "logmass": float(
                    logsumexp(logp)
                ),
            }

        if (
            width
            <= self.cprime_support_max
        ):
            return {
                "kind": "discrete_chunked",
                "mu": mu,
                "r": r,
                "lo": lo,
                "hi": hi,
            }

        if self.continuum_allowed(
            mu,
            r,
        ):
            return {
                "kind": "continuum",
                "mu": mu,
                "r": r,
                "quad_cache": {},
            }

        raise RuntimeError(
            "Exact-theta reference window requires "
            f"{width:,} points for "
            f"mu={mu:.6g}, r={r:.6g}, "
            "beyond cprime_support_max and "
            "outside the audited high-count continuum regime."
        )

    def _combine_logZ_mean(
        self,
        state,
        logw,
        s,
    ):
        chunk_logZ = float(
            logsumexp(logw)
        )

        if not np.isfinite(
            chunk_logZ
        ):
            return state

        q = np.exp(
            logw - chunk_logZ
        )

        chunk_mean = float(
            np.sum(q * s)
        )

        if state is None:
            return [
                chunk_logZ,
                chunk_mean,
            ]

        old_logZ, old_mean = state

        new_logZ = float(
            np.logaddexp(
                old_logZ,
                chunk_logZ,
            )
        )

        a = float(
            np.exp(
                old_logZ
                - new_logZ
            )
        )

        b = float(
            np.exp(
                chunk_logZ
                - new_logZ
            )
        )

        return [
            new_logZ,
            a * old_mean
            + b * chunk_mean,
        ]

    def _chunked_window_state(
        self,
        mu,
        r,
        theta,
        lo,
        hi,
    ):
        tilted = None
        ref_logZ = None

        for start in range(
            int(lo),
            int(hi) + 1,
            self.cprime_chunk_size,
        ):
            stop = min(
                int(hi) + 1,
                start
                + self.cprime_chunk_size,
            )

            yv = np.arange(
                start,
                stop,
                dtype=float,
            )

            logp = self.log_nb_pmf(
                yv,
                mu,
                r,
            )

            s = _half_nb_deviance(
                yv,
                mu,
                r,
            )

            tilted = self._combine_logZ_mean(
                tilted,
                logp
                - (theta - 1.0) * s,
                s,
            )

            lref = float(
                logsumexp(logp)
            )

            ref_logZ = (
                lref
                if ref_logZ is None
                else float(
                    np.logaddexp(
                        ref_logZ,
                        lref,
                    )
                )
            )

        return (
            tilted,
            ref_logZ,
        )

    def _eval_continuum(
        self,
        state,
        theta,
    ):
        n_fine = int(
            self.cprime_continuum_audit_quad_n
        )

        n_max = int(
            self.cprime_continuum_audit_quad_n_max
        )

        while True:
            n_coarse = max(
                64,
                n_fine // 2,
            )

            for n_quad in (
                n_coarse,
                n_fine,
            ):
                if (
                    n_quad
                    not in state["quad_cache"]
                ):
                    state[
                        "quad_cache"
                    ][
                        n_quad
                    ] = self.make_continuum_arrays(
                        state["mu"],
                        state["r"],
                        n_quad,
                    )

            def at_order(n_quad):
                s, logw = state[
                    "quad_cache"
                ][
                    n_quad
                ]

                lw = (
                    logw
                    - (theta - 1.0) * s
                )

                logZ = float(
                    logsumexp(lw)
                )

                q = np.exp(
                    lw - logZ
                )

                mean_s = float(
                    np.sum(q * s)
                )

                cprime_theta = (
                    mean_s
                    - 0.5 / theta
                )

                logC = (
                    -0.5 * np.log(theta)
                    - logZ
                )

                return (
                    cprime_theta,
                    logC,
                )

            cp_coarse, _ = at_order(
                n_coarse
            )

            cp_fine, lc_fine = at_order(
                n_fine
            )

            err = abs(
                cp_fine
                - cp_coarse
            )

            if (
                np.isfinite(err)
                and err
                <= self.cprime_continuum_audit_selfcheck_tol
            ):
                return (
                    cp_fine,
                    lc_fine,
                    f"continuum_{n_fine}",
                    n_fine,
                )

            if n_fine >= n_max:
                raise RuntimeError(
                    "Exact-theta continuum self-check failed for "
                    f"mu={state['mu']:.6g}, "
                    f"r={state['r']:.6g}, "
                    f"theta={theta:.6g}: "
                    f"|fine-coarse|={err:.6g} > "
                    f"{self.cprime_continuum_audit_selfcheck_tol:.6g}."
                )

            n_fine = min(
                2 * n_fine,
                n_max,
            )

    def eval_point_state(
        self,
        state,
        theta,
    ):
        theta = float(theta)

        if (
            not np.isfinite(theta)
            or theta <= 0
        ):
            raise ValueError(
                "theta must be finite and positive."
            )

        if state["kind"] == "continuum":
            return self._eval_continuum(
                state,
                theta,
            )

        mu = state["mu"]
        r = state["r"]
        lo = state["lo"]
        hi = state["hi"]

        if (
            state["kind"]
            == "discrete_cached"
            and theta >= 1.0
        ):
            logp = state["logp"]
            s = state["s"]

            lw = (
                logp
                - (theta - 1.0) * s
            )

            logZraw = float(
                logsumexp(lw)
            )

            q = np.exp(
                lw - logZraw
            )

            mean_s = float(
                np.sum(q * s)
            )

            logZnorm = (
                logZraw
                - state["logmass"]
            )

            return (
                mean_s
                - 0.5 / theta,
                -0.5 * np.log(theta)
                - logZnorm,
                "discrete_cached",
                hi - lo + 1,
            )

        tilted, ref_logZ = (
            self._chunked_window_state(
                mu,
                r,
                theta,
                lo,
                hi,
            )
        )

        if (
            tilted is None
            or ref_logZ is None
        ):
            raise RuntimeError(
                "Initial exact-theta discrete sum failed."
            )

        total_points = (
            hi - lo + 1
        )

        if theta < 1.0:
            step = max(
                1024,
                min(
                    total_points // 2,
                    250_000,
                ),
            )

            stable_rounds = 0

            mass_tol = max(
                self.cprime_tail_probability,
                1e-12,
            )

            mean_tol = max(
                10.0
                * self.cprime_tail_probability,
                1e-10,
            )

            while stable_rounds < 2:
                old_logZ, old_mean = tilted

                added_tilted = None
                added_ref_logZ = None
                added_points = 0

                if lo > 0:
                    new_lo = max(
                        0,
                        lo - step,
                    )

                    left_t, left_r = (
                        self._chunked_window_state(
                            mu,
                            r,
                            theta,
                            new_lo,
                            lo - 1,
                        )
                    )

                    if left_t is not None:
                        added_tilted = left_t

                    if left_r is not None:
                        added_ref_logZ = left_r

                    added_points += (
                        lo - new_lo
                    )

                    lo = new_lo

                new_hi = (
                    hi + step
                )

                right_t, right_r = (
                    self._chunked_window_state(
                        mu,
                        r,
                        theta,
                        hi + 1,
                        new_hi,
                    )
                )

                if right_t is not None:
                    if added_tilted is None:
                        added_tilted = right_t
                    else:
                        lz, lm = added_tilted
                        rz, rm = right_t

                        nz = float(
                            np.logaddexp(
                                lz,
                                rz,
                            )
                        )

                        added_tilted = [
                            nz,
                            float(
                                np.exp(
                                    lz - nz
                                )
                            )
                            * lm
                            + float(
                                np.exp(
                                    rz - nz
                                )
                            )
                            * rm,
                        ]

                if right_r is not None:
                    added_ref_logZ = (
                        right_r
                        if added_ref_logZ
                        is None
                        else float(
                            np.logaddexp(
                                added_ref_logZ,
                                right_r,
                            )
                        )
                    )

                added_points += (
                    new_hi - hi
                )

                hi = new_hi

                if added_tilted is None:
                    stable_rounds += 1
                    total_points += added_points
                    continue

                add_logZ, add_mean = (
                    added_tilted
                )

                new_logZ = float(
                    np.logaddexp(
                        old_logZ,
                        add_logZ,
                    )
                )

                old_weight = float(
                    np.exp(
                        old_logZ
                        - new_logZ
                    )
                )

                add_weight = float(
                    np.exp(
                        add_logZ
                        - new_logZ
                    )
                )

                new_mean = (
                    old_weight * old_mean
                    + add_weight * add_mean
                )

                tilted = [
                    new_logZ,
                    new_mean,
                ]

                if (
                    added_ref_logZ
                    is not None
                ):
                    ref_logZ = float(
                        np.logaddexp(
                            ref_logZ,
                            added_ref_logZ,
                        )
                    )

                if (
                    add_weight <= mass_tol
                    and abs(
                        new_mean
                        - old_mean
                    )
                    <= mean_tol
                ):
                    stable_rounds += 1
                else:
                    stable_rounds = 0

                total_points += (
                    added_points
                )

                if (
                    total_points
                    > self.cprime_support_max
                ):
                    if self.continuum_allowed(
                        mu,
                        r,
                    ):
                        cont_state = {
                            "kind": "continuum",
                            "mu": mu,
                            "r": r,
                            "quad_cache": {},
                        }

                        return self._eval_continuum(
                            cont_state,
                            theta,
                        )

                    raise RuntimeError(
                        "theta<1 exact discrete expansion exceeded "
                        f"{self.cprime_support_max:,} points."
                    )

                step = min(
                    step * 2,
                    1_000_000,
                )

        logZraw, mean_s = tilted

        logZnorm = float(
            logZraw
            - ref_logZ
        )

        return (
            float(
                mean_s
                - 0.5 / theta
            ),
            float(
                -0.5 * np.log(theta)
                - logZnorm
            ),
            "discrete_dynamic",
            int(total_points),
        )

    def prepare_model_states(
        self,
        mu_row,
        r_value,
    ):
        return [
            self.prepare_point_state(
                mu,
                r_value,
            )
            for mu in np.asarray(
                mu_row,
                dtype=float,
            )
        ]

    def eval_model(
        self,
        states,
        theta,
    ):
        cpsum = 0.0
        logCsum = 0.0
        max_work = 0
        sources = set()

        for state in states:
            cp, lc, source, work = (
                self.eval_point_state(
                    state,
                    theta,
                )
            )

            cpsum += float(cp)
            logCsum += float(lc)
            sources.add(
                str(source)
            )

            if np.isfinite(work):
                max_work = max(
                    max_work,
                    int(work),
                )

        return (
            cpsum,
            logCsum,
            "+".join(
                sorted(sources)
            ),
            max_work,
        )

    def model_smin_sum(
        self,
        mu_row,
        r_value,
    ):
        mu_row = np.asarray(
            mu_row,
            dtype=float,
        )

        fl = np.maximum(
            np.floor(mu_row),
            0.0,
        )

        ce = np.maximum(
            np.ceil(mu_row),
            0.0,
        )

        sf = _half_nb_deviance(
            fl,
            mu_row,
            float(r_value),
        )

        sc = _half_nb_deviance(
            ce,
            mu_row,
            float(r_value),
        )

        return float(
            np.minimum(
                sf,
                sc,
            ).sum()
        )

    def solve_precision_profile(
        self,
        S,
        a,
        states,
        smin_sum,
    ):
        S = float(S)
        a = float(a)

        cache = {}

        def evaluated(theta):
            theta = float(theta)

            if theta not in cache:
                cp, lc, source, work = (
                    self.eval_model(
                        states,
                        theta,
                    )
                )

                score = (
                    a / theta
                    - S
                    + cp
                )

                profile = (
                    a * np.log(theta)
                    - (theta - 1.0) * S
                    + lc
                )

                cache[theta] = (
                    score,
                    profile,
                    cp,
                    lc,
                    source,
                    work,
                )

            return cache[theta]

        score1, obj1, cp1, lc1, source1, work1 = (
            evaluated(1.0)
        )

        result = {
            "root_status": None,
            "theta_hat_exact": np.nan,
            "score_at_1": float(score1),
            "profile_at_1": float(obj1),
            "cprime_at_1": float(cp1),
            "logC_at_1": float(lc1),
            "smin_sum": float(smin_sum),
            "S_minus_smin_sum": float(
                S - smin_sum
            ),
            "score_at_hat": np.nan,
            "profile_at_hat": np.nan,
            "logC_at_hat": np.nan,
            "cprime_at_hat": np.nan,
            "local_profile_max": False,
            "normalizer_source_at_hat": None,
            "normalizer_work_at_hat": np.nan,
            "n_score_evaluations": 1,
        }

        try:
            if (
                abs(score1)
                <= self.exact_score_zero_tol
            ):
                root = 1.0
                bracket = (
                    1.0,
                    1.0,
                )

            elif score1 > 0:
                lo = 1.0
                hi = 2.0

                shi = evaluated(
                    hi
                )[0]

                while (
                    shi > 0
                    and hi
                    < self.exact_theta_search_max
                ):
                    lo = hi

                    hi = min(
                        2.0 * hi,
                        self.exact_theta_search_max,
                    )

                    shi = evaluated(
                        hi
                    )[0]

                    if (
                        hi
                        >= self.exact_theta_search_max
                    ):
                        break

                if shi > 0:
                    result[
                        "root_status"
                    ] = (
                        "root_above_search_max"
                    )
                    return result

                root = brentq(
                    lambda th: evaluated(
                        th
                    )[0],
                    lo,
                    hi,
                    xtol=self.exact_score_root_xtol,
                    rtol=self.exact_score_root_rtol,
                    maxiter=200,
                )

                bracket = (
                    lo,
                    hi,
                )

            else:
                hi = 1.0
                lo = 0.5

                slo = evaluated(
                    lo
                )[0]

                while (
                    slo < 0
                    and lo
                    > self.exact_theta_search_min
                ):
                    hi = lo

                    lo = max(
                        0.5 * lo,
                        self.exact_theta_search_min,
                    )

                    slo = evaluated(
                        lo
                    )[0]

                    if (
                        lo
                        <= self.exact_theta_search_min
                    ):
                        break

                if slo < 0:
                    result[
                        "root_status"
                    ] = (
                        "root_below_search_min"
                    )
                    return result

                root = brentq(
                    lambda th: evaluated(
                        th
                    )[0],
                    lo,
                    hi,
                    xtol=self.exact_score_root_xtol,
                    rtol=self.exact_score_root_rtol,
                    maxiter=200,
                )

                bracket = (
                    lo,
                    hi,
                )

            (
                scoreh,
                objh,
                cph,
                lch,
                sourceh,
                workh,
            ) = evaluated(root)

            result.update(
                {
                    "root_status": "finite_root",
                    "theta_hat_exact": float(
                        root
                    ),
                    "score_at_hat": float(
                        scoreh
                    ),
                    "profile_at_hat": float(
                        objh
                    ),
                    "logC_at_hat": float(
                        lch
                    ),
                    "cprime_at_hat": float(
                        cph
                    ),
                    "normalizer_source_at_hat": str(
                        sourceh
                    ),
                    "normalizer_work_at_hat": float(
                        workh
                    ),
                }
            )

            left = max(
                self.exact_theta_search_min,
                root
                * np.exp(
                    -self.exact_profile_local_step
                ),
            )

            right = min(
                self.exact_theta_search_max,
                root
                * np.exp(
                    self.exact_profile_local_step
                ),
            )

            obj_left = evaluated(
                left
            )[1]

            obj_right = evaluated(
                right
            )[1]

            tol_obj = (
                1e-8
                * max(
                    1.0,
                    abs(objh),
                )
            )

            result[
                "local_profile_max"
            ] = bool(
                objh
                >= obj_left
                - tol_obj
                and objh
                >= obj_right
                - tol_obj
            )

            result[
                "root_bracket_lo"
            ] = float(
                bracket[0]
            )

            result[
                "root_bracket_hi"
            ] = float(
                bracket[1]
            )

            result[
                "n_score_evaluations"
            ] = int(
                len(cache)
            )

            return result

        except Exception as exc:
            result[
                "root_status"
            ] = (
                "numerical_failure"
            )

            result[
                "error_type"
            ] = type(exc).__name__

            result[
                "error_message"
            ] = str(exc)

            result[
                "n_score_evaluations"
            ] = int(
                len(cache)
            )

            return result


# =============================================================================
# DISCOVER COMPLETED 40v40 SPLITS
# =============================================================================

def _discover_completed_40v40():
    if not CELL_DIR.exists():
        raise RuntimeError(
            f"Cell-result directory does not exist: {CELL_DIR}"
        )

    rows = []

    for path in sorted(
        CELL_DIR.glob("*.csv")
    ):
        try:
            x = pd.read_csv(path)
        except Exception:
            continue

        if len(x) != 1:
            continue

        row = x.iloc[0]

        if str(
            row.get(
                "status",
                "",
            )
        ) != "success":
            continue

        if int(
            row.get(
                "n_per_group",
                -1,
            )
        ) != 40:
            continue

        d = row.to_dict()
        d["_cell_path"] = str(path)
        rows.append(d)

    if not rows:
        raise RuntimeError(
            "No completed 40v40 split results were found."
        )

    out = pd.DataFrame(rows)

    out["split_id"] = (
        out["split_id"]
        .astype(str)
    )

    out["dataset"] = (
        out["dataset"]
        .astype(str)
    )

    out["replicate"] = (
        out["replicate"]
        .astype(int)
    )

    return (
        out.sort_values(
            [
                "dataset",
                "replicate",
            ],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )


# =============================================================================
# COVERAGE-BIN KAPPA
# =============================================================================

def _coverage_bins_and_kappa(
    core,
    full,
    geom,
):
    y = np.asarray(
        core["y"],
        dtype=float,
    )

    exposure = np.asarray(
        core["exposure"],
        dtype=float,
    )

    theta_F = np.asarray(
        full["theta_F_first"],
        dtype=float,
    )

    delta = np.asarray(
        full["delta"],
        dtype=float,
    )

    weights = np.asarray(
        geom["weights"],
        dtype=float,
    )

    geom_valid = np.asarray(
        geom["geom_valid"],
        dtype=bool,
    )

    normalized_counts = (
        y
        / exposure[None, :]
    )

    total_coverage = np.sum(
        normalized_counts,
        axis=1,
    )

    log_coverage = np.log1p(
        total_coverage
    )

    bin_raw = pd.qcut(
        pd.Series(
            log_coverage
        ),
        q=N_COVERAGE_BINS,
        labels=False,
        duplicates="drop",
    )

    bin_id = np.asarray(
        bin_raw,
        dtype=int,
    )

    n_bins_actual = (
        int(
            np.max(bin_id)
        )
        + 1
    )

    Q_c = (
        2.0
        * theta_F
        * delta
    )

    production_mask = (
        geom_valid
        & np.isfinite(Q_c)
        & np.isfinite(weights)
        & (weights >= 0)
    )

    global_kappa = float(
        geom["kappa_c"]
    )

    global_reconstructed = (
        _weighted_mean(
            Q_c[production_mask],
            weights[production_mask],
        )
    )

    relerr = abs(
        global_reconstructed
        - global_kappa
    ) / max(
        abs(global_kappa),
        1e-15,
    )

    if (
        not np.isfinite(
            global_reconstructed
        )
        or relerr > 1e-9
    ):
        raise RuntimeError(
            "Global kappa reconstruction failed: "
            f"saved={global_kappa}, "
            f"reconstructed={global_reconstructed}, "
            f"relative error={relerr:.3e}."
        )

    kappa_by_bin = {}

    bin_rows = []

    for b in range(
        n_bins_actual
    ):
        in_bin = (
            bin_id == b
        )

        use = (
            in_bin
            & production_mask
        )

        w = weights[use]
        qc = Q_c[use]

        kappa_bin = (
            _weighted_mean(
                qc,
                w,
            )
        )

        if (
            not np.isfinite(
                kappa_bin
            )
            or kappa_bin <= 0
        ):
            raise RuntimeError(
                f"Coverage bin {b+1} has invalid "
                f"kappa_c={kappa_bin}."
            )

        kappa_by_bin[b] = float(
            kappa_bin
        )

        bin_rows.append(
            {
                "coverage_bin": int(
                    b + 1
                ),
                "n_genes": int(
                    np.sum(
                        in_bin
                    )
                ),
                "n_geometry_valid": int(
                    np.sum(
                        use
                    )
                ),
                "median_log1p_total_normalized_coverage": float(
                    np.median(
                        log_coverage[
                            in_bin
                        ]
                    )
                ),
                "kappa_c_global": global_kappa,
                "kappa_c_bin": float(
                    kappa_bin
                ),
                "kappa_bin_over_global": float(
                    kappa_bin
                    / global_kappa
                ),
                "null_weight_sum": float(
                    np.sum(w)
                ),
                "effective_null_n": (
                    _effective_weight_n(
                        w
                    )
                ),
            }
        )

    lambda_g = np.asarray(
        [
            kappa_by_bin[
                int(b)
            ]
            for b in bin_id
        ],
        dtype=float,
    )

    # Partition identity: the null-weighted average of bin means should
    # reproduce the original global kappa_c.
    weighted_numer = 0.0
    weighted_denom = 0.0

    for b in range(
        n_bins_actual
    ):
        use = (
            (bin_id == b)
            & production_mask
        )

        sw = float(
            np.sum(
                weights[use]
            )
        )

        weighted_numer += (
            sw
            * kappa_by_bin[b]
        )

        weighted_denom += sw

    binned_reconstructed_global = (
        weighted_numer
        / weighted_denom
    )

    bin_partition_relerr = abs(
        binned_reconstructed_global
        - global_kappa
    ) / max(
        abs(global_kappa),
        1e-15,
    )

    if (
        not np.isfinite(
            binned_reconstructed_global
        )
        or bin_partition_relerr
        > 1e-9
    ):
        raise RuntimeError(
            "Coverage-bin partition does not reproduce global kappa_c: "
            f"global={global_kappa}, "
            f"bin-weighted={binned_reconstructed_global}, "
            f"relative error={bin_partition_relerr:.3e}."
        )

    return {
        "total_coverage": total_coverage,
        "log_coverage": log_coverage,
        "bin_id": bin_id,
        "n_bins_actual": n_bins_actual,
        "lambda_g": lambda_g,
        "global_kappa": global_kappa,
        "bin_table": pd.DataFrame(
            bin_rows
        ),
    }

def _counterfactual_exact_gene(
    split_id,
    i,
    core,
    full,
    S0dag_cf,
    S0_min,
    backend,
):
    i = int(i)

    S_F = np.asarray(
        core["S_F"],
        dtype=float,
    )

    mu_F = np.asarray(
        core["mu_F"],
        dtype=float,
    )

    mu_0 = np.asarray(
        core["mu_0"],
        dtype=float,
    )

    r_hat = np.asarray(
        core["r_hat"],
        dtype=float,
    )

    a_F = float(
        core["a_F"]
    )

    a_0 = float(
        core["a_0"]
    )

    n = int(
        core["n"]
    )

    # -------------------------------------------------------------
    # FULL exact profile is independent of lambda_c.
    # Reuse the original exact checkpoint if it is valid.
    # -------------------------------------------------------------
    old = _load_existing_exact_payload(
        split_id,
        i,
    )

    full_fit = None
    reused_full = False

    if (
        isinstance(
            old,
            dict,
        )
        and str(
            old.get(
                "status",
                "",
            )
        ).startswith(
            "success"
        )
    ):
        candidate = old.get(
            "full_fit"
        )

        if (
            isinstance(
                candidate,
                dict,
            )
            and candidate.get(
                "root_status"
            )
            == "finite_root"
        ):
            full_fit = candidate
            reused_full = True

    if full_fit is None:
        full_states = (
            backend.prepare_model_states(
                mu_F[i],
                r_hat[i],
            )
        )

        smin_F = (
            backend.model_smin_sum(
                mu_F[i],
                r_hat[i],
            )
        )

        full_fit = (
            backend.solve_precision_profile(
                S_F[i],
                a_F,
                full_states,
                smin_F,
            )
        )

    if (
        full_fit.get(
            "root_status"
        )
        != "finite_root"
    ):
        return {
            "status": "statistical_failure_full_exact_root",
            "gene_index": i,
            "reused_full_exact": reused_full,
        }

    # -------------------------------------------------------------
    # NULL exact profile MUST be recomputed because S0dag changed.
    # -------------------------------------------------------------
    null_states = (
        backend.prepare_model_states(
            mu_0[i],
            r_hat[i],
        )
    )

    null_fit = (
        backend.solve_precision_profile(
            S0dag_cf[i],
            a_0,
            null_states,
            S0_min[i],
        )
    )

    if (
        null_fit.get(
            "root_status"
        )
        != "finite_root"
    ):
        return {
            "status": "statistical_failure_null_exact_root",
            "gene_index": i,
            "reused_full_exact": reused_full,
        }

    thF = float(
        full_fit[
            "theta_hat_exact"
        ]
    )

    th0 = float(
        null_fit[
            "theta_hat_exact"
        ]
    )

    logCF = float(
        full_fit[
            "logC_at_hat"
        ]
    )

    logC0 = float(
        null_fit[
            "logC_at_hat"
        ]
    )

    T_raw = (
        2.0
        * (
            th0
            * S0dag_cf[i]
            - thF
            * S_F[i]
        )
        + float(n)
        * np.log(
            thF / th0
        )
        + 2.0
        * (
            logCF
            - logC0
        )
    )

    T = max(
        float(T_raw),
        0.0,
    )

    p = float(
        chi2.sf(
            T,
            1,
        )
    )

    return {
        "status": "success",
        "gene_index": i,
        "p": p,
        "T_raw": float(
            T_raw
        ),
        "T": float(T),
        "theta_F": thF,
        "theta_0": th0,
        "negative_T": bool(
            T_raw < 0
        ),
        "reused_full_exact": reused_full,
    }

# =============================================================================
# LOCAL kappa_c(coverage) COUNTERFACTUAL FOR ONE PICKRELL SPLIT
# =============================================================================

def _local_kappa_pvalues_for_replicate(replicate: int):
    cache_path = _local_cache_path(replicate)

    if RESUME and cache_path.exists():
        try:
            cached = _load_pickle(cache_path)

            if (
                isinstance(cached, dict)
                and cached.get("version")
                == "pickrell-local-kappa-comparison-v1"
            ):
                print(
                    f"[local reload] {_now()} | rep={replicate:02d}",
                    flush=True,
                )
                return cached
        except Exception:
            pass

    t0 = time.perf_counter()

    core = _load_success_stage(replicate, "01_core")
    full = _load_success_stage(replicate, "02_full_firstorder")
    geom = _load_success_stage(replicate, "03_geometry_support")
    manifest = _load_manifest(replicate)

    settings = dict(
        manifest["numerical_settings"]
    )

    support_floor_tolerance = float(
        settings["support_floor_tolerance"]
    )

    gene_ids = np.asarray(
        core["gene_ids"]
    ).astype(str)

    S_F = np.asarray(
        core["S_F"],
        dtype=float,
    )

    S_0 = np.asarray(
        core["S_0"],
        dtype=float,
    )

    C_F = np.asarray(
        full["C_F"],
        dtype=float,
    )

    C_0 = np.asarray(
        full["C_0"],
        dtype=float,
    )

    theta_F_first = np.asarray(
        full["theta_F_first"],
        dtype=float,
    )

    mask_Rnonpos = np.asarray(
        full["mask_Rnonpos"],
        dtype=bool,
    )

    delta = np.asarray(
        full["delta"],
        dtype=float,
    )

    S0_min = np.asarray(
        geom["S0_min"],
        dtype=float,
    )

    a_0 = float(
        core["a_0"]
    )

    n = int(
        core["n"]
    )

    cov = _coverage_bin_kappa(
        core,
        full,
        geom,
    )

    lambda_g = np.asarray(
        cov["lambda_g"],
        dtype=float,
    )

    current_S0dag = (
        S_F
        + delta / lambda_g
    )

    support_violation = (
        np.isfinite(current_S0dag)
        & np.isfinite(S0_min)
        & (
            current_S0dag
            < S0_min
            - support_floor_tolerance
        )
    )

    B = np.maximum(
        S_F,
        S0_min,
    )

    shrinkable = np.maximum(
        S_0 - B,
        0.0,
    )

    feasible_S0dag = (
        B
        + shrinkable / lambda_g
    )

    S0dag = np.where(
        support_violation,
        feasible_S0dag,
        current_S0dag,
    )

    if np.any(
        S0dag
        < S0_min
        - support_floor_tolerance
    ):
        raise RuntimeError(
            f"Pickrell replicate {replicate}: local-kappa support-floor correction failed."
        )

    R0_local = (
        S0dag
        - C_0
    )

    null_fit = _fit_deb_masked(
        R0_local,
        a_0,
        settings,
        f"Pickrell replicate {replicate} local-kappa null DEB",
    )

    theta_0 = np.asarray(
        null_fit["theta"],
        dtype=float,
    )

    test = _normalized_statistic(
        S_F,
        S0dag,
        C_F,
        C_0,
        theta_F_first,
        theta_0,
        n,
        settings,
    )

    p = np.asarray(
        test["p"],
        dtype=float,
    ).copy()

    # -------------------------------------------------------------
    # Mandatory exact branch.
    # -------------------------------------------------------------
    mandatory = np.flatnonzero(
        mask_Rnonpos
    )

    p[mask_Rnonpos] = np.nan

    exact_results = []

    if len(mandatory):
        checkpoint_stem = f"pickrell_rep{int(replicate):02d}"

        if int(N_JOBS_EXACT) <= 1:
            backend = _ExactBackend(settings)

            for i in mandatory:
                exact_results.append(
                    _counterfactual_exact_gene(
                        checkpoint_stem,
                        int(i),
                        core,
                        full,
                        S0dag,
                        S0_min,
                        backend,
                    )
                )
        else:
            with ThreadPoolExecutor(
                max_workers=int(N_JOBS_EXACT)
            ) as executor:
                future_map = {
                    executor.submit(
                        _counterfactual_exact_gene,
                        checkpoint_stem,
                        int(i),
                        core,
                        full,
                        S0dag,
                        S0_min,
                        _ExactBackend(settings),
                    ): int(i)
                    for i in mandatory
                }

                for future in as_completed(
                    future_map
                ):
                    exact_results.append(
                        future.result()
                    )

    n_exact_success = 0
    n_exact_failure = 0
    n_exact_full_reused = 0

    for result in exact_results:
        i = int(
            result["gene_index"]
        )

        n_exact_full_reused += int(
            bool(
                result.get(
                    "reused_full_exact",
                    False,
                )
            )
        )

        if result.get("status") == "success":
            p[i] = float(
                result["p"]
            )

            n_exact_success += 1
        else:
            n_exact_failure += 1

    q = _bh_adjust(p)

    pvalues = pd.DataFrame(
        {
            "replicate": int(replicate),
            "method": METHOD_LOCAL,
            "gene_id": gene_ids,
            "p_value": p,
            "q_value": q,
        }
    )

    payload = {
        "version": "pickrell-local-kappa-comparison-v1",
        "replicate": int(replicate),
        "pvalues": pvalues,
        "bin_table": cov["bin_table"],
        "global_kappa_c": float(
            cov["global_kappa"]
        ),
        "n_support_floor_violations": int(
            np.sum(support_violation)
        ),
        "n_mandatory_exact": int(
            len(mandatory)
        ),
        "n_mandatory_exact_success": int(
            n_exact_success
        ),
        "n_mandatory_exact_failure": int(
            n_exact_failure
        ),
        "n_exact_full_profiles_reused": int(
            n_exact_full_reused
        ),
        "elapsed_seconds": float(
            time.perf_counter() - t0
        ),
    }

    _atomic_pickle(
        payload,
        cache_path,
    )

    print(
        f"[local end] {_now()} | rep={replicate:02d} | "
        f"{payload['elapsed_seconds']:.1f}s | "
        f"kappa_global={payload['global_kappa_c']:.6f} | "
        f"mandatory exact={n_exact_success}/{len(mandatory)}",
        flush=True,
    )

    return payload


# =============================================================================
# ORIGINAL-COMPARISON SCORING
# =============================================================================

def _summarize_one_method(method_df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for replicate, d in method_df.groupby(
        "replicate",
        sort=True,
    ):
        p = pd.to_numeric(
            d["p_value"],
            errors="coerce",
        ).to_numpy(dtype=float)

        ok = np.isfinite(p)
        p_use = p[ok]

        q = _bh_adjust(p)

        rows.append(
            {
                "replicate": int(replicate),
                "method": str(
                    d["method"].iloc[0]
                ),
                "n_genes": int(
                    len(d)
                ),
                "n_finite_p": int(
                    np.sum(ok)
                ),
                "finite_p_fraction": float(
                    np.mean(ok)
                ),
                "FPR_001": (
                    float(
                        np.mean(
                            p_use < RAW_P_THRESHOLD
                        )
                    )
                    if len(p_use)
                    else np.nan
                ),
                "BH_05_discoveries": int(
                    np.sum(
                        np.isfinite(q)
                        & (q < ALPHA)
                    )
                ),
                "any_BH_05": int(
                    np.any(
                        np.isfinite(q)
                        & (q < ALPHA)
                    )
                ),
                "median_p": (
                    float(
                        np.median(p_use)
                    )
                    if len(p_use)
                    else np.nan
                ),
            }
        )

    return pd.DataFrame(rows)


def _threshold_calibration(
    all_pvalues: pd.DataFrame,
) -> pd.DataFrame:
    rows = []

    for method, dm in all_pvalues.groupby(
        "method",
        sort=False,
    ):
        for threshold in THRESHOLD_GRID:
            fractions = []

            for _, dr in dm.groupby(
                "replicate",
                sort=True,
            ):
                p = pd.to_numeric(
                    dr["p_value"],
                    errors="coerce",
                ).to_numpy(dtype=float)

                p = p[
                    np.isfinite(p)
                ]

                if len(p):
                    fractions.append(
                        float(
                            np.mean(
                                p <= threshold
                            )
                        )
                    )

            if not fractions:
                continue

            rows.append(
                {
                    "method": str(method),
                    "threshold": float(
                        threshold
                    ),
                    "mean_fraction_le_threshold": float(
                        np.mean(fractions)
                    ),
                    "median_fraction_le_threshold": float(
                        np.median(fractions)
                    ),
                    "q25_fraction_le_threshold": float(
                        np.quantile(
                            fractions,
                            0.25,
                        )
                    ),
                    "q75_fraction_le_threshold": float(
                        np.quantile(
                            fractions,
                            0.75,
                        )
                    ),
                    "ratio_to_uniform": float(
                        np.mean(fractions)
                        / threshold
                    ),
                }
            )

    return pd.DataFrame(rows)


def _method_summary(
    split_summary: pd.DataFrame,
) -> pd.DataFrame:
    rows = []

    for method, x in split_summary.groupby(
        "method",
        sort=False,
    ):
        fpr = x["FPR_001"].astype(float)
        bh = x["BH_05_discoveries"].astype(float)

        rows.append(
            {
                "method": str(method),
                "n_splits": int(
                    len(x)
                ),
                "mean_FPR_001": float(
                    fpr.mean()
                ),
                "median_FPR_001": float(
                    fpr.median()
                ),
                "sd_FPR_001": float(
                    fpr.std(ddof=1)
                ),
                "q25_FPR_001": float(
                    fpr.quantile(0.25)
                ),
                "q75_FPR_001": float(
                    fpr.quantile(0.75)
                ),
                "q975_FPR_001": float(
                    fpr.quantile(0.975)
                ),
                "FPR_001_ratio_to_nominal": float(
                    fpr.mean()
                    / RAW_P_THRESHOLD
                ),
                "mean_BH_05_discoveries": float(
                    bh.mean()
                ),
                "median_BH_05_discoveries": float(
                    bh.median()
                ),
                "q975_BH_05_discoveries": float(
                    bh.quantile(0.975)
                ),
                # Under the complete random-label null:
                # FDR = P(R > 0).
                "P_any_BH_05_complete_null_FDR": float(
                    x["any_BH_05"]
                    .astype(float)
                    .mean()
                ),
                "mean_median_p": float(
                    x["median_p"]
                    .astype(float)
                    .mean()
                ),
                "median_of_median_p": float(
                    x["median_p"]
                    .astype(float)
                    .median()
                ),
                "mean_finite_p_fraction": float(
                    x["finite_p_fraction"]
                    .astype(float)
                    .mean()
                ),
            }
        )

    return pd.DataFrame(rows)


# =============================================================================
# DISPLAY PLOTS — NOTHING SAVED
# =============================================================================

def _show_comparison_plots(
    split_summary: pd.DataFrame,
    threshold_calibration: pd.DataFrame,
):
    method_order = (
        split_summary[
            "method"
        ]
        .drop_duplicates()
        .tolist()
    )

    # -------------------------------------------------------------
    # Raw p<.01 by split
    # -------------------------------------------------------------
    fig, ax = plt.subplots(
        figsize=(16, 9)
    )

    data = [
        split_summary.loc[
            split_summary["method"] == method,
            "FPR_001",
        ].astype(float).to_numpy()
        for method in method_order
    ]

    ax.boxplot(
        data,
        labels=method_order,
        showfliers=True,
    )

    ax.axhline(
        RAW_P_THRESHOLD,
        linestyle="--",
        linewidth=1.8,
        label="Nominal 1%",
    )

    ax.set_ylabel(
        "Fraction of finite p-values < .01",
        fontsize=14,
    )

    ax.set_title(
        "Pickrell Complete Null: Raw-p Calibration Across 30 Frozen 5v5 Splits",
        fontsize=18,
        pad=16,
    )

    ax.tick_params(
        axis="x",
        labelrotation=20,
        labelsize=11,
    )

    ax.tick_params(
        axis="y",
        labelsize=12,
    )

    ax.grid(
        axis="y",
        alpha=0.20,
    )

    ax.legend(
        fontsize=11,
    )

    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------------
    # BH discoveries by split
    # -------------------------------------------------------------
    fig, ax = plt.subplots(
        figsize=(16, 9)
    )

    data = [
        split_summary.loc[
            split_summary["method"] == method,
            "BH_05_discoveries",
        ].astype(float).to_numpy()
        for method in method_order
    ]

    ax.boxplot(
        data,
        labels=method_order,
        showfliers=True,
    )

    ax.set_ylabel(
        "BH(.05) discoveries",
        fontsize=14,
    )

    ax.set_title(
        "Pickrell Complete Null: BH(.05) Discoveries Across 30 Frozen 5v5 Splits",
        fontsize=18,
        pad=16,
    )

    ax.tick_params(
        axis="x",
        labelrotation=20,
        labelsize=11,
    )

    ax.tick_params(
        axis="y",
        labelsize=12,
    )

    ax.grid(
        axis="y",
        alpha=0.20,
    )

    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------------
    # Complete-null BH FDR = P(any rejection)
    # -------------------------------------------------------------
    fdr = (
        split_summary.groupby(
            "method",
            sort=False,
        )["any_BH_05"]
        .mean()
        .reindex(method_order)
    )

    fig, ax = plt.subplots(
        figsize=(16, 9)
    )

    ax.bar(
        np.arange(
            len(method_order)
        ),
        fdr.to_numpy(float),
    )

    ax.axhline(
        ALPHA,
        linestyle="--",
        linewidth=1.8,
        label="Nominal 5%",
    )

    ax.set_xticks(
        np.arange(
            len(method_order)
        )
    )

    ax.set_xticklabels(
        method_order,
        rotation=20,
        ha="right",
    )

    ax.set_ylabel(
        "P(any BH(.05) rejection)",
        fontsize=14,
    )

    ax.set_title(
        "Pickrell Complete Null: BH FDR = P(R > 0)",
        fontsize=18,
        pad=16,
    )

    ax.grid(
        axis="y",
        alpha=0.20,
    )

    ax.legend(
        fontsize=11,
    )

    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------------
    # Threshold calibration
    # -------------------------------------------------------------
    fig, ax = plt.subplots(
        figsize=(16, 9)
    )

    for method, x in threshold_calibration.groupby(
        "method",
        sort=False,
    ):
        x = x.sort_values(
            "threshold"
        )

        ax.plot(
            x["threshold"],
            x["mean_fraction_le_threshold"],
            marker="o",
            linewidth=2.2,
            label=str(method),
        )

    t = THRESHOLD_GRID

    ax.plot(
        t,
        t,
        linestyle="--",
        linewidth=1.8,
        label="Uniform null target",
    )

    ax.set_xscale(
        "log"
    )

    ax.set_yscale(
        "log"
    )

    ax.set_xlabel(
        "Nominal p-value threshold",
        fontsize=14,
    )

    ax.set_ylabel(
        "Mean fraction of p-values <= threshold",
        fontsize=14,
    )

    ax.set_title(
        "Pickrell Complete Null: Full Raw-p Calibration Curve",
        fontsize=18,
        pad=16,
    )

    ax.grid(
        alpha=0.20,
        which="both",
    )

    ax.legend(
        fontsize=10,
        ncol=2,
    )

    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------------
    # Paired M03 global vs local-kappa raw-p FPR
    # -------------------------------------------------------------
    pair = (
        split_summary.loc[
            split_summary["method"].isin(
                [
                    METHOD_GLOBAL,
                    METHOD_LOCAL,
                ]
            )
        ]
        .pivot(
            index="replicate",
            columns="method",
            values="FPR_001",
        )
        .dropna()
    )

    if not pair.empty:
        fig, ax = plt.subplots(
            figsize=(16, 9)
        )

        ax.scatter(
            pair[METHOD_GLOBAL],
            pair[METHOD_LOCAL],
            s=70,
            alpha=0.85,
        )

        vmax = float(
            np.nanmax(
                pair.to_numpy(float)
            )
        )

        ax.plot(
            [0.0, vmax * 1.05],
            [0.0, vmax * 1.05],
            linestyle="--",
            linewidth=1.8,
            label="No change",
        )

        ax.set_xlabel(
            "M03 global kappa_c: fraction p<.01",
            fontsize=14,
        )

        ax.set_ylabel(
            "M03 kappa_c(coverage): fraction p<.01",
            fontsize=14,
        )

        ax.set_title(
            "Pickrell: Paired Global vs Coverage-Conditioned M03 Calibration",
            fontsize=18,
            pad=16,
        )

        ax.grid(
            alpha=0.20,
        )

        ax.legend(
            fontsize=11,
        )

        plt.tight_layout()
        plt.show()


# =============================================================================
# TOP-LEVEL RUNNER
# =============================================================================

def run_pickrell_full_method_comparison():
    started = datetime.now().astimezone()

    print(
        f"[start] {_now()}",
        flush=True,
    )

    try:
        # -------------------------------------------------------------
        # Validate inputs / namespace
        # -------------------------------------------------------------
        if not BASELINE_PVALUES_FILE.exists():
            raise FileNotFoundError(
                BASELINE_PVALUES_FILE
            )

        if not COUNTS_FILE.exists():
            raise FileNotFoundError(
                COUNTS_FILE
            )

        completed = _discover_completed_global_replicates()

        if len(completed) != 30:
            raise RuntimeError(
                f"Current Pickrell M03 namespace contains {len(completed)} "
                "successful replicates; expected all 30 before the full comparison."
            )

        OUTPUT_DIR.mkdir(
            parents=True,
            exist_ok=True,
        )

        LOCAL_CACHE_DIR.mkdir(
            parents=True,
            exist_ok=True,
        )

        if (
            OUTPUT_DIR.exists()
            and any(OUTPUT_DIR.iterdir())
            and not SENTINEL.exists()
        ):
            # The cache directory may just have been created above, so ignore that
            # single empty directory when deciding whether this is a foreign namespace.
            foreign = [
                p
                for p in OUTPUT_DIR.iterdir()
                if p != LOCAL_CACHE_DIR
            ]

            if foreign:
                raise RuntimeError(
                    "Comparison output namespace already contains unrelated files "
                    "without this runner's sentinel. Choose a new COMPARISON_NAMESPACE."
                )

        if not SENTINEL.exists():
            SENTINEL.write_text(
                json.dumps(
                    {
                        "initialized_local_time": _now(),
                        "comparison_namespace": COMPARISON_NAMESPACE,
                        "current_m03_namespace": CURRENT_M03_NAMESPACE,
                        "competitor_source": str(
                            BASELINE_PVALUES_FILE
                        ),
                        "resume_enabled": bool(
                            RESUME
                        ),
                    },
                    indent=2,
                    sort_keys=True,
                ),
                encoding="utf-8",
            )

        # -------------------------------------------------------------
        # Fixed supplied Pickrell gene universe.
        # -------------------------------------------------------------
        counts_header = pd.read_csv(
            COUNTS_FILE,
            nrows=0,
        )

        if "gene_id" not in counts_header.columns:
            raise RuntimeError(
                "Pickrell counts file does not contain gene_id."
            )

        fixed_gene_ids = (
            pd.read_csv(
                COUNTS_FILE,
                usecols=["gene_id"],
            )["gene_id"]
            .astype(str)
            .to_numpy()
        )

        n_fixed = int(
            len(
                fixed_gene_ids
            )
        )

        print(
            "\nPICKRELL FULL METHOD COMPARISON\n"
            f"frozen splits:        {len(completed)}\n"
            f"fixed supplied genes: {n_fixed:,}\n"
            f"baseline p-values:    {BASELINE_PVALUES_FILE}\n"
            f"global M03 source:    {GLOBAL_RUN_DIR}\n"
            f"comparison output:    {OUTPUT_DIR}\n",
            flush=True,
        )

        # -------------------------------------------------------------
        # Competitor baseline p-values.
        # -------------------------------------------------------------
        baseline = pd.read_csv(
            BASELINE_PVALUES_FILE
        )

        required_baseline = {
            "replicate",
            "method",
            "gene_id",
            "p_value",
        }

        missing = (
            required_baseline
            - set(
                baseline.columns
            )
        )

        if missing:
            raise RuntimeError(
                "Saved Pickrell competitor p-values are missing columns: "
                + ", ".join(
                    sorted(
                        missing
                    )
                )
            )

        baseline = baseline.copy()

        baseline["replicate"] = (
            baseline["replicate"]
            .astype(int)
        )

        baseline["gene_id"] = (
            baseline["gene_id"]
            .astype(str)
        )

        baseline["p_value"] = pd.to_numeric(
            baseline["p_value"],
            errors="coerce",
        )

        baseline["method_raw"] = (
            baseline["method"]
            .astype(str)
        )

        baseline["method"] = (
            baseline["method_raw"]
            .map(
                _canonical_competitor_label
            )
        )

        baseline = baseline.loc[
            baseline["replicate"].isin(
                completed["replicate"]
            )
        ].copy()

        print(
            "\nSAVED COMPETITOR METHOD LABELS\n",
            flush=True,
        )

        display(
            baseline[
                [
                    "method_raw",
                    "method",
                ]
            ]
            .drop_duplicates()
            .sort_values(
                [
                    "method",
                    "method_raw",
                ],
                kind="stable",
            )
            .reset_index(
                drop=True
            )
        )

        # -------------------------------------------------------------
        # Current global M03 + local-kappa counterfactual.
        # -------------------------------------------------------------
        global_frames = []
        local_frames = []
        local_bin_frames = []

        for j, replicate in enumerate(
            completed["replicate"],
            start=1,
        ):
            replicate = int(
                replicate
            )

            global_frames.append(
                _global_m03_pvalues_for_replicate(
                    replicate
                )
            )

            local_payload = (
                _local_kappa_pvalues_for_replicate(
                    replicate
                )
            )

            local_frames.append(
                local_payload[
                    "pvalues"
                ]
            )

            b = (
                local_payload[
                    "bin_table"
                ]
                .copy()
            )

            b["replicate"] = (
                replicate
            )

            local_bin_frames.append(
                b
            )

            print(
                f"[comparison progress] {j:02d}/30",
                flush=True,
            )

        global_pvalues = pd.concat(
            global_frames,
            ignore_index=True,
        )

        local_pvalues = pd.concat(
            local_frames,
            ignore_index=True,
        )

        local_bins = pd.concat(
            local_bin_frames,
            ignore_index=True,
        )

        # -------------------------------------------------------------
        # Use the same columns for every method.
        # -------------------------------------------------------------
        competitor_pvalues = baseline[
            [
                "replicate",
                "method",
                "gene_id",
                "p_value",
            ]
        ].copy()

        all_pvalues = pd.concat(
            [
                competitor_pvalues,
                global_pvalues,
                local_pvalues[
                    [
                        "replicate",
                        "method",
                        "gene_id",
                        "p_value",
                    ]
                ],
            ],
            ignore_index=True,
        )

        # Guard against accidental duplicate method/gene rows.
        duplicates = all_pvalues.duplicated(
            [
                "replicate",
                "method",
                "gene_id",
            ]
        )

        if duplicates.any():
            preview = all_pvalues.loc[
                duplicates,
                [
                    "replicate",
                    "method",
                    "gene_id",
                ],
            ].head(20)

            raise RuntimeError(
                "Duplicate replicate/method/gene rows were found in the comparison:\n"
                + preview.to_string(index=False)
            )

        # -------------------------------------------------------------
        # Reproduce the old comparison scoring.
        # -------------------------------------------------------------
        split_frames = []

        for _, dm in all_pvalues.groupby(
            "method",
            sort=False,
        ):
            split_frames.append(
                _summarize_one_method(
                    dm
                )
            )

        split_summary = pd.concat(
            split_frames,
            ignore_index=True,
        )

        method_summary = _method_summary(
            split_summary
        )

        calibration = _threshold_calibration(
            all_pvalues
        )

        paired_FPR = (
            split_summary.pivot(
                index="replicate",
                columns="method",
                values="FPR_001",
            )
            .reset_index()
        )

        paired_BH = (
            split_summary.pivot(
                index="replicate",
                columns="method",
                values="BH_05_discoveries",
            )
            .reset_index()
        )

        paired_any_BH = (
            split_summary.pivot(
                index="replicate",
                columns="method",
                values="any_BH_05",
            )
            .reset_index()
        )

        # -------------------------------------------------------------
        # Save numerical results; figures remain display-only.
        # -------------------------------------------------------------
        _atomic_csv(
            method_summary,
            OUTPUT_DIR
            / "method_summary.csv",
        )

        _atomic_csv(
            split_summary,
            OUTPUT_DIR
            / "split_summary.csv",
        )

        _atomic_csv(
            calibration,
            OUTPUT_DIR
            / "threshold_calibration.csv",
        )

        _atomic_csv(
            paired_FPR,
            OUTPUT_DIR
            / "paired_raw_p01.csv",
        )

        _atomic_csv(
            paired_BH,
            OUTPUT_DIR
            / "paired_BH05_discoveries.csv",
        )

        _atomic_csv(
            paired_any_BH,
            OUTPUT_DIR
            / "paired_any_BH05.csv",
        )

        _atomic_csv(
            local_bins,
            OUTPUT_DIR
            / "local_kappa_coverage_bins.csv",
        )

        # -------------------------------------------------------------
        # Console / notebook output.
        # -------------------------------------------------------------
        print(
            "\nFULL PICKRELL METHOD SUMMARY\n",
            flush=True,
        )

        display(
            method_summary
        )

        print(
            "\nTHRESHOLD CALIBRATION AT p=.001, .005, .01, .025, .05\n",
            flush=True,
        )

        display(
            calibration.loc[
                calibration[
                    "threshold"
                ].isin(
                    [
                        0.001,
                        0.005,
                        0.01,
                        0.025,
                        0.05,
                    ]
                )
            ]
            .sort_values(
                [
                    "threshold",
                    "method",
                ],
                kind="stable",
            )
            .reset_index(
                drop=True
            )
        )

        if SHOW_PLOTS:
            _show_comparison_plots(
                split_summary,
                calibration,
            )

        return {
            "method_summary": method_summary,
            "split_summary": split_summary,
            "threshold_calibration": calibration,
            "paired_FPR": paired_FPR,
            "paired_BH": paired_BH,
            "paired_any_BH": paired_any_BH,
            "local_kappa_bins": local_bins,
            "all_pvalues": all_pvalues,
            "output_dir": OUTPUT_DIR,
        }

    finally:
        ended = datetime.now().astimezone()

        print(
            f"[end]   {_now()} | elapsed={ended-started}",
            flush=True,
        )


# =============================================================================
# EXECUTION
# =============================================================================

pickrell_full_comparison = (
    run_pickrell_full_method_comparison()
)


In [ ]:
from __future__ import annotations

"""
Plot Pickrell M03 parameter behavior versus gene coverage.

Goal
----
For the completed Pickrell M03 run, map how the fitted precision / dispersion
behavior changes with coverage across the 30 frozen 5v5 null splits.

This script focuses on the quantities Sean asked about:
    - theta_F (first-order full fit precision)
    - theta_0 (null DEB precision)
    - theta_0 / theta_F
    - log(theta_0 / theta_F)
    - dispersion proxy 1/theta_F
    - dispersion proxy 1/theta_0
    - raw p<.01 fraction by coverage

All plots are DISPLAYED to screen only. Nothing is saved.

Assumptions
-----------
- Run from either:
    <repo>/notebook
    <repo>/notebooks
    <repo root>  (must contain ./data)
- Current completed Pickrell M03 run exists under:
    ./data/pickrell_m03_seam_safe_20M_kappa_coverage_v1
"""

from datetime import datetime
from pathlib import Path
import pickle
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


# =============================================================================
# SETTINGS
# =============================================================================

CURRENT_M03_NAMESPACE = "pickrell_m03_seam_safe_20M_kappa_coverage_v1"
N_COVERAGE_BINS = 12
RAW_P_THRESHOLD = 0.01
SHOW_PER_SPLIT_FAINT_LINES = True
USE_LOG_Y_FOR_THETA_AND_DISPERSION = True


# =============================================================================
# TIME / PATH HELPERS
# =============================================================================

def _now() -> str:
    return datetime.now().astimezone().strftime("%Y-%m-%d %I:%M:%S %p %Z")


CWD = Path.cwd().resolve()

if CWD.name.lower() in {"notebook", "notebooks"}:
    PROJECT_ROOT = CWD.parent
elif (CWD / "data").is_dir():
    PROJECT_ROOT = CWD
else:
    raise RuntimeError(
        "Could not resolve repository root. Run from <repo>/notebook, "
        "<repo>/notebooks, or the repository root containing ./data. "
        f"Current working directory: {CWD}"
    )

PROJECT_DATA_DIR = PROJECT_ROOT / "data"
RUN_DIR = PROJECT_DATA_DIR / CURRENT_M03_NAMESPACE
CELL_DIR = RUN_DIR / "cells"
CHECKPOINT_ROOT = RUN_DIR / "m03_checkpoints"


# =============================================================================
# CHECKPOINT HELPERS
# =============================================================================

def _load_pickle(path: Path):
    with open(path, "rb") as f:
        return pickle.load(f)


def _checkpoint_dir(replicate: int) -> Path:
    return CHECKPOINT_ROOT / f"pickrell_rep{int(replicate):02d}"


def _load_success_stage(replicate: int, stage_name: str):
    path = _checkpoint_dir(replicate) / "stages" / f"{stage_name}.pkl"

    if not path.exists():
        raise FileNotFoundError(f"Missing stage checkpoint: {path}")

    payload = _load_pickle(path)

    if not isinstance(payload, dict) or payload.get("status") != "success":
        raise RuntimeError(
            f"Replicate {replicate}: stage {stage_name} is not a successful checkpoint."
        )

    return payload["data"]


def _discover_completed_replicates() -> list[int]:
    if not CELL_DIR.exists():
        raise RuntimeError(f"Cell directory does not exist: {CELL_DIR}")

    reps = []

    for path in sorted(CELL_DIR.glob("pickrell_rep*.csv")):
        try:
            x = pd.read_csv(path)
        except Exception:
            continue

        if len(x) != 1:
            continue

        status = str(x.loc[0, "status"]).strip().lower()
        if status != "success":
            continue

        reps.append(int(x.loc[0, "replicate"]))

    reps = sorted(set(reps))

    if not reps:
        raise RuntimeError("No completed Pickrell replicates were found.")

    return reps


# =============================================================================
# DATA EXTRACTION
# =============================================================================

def _extract_null_theta(stage04: dict) -> np.ndarray:
    # Current M03 Stage 04 stores the first-order null precision vector as
    # theta_0_first.  Keep a few historical aliases for compatibility.
    for key in (
        "theta_0_first",
        "theta",
        "theta_0",
        "theta0",
        "theta_null",
    ):
        if key in stage04:
            out = np.asarray(stage04[key], dtype=float)
            if out.ndim != 1:
                raise RuntimeError(f"Stage 04 key {key} is not a 1D array.")
            return out

    raise RuntimeError(
        "Could not find a null-theta vector in stage 04. "
        f"Available keys: {sorted(stage04.keys())}"
    )


def _extract_pvalues(stage05: dict) -> np.ndarray:
    for key in ("p_value_m03", "p", "p_value", "p_value_web"):
        if key in stage05:
            out = np.asarray(stage05[key], dtype=float)
            if out.ndim != 1:
                raise RuntimeError(f"Stage 05 key {key} is not a 1D array.")
            return out

    raise RuntimeError(
        "Could not find a p-value vector in stage 05. "
        f"Available keys: {sorted(stage05.keys())}"
    )


def _extract_one_replicate(replicate: int) -> tuple[pd.DataFrame, pd.DataFrame]:
    core = _load_success_stage(replicate, "01_core")
    full = _load_success_stage(replicate, "02_full_firstorder")
    null_stage = _load_success_stage(replicate, "04_null_deb")
    base = _load_success_stage(replicate, "05_base_m03_result")

    y = np.asarray(core["y"], dtype=float)
    exposure = np.asarray(core["exposure"], dtype=float)

    theta_F = np.asarray(full["theta_F_first"], dtype=float)
    theta_0 = _extract_null_theta(null_stage)
    p_value = _extract_pvalues(base)

    n_genes = y.shape[0]

    if len(theta_F) != n_genes or len(theta_0) != n_genes or len(p_value) != n_genes:
        raise RuntimeError(
            f"Replicate {replicate}: gene-length mismatch among stage arrays."
        )

    total_normalized_coverage = np.sum(y / exposure[None, :], axis=1)
    log_coverage = np.log1p(total_normalized_coverage)

    bin_id = np.asarray(
        pd.qcut(
            pd.Series(log_coverage),
            q=N_COVERAGE_BINS,
            labels=False,
            duplicates="drop",
        ),
        dtype=int,
    )

    theta_ratio = np.full(n_genes, np.nan, dtype=float)
    ok_ratio = (
        np.isfinite(theta_0)
        & np.isfinite(theta_F)
        & (theta_0 > 0)
        & (theta_F > 0)
    )
    theta_ratio[ok_ratio] = theta_0[ok_ratio] / theta_F[ok_ratio]

    log_theta_ratio = np.full(n_genes, np.nan, dtype=float)
    log_theta_ratio[ok_ratio] = np.log(theta_ratio[ok_ratio])

    disp_F = np.full(n_genes, np.nan, dtype=float)
    disp_0 = np.full(n_genes, np.nan, dtype=float)

    ok_F = np.isfinite(theta_F) & (theta_F > 0)
    ok_0 = np.isfinite(theta_0) & (theta_0 > 0)

    disp_F[ok_F] = 1.0 / theta_F[ok_F]
    disp_0[ok_0] = 1.0 / theta_0[ok_0]

    gene_level = pd.DataFrame(
        {
            "replicate": int(replicate),
            "log1p_total_normalized_coverage": log_coverage,
            "coverage_bin": bin_id + 1,
            "theta_F": theta_F,
            "theta_0": theta_0,
            "theta0_over_thetaF": theta_ratio,
            "log_theta0_over_thetaF": log_theta_ratio,
            "dispersion_proxy_F": disp_F,
            "dispersion_proxy_0": disp_0,
            "is_raw_p_lt_0_01": np.isfinite(p_value) & (p_value < RAW_P_THRESHOLD),
            "p_value": p_value,
        }
    )

    bin_rows = []

    for b, d in gene_level.groupby("coverage_bin", sort=True):
        q25_cov, q75_cov = np.quantile(d["log1p_total_normalized_coverage"], [0.25, 0.75])

        def _q(series, q):
            x = pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)
            x = x[np.isfinite(x)]
            return float(np.quantile(x, q)) if len(x) else np.nan

        def _med(series):
            x = pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)
            x = x[np.isfinite(x)]
            return float(np.median(x)) if len(x) else np.nan

        bin_rows.append(
            {
                "replicate": int(replicate),
                "coverage_bin": int(b),
                "n_genes": int(len(d)),
                "median_log1p_total_normalized_coverage": float(np.median(d["log1p_total_normalized_coverage"])),
                "q25_log1p_total_normalized_coverage": float(q25_cov),
                "q75_log1p_total_normalized_coverage": float(q75_cov),
                "theta_F_median": _med(d["theta_F"]),
                "theta_F_q25": _q(d["theta_F"], 0.25),
                "theta_F_q75": _q(d["theta_F"], 0.75),
                "theta_0_median": _med(d["theta_0"]),
                "theta_0_q25": _q(d["theta_0"], 0.25),
                "theta_0_q75": _q(d["theta_0"], 0.75),
                "theta_ratio_median": _med(d["theta0_over_thetaF"]),
                "theta_ratio_q25": _q(d["theta0_over_thetaF"], 0.25),
                "theta_ratio_q75": _q(d["theta0_over_thetaF"], 0.75),
                "log_theta_ratio_median": _med(d["log_theta0_over_thetaF"]),
                "log_theta_ratio_q25": _q(d["log_theta0_over_thetaF"], 0.25),
                "log_theta_ratio_q75": _q(d["log_theta0_over_thetaF"], 0.75),
                "dispersion_F_median": _med(d["dispersion_proxy_F"]),
                "dispersion_F_q25": _q(d["dispersion_proxy_F"], 0.25),
                "dispersion_F_q75": _q(d["dispersion_proxy_F"], 0.75),
                "dispersion_0_median": _med(d["dispersion_proxy_0"]),
                "dispersion_0_q25": _q(d["dispersion_proxy_0"], 0.25),
                "dispersion_0_q75": _q(d["dispersion_proxy_0"], 0.75),
                "raw_p_lt_0_01_fraction": float(np.mean(d["is_raw_p_lt_0_01"])),
            }
        )

    bin_level = pd.DataFrame(bin_rows)
    return gene_level, bin_level


# =============================================================================
# AGGREGATION
# =============================================================================

def _aggregate_across_replicates(bin_level: pd.DataFrame) -> pd.DataFrame:
    rows = []

    agg_cols = [
        "median_log1p_total_normalized_coverage",
        "theta_F_median",
        "theta_0_median",
        "theta_ratio_median",
        "log_theta_ratio_median",
        "dispersion_F_median",
        "dispersion_0_median",
        "raw_p_lt_0_01_fraction",
    ]

    for b, d in bin_level.groupby("coverage_bin", sort=True):
        row = {
            "coverage_bin": int(b),
            "n_splits": int(d["replicate"].nunique()),
        }

        for col in agg_cols:
            x = pd.to_numeric(d[col], errors="coerce").to_numpy(dtype=float)
            x = x[np.isfinite(x)]
            row[f"{col}__median"] = float(np.median(x)) if len(x) else np.nan
            row[f"{col}__q25"] = float(np.quantile(x, 0.25)) if len(x) else np.nan
            row[f"{col}__q75"] = float(np.quantile(x, 0.75)) if len(x) else np.nan

        rows.append(row)

    return pd.DataFrame(rows).sort_values("coverage_bin", kind="stable").reset_index(drop=True)


# =============================================================================
# PLOTTING
# =============================================================================

def _plot_with_band(ax, x, y, lo, hi, label):
    ax.plot(x, y, marker="o", linewidth=2.2, label=label)
    ax.fill_between(x, lo, hi, alpha=0.15)


def _maybe_add_faint_split_lines(ax, bin_level: pd.DataFrame, y_col: str):
    if not SHOW_PER_SPLIT_FAINT_LINES:
        return

    for rep, d in bin_level.groupby("replicate", sort=True):
        d = d.sort_values("coverage_bin", kind="stable")
        ax.plot(
            d["median_log1p_total_normalized_coverage"],
            d[y_col],
            linewidth=0.8,
            alpha=0.15,
        )


def _plot_pickrell_parameter_maps(bin_level: pd.DataFrame, agg: pd.DataFrame):
    x = agg["median_log1p_total_normalized_coverage__median"].to_numpy(dtype=float)

    # -------------------------------------------------------------------------
    # 1) theta_F and theta_0
    # -------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(16, 9))

    _maybe_add_faint_split_lines(ax, bin_level, "theta_F_median")
    _maybe_add_faint_split_lines(ax, bin_level, "theta_0_median")

    _plot_with_band(
        ax,
        x,
        agg["theta_F_median__median"].to_numpy(dtype=float),
        agg["theta_F_median__q25"].to_numpy(dtype=float),
        agg["theta_F_median__q75"].to_numpy(dtype=float),
        label=r"$\theta_F$ median across splits",
    )

    _plot_with_band(
        ax,
        x,
        agg["theta_0_median__median"].to_numpy(dtype=float),
        agg["theta_0_median__q25"].to_numpy(dtype=float),
        agg["theta_0_median__q75"].to_numpy(dtype=float),
        label=r"$\theta_0$ median across splits",
    )

    if USE_LOG_Y_FOR_THETA_AND_DISPERSION:
        ax.set_yscale("log")

    ax.set_xlabel("log1p(total normalized gene coverage)", fontsize=14)
    ax.set_ylabel(r"Precision parameter $\theta$", fontsize=14)
    ax.set_title("Pickrell: Full and Null Precision Estimates by Coverage", fontsize=18, pad=16)
    ax.grid(alpha=0.20, which="both")
    ax.legend(fontsize=11)
    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------------------------
    # 2) dispersion proxy 1/theta
    # -------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(16, 9))

    _maybe_add_faint_split_lines(ax, bin_level, "dispersion_F_median")
    _maybe_add_faint_split_lines(ax, bin_level, "dispersion_0_median")

    _plot_with_band(
        ax,
        x,
        agg["dispersion_F_median__median"].to_numpy(dtype=float),
        agg["dispersion_F_median__q25"].to_numpy(dtype=float),
        agg["dispersion_F_median__q75"].to_numpy(dtype=float),
        label=r"Dispersion proxy $1/\theta_F$",
    )

    _plot_with_band(
        ax,
        x,
        agg["dispersion_0_median__median"].to_numpy(dtype=float),
        agg["dispersion_0_median__q25"].to_numpy(dtype=float),
        agg["dispersion_0_median__q75"].to_numpy(dtype=float),
        label=r"Dispersion proxy $1/\theta_0$",
    )

    if USE_LOG_Y_FOR_THETA_AND_DISPERSION:
        ax.set_yscale("log")

    ax.set_xlabel("log1p(total normalized gene coverage)", fontsize=14)
    ax.set_ylabel(r"Dispersion proxy $1/\theta$", fontsize=14)
    ax.set_title("Pickrell: Dispersion Proxy by Coverage", fontsize=18, pad=16)
    ax.grid(alpha=0.20, which="both")
    ax.legend(fontsize=11)
    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------------------------
    # 3) theta0/thetaF ratio
    # -------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(16, 9))

    _maybe_add_faint_split_lines(ax, bin_level, "theta_ratio_median")

    _plot_with_band(
        ax,
        x,
        agg["theta_ratio_median__median"].to_numpy(dtype=float),
        agg["theta_ratio_median__q25"].to_numpy(dtype=float),
        agg["theta_ratio_median__q75"].to_numpy(dtype=float),
        label=r"Median $\theta_0/\theta_F$",
    )

    ax.axhline(1.0, linestyle="--", linewidth=1.7, label=r"$\theta_0=\theta_F$")

    ax.set_xlabel("log1p(total normalized gene coverage)", fontsize=14)
    ax.set_ylabel(r"$\theta_0/\theta_F$", fontsize=14)
    ax.set_title("Pickrell: Null-to-Full Precision Ratio by Coverage", fontsize=18, pad=16)
    ax.grid(alpha=0.20)
    ax.legend(fontsize=11)
    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------------------------
    # 4) log(theta0/thetaF)
    # -------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(16, 9))

    _maybe_add_faint_split_lines(ax, bin_level, "log_theta_ratio_median")

    _plot_with_band(
        ax,
        x,
        agg["log_theta_ratio_median__median"].to_numpy(dtype=float),
        agg["log_theta_ratio_median__q25"].to_numpy(dtype=float),
        agg["log_theta_ratio_median__q75"].to_numpy(dtype=float),
        label=r"Median $\log(\theta_0/\theta_F)$",
    )

    ax.axhline(0.0, linestyle="--", linewidth=1.7, label=r"$\log(\theta_0/\theta_F)=0$")

    ax.set_xlabel("log1p(total normalized gene coverage)", fontsize=14)
    ax.set_ylabel(r"$\log(\theta_0/\theta_F)$", fontsize=14)
    ax.set_title("Pickrell: Log Null-to-Full Precision Ratio by Coverage", fontsize=18, pad=16)
    ax.grid(alpha=0.20)
    ax.legend(fontsize=11)
    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------------------------
    # 5) raw p<.01 fraction by coverage
    # -------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(16, 9))

    _maybe_add_faint_split_lines(ax, bin_level, "raw_p_lt_0_01_fraction")

    _plot_with_band(
        ax,
        x,
        agg["raw_p_lt_0_01_fraction__median"].to_numpy(dtype=float),
        agg["raw_p_lt_0_01_fraction__q25"].to_numpy(dtype=float),
        agg["raw_p_lt_0_01_fraction__q75"].to_numpy(dtype=float),
        label=r"Median fraction with raw $p<0.01$",
    )

    ax.axhline(RAW_P_THRESHOLD, linestyle="--", linewidth=1.7, label="Nominal 1%")

    ax.set_xlabel("log1p(total normalized gene coverage)", fontsize=14)
    ax.set_ylabel("Fraction of genes with raw p < .01", fontsize=14)
    ax.set_title("Pickrell: Raw-p Calibration by Coverage", fontsize=18, pad=16)
    ax.grid(alpha=0.20)
    ax.legend(fontsize=11)
    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------------------------
    # 6) Overlay: log(theta0/thetaF) vs raw-p fraction (separate y axes)
    # -------------------------------------------------------------------------
    fig, ax1 = plt.subplots(figsize=(16, 9))
    ax2 = ax1.twinx()

    y1 = agg["log_theta_ratio_median__median"].to_numpy(dtype=float)
    y2 = agg["raw_p_lt_0_01_fraction__median"].to_numpy(dtype=float)

    ax1.plot(x, y1, marker="o", linewidth=2.4, label=r"Median $\log(\theta_0/\theta_F)$")
    ax1.fill_between(
        x,
        agg["log_theta_ratio_median__q25"].to_numpy(dtype=float),
        agg["log_theta_ratio_median__q75"].to_numpy(dtype=float),
        alpha=0.15,
    )
    ax1.axhline(0.0, linestyle="--", linewidth=1.5)

    ax2.plot(x, y2, marker="s", linewidth=2.2, linestyle="--", label=r"Median fraction raw $p<.01$")
    ax2.fill_between(
        x,
        agg["raw_p_lt_0_01_fraction__q25"].to_numpy(dtype=float),
        agg["raw_p_lt_0_01_fraction__q75"].to_numpy(dtype=float),
        alpha=0.12,
    )
    ax2.axhline(RAW_P_THRESHOLD, linestyle=":", linewidth=1.5)

    ax1.set_xlabel("log1p(total normalized gene coverage)", fontsize=14)
    ax1.set_ylabel(r"Median $\log(\theta_0/\theta_F)$", fontsize=14)
    ax2.set_ylabel("Median fraction raw p < .01", fontsize=14)
    ax1.set_title(
        "Pickrell: Coverage Pattern of Precision-Ratio Shift vs Raw-p Inflation",
        fontsize=18,
        pad=16,
    )
    ax1.grid(alpha=0.20)

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=11, loc="upper right")
    plt.tight_layout()
    plt.show()


# =============================================================================
# TOP-LEVEL RUNNER
# =============================================================================

def run_pickrell_theta_dispersion_coverage_audit():
    started = datetime.now().astimezone()
    print(f"[start] {_now()}", flush=True)

    try:
        reps = _discover_completed_replicates()

        print(
            "\nPICKRELL THETA / DISPERSION VS COVERAGE AUDIT\n"
            f"namespace:      {CURRENT_M03_NAMESPACE}\n"
            f"run directory:   {RUN_DIR}\n"
            f"completed reps:  {len(reps)}\n"
            f"replicates:      {reps}\n",
            flush=True,
        )

        gene_frames = []
        bin_frames = []

        for j, rep in enumerate(reps, start=1):
            gene_df, bin_df = _extract_one_replicate(rep)
            gene_frames.append(gene_df)
            bin_frames.append(bin_df)
            print(f"[progress] {j:02d}/{len(reps):02d} | rep={rep:02d}", flush=True)

        gene_level = pd.concat(gene_frames, ignore_index=True)
        bin_level = pd.concat(bin_frames, ignore_index=True)
        agg = _aggregate_across_replicates(bin_level)

        summary_cols = [
            "coverage_bin",
            "n_splits",
            "median_log1p_total_normalized_coverage__median",
            "theta_F_median__median",
            "theta_0_median__median",
            "theta_ratio_median__median",
            "log_theta_ratio_median__median",
            "dispersion_F_median__median",
            "dispersion_0_median__median",
            "raw_p_lt_0_01_fraction__median",
        ]

        print("\nBIN-LEVEL MEDIAN SUMMARY ACROSS PICKRELL SPLITS\n", flush=True)
        display(agg[summary_cols])

        per_split_summary = (
            bin_level.groupby("replicate", sort=True)
            .agg(
                theta_ratio_low_cov=("theta_ratio_median", "first"),
                theta_ratio_mid_cov=("theta_ratio_median", lambda x: float(np.median(np.asarray(x, dtype=float)))),
                theta_ratio_high_cov=("theta_ratio_median", "last"),
                raw_p01_peak=("raw_p_lt_0_01_fraction", "max"),
            )
            .reset_index()
        )

        print("\nPER-SPLIT QUICK SUMMARY\n", flush=True)
        display(per_split_summary)

        _plot_pickrell_parameter_maps(bin_level, agg)

        return {
            "gene_level": gene_level,
            "bin_level": bin_level,
            "aggregate": agg,
            "per_split_summary": per_split_summary,
        }

    finally:
        ended = datetime.now().astimezone()
        print(f"[end]   {_now()} | elapsed={ended - started}", flush=True)


# =============================================================================
# EXECUTION
# =============================================================================

pickrell_theta_dispersion_coverage_audit = run_pickrell_theta_dispersion_coverage_audit()


In [ ]:
from __future__ import annotations

"""
Pickrell M03 upstream variance / precision audit by coverage.

This version uses the ACTUAL current M03 checkpoint structure:

Stage 01_core
    r_fit
    r_hat
    alpha_hat
    a_F
    a_0
    y
    exposure

Stage 02_full_firstorder
    R_F
    full_theta_fit
    theta_F_first

Stage 04_null_deb
    R_0_dagger_first
    null_theta_fit
    theta_0_first

Stage 05_base_m03_result
    p_value_m03

It plots:
    1. r_g by coverage
    2. alpha_g = 1/r_g by coverage
    3. DP Fano-factor chain:
           phi_raw_DP
           phi_trend_DP
           phi_hat_DP
    4. raw full precision a_F/R_F vs DEB theta_F
    5. raw null precision a_0/R_0^dagger vs DEB theta_0
    6. raw theta_0/theta_F vs DEB theta_0/theta_F
    7. raw p<.01 fraction by coverage
    8. alpha_g overlaid with raw-p inflation
    9. phi_raw / phi_trend / phi_hat overlaid with raw-p inflation
    10. DEB hyperparameter summaries across splits

Plots are DISPLAYED ONLY. Nothing is saved.
"""

from datetime import datetime
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


# =============================================================================
# SETTINGS
# =============================================================================

CURRENT_M03_NAMESPACE = "pickrell_m03_seam_safe_20M_kappa_coverage_v1"
N_COVERAGE_BINS = 12
RAW_P_THRESHOLD = 0.01

SHOW_PER_SPLIT_FAINT_LINES = True
FIGSIZE = (16, 9)


# =============================================================================
# TIME / PATH HELPERS
# =============================================================================

def _now() -> str:
    return datetime.now().astimezone().strftime(
        "%Y-%m-%d %I:%M:%S %p %Z"
    )


CWD = Path.cwd().resolve()

if CWD.name.lower() in {"notebook", "notebooks"}:
    PROJECT_ROOT = CWD.parent
elif (CWD / "data").is_dir():
    PROJECT_ROOT = CWD
else:
    raise RuntimeError(
        "Could not resolve repository root. Run from <repo>/notebook, "
        "<repo>/notebooks, or the repository root containing ./data. "
        f"Current working directory: {CWD}"
    )

PROJECT_DATA_DIR = PROJECT_ROOT / "data"
RUN_DIR = PROJECT_DATA_DIR / CURRENT_M03_NAMESPACE
CELL_DIR = RUN_DIR / "cells"
CHECKPOINT_ROOT = RUN_DIR / "m03_checkpoints"


# =============================================================================
# CHECKPOINT HELPERS
# =============================================================================

def _load_pickle(path: Path):
    with open(path, "rb") as f:
        return pickle.load(f)


def _checkpoint_dir(replicate: int) -> Path:
    return (
        CHECKPOINT_ROOT
        / f"pickrell_rep{int(replicate):02d}"
    )


def _load_success_stage(
    replicate: int,
    stage_name: str,
):
    path = (
        _checkpoint_dir(replicate)
        / "stages"
        / f"{stage_name}.pkl"
    )

    if not path.exists():
        raise FileNotFoundError(
            f"Missing checkpoint: {path}"
        )

    payload = _load_pickle(path)

    if (
        not isinstance(payload, dict)
        or payload.get("status") != "success"
    ):
        raise RuntimeError(
            f"Replicate {replicate}: "
            f"{stage_name} is not a successful checkpoint."
        )

    return payload["data"]


def _discover_completed_replicates():
    reps = []

    for path in sorted(
        CELL_DIR.glob("pickrell_rep*.csv")
    ):
        try:
            x = pd.read_csv(path)
        except Exception:
            continue

        if len(x) != 1:
            continue

        if str(
            x.loc[0, "status"]
        ).strip().lower() != "success":
            continue

        reps.append(
            int(
                x.loc[0, "replicate"]
            )
        )

    reps = sorted(
        set(reps)
    )

    if not reps:
        raise RuntimeError(
            "No completed Pickrell replicates were found."
        )

    return reps


# =============================================================================
# NUMERIC HELPERS
# =============================================================================

def _finite_positive(x):
    x = np.asarray(
        x,
        dtype=float,
    )

    return (
        np.isfinite(x)
        & (x > 0)
    )


def _raw_theta(
    a,
    R,
):
    R = np.asarray(
        R,
        dtype=float,
    )

    out = np.full(
        len(R),
        np.nan,
        dtype=float,
    )

    ok = (
        np.isfinite(R)
        & (R > 0)
    )

    out[ok] = (
        float(a)
        / R[ok]
    )

    return out


def _safe_ratio(
    num,
    den,
):
    num = np.asarray(
        num,
        dtype=float,
    )

    den = np.asarray(
        den,
        dtype=float,
    )

    out = np.full(
        len(num),
        np.nan,
        dtype=float,
    )

    ok = (
        np.isfinite(num)
        & np.isfinite(den)
        & (den != 0)
    )

    out[ok] = (
        num[ok]
        / den[ok]
    )

    return out


def _med(x):
    x = pd.to_numeric(
        x,
        errors="coerce",
    ).to_numpy(
        dtype=float
    )

    x = x[
        np.isfinite(x)
    ]

    return (
        float(
            np.median(x)
        )
        if len(x)
        else np.nan
    )


def _q(
    x,
    q,
):
    x = pd.to_numeric(
        x,
        errors="coerce",
    ).to_numpy(
        dtype=float
    )

    x = x[
        np.isfinite(x)
    ]

    return (
        float(
            np.quantile(
                x,
                q,
            )
        )
        if len(x)
        else np.nan
    )


# =============================================================================
# EXTRACT ONE REPLICATE
# =============================================================================

def _extract_one_replicate(
    replicate: int,
):
    core = _load_success_stage(
        replicate,
        "01_core",
    )

    full = _load_success_stage(
        replicate,
        "02_full_firstorder",
    )

    null = _load_success_stage(
        replicate,
        "04_null_deb",
    )

    base = _load_success_stage(
        replicate,
        "05_base_m03_result",
    )

    # -----------------------------------------------------------------
    # Exact current checkpoint keys
    # -----------------------------------------------------------------

    y = np.asarray(
        core["y"],
        dtype=float,
    )

    exposure = np.asarray(
        core["exposure"],
        dtype=float,
    )

    r_fit = core["r_fit"]

    r_hat = np.asarray(
        core["r_hat"],
        dtype=float,
    )

    alpha_hat = np.asarray(
        core["alpha_hat"],
        dtype=float,
    )

    a_F = float(
        core["a_F"]
    )

    a_0 = float(
        core["a_0"]
    )

    R_F = np.asarray(
        full["R_F"],
        dtype=float,
    )

    full_theta_fit = (
        full["full_theta_fit"]
    )

    theta_F = np.asarray(
        full["theta_F_first"],
        dtype=float,
    )

    R_0 = np.asarray(
        null[
            "R_0_dagger_first"
        ],
        dtype=float,
    )

    null_theta_fit = (
        null[
            "null_theta_fit"
        ]
    )

    theta_0 = np.asarray(
        null[
            "theta_0_first"
        ],
        dtype=float,
    )

    p = np.asarray(
        base[
            "p_value_m03"
        ],
        dtype=float,
    )

    n_genes = (
        y.shape[0]
    )

    required_lengths = {
        "r_hat": len(r_hat),
        "alpha_hat": len(alpha_hat),
        "R_F": len(R_F),
        "theta_F": len(theta_F),
        "R_0": len(R_0),
        "theta_0": len(theta_0),
        "p": len(p),
    }

    bad = {
        k: v
        for k, v
        in required_lengths.items()
        if v != n_genes
    }

    if bad:
        raise RuntimeError(
            f"Replicate {replicate}: "
            f"length mismatch vs n_genes={n_genes}: {bad}"
        )

    # Verify alpha_hat = 1/r_hat as intended.
    ok_r = (
        _finite_positive(
            r_hat
        )
        & _finite_positive(
            alpha_hat
        )
    )

    if np.any(ok_r):
        max_relerr = float(
            np.max(
                np.abs(
                    alpha_hat[ok_r]
                    - 1.0 / r_hat[ok_r]
                )
                / np.maximum(
                    np.abs(
                        alpha_hat[ok_r]
                    ),
                    1e-15,
                )
            )
        )

        if max_relerr > 1e-10:
            raise RuntimeError(
                f"Replicate {replicate}: "
                f"alpha_hat != 1/r_hat; max relative error={max_relerr:.3e}"
            )

    # -----------------------------------------------------------------
    # Upstream DP / Fano quantities used to estimate r_g
    # -----------------------------------------------------------------

    phi_raw = np.asarray(
        r_fit[
            "phi_raw_DP"
        ],
        dtype=float,
    )

    phi_trend = np.asarray(
        r_fit[
            "phi_trend_DP"
        ],
        dtype=float,
    )

    phi_hat = np.asarray(
        r_fit[
            "phi_hat_DP"
        ],
        dtype=float,
    )

    mean_mu_DP = np.asarray(
        r_fit[
            "mean_mu_DP"
        ],
        dtype=float,
    )

    for name, arr in {
        "phi_raw_DP": phi_raw,
        "phi_trend_DP": phi_trend,
        "phi_hat_DP": phi_hat,
        "mean_mu_DP": mean_mu_DP,
    }.items():
        if len(arr) != n_genes:
            raise RuntimeError(
                f"Replicate {replicate}: "
                f"{name} length={len(arr)} != n_genes={n_genes}"
            )

    # -----------------------------------------------------------------
    # Raw pre-DEB precision estimates
    # -----------------------------------------------------------------

    raw_theta_F = (
        _raw_theta(
            a_F,
            R_F,
        )
    )

    raw_theta_0 = (
        _raw_theta(
            a_0,
            R_0,
        )
    )

    raw_theta_ratio = (
        _safe_ratio(
            raw_theta_0,
            raw_theta_F,
        )
    )

    deb_theta_ratio = (
        _safe_ratio(
            theta_0,
            theta_F,
        )
    )

    # -----------------------------------------------------------------
    # Coverage bins
    # -----------------------------------------------------------------

    total_normalized_coverage = np.sum(
        y
        / exposure[None, :],
        axis=1,
    )

    log_coverage = np.log1p(
        total_normalized_coverage
    )

    bin_id = (
        np.asarray(
            pd.qcut(
                pd.Series(
                    log_coverage
                ),
                q=N_COVERAGE_BINS,
                labels=False,
                duplicates="drop",
            ),
            dtype=int,
        )
        + 1
    )

    gene_level = pd.DataFrame(
        {
            "replicate": int(
                replicate
            ),
            "coverage_bin": bin_id,
            "log1p_total_normalized_coverage": log_coverage,

            "mean_mu_DP": mean_mu_DP,

            "r_g": r_hat,
            "alpha_g": alpha_hat,

            "phi_raw_DP": phi_raw,
            "phi_trend_DP": phi_trend,
            "phi_hat_DP": phi_hat,

            "raw_theta_F": raw_theta_F,
            "deb_theta_F": theta_F,

            "raw_theta_0": raw_theta_0,
            "deb_theta_0": theta_0,

            "raw_theta_ratio": raw_theta_ratio,
            "deb_theta_ratio": deb_theta_ratio,

            "raw_p_lt_0_01": (
                np.isfinite(p)
                & (
                    p
                    < RAW_P_THRESHOLD
                )
            ),
        }
    )

    # -----------------------------------------------------------------
    # Within-split coverage-bin summaries
    # -----------------------------------------------------------------

    rows = []

    for b, d in gene_level.groupby(
        "coverage_bin",
        sort=True,
    ):
        rows.append(
            {
                "replicate": int(
                    replicate
                ),
                "coverage_bin": int(
                    b
                ),
                "n_genes": int(
                    len(d)
                ),

                "median_log1p_total_normalized_coverage":
                    _med(
                        d[
                            "log1p_total_normalized_coverage"
                        ]
                    ),

                "mean_mu_DP_median":
                    _med(
                        d[
                            "mean_mu_DP"
                        ]
                    ),

                "r_g_median":
                    _med(
                        d[
                            "r_g"
                        ]
                    ),

                "alpha_g_median":
                    _med(
                        d[
                            "alpha_g"
                        ]
                    ),

                "phi_raw_DP_median":
                    _med(
                        d[
                            "phi_raw_DP"
                        ]
                    ),

                "phi_trend_DP_median":
                    _med(
                        d[
                            "phi_trend_DP"
                        ]
                    ),

                "phi_hat_DP_median":
                    _med(
                        d[
                            "phi_hat_DP"
                        ]
                    ),

                "raw_theta_F_median":
                    _med(
                        d[
                            "raw_theta_F"
                        ]
                    ),

                "deb_theta_F_median":
                    _med(
                        d[
                            "deb_theta_F"
                        ]
                    ),

                "raw_theta_0_median":
                    _med(
                        d[
                            "raw_theta_0"
                        ]
                    ),

                "deb_theta_0_median":
                    _med(
                        d[
                            "deb_theta_0"
                        ]
                    ),

                "raw_theta_ratio_median":
                    _med(
                        d[
                            "raw_theta_ratio"
                        ]
                    ),

                "deb_theta_ratio_median":
                    _med(
                        d[
                            "deb_theta_ratio"
                        ]
                    ),

                "raw_p_lt_0_01_fraction":
                    float(
                        np.mean(
                            d[
                                "raw_p_lt_0_01"
                            ]
                        )
                    ),
            }
        )

    bin_level = pd.DataFrame(
        rows
    )

    # -----------------------------------------------------------------
    # Split-level hyperparameters
    # -----------------------------------------------------------------

    meta = {
        "replicate": int(
            replicate
        ),

        "r_estimator":
            str(
                r_fit[
                    "r_estimator"
                ]
            ),

        "WEB_weight_r":
            float(
                r_fit.get(
                    "WEB_weight_r",
                    np.nan,
                )
            ),

        "trend_intercept":
            float(
                r_fit.get(
                    "trend_intercept",
                    np.nan,
                )
            ),

        "trend_alpha_inf":
            float(
                r_fit.get(
                    "trend_alpha_inf",
                    np.nan,
                )
            ),

        "full_DEB_alpha0":
            float(
                full_theta_fit.get(
                    "alpha0",
                    np.nan,
                )
            ),

        "full_DEB_beta0":
            float(
                full_theta_fit.get(
                    "beta0",
                    np.nan,
                )
            ),

        "full_DEB_prior_mean_theta":
            float(
                full_theta_fit.get(
                    "prior_mean_theta",
                    np.nan,
                )
            ),

        "null_DEB_alpha0":
            float(
                null_theta_fit.get(
                    "alpha0",
                    np.nan,
                )
            ),

        "null_DEB_beta0":
            float(
                null_theta_fit.get(
                    "beta0",
                    np.nan,
                )
            ),

        "null_DEB_prior_mean_theta":
            float(
                null_theta_fit.get(
                    "prior_mean_theta",
                    np.nan,
                )
            ),

        "full_DEB_invalid_fraction":
            float(
                full_theta_fit.get(
                    "invalid_fraction",
                    np.nan,
                )
            ),

        "null_DEB_invalid_fraction":
            float(
                null_theta_fit.get(
                    "invalid_fraction",
                    np.nan,
                )
            ),
    }

    return (
        gene_level,
        bin_level,
        meta,
    )


# =============================================================================
# AGGREGATION ACROSS SPLITS
# =============================================================================

def _aggregate_across_replicates(
    bin_level: pd.DataFrame,
):
    metrics = [
        "median_log1p_total_normalized_coverage",
        "mean_mu_DP_median",
        "r_g_median",
        "alpha_g_median",
        "phi_raw_DP_median",
        "phi_trend_DP_median",
        "phi_hat_DP_median",
        "raw_theta_F_median",
        "deb_theta_F_median",
        "raw_theta_0_median",
        "deb_theta_0_median",
        "raw_theta_ratio_median",
        "deb_theta_ratio_median",
        "raw_p_lt_0_01_fraction",
    ]

    rows = []

    for b, d in bin_level.groupby(
        "coverage_bin",
        sort=True,
    ):
        row = {
            "coverage_bin": int(
                b
            ),
            "n_splits": int(
                d[
                    "replicate"
                ].nunique()
            ),
        }

        for col in metrics:
            row[
                f"{col}__median"
            ] = _med(
                d[col]
            )

            row[
                f"{col}__q25"
            ] = _q(
                d[col],
                0.25,
            )

            row[
                f"{col}__q75"
            ] = _q(
                d[col],
                0.75,
            )

        rows.append(
            row
        )

    return (
        pd.DataFrame(
            rows
        )
        .sort_values(
            "coverage_bin",
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )


# =============================================================================
# PLOTTING HELPERS
# =============================================================================

def _add_faint_split_lines(
    ax,
    bin_level,
    y_col,
):
    if not SHOW_PER_SPLIT_FAINT_LINES:
        return

    for _, d in bin_level.groupby(
        "replicate",
        sort=True,
    ):
        d = d.sort_values(
            "coverage_bin",
            kind="stable",
        )

        ax.plot(
            d[
                "median_log1p_total_normalized_coverage"
            ],
            d[
                y_col
            ],
            linewidth=0.8,
            alpha=0.12,
        )


def _plot_band(
    ax,
    x,
    agg,
    metric,
    label,
    *,
    linestyle="-",
    marker="o",
):
    ax.plot(
        x,
        agg[
            f"{metric}__median"
        ],
        marker=marker,
        linewidth=2.3,
        linestyle=linestyle,
        label=label,
    )

    ax.fill_between(
        x,
        agg[
            f"{metric}__q25"
        ],
        agg[
            f"{metric}__q75"
        ],
        alpha=0.14,
    )


def _finish_plot(
    ax,
    title,
    ylabel,
    *,
    log_y=False,
    hline=None,
    hline_label=None,
):
    if log_y:
        ax.set_yscale(
            "log"
        )

    if hline is not None:
        ax.axhline(
            hline,
            linestyle=":",
            linewidth=1.5,
            label=hline_label,
        )

    ax.set_xlabel(
        "log1p(total normalized gene coverage)",
        fontsize=14,
    )

    ax.set_ylabel(
        ylabel,
        fontsize=14,
    )

    ax.set_title(
        title,
        fontsize=18,
        pad=16,
    )

    ax.grid(
        alpha=0.20,
        which="both",
    )

    ax.legend(
        fontsize=11,
    )

    plt.tight_layout()
    plt.show()


# =============================================================================
# PLOTS
# =============================================================================

def _show_plots(
    bin_level,
    agg,
):
    x = (
        agg[
            "median_log1p_total_normalized_coverage__median"
        ]
        .to_numpy(
            dtype=float
        )
    )

    # -------------------------------------------------------------
    # 1. r_g
    # -------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    _add_faint_split_lines(
        ax,
        bin_level,
        "r_g_median",
    )

    _plot_band(
        ax,
        x,
        agg,
        "r_g_median",
        r"Median $r_g$",
    )

    _finish_plot(
        ax,
        r"Pickrell: NB size parameter $r_g$ by coverage",
        r"$r_g$",
        log_y=True,
    )

    # -------------------------------------------------------------
    # 2. alpha_g = 1/r_g
    # -------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    _add_faint_split_lines(
        ax,
        bin_level,
        "alpha_g_median",
    )

    _plot_band(
        ax,
        x,
        agg,
        "alpha_g_median",
        r"Median $\alpha_g = 1/r_g$",
    )

    _finish_plot(
        ax,
        r"Pickrell: NB dispersion $\alpha_g = 1/r_g$ by coverage",
        r"$\alpha_g = 1/r_g$",
        log_y=True,
    )

    # -------------------------------------------------------------
    # 3. DP Fano-factor chain
    # -------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    _plot_band(
        ax,
        x,
        agg,
        "phi_raw_DP_median",
        r"Raw DP Fano $\phi_{\rm raw}$",
    )

    _plot_band(
        ax,
        x,
        agg,
        "phi_trend_DP_median",
        r"Trend target $\phi_{\rm trend}$",
        linestyle="--",
        marker="s",
    )

    _plot_band(
        ax,
        x,
        agg,
        "phi_hat_DP_median",
        r"Shrunk $\widehat\phi$ used for $r_g$",
        linestyle="-.",
        marker="^",
    )

    _finish_plot(
        ax,
        "Pickrell: DP Fano-factor estimation by coverage",
        "Fano factor",
        log_y=True,
    )

    # -------------------------------------------------------------
    # 4. Raw vs DEB full precision
    # -------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    _plot_band(
        ax,
        x,
        agg,
        "raw_theta_F_median",
        r"Raw $\widetilde\theta_F = a_F/R_F$",
    )

    _plot_band(
        ax,
        x,
        agg,
        "deb_theta_F_median",
        r"DEB $\widehat\theta_F$",
        linestyle="--",
        marker="s",
    )

    _finish_plot(
        ax,
        "Pickrell: Full precision before vs after DEB shrinkage",
        "Full precision",
        log_y=True,
    )

    # -------------------------------------------------------------
    # 5. Raw vs DEB null precision
    # -------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    _plot_band(
        ax,
        x,
        agg,
        "raw_theta_0_median",
        r"Raw $\widetilde\theta_0 = a_0/R_0^\dagger$",
    )

    _plot_band(
        ax,
        x,
        agg,
        "deb_theta_0_median",
        r"DEB $\widehat\theta_0$",
        linestyle="--",
        marker="s",
    )

    _finish_plot(
        ax,
        "Pickrell: Null precision before vs after DEB shrinkage",
        "Null precision",
        log_y=True,
    )

    # -------------------------------------------------------------
    # 6. Raw vs DEB null/full precision ratio
    # -------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    _plot_band(
        ax,
        x,
        agg,
        "raw_theta_ratio_median",
        r"Raw $\widetilde\theta_0/\widetilde\theta_F$",
    )

    _plot_band(
        ax,
        x,
        agg,
        "deb_theta_ratio_median",
        r"DEB $\widehat\theta_0/\widehat\theta_F$",
        linestyle="--",
        marker="s",
    )

    _finish_plot(
        ax,
        "Pickrell: Raw vs DEB null/full precision ratio",
        "Null / full precision ratio",
        hline=1.0,
        hline_label="ratio = 1",
    )

    # -------------------------------------------------------------
    # 7. raw p<.01
    # -------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    _plot_band(
        ax,
        x,
        agg,
        "raw_p_lt_0_01_fraction",
        r"Median fraction with raw $p<.01$",
    )

    _finish_plot(
        ax,
        "Pickrell: Raw-p calibration by coverage",
        "Fraction with raw p < .01",
        hline=RAW_P_THRESHOLD,
        hline_label="Nominal 1%",
    )

    # -------------------------------------------------------------
    # 8. alpha_g vs raw-p inflation
    # -------------------------------------------------------------

    fig, ax1 = plt.subplots(
        figsize=FIGSIZE
    )

    ax2 = (
        ax1.twinx()
    )

    ax1.plot(
        x,
        agg[
            "raw_p_lt_0_01_fraction__median"
        ],
        marker="o",
        linewidth=2.3,
        label="Raw p<.01 fraction",
    )

    ax1.fill_between(
        x,
        agg[
            "raw_p_lt_0_01_fraction__q25"
        ],
        agg[
            "raw_p_lt_0_01_fraction__q75"
        ],
        alpha=0.12,
    )

    ax1.axhline(
        RAW_P_THRESHOLD,
        linestyle=":",
        linewidth=1.4,
    )

    ax2.plot(
        x,
        agg[
            "alpha_g_median__median"
        ],
        marker="s",
        linewidth=2.3,
        linestyle="--",
        label=r"Median $\alpha_g$",
    )

    ax2.fill_between(
        x,
        agg[
            "alpha_g_median__q25"
        ],
        agg[
            "alpha_g_median__q75"
        ],
        alpha=0.12,
    )

    ax2.set_yscale(
        "log"
    )

    ax1.set_xlabel(
        "log1p(total normalized gene coverage)",
        fontsize=14,
    )

    ax1.set_ylabel(
        "Median fraction raw p < .01",
        fontsize=14,
    )

    ax2.set_ylabel(
        r"Median $\alpha_g$",
        fontsize=14,
    )

    ax1.set_title(
        "Pickrell: NB dispersion vs raw-p inflation",
        fontsize=18,
        pad=16,
    )

    ax1.grid(
        alpha=0.20,
    )

    lines1, labels1 = (
        ax1.get_legend_handles_labels()
    )

    lines2, labels2 = (
        ax2.get_legend_handles_labels()
    )

    ax1.legend(
        lines1 + lines2,
        labels1 + labels2,
        fontsize=11,
        loc="upper right",
    )

    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------------
    # 9. phi chain vs raw-p inflation
    # -------------------------------------------------------------

    fig, ax1 = plt.subplots(
        figsize=FIGSIZE
    )

    ax2 = (
        ax1.twinx()
    )

    ax1.plot(
        x,
        agg[
            "raw_p_lt_0_01_fraction__median"
        ],
        marker="o",
        linewidth=2.3,
        label="Raw p<.01 fraction",
    )

    ax1.fill_between(
        x,
        agg[
            "raw_p_lt_0_01_fraction__q25"
        ],
        agg[
            "raw_p_lt_0_01_fraction__q75"
        ],
        alpha=0.12,
    )

    ax1.axhline(
        RAW_P_THRESHOLD,
        linestyle=":",
        linewidth=1.4,
    )

    ax2.plot(
        x,
        agg[
            "phi_raw_DP_median__median"
        ],
        marker="s",
        linewidth=2.1,
        label=r"$\phi_{\rm raw}$",
    )

    ax2.plot(
        x,
        agg[
            "phi_trend_DP_median__median"
        ],
        marker="^",
        linewidth=2.1,
        linestyle="--",
        label=r"$\phi_{\rm trend}$",
    )

    ax2.plot(
        x,
        agg[
            "phi_hat_DP_median__median"
        ],
        marker="D",
        linewidth=2.1,
        linestyle="-.",
        label=r"$\widehat\phi$",
    )

    ax2.set_yscale(
        "log"
    )

    ax1.set_xlabel(
        "log1p(total normalized gene coverage)",
        fontsize=14,
    )

    ax1.set_ylabel(
        "Median fraction raw p < .01",
        fontsize=14,
    )

    ax2.set_ylabel(
        "DP Fano factor",
        fontsize=14,
    )

    ax1.set_title(
        "Pickrell: DP dispersion-estimation path vs raw-p inflation",
        fontsize=18,
        pad=16,
    )

    ax1.grid(
        alpha=0.20,
    )

    lines1, labels1 = (
        ax1.get_legend_handles_labels()
    )

    lines2, labels2 = (
        ax2.get_legend_handles_labels()
    )

    ax1.legend(
        lines1 + lines2,
        labels1 + labels2,
        fontsize=11,
        loc="upper right",
    )

    plt.tight_layout()
    plt.show()


# =============================================================================
# TOP-LEVEL
# =============================================================================

def run_pickrell_upstream_variance_precision_audit():
    started = (
        datetime.now()
        .astimezone()
    )

    print(
        f"[start] {_now()}",
        flush=True,
    )

    try:
        reps = (
            _discover_completed_replicates()
        )

        print(
            "\nPICKRELL UPSTREAM VARIANCE / PRECISION AUDIT\n"
            f"namespace:      {CURRENT_M03_NAMESPACE}\n"
            f"completed reps: {len(reps)}\n"
            f"replicates:     {reps}\n",
            flush=True,
        )

        gene_frames = []
        bin_frames = []
        meta_rows = []

        for j, rep in enumerate(
            reps,
            start=1,
        ):
            (
                gene_df,
                bin_df,
                meta,
            ) = (
                _extract_one_replicate(
                    rep
                )
            )

            gene_frames.append(
                gene_df
            )

            bin_frames.append(
                bin_df
            )

            meta_rows.append(
                meta
            )

            print(
                f"[progress] "
                f"{j:02d}/{len(reps):02d} "
                f"| rep={rep:02d} "
                f"| r_estimator={meta['r_estimator']} "
                f"| WEB_weight_r={meta['WEB_weight_r']:.6g}",
                flush=True,
            )

        gene_level = pd.concat(
            gene_frames,
            ignore_index=True,
        )

        bin_level = pd.concat(
            bin_frames,
            ignore_index=True,
        )

        meta = (
            pd.DataFrame(
                meta_rows
            )
            .sort_values(
                "replicate",
                kind="stable",
            )
            .reset_index(
                drop=True
            )
        )

        aggregate = (
            _aggregate_across_replicates(
                bin_level
            )
        )

        print(
            "\nBIN-LEVEL MEDIAN SUMMARY ACROSS PICKRELL SPLITS\n",
            flush=True,
        )

        display(
            aggregate[
                [
                    "coverage_bin",
                    "n_splits",
                    "median_log1p_total_normalized_coverage__median",

                    "r_g_median__median",
                    "alpha_g_median__median",

                    "phi_raw_DP_median__median",
                    "phi_trend_DP_median__median",
                    "phi_hat_DP_median__median",

                    "raw_theta_F_median__median",
                    "deb_theta_F_median__median",

                    "raw_theta_0_median__median",
                    "deb_theta_0_median__median",

                    "raw_theta_ratio_median__median",
                    "deb_theta_ratio_median__median",

                    "raw_p_lt_0_01_fraction__median",
                ]
            ]
        )

        print(
            "\nR-ESTIMATION / DEB HYPERPARAMETERS BY SPLIT\n",
            flush=True,
        )

        display(
            meta[
                [
                    "replicate",
                    "r_estimator",
                    "WEB_weight_r",
                    "trend_intercept",
                    "trend_alpha_inf",

                    "full_DEB_alpha0",
                    "full_DEB_beta0",
                    "full_DEB_prior_mean_theta",

                    "null_DEB_alpha0",
                    "null_DEB_beta0",
                    "null_DEB_prior_mean_theta",
                ]
            ]
        )

        _show_plots(
            bin_level,
            aggregate,
        )

        return {
            "gene_level": gene_level,
            "bin_level": bin_level,
            "aggregate": aggregate,
            "meta": meta,
        }

    finally:
        ended = (
            datetime.now()
            .astimezone()
        )

        print(
            f"[end]   {_now()} | "
            f"elapsed={ended-started}",
            flush=True,
        )


# =============================================================================
# EXECUTION
# =============================================================================

pickrell_upstream_variance_precision_audit = (
    run_pickrell_upstream_variance_precision_audit()
)


In [ ]:
from __future__ import annotations

"""
Pickrell M03 residual-component audit by coverage.

Directly plots the quantities requested:

FULL MODEL
    S_F / a_F
    C_F / a_F
    R_F / a_F = (S_F - C_F) / a_F

NULL MODEL
    S_0 / a_0                     [raw unconstrained null deviance]
    S_0^dagger / a_0              [geometry-adjusted null deviance]
    C_0 / a_0
    R_0^dagger / a_0
        = (S_0^dagger - C_0) / a_0

Also plots:
    geometry adjustment (S_0 - S_0^dagger) / a_0
    raw p<.01 fraction by coverage
    R_F/a_F and R_0^dagger/a_0 overlaid against raw-p inflation

All plots are DISPLAYED ONLY. Nothing is saved.

This version uses the actual current M03 checkpoint keys:
    Stage 01_core:
        y, exposure, S_F, S_0, a_F, a_0

    Stage 02_full_firstorder:
        C_F, C_0, R_F

    Stage 03_geometry_support:
        S0dag_final

    Stage 04_null_deb:
        R_0_dagger_first

    Stage 05_base_m03_result:
        p_value_m03
"""

from datetime import datetime
from pathlib import Path
import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


# =============================================================================
# SETTINGS
# =============================================================================

CURRENT_M03_NAMESPACE = "pickrell_m03_seam_safe_20M_kappa_coverage_v1"

N_COVERAGE_BINS = 12
RAW_P_THRESHOLD = 0.01

SHOW_PER_SPLIT_FAINT_LINES = True
FIGSIZE = (16, 9)


# =============================================================================
# PATHS / TIME
# =============================================================================

def _now() -> str:
    return datetime.now().astimezone().strftime(
        "%Y-%m-%d %I:%M:%S %p %Z"
    )


CWD = Path.cwd().resolve()

if CWD.name.lower() in {"notebook", "notebooks"}:
    PROJECT_ROOT = CWD.parent
elif (CWD / "data").is_dir():
    PROJECT_ROOT = CWD
else:
    raise RuntimeError(
        "Could not resolve repository root. Run from <repo>/notebook, "
        "<repo>/notebooks, or the repository root containing ./data. "
        f"Current working directory: {CWD}"
    )

RUN_DIR = (
    PROJECT_ROOT
    / "data"
    / CURRENT_M03_NAMESPACE
)

CELL_DIR = (
    RUN_DIR
    / "cells"
)

CHECKPOINT_ROOT = (
    RUN_DIR
    / "m03_checkpoints"
)


# =============================================================================
# CHECKPOINT HELPERS
# =============================================================================

def _load_pickle(path: Path):
    with open(path, "rb") as f:
        return pickle.load(f)


def _checkpoint_dir(
    replicate: int,
) -> Path:
    return (
        CHECKPOINT_ROOT
        / f"pickrell_rep{int(replicate):02d}"
    )


def _load_success_stage(
    replicate: int,
    stage_name: str,
):
    path = (
        _checkpoint_dir(
            replicate
        )
        / "stages"
        / f"{stage_name}.pkl"
    )

    if not path.exists():
        raise FileNotFoundError(
            f"Missing checkpoint: {path}"
        )

    payload = _load_pickle(
        path
    )

    if (
        not isinstance(
            payload,
            dict,
        )
        or payload.get(
            "status"
        )
        != "success"
    ):
        raise RuntimeError(
            f"Replicate {replicate}: "
            f"{stage_name} is not a successful checkpoint."
        )

    return payload[
        "data"
    ]


def _discover_completed_replicates():
    if not CELL_DIR.exists():
        raise RuntimeError(
            f"Cell directory does not exist: {CELL_DIR}"
        )

    reps = []

    for path in sorted(
        CELL_DIR.glob(
            "pickrell_rep*.csv"
        )
    ):
        try:
            x = pd.read_csv(
                path
            )
        except Exception:
            continue

        if len(x) != 1:
            continue

        if (
            str(
                x.loc[
                    0,
                    "status",
                ]
            )
            .strip()
            .lower()
            != "success"
        ):
            continue

        reps.append(
            int(
                x.loc[
                    0,
                    "replicate",
                ]
            )
        )

    reps = sorted(
        set(
            reps
        )
    )

    if not reps:
        raise RuntimeError(
            "No completed Pickrell replicates were found."
        )

    return reps


# =============================================================================
# NUMERIC HELPERS
# =============================================================================

def _med(
    x,
):
    x = pd.to_numeric(
        x,
        errors="coerce",
    ).to_numpy(
        dtype=float
    )

    x = x[
        np.isfinite(
            x
        )
    ]

    return (
        float(
            np.median(
                x
            )
        )
        if len(
            x
        )
        else np.nan
    )


def _q(
    x,
    q,
):
    x = pd.to_numeric(
        x,
        errors="coerce",
    ).to_numpy(
        dtype=float
    )

    x = x[
        np.isfinite(
            x
        )
    ]

    return (
        float(
            np.quantile(
                x,
                q,
            )
        )
        if len(
            x
        )
        else np.nan
    )


def _assert_same_length(
    n,
    **arrays,
):
    bad = {
        key: len(
            np.asarray(
                value
            )
        )
        for key, value
        in arrays.items()
        if len(
            np.asarray(
                value
            )
        )
        != n
    }

    if bad:
        raise RuntimeError(
            f"Length mismatch versus n={n}: {bad}"
        )


# =============================================================================
# ONE REPLICATE
# =============================================================================

def _extract_one_replicate(
    replicate: int,
):
    core = _load_success_stage(
        replicate,
        "01_core",
    )

    full = _load_success_stage(
        replicate,
        "02_full_firstorder",
    )

    geom = _load_success_stage(
        replicate,
        "03_geometry_support",
    )

    null = _load_success_stage(
        replicate,
        "04_null_deb",
    )

    base = _load_success_stage(
        replicate,
        "05_base_m03_result",
    )

    # -----------------------------------------------------------------
    # Exact current keys.
    # -----------------------------------------------------------------

    y = np.asarray(
        core[
            "y"
        ],
        dtype=float,
    )

    exposure = np.asarray(
        core[
            "exposure"
        ],
        dtype=float,
    )

    S_F = np.asarray(
        core[
            "S_F"
        ],
        dtype=float,
    )

    S_0 = np.asarray(
        core[
            "S_0"
        ],
        dtype=float,
    )

    a_F = float(
        core[
            "a_F"
        ]
    )

    a_0 = float(
        core[
            "a_0"
        ]
    )

    C_F = np.asarray(
        full[
            "C_F"
        ],
        dtype=float,
    )

    C_0 = np.asarray(
        full[
            "C_0"
        ],
        dtype=float,
    )

    R_F = np.asarray(
        full[
            "R_F"
        ],
        dtype=float,
    )

    S0dag = np.asarray(
        geom[
            "S0dag_final"
        ],
        dtype=float,
    )

    R0dag = np.asarray(
        null[
            "R_0_dagger_first"
        ],
        dtype=float,
    )

    p = np.asarray(
        base[
            "p_value_m03"
        ],
        dtype=float,
    )

    n_genes = int(
        y.shape[
            0
        ]
    )

    _assert_same_length(
        n_genes,
        S_F=S_F,
        S_0=S_0,
        C_F=C_F,
        C_0=C_0,
        R_F=R_F,
        S0dag=S0dag,
        R0dag=R0dag,
        p=p,
    )

    # -----------------------------------------------------------------
    # Internal identity checks.
    # -----------------------------------------------------------------

    rf_err = np.nanmax(
        np.abs(
            R_F
            - (
                S_F
                - C_F
            )
        )
    )

    r0_err = np.nanmax(
        np.abs(
            R0dag
            - (
                S0dag
                - C_0
            )
        )
    )

    if (
        not np.isfinite(
            rf_err
        )
        or rf_err
        > 1e-10
    ):
        raise RuntimeError(
            f"Replicate {replicate}: "
            f"R_F != S_F-C_F; max error={rf_err:.3e}"
        )

    if (
        not np.isfinite(
            r0_err
        )
        or r0_err
        > 1e-10
    ):
        raise RuntimeError(
            f"Replicate {replicate}: "
            f"R_0^dagger != S_0^dagger-C_0; "
            f"max error={r0_err:.3e}"
        )

    # -----------------------------------------------------------------
    # Coverage.
    # -----------------------------------------------------------------

    total_normalized_coverage = np.sum(
        y
        / exposure[
            None,
            :
        ],
        axis=1,
    )

    log_coverage = np.log1p(
        total_normalized_coverage
    )

    coverage_bin = (
        np.asarray(
            pd.qcut(
                pd.Series(
                    log_coverage
                ),
                q=N_COVERAGE_BINS,
                labels=False,
                duplicates="drop",
            ),
            dtype=int,
        )
        + 1
    )

    # -----------------------------------------------------------------
    # Normalized residual components.
    # -----------------------------------------------------------------

    S_F_over_aF = (
        S_F
        / a_F
    )

    C_F_over_aF = (
        C_F
        / a_F
    )

    R_F_over_aF = (
        R_F
        / a_F
    )

    S_0_over_a0 = (
        S_0
        / a_0
    )

    S0dag_over_a0 = (
        S0dag
        / a_0
    )

    C_0_over_a0 = (
        C_0
        / a_0
    )

    R0dag_over_a0 = (
        R0dag
        / a_0
    )

    geometry_removed_over_a0 = (
        (
            S_0
            - S0dag
        )
        / a_0
    )

    gene_level = pd.DataFrame(
        {
            "replicate":
                int(
                    replicate
                ),

            "coverage_bin":
                coverage_bin,

            "log1p_total_normalized_coverage":
                log_coverage,

            "S_F_over_aF":
                S_F_over_aF,

            "C_F_over_aF":
                C_F_over_aF,

            "R_F_over_aF":
                R_F_over_aF,

            "S_0_over_a0":
                S_0_over_a0,

            "S0dag_over_a0":
                S0dag_over_a0,

            "C_0_over_a0":
                C_0_over_a0,

            "R0dag_over_a0":
                R0dag_over_a0,

            "geometry_removed_over_a0":
                geometry_removed_over_a0,

            "raw_p_lt_0_01":
                (
                    np.isfinite(
                        p
                    )
                    & (
                        p
                        < RAW_P_THRESHOLD
                    )
                ),
        }
    )

    # -----------------------------------------------------------------
    # Within-split bin summaries.
    # -----------------------------------------------------------------

    rows = []

    for b, d in gene_level.groupby(
        "coverage_bin",
        sort=True,
    ):
        row = {
            "replicate":
                int(
                    replicate
                ),

            "coverage_bin":
                int(
                    b
                ),

            "n_genes":
                int(
                    len(
                        d
                    )
                ),

            "median_log1p_total_normalized_coverage":
                _med(
                    d[
                        "log1p_total_normalized_coverage"
                    ]
                ),

            "raw_p_lt_0_01_fraction":
                float(
                    np.mean(
                        d[
                            "raw_p_lt_0_01"
                        ]
                    )
                ),
        }

        for col in (
            "S_F_over_aF",
            "C_F_over_aF",
            "R_F_over_aF",
            "S_0_over_a0",
            "S0dag_over_a0",
            "C_0_over_a0",
            "R0dag_over_a0",
            "geometry_removed_over_a0",
        ):
            row[
                f"{col}_median"
            ] = _med(
                d[
                    col
                ]
            )

            row[
                f"{col}_q25"
            ] = _q(
                d[
                    col
                ],
                0.25,
            )

            row[
                f"{col}_q75"
            ] = _q(
                d[
                    col
                ],
                0.75,
            )

        rows.append(
            row
        )

    return (
        gene_level,
        pd.DataFrame(
            rows
        ),
    )


# =============================================================================
# AGGREGATE ACROSS SPLITS
# =============================================================================

def _aggregate_across_replicates(
    bin_level,
):
    metrics = [
        "median_log1p_total_normalized_coverage",
        "S_F_over_aF_median",
        "C_F_over_aF_median",
        "R_F_over_aF_median",
        "S_0_over_a0_median",
        "S0dag_over_a0_median",
        "C_0_over_a0_median",
        "R0dag_over_a0_median",
        "geometry_removed_over_a0_median",
        "raw_p_lt_0_01_fraction",
    ]

    rows = []

    for b, d in bin_level.groupby(
        "coverage_bin",
        sort=True,
    ):
        row = {
            "coverage_bin":
                int(
                    b
                ),

            "n_splits":
                int(
                    d[
                        "replicate"
                    ]
                    .nunique()
                ),
        }

        for col in metrics:
            row[
                f"{col}__median"
            ] = _med(
                d[
                    col
                ]
            )

            row[
                f"{col}__q25"
            ] = _q(
                d[
                    col
                ],
                0.25,
            )

            row[
                f"{col}__q75"
            ] = _q(
                d[
                    col
                ],
                0.75,
            )

        rows.append(
            row
        )

    return (
        pd.DataFrame(
            rows
        )
        .sort_values(
            "coverage_bin",
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )


# =============================================================================
# PLOT HELPERS
# =============================================================================

def _add_split_lines(
    ax,
    bin_level,
    metric,
):
    if not SHOW_PER_SPLIT_FAINT_LINES:
        return

    y_col = (
        f"{metric}_median"
    )

    for _, d in bin_level.groupby(
        "replicate",
        sort=True,
    ):
        d = d.sort_values(
            "coverage_bin",
            kind="stable",
        )

        ax.plot(
            d[
                "median_log1p_total_normalized_coverage"
            ],
            d[
                y_col
            ],
            linewidth=0.8,
            alpha=0.12,
        )


def _plot_metric(
    ax,
    x,
    agg,
    metric,
    label,
    *,
    linestyle="-",
    marker="o",
):
    ax.plot(
        x,
        agg[
            f"{metric}_median__median"
        ],
        linewidth=2.4,
        linestyle=linestyle,
        marker=marker,
        label=label,
    )

    ax.fill_between(
        x,
        agg[
            f"{metric}_median__q25"
        ],
        agg[
            f"{metric}_median__q75"
        ],
        alpha=0.14,
    )


def _finish(
    ax,
    title,
    ylabel,
):
    ax.set_xlabel(
        "log1p(total normalized gene coverage)",
        fontsize=14,
    )

    ax.set_ylabel(
        ylabel,
        fontsize=14,
    )

    ax.set_title(
        title,
        fontsize=18,
        pad=16,
    )

    ax.axhline(
        0.0,
        linestyle=":",
        linewidth=1.0,
    )

    ax.grid(
        alpha=0.20,
    )

    ax.legend(
        fontsize=11,
    )

    plt.tight_layout()
    plt.show()


# =============================================================================
# PLOTS
# =============================================================================

def _show_plots(
    bin_level,
    agg,
):
    x = (
        agg[
            "median_log1p_total_normalized_coverage__median"
        ]
        .to_numpy(
            dtype=float
        )
    )

    # -----------------------------------------------------------------
    # 1. Full model components.
    # -----------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    _plot_metric(
        ax,
        x,
        agg,
        "S_F_over_aF",
        r"$S_F/a_F$",
    )

    _plot_metric(
        ax,
        x,
        agg,
        "C_F_over_aF",
        r"$C_F/a_F$",
        linestyle="--",
        marker="s",
    )

    _plot_metric(
        ax,
        x,
        agg,
        "R_F_over_aF",
        r"$R_F/a_F=(S_F-C_F)/a_F$",
        linestyle="-.",
        marker="^",
    )

    _finish(
        ax,
        "Pickrell: Full-model residual components by coverage",
        "Component divided by $a_F$",
    )

    # -----------------------------------------------------------------
    # 2. Null model components.
    # -----------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    _plot_metric(
        ax,
        x,
        agg,
        "S0dag_over_a0",
        r"$S_0^\dagger/a_0$",
    )

    _plot_metric(
        ax,
        x,
        agg,
        "C_0_over_a0",
        r"$C_0/a_0$",
        linestyle="--",
        marker="s",
    )

    _plot_metric(
        ax,
        x,
        agg,
        "R0dag_over_a0",
        r"$R_0^\dagger/a_0=(S_0^\dagger-C_0)/a_0$",
        linestyle="-.",
        marker="^",
    )

    _finish(
        ax,
        "Pickrell: Geometry-adjusted null residual components by coverage",
        "Component divided by $a_0$",
    )

    # -----------------------------------------------------------------
    # 3. Raw null S0 vs geometry-adjusted S0dag.
    # -----------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    _plot_metric(
        ax,
        x,
        agg,
        "S_0_over_a0",
        r"Raw $S_0/a_0$",
    )

    _plot_metric(
        ax,
        x,
        agg,
        "S0dag_over_a0",
        r"Adjusted $S_0^\dagger/a_0$",
        linestyle="--",
        marker="s",
    )

    _plot_metric(
        ax,
        x,
        agg,
        "geometry_removed_over_a0",
        r"Removed $(S_0-S_0^\dagger)/a_0$",
        linestyle="-.",
        marker="^",
    )

    _finish(
        ax,
        "Pickrell: Null geometry adjustment by coverage",
        "Null deviance component divided by $a_0$",
    )

    # -----------------------------------------------------------------
    # 4. Corrected residuals R_F/aF and R0dag/a0 together.
    # -----------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    _plot_metric(
        ax,
        x,
        agg,
        "R_F_over_aF",
        r"$R_F/a_F$",
    )

    _plot_metric(
        ax,
        x,
        agg,
        "R0dag_over_a0",
        r"$R_0^\dagger/a_0$",
        linestyle="--",
        marker="s",
    )

    # Reference 1 corresponds to raw theta = a/R = 1.
    ax.axhline(
        1.0,
        linestyle=":",
        linewidth=1.5,
        label=r"$R/a=1$  ($\widetilde\theta=1$)",
    )

    ax.set_xlabel(
        "log1p(total normalized gene coverage)",
        fontsize=14,
    )

    ax.set_ylabel(
        "Corrected residual score / shape",
        fontsize=14,
    )

    ax.set_title(
        r"Pickrell: Corrected residual information $R/a$ by coverage",
        fontsize=18,
        pad=16,
    )

    ax.grid(
        alpha=0.20,
    )

    ax.legend(
        fontsize=11,
    )

    plt.tight_layout()
    plt.show()

    # -----------------------------------------------------------------
    # 5. R/a versus raw-p hump.
    # -----------------------------------------------------------------

    fig, ax1 = plt.subplots(
        figsize=FIGSIZE
    )

    ax2 = (
        ax1.twinx()
    )

    ax1.plot(
        x,
        agg[
            "raw_p_lt_0_01_fraction__median"
        ],
        marker="o",
        linewidth=2.4,
        label="Raw p<.01 fraction",
    )

    ax1.fill_between(
        x,
        agg[
            "raw_p_lt_0_01_fraction__q25"
        ],
        agg[
            "raw_p_lt_0_01_fraction__q75"
        ],
        alpha=0.12,
    )

    ax1.axhline(
        RAW_P_THRESHOLD,
        linestyle=":",
        linewidth=1.5,
    )

    ax2.plot(
        x,
        agg[
            "R_F_over_aF_median__median"
        ],
        marker="s",
        linewidth=2.2,
        linestyle="--",
        label=r"$R_F/a_F$",
    )

    ax2.plot(
        x,
        agg[
            "R0dag_over_a0_median__median"
        ],
        marker="^",
        linewidth=2.2,
        linestyle="-.",
        label=r"$R_0^\dagger/a_0$",
    )

    ax2.axhline(
        1.0,
        linestyle=":",
        linewidth=1.2,
    )

    ax1.set_xlabel(
        "log1p(total normalized gene coverage)",
        fontsize=14,
    )

    ax1.set_ylabel(
        "Median fraction raw p < .01",
        fontsize=14,
    )

    ax2.set_ylabel(
        "Median corrected residual score / shape",
        fontsize=14,
    )

    ax1.set_title(
        "Pickrell: Corrected residual-score trend vs raw-p inflation",
        fontsize=18,
        pad=16,
    )

    ax1.grid(
        alpha=0.20,
    )

    lines1, labels1 = (
        ax1.get_legend_handles_labels()
    )

    lines2, labels2 = (
        ax2.get_legend_handles_labels()
    )

    ax1.legend(
        lines1 + lines2,
        labels1 + labels2,
        fontsize=11,
        loc="upper right",
    )

    plt.tight_layout()
    plt.show()

    # -----------------------------------------------------------------
    # 6. C/a versus raw-p hump.
    # -----------------------------------------------------------------

    fig, ax1 = plt.subplots(
        figsize=FIGSIZE
    )

    ax2 = (
        ax1.twinx()
    )

    ax1.plot(
        x,
        agg[
            "raw_p_lt_0_01_fraction__median"
        ],
        marker="o",
        linewidth=2.4,
        label="Raw p<.01 fraction",
    )

    ax1.fill_between(
        x,
        agg[
            "raw_p_lt_0_01_fraction__q25"
        ],
        agg[
            "raw_p_lt_0_01_fraction__q75"
        ],
        alpha=0.12,
    )

    ax1.axhline(
        RAW_P_THRESHOLD,
        linestyle=":",
        linewidth=1.5,
    )

    ax2.plot(
        x,
        agg[
            "C_F_over_aF_median__median"
        ],
        marker="s",
        linewidth=2.2,
        linestyle="--",
        label=r"$C_F/a_F$",
    )

    ax2.plot(
        x,
        agg[
            "C_0_over_a0_median__median"
        ],
        marker="^",
        linewidth=2.2,
        linestyle="-.",
        label=r"$C_0/a_0$",
    )

    ax1.set_xlabel(
        "log1p(total normalized gene coverage)",
        fontsize=14,
    )

    ax1.set_ylabel(
        "Median fraction raw p < .01",
        fontsize=14,
    )

    ax2.set_ylabel(
        "Median normalizer correction / shape",
        fontsize=14,
    )

    ax1.set_title(
        "Pickrell: Normalizer correction trend vs raw-p inflation",
        fontsize=18,
        pad=16,
    )

    ax1.grid(
        alpha=0.20,
    )

    lines1, labels1 = (
        ax1.get_legend_handles_labels()
    )

    lines2, labels2 = (
        ax2.get_legend_handles_labels()
    )

    ax1.legend(
        lines1 + lines2,
        labels1 + labels2,
        fontsize=11,
        loc="upper right",
    )

    plt.tight_layout()
    plt.show()


# =============================================================================
# TOP-LEVEL
# =============================================================================

def run_pickrell_residual_component_coverage_audit():
    started = (
        datetime.now()
        .astimezone()
    )

    print(
        f"[start] {_now()}",
        flush=True,
    )

    try:
        reps = (
            _discover_completed_replicates()
        )

        print(
            "\nPICKRELL RESIDUAL-COMPONENT VS COVERAGE AUDIT\n"
            f"namespace:      {CURRENT_M03_NAMESPACE}\n"
            f"completed reps: {len(reps)}\n"
            f"replicates:     {reps}\n",
            flush=True,
        )

        gene_frames = []
        bin_frames = []

        for j, rep in enumerate(
            reps,
            start=1,
        ):
            (
                gene_df,
                bin_df,
            ) = (
                _extract_one_replicate(
                    rep
                )
            )

            gene_frames.append(
                gene_df
            )

            bin_frames.append(
                bin_df
            )

            print(
                f"[progress] "
                f"{j:02d}/{len(reps):02d} "
                f"| rep={rep:02d}",
                flush=True,
            )

        gene_level = pd.concat(
            gene_frames,
            ignore_index=True,
        )

        bin_level = pd.concat(
            bin_frames,
            ignore_index=True,
        )

        aggregate = (
            _aggregate_across_replicates(
                bin_level
            )
        )

        print(
            "\nBIN-LEVEL MEDIAN SUMMARY ACROSS PICKRELL SPLITS\n",
            flush=True,
        )

        display(
            aggregate[
                [
                    "coverage_bin",
                    "n_splits",
                    "median_log1p_total_normalized_coverage__median",

                    "S_F_over_aF_median__median",
                    "C_F_over_aF_median__median",
                    "R_F_over_aF_median__median",

                    "S_0_over_a0_median__median",
                    "S0dag_over_a0_median__median",
                    "C_0_over_a0_median__median",
                    "R0dag_over_a0_median__median",

                    "geometry_removed_over_a0_median__median",

                    "raw_p_lt_0_01_fraction__median",
                ]
            ]
        )

        _show_plots(
            bin_level,
            aggregate,
        )

        return {
            "gene_level":
                gene_level,

            "bin_level":
                bin_level,

            "aggregate":
                aggregate,
        }

    finally:
        ended = (
            datetime.now()
            .astimezone()
        )

        print(
            f"[end]   {_now()} | "
            f"elapsed={ended-started}",
            flush=True,
        )


# =============================================================================
# EXECUTION
# =============================================================================

pickrell_residual_component_coverage_audit = (
    run_pickrell_residual_component_coverage_audit()
)


In [ ]:
from __future__ import annotations

"""
Pickrell 12-bin DEB hyperparameter counterfactual.

Question
--------
Does the Pickrell raw-p calibration hump arise because the current DEB precision
model fits ONE genome-wide Gamma prior for theta even though the residual
precision information changes strongly with gene coverage?

Diagnostic intervention
-----------------------
Keep ALL upstream M03 quantities frozen, including:

    - r_g / alpha_g
    - fitted full/null means
    - C_F and C_0
    - the current GLOBAL kappa_c / lambda_c
    - S_0^dagger from Stage 03
    - the support-floor correction
    - the mandatory exact branch for R_F <= 0 genes

Change ONLY the DEB precision prior:

    CURRENT:
        one full-model DEB prior across all genes
        one null-model DEB prior across all genes

    COUNTERFACTUAL:
        fit one full-model DEB prior inside each of 12 equal-count coverage bins
        fit one null-model DEB prior inside each of 12 equal-count coverage bins

Then recompute the ordinary first-order M03 statistic using those bin-specific
theta_F and theta_0 estimates.

For mandatory exact genes (R_F <= 0), the exact normalized-DNB profile does not
depend on the DEB prior. Their existing canonical M03 p-values are therefore
reused unchanged.

This is an in-sample diagnostic, not yet the proposed production estimator.

Outputs
-------
The computation is resumable. Per-split numerical results are checkpointed under

    <repo>/data/pickrell_m03_12bin_deb_counterfactual_v1/

Plots are DISPLAYED ONLY and are not saved.

Displayed diagnostics include:

    1. raw p<.01 by coverage: global-DEB vs 12-bin-DEB
    2. theta_F by coverage: global-DEB vs 12-bin-DEB vs raw a_F/R_F
    3. theta_0 by coverage: global-DEB vs 12-bin-DEB vs raw a_0/R_0^dagger
    4. full DEB prior mean theta by coverage
    5. null DEB prior mean theta by coverage
    6. full/null DEB alpha0 by coverage
    7. paired split-level raw-p calibration
    8. BH(.05) discovery counts
    9. complete-null BH FDR = P(any rejection)
"""

from datetime import datetime
from pathlib import Path
from scipy.optimize import minimize
from scipy.special import betaln
from scipy.stats import chi2

import json
import os
import pickle
import tempfile
import time
import traceback

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


# =============================================================================
# SETTINGS
# =============================================================================

CURRENT_M03_NAMESPACE = "pickrell_m03_seam_safe_20M_kappa_coverage_v1"
COUNTERFACTUAL_NAMESPACE = "pickrell_m03_12bin_deb_counterfactual_v1"

N_COVERAGE_BINS = 12

RAW_P_THRESHOLD = 0.01
FDR_CUTOFF = 0.05

RESUME = True
SHOW_PLOTS = True

FIGSIZE = (16, 9)


# =============================================================================
# PATHS / TIME
# =============================================================================

def _now() -> str:
    return datetime.now().astimezone().strftime(
        "%Y-%m-%d %I:%M:%S %p %Z"
    )


CWD = Path.cwd().resolve()

if CWD.name.lower() in {"notebook", "notebooks"}:
    PROJECT_ROOT = CWD.parent
elif (CWD / "data").is_dir():
    PROJECT_ROOT = CWD
else:
    raise RuntimeError(
        "Could not resolve repository root. Run from <repo>/notebook, "
        "<repo>/notebooks, or the repository root containing ./data. "
        f"Current working directory: {CWD}"
    )

PROJECT_DATA_DIR = PROJECT_ROOT / "data"

GLOBAL_RUN_DIR = (
    PROJECT_DATA_DIR
    / CURRENT_M03_NAMESPACE
)

GLOBAL_CELL_DIR = (
    GLOBAL_RUN_DIR
    / "cells"
)

M03_CHECKPOINT_ROOT = (
    GLOBAL_RUN_DIR
    / "m03_checkpoints"
)

OUTPUT_DIR = (
    PROJECT_DATA_DIR
    / COUNTERFACTUAL_NAMESPACE
)

SPLIT_CACHE_DIR = (
    OUTPUT_DIR
    / "per_split"
)

SENTINEL = (
    OUTPUT_DIR
    / "_pickrell_12bin_deb_counterfactual_v1_initialized.json"
)


# =============================================================================
# FILE HELPERS
# =============================================================================

def _load_pickle(
    path: Path,
):
    with open(
        path,
        "rb",
    ) as f:
        return pickle.load(
            f
        )


def _atomic_pickle(
    obj,
    path: Path,
):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    fd, tmp_name = tempfile.mkstemp(
        prefix=f".{path.name}.",
        dir=str(
            path.parent
        ),
    )

    os.close(
        fd
    )

    tmp = Path(
        tmp_name
    )

    try:
        with open(
            tmp,
            "wb",
        ) as f:
            pickle.dump(
                obj,
                f,
                protocol=pickle.HIGHEST_PROTOCOL,
            )

            f.flush()

            try:
                os.fsync(
                    f.fileno()
                )
            except OSError:
                pass

        os.replace(
            tmp,
            path,
        )

    finally:
        if tmp.exists():
            try:
                tmp.unlink()
            except OSError:
                pass


def _atomic_csv(
    df: pd.DataFrame,
    path: Path,
):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    fd, tmp_name = tempfile.mkstemp(
        prefix=f".{path.name}.",
        dir=str(
            path.parent
        ),
    )

    os.close(
        fd
    )

    tmp = Path(
        tmp_name
    )

    try:
        df.to_csv(
            tmp,
            index=False,
        )

        os.replace(
            tmp,
            path,
        )

    finally:
        if tmp.exists():
            try:
                tmp.unlink()
            except OSError:
                pass


def _checkpoint_dir(
    replicate: int,
):
    return (
        M03_CHECKPOINT_ROOT
        / f"pickrell_rep{int(replicate):02d}"
    )


def _load_success_stage(
    replicate: int,
    stage_name: str,
):
    path = (
        _checkpoint_dir(
            replicate
        )
        / "stages"
        / f"{stage_name}.pkl"
    )

    if not path.exists():
        raise FileNotFoundError(
            f"Missing checkpoint: {path}"
        )

    payload = _load_pickle(
        path
    )

    if (
        not isinstance(
            payload,
            dict,
        )
        or payload.get(
            "status"
        )
        != "success"
    ):
        raise RuntimeError(
            f"Replicate {replicate}: "
            f"{stage_name} is not a successful checkpoint."
        )

    return payload[
        "data"
    ]


def _load_manifest(
    replicate: int,
):
    path = (
        _checkpoint_dir(
            replicate
        )
        / "m03_manifest.pkl"
    )

    if not path.exists():
        raise FileNotFoundError(
            f"Missing manifest: {path}"
        )

    return _load_pickle(
        path
    )


def _discover_completed_replicates():
    reps = []

    for path in sorted(
        GLOBAL_CELL_DIR.glob(
            "pickrell_rep*.csv"
        )
    ):
        try:
            x = pd.read_csv(
                path
            )
        except Exception:
            continue

        if len(
            x
        ) != 1:
            continue

        if (
            str(
                x.loc[
                    0,
                    "status",
                ]
            )
            .strip()
            .lower()
            != "success"
        ):
            continue

        reps.append(
            int(
                x.loc[
                    0,
                    "replicate",
                ]
            )
        )

    reps = sorted(
        set(
            reps
        )
    )

    if not reps:
        raise RuntimeError(
            "No completed Pickrell replicates were found."
        )

    return reps


# =============================================================================
# NUMERIC HELPERS
# =============================================================================

def _bh_adjust(
    p,
):
    p = np.asarray(
        p,
        dtype=float,
    )

    q = np.full_like(
        p,
        np.nan,
    )

    ok = (
        np.isfinite(
            p
        )
        & (
            p
            >= 0
        )
        & (
            p
            <= 1
        )
    )

    if not np.any(
        ok
    ):
        return q

    p0 = p[
        ok
    ]

    order = np.argsort(
        p0,
        kind="stable",
    )

    ranked = p0[
        order
    ]

    n = len(
        ranked
    )

    adjusted = (
        ranked
        * n
        / np.arange(
            1,
            n + 1,
        )
    )

    adjusted = np.minimum.accumulate(
        adjusted[
            ::-1
        ]
    )[
        ::-1
    ]

    adjusted = np.minimum(
        adjusted,
        1.0,
    )

    restored = np.empty_like(
        adjusted
    )

    restored[
        order
    ] = adjusted

    q[
        ok
    ] = restored

    return q


def _med(
    x,
):
    x = pd.to_numeric(
        x,
        errors="coerce",
    ).to_numpy(
        dtype=float
    )

    x = x[
        np.isfinite(
            x
        )
    ]

    return (
        float(
            np.median(
                x
            )
        )
        if len(
            x
        )
        else np.nan
    )


def _q(
    x,
    q,
):
    x = pd.to_numeric(
        x,
        errors="coerce",
    ).to_numpy(
        dtype=float
    )

    x = x[
        np.isfinite(
            x
        )
    ]

    return (
        float(
            np.quantile(
                x,
                q,
            )
        )
        if len(
            x
        )
        else np.nan
    )


def _raw_theta(
    a,
    R,
):
    R = np.asarray(
        R,
        dtype=float,
    )

    out = np.full(
        len(
            R
        ),
        np.nan,
        dtype=float,
    )

    ok = (
        np.isfinite(
            R
        )
        & (
            R
            > 0
        )
    )

    out[
        ok
    ] = (
        float(
            a
        )
        / R[
            ok
        ]
    )

    return out


# =============================================================================
# SAME DEB FIT USED BY CURRENT M03
# =============================================================================

def _fit_deb_masked(
    S,
    a,
    settings,
    label,
):
    S = np.asarray(
        S,
        dtype=float,
    )

    corrected_score_tolerance = float(
        settings[
            "corrected_score_tolerance"
        ]
    )

    theta_floor = float(
        settings[
            "theta_floor"
        ]
    )

    theta_cap = float(
        settings[
            "theta_cap"
        ]
    )

    deb_precision_maxiter = int(
        settings[
            "deb_precision_maxiter"
        ]
    )

    valid = (
        np.isfinite(
            S
        )
        & (
            S
            > corrected_score_tolerance
        )
    )

    if valid.sum() < 100:
        raise RuntimeError(
            f"{label}: too few positive corrected scores for DEB "
            f"({int(valid.sum())})."
        )

    x = S[
        valid
    ]

    a0 = float(
        a
    )

    theta_pool = float(
        np.clip(
            a0
            * len(
                x
            )
            / np.sum(
                x
            ),
            theta_floor,
            theta_cap,
        )
    )

    def objective(
        par,
    ):
        alpha0 = float(
            np.exp(
                par[
                    0
                ]
            )
        )

        theta_bar = float(
            np.exp(
                par[
                    1
                ]
            )
        )

        beta0 = (
            alpha0
            / theta_bar
        )

        ll = (
            (
                a0
                - 1.0
            )
            * np.log(
                x
            )
            - a0
            * np.log(
                beta0
            )
            - (
                a0
                + alpha0
            )
            * np.log1p(
                x
                / beta0
            )
            - betaln(
                a0,
                alpha0,
            )
        )

        return (
            -float(
                np.mean(
                    ll
                )
            )
            if np.all(
                np.isfinite(
                    ll
                )
            )
            else 1e100
        )

    fits = []

    theta_median = float(
        np.clip(
            np.median(
                a0
                / x
            ),
            theta_floor,
            theta_cap,
        )
    )

    for alpha_start in (
        0.05,
        0.2,
        1.0,
        5.0,
        25.0,
        100.0,
        1000.0,
        1e4,
    ):
        for theta_start in (
            theta_pool,
            theta_median,
        ):
            fit = minimize(
                objective,
                x0=np.log(
                    [
                        alpha_start,
                        theta_start,
                    ]
                ),
                method="L-BFGS-B",
                bounds=[
                    (
                        np.log(
                            1e-8
                        ),
                        np.log(
                            1e9
                        ),
                    ),
                    (
                        np.log(
                            theta_floor
                        ),
                        np.log(
                            theta_cap
                        ),
                    ),
                ],
                options={
                    "maxiter":
                        deb_precision_maxiter,

                    "ftol":
                        1e-12,
                },
            )

            if np.isfinite(
                fit.fun
            ):
                fits.append(
                    fit
                )

    if not fits:
        raise RuntimeError(
            f"{label}: DEB fit failed."
        )

    best = min(
        fits,
        key=lambda x: x.fun,
    )

    alpha0 = float(
        np.exp(
            best.x[
                0
            ]
        )
    )

    theta_bar = float(
        np.exp(
            best.x[
                1
            ]
        )
    )

    beta0 = (
        alpha0
        / theta_bar
    )

    theta_valid = (
        a0
        + alpha0
    ) / (
        x
        + beta0
    )

    theta = np.full(
        len(
            S
        ),
        np.nan,
    )

    theta[
        valid
    ] = np.clip(
        theta_valid,
        theta_floor,
        theta_cap,
    )

    return {
        "theta":
            theta,

        "valid_mask":
            valid,

        "alpha0":
            alpha0,

        "beta0":
            beta0,

        "prior_mean_theta":
            theta_bar,

        "objective":
            float(
                best.fun
            ),

        "n_valid":
            int(
                valid.sum()
            ),

        "n_invalid":
            int(
                (
                    ~valid
                ).sum()
            ),
    }


# =============================================================================
# SAME FIRST-ORDER NORMALIZED STATISTIC USED BY CURRENT M03
# =============================================================================

def _normalized_statistic(
    S_F,
    S_0_used,
    C_F,
    C_0,
    theta_F,
    theta_0,
    n,
    settings,
):
    corrected_score_tolerance = float(
        settings[
            "corrected_score_tolerance"
        ]
    )

    R_F = (
        S_F
        - C_F
    )

    R_0 = (
        S_0_used
        - C_0
    )

    ok = (
        np.isfinite(
            R_F
        )
        & np.isfinite(
            R_0
        )
        & np.isfinite(
            theta_F
        )
        & np.isfinite(
            theta_0
        )
        & (
            R_F
            > corrected_score_tolerance
        )
        & (
            R_0
            > corrected_score_tolerance
        )
        & (
            theta_F
            > 0
        )
        & (
            theta_0
            > 0
        )
    )

    T_raw = np.full(
        len(
            S_F
        ),
        np.nan,
    )

    T_raw[
        ok
    ] = (
        2.0
        * (
            theta_0[
                ok
            ]
            * R_0[
                ok
            ]
            - theta_F[
                ok
            ]
            * R_F[
                ok
            ]
        )
        + float(
            n
        )
        * np.log(
            theta_F[
                ok
            ]
            / theta_0[
                ok
            ]
        )
        + 2.0
        * (
            C_0[
                ok
            ]
            - C_F[
                ok
            ]
        )
    )

    T = np.full(
        len(
            S_F
        ),
        np.nan,
    )

    T[
        ok
    ] = np.maximum(
        T_raw[
            ok
        ],
        0.0,
    )

    p = np.full(
        len(
            S_F
        ),
        np.nan,
    )

    p[
        ok
    ] = chi2.sf(
        T[
            ok
        ],
        1,
    )

    return {
        "valid_mask":
            ok,

        "T_raw":
            T_raw,

        "T":
            T,

        "p":
            p,

        "R_F":
            R_F,

        "R_0":
            R_0,
    }


# =============================================================================
# ONE-SPLIT COUNTERFACTUAL
# =============================================================================

def _run_one_replicate(
    replicate: int,
):
    cache_path = (
        SPLIT_CACHE_DIR
        / f"pickrell_rep{int(replicate):02d}.pkl"
    )

    if (
        RESUME
        and cache_path.exists()
    ):
        try:
            payload = _load_pickle(
                cache_path
            )

            if (
                isinstance(
                    payload,
                    dict,
                )
                and payload.get(
                    "version"
                )
                == "pickrell-12bin-deb-v1"
            ):
                print(
                    f"[reload] {_now()} | "
                    f"rep={replicate:02d}",
                    flush=True,
                )

                return (
                    payload,
                    True,
                )

        except Exception:
            pass

    t0 = time.perf_counter()

    core = _load_success_stage(
        replicate,
        "01_core",
    )

    full = _load_success_stage(
        replicate,
        "02_full_firstorder",
    )

    geom = _load_success_stage(
        replicate,
        "03_geometry_support",
    )

    null = _load_success_stage(
        replicate,
        "04_null_deb",
    )

    base = _load_success_stage(
        replicate,
        "05_base_m03_result",
    )

    manifest = _load_manifest(
        replicate
    )

    settings = dict(
        manifest[
            "numerical_settings"
        ]
    )

    y = np.asarray(
        core[
            "y"
        ],
        dtype=float,
    )

    exposure = np.asarray(
        core[
            "exposure"
        ],
        dtype=float,
    )

    S_F = np.asarray(
        core[
            "S_F"
        ],
        dtype=float,
    )

    a_F = float(
        core[
            "a_F"
        ]
    )

    a_0 = float(
        core[
            "a_0"
        ]
    )

    n = int(
        core[
            "n"
        ]
    )

    C_F = np.asarray(
        full[
            "C_F"
        ],
        dtype=float,
    )

    C_0 = np.asarray(
        full[
            "C_0"
        ],
        dtype=float,
    )

    R_F = np.asarray(
        full[
            "R_F"
        ],
        dtype=float,
    )

    theta_F_global = np.asarray(
        full[
            "theta_F_first"
        ],
        dtype=float,
    )

    mask_Rnonpos = np.asarray(
        full[
            "mask_Rnonpos"
        ],
        dtype=bool,
    )

    S0dag = np.asarray(
        geom[
            "S0dag_final"
        ],
        dtype=float,
    )

    R0dag = np.asarray(
        null[
            "R_0_dagger_first"
        ],
        dtype=float,
    )

    theta_0_global = np.asarray(
        null[
            "theta_0_first"
        ],
        dtype=float,
    )

    original_test = null[
        "test_first"
    ]

    p_global = np.asarray(
        base[
            "p_value_m03"
        ],
        dtype=float,
    )

    q_global = np.asarray(
        base[
            "q_value_m03"
        ],
        dtype=float,
    )

    n_genes = int(
        y.shape[
            0
        ]
    )

    # -----------------------------------------------------------------
    # Validate our statistic implementation against the saved first-order
    # statistic BEFORE changing any DEB prior.
    # -----------------------------------------------------------------

    reconstructed = _normalized_statistic(
        S_F,
        S0dag,
        C_F,
        C_0,
        theta_F_global,
        theta_0_global,
        n,
        settings,
    )

    p_first_saved = np.asarray(
        original_test[
            "p"
        ],
        dtype=float,
    )

    finite_compare = (
        np.isfinite(
            reconstructed[
                "p"
            ]
        )
        & np.isfinite(
            p_first_saved
        )
    )

    if np.any(
        finite_compare
    ):
        max_p_error = float(
            np.max(
                np.abs(
                    reconstructed[
                        "p"
                    ][
                        finite_compare
                    ]
                    - p_first_saved[
                        finite_compare
                    ]
                )
            )
        )

        if max_p_error > 1e-12:
            raise RuntimeError(
                f"Replicate {replicate}: "
                f"saved first-order p-value reconstruction failed; "
                f"max abs error={max_p_error:.3e}"
            )

    # -----------------------------------------------------------------
    # Coverage bins.
    # -----------------------------------------------------------------

    total_normalized_coverage = np.sum(
        y
        / exposure[
            None,
            :
        ],
        axis=1,
    )

    log_coverage = np.log1p(
        total_normalized_coverage
    )

    coverage_bin = (
        np.asarray(
            pd.qcut(
                pd.Series(
                    log_coverage
                ),
                q=N_COVERAGE_BINS,
                labels=False,
                duplicates="drop",
            ),
            dtype=int,
        )
        + 1
    )

    actual_bins = sorted(
        int(
            b
        )
        for b in np.unique(
            coverage_bin
        )
    )

    # -----------------------------------------------------------------
    # Fit separate full/null DEB priors within each coverage bin.
    # -----------------------------------------------------------------

    theta_F_bin = np.full(
        n_genes,
        np.nan,
        dtype=float,
    )

    theta_0_bin = np.full(
        n_genes,
        np.nan,
        dtype=float,
    )

    hyper_rows = []

    for b in actual_bins:
        idx = (
            coverage_bin
            == b
        )

        full_fit = _fit_deb_masked(
            R_F[
                idx
            ],
            a_F,
            settings,
            (
                f"Pickrell rep {replicate} "
                f"coverage bin {b} full DEB"
            ),
        )

        null_fit = _fit_deb_masked(
            R0dag[
                idx
            ],
            a_0,
            settings,
            (
                f"Pickrell rep {replicate} "
                f"coverage bin {b} null DEB"
            ),
        )

        theta_F_bin[
            idx
        ] = np.asarray(
            full_fit[
                "theta"
            ],
            dtype=float,
        )

        theta_0_bin[
            idx
        ] = np.asarray(
            null_fit[
                "theta"
            ],
            dtype=float,
        )

        hyper_rows.append(
            {
                "replicate":
                    int(
                        replicate
                    ),

                "coverage_bin":
                    int(
                        b
                    ),

                "n_genes":
                    int(
                        np.sum(
                            idx
                        )
                    ),

                "median_log1p_total_normalized_coverage":
                    float(
                        np.median(
                            log_coverage[
                                idx
                            ]
                        )
                    ),

                "full_alpha0":
                    float(
                        full_fit[
                            "alpha0"
                        ]
                    ),

                "full_beta0":
                    float(
                        full_fit[
                            "beta0"
                        ]
                    ),

                "full_prior_mean_theta":
                    float(
                        full_fit[
                            "prior_mean_theta"
                        ]
                    ),

                "full_n_valid":
                    int(
                        full_fit[
                            "n_valid"
                        ]
                    ),

                "null_alpha0":
                    float(
                        null_fit[
                            "alpha0"
                        ]
                    ),

                "null_beta0":
                    float(
                        null_fit[
                            "beta0"
                        ]
                    ),

                "null_prior_mean_theta":
                    float(
                        null_fit[
                            "prior_mean_theta"
                        ]
                    ),

                "null_n_valid":
                    int(
                        null_fit[
                            "n_valid"
                        ]
                    ),
            }
        )

    # -----------------------------------------------------------------
    # Recompute ordinary first-order test with binned DEB theta estimates.
    # Global geometry is intentionally frozen.
    # -----------------------------------------------------------------

    test_bin = _normalized_statistic(
        S_F,
        S0dag,
        C_F,
        C_0,
        theta_F_bin,
        theta_0_bin,
        n,
        settings,
    )

    p_bin = np.asarray(
        test_bin[
            "p"
        ],
        dtype=float,
    ).copy()

    # -----------------------------------------------------------------
    # Mandatory exact genes are independent of the DEB prior.
    # Reuse their canonical exact M03 p-values unchanged.
    # -----------------------------------------------------------------

    p_bin[
        mask_Rnonpos
    ] = p_global[
        mask_Rnonpos
    ]

    q_bin = _bh_adjust(
        p_bin
    )

    # -----------------------------------------------------------------
    # Raw pre-DEB theta for diagnostic plots.
    # -----------------------------------------------------------------

    raw_theta_F = _raw_theta(
        a_F,
        R_F,
    )

    raw_theta_0 = _raw_theta(
        a_0,
        R0dag,
    )

    # -----------------------------------------------------------------
    # Bin-level before/after summary.
    # -----------------------------------------------------------------

    bin_rows = []

    hyper_df = pd.DataFrame(
        hyper_rows
    )

    for b in actual_bins:
        idx = (
            coverage_bin
            == b
        )

        n_bin = int(
            np.sum(
                idx
            )
        )

        h = hyper_df.loc[
            hyper_df[
                "coverage_bin"
            ]
            == b
        ].iloc[
            0
        ]

        bin_rows.append(
            {
                "replicate":
                    int(
                        replicate
                    ),

                "coverage_bin":
                    int(
                        b
                    ),

                "n_genes":
                    n_bin,

                "median_log1p_total_normalized_coverage":
                    float(
                        np.median(
                            log_coverage[
                                idx
                            ]
                        )
                    ),

                "raw_theta_F_median":
                    _med(
                        pd.Series(
                            raw_theta_F[
                                idx
                            ]
                        )
                    ),

                "global_theta_F_median":
                    _med(
                        pd.Series(
                            theta_F_global[
                                idx
                            ]
                        )
                    ),

                "binned_theta_F_median":
                    _med(
                        pd.Series(
                            theta_F_bin[
                                idx
                            ]
                        )
                    ),

                "raw_theta_0_median":
                    _med(
                        pd.Series(
                            raw_theta_0[
                                idx
                            ]
                        )
                    ),

                "global_theta_0_median":
                    _med(
                        pd.Series(
                            theta_0_global[
                                idx
                            ]
                        )
                    ),

                "binned_theta_0_median":
                    _med(
                        pd.Series(
                            theta_0_bin[
                                idx
                            ]
                        )
                    ),

                "global_raw_p_lt_0_01_fraction":
                    float(
                        np.mean(
                            np.isfinite(
                                p_global[
                                    idx
                                ]
                            )
                            & (
                                p_global[
                                    idx
                                ]
                                < RAW_P_THRESHOLD
                            )
                        )
                    ),

                "binned_raw_p_lt_0_01_fraction":
                    float(
                        np.mean(
                            np.isfinite(
                                p_bin[
                                    idx
                                ]
                            )
                            & (
                                p_bin[
                                    idx
                                ]
                                < RAW_P_THRESHOLD
                            )
                        )
                    ),

                "full_alpha0":
                    float(
                        h[
                            "full_alpha0"
                        ]
                    ),

                "full_beta0":
                    float(
                        h[
                            "full_beta0"
                        ]
                    ),

                "full_prior_mean_theta":
                    float(
                        h[
                            "full_prior_mean_theta"
                        ]
                    ),

                "null_alpha0":
                    float(
                        h[
                            "null_alpha0"
                        ]
                    ),

                "null_beta0":
                    float(
                        h[
                            "null_beta0"
                        ]
                    ),

                "null_prior_mean_theta":
                    float(
                        h[
                            "null_prior_mean_theta"
                        ]
                    ),
            }
        )

    bin_df = pd.DataFrame(
        bin_rows
    )

    # -----------------------------------------------------------------
    # Split-level overall performance.
    # Original canonical p-values include the same mandatory exact branch.
    # -----------------------------------------------------------------

    global_finite = np.isfinite(
        p_global
    )

    bin_finite = np.isfinite(
        p_bin
    )

    summary = {
        "replicate":
            int(
                replicate
            ),

        "n_genes":
            int(
                n_genes
            ),

        "global_fraction_p_lt_0_01_finite":
            float(
                np.mean(
                    p_global[
                        global_finite
                    ]
                    < RAW_P_THRESHOLD
                )
            ),

        "binned_fraction_p_lt_0_01_finite":
            float(
                np.mean(
                    p_bin[
                        bin_finite
                    ]
                    < RAW_P_THRESHOLD
                )
            ),

        "global_fraction_p_lt_0_01_fixed":
            float(
                np.sum(
                    global_finite
                    & (
                        p_global
                        < RAW_P_THRESHOLD
                    )
                )
                / n_genes
            ),

        "binned_fraction_p_lt_0_01_fixed":
            float(
                np.sum(
                    bin_finite
                    & (
                        p_bin
                        < RAW_P_THRESHOLD
                    )
                )
                / n_genes
            ),

        "global_BH05_discoveries":
            int(
                np.sum(
                    np.isfinite(
                        q_global
                    )
                    & (
                        q_global
                        < FDR_CUTOFF
                    )
                )
            ),

        "binned_BH05_discoveries":
            int(
                np.sum(
                    np.isfinite(
                        q_bin
                    )
                    & (
                        q_bin
                        < FDR_CUTOFF
                    )
                )
            ),

        "global_any_BH05":
            int(
                np.any(
                    np.isfinite(
                        q_global
                    )
                    & (
                        q_global
                        < FDR_CUTOFF
                    )
                )
            ),

        "binned_any_BH05":
            int(
                np.any(
                    np.isfinite(
                        q_bin
                    )
                    & (
                        q_bin
                        < FDR_CUTOFF
                    )
                )
            ),

        "global_median_p":
            float(
                np.nanmedian(
                    p_global
                )
            ),

        "binned_median_p":
            float(
                np.nanmedian(
                    p_bin
                )
            ),

        "n_mandatory_exact_reused":
            int(
                np.sum(
                    mask_Rnonpos
                )
            ),

        "elapsed_seconds":
            float(
                time.perf_counter()
                - t0
            ),
    }

    payload = {
        "version":
            "pickrell-12bin-deb-v1",

        "summary":
            summary,

        "bin_detail":
            bin_df,

        "hyper_detail":
            hyper_df,

        "p_binned":
            p_bin,

        "q_binned":
            q_bin,
    }

    _atomic_pickle(
        payload,
        cache_path,
    )

    print(
        f"[rep end] {_now()} | "
        f"rep={replicate:02d} | "
        f"p<.01 "
        f"{summary['global_fraction_p_lt_0_01_finite']:.6f} "
        f"-> "
        f"{summary['binned_fraction_p_lt_0_01_finite']:.6f} | "
        f"BH "
        f"{summary['global_BH05_discoveries']} "
        f"-> "
        f"{summary['binned_BH05_discoveries']} | "
        f"{summary['elapsed_seconds']:.1f}s",
        flush=True,
    )

    return (
        payload,
        False,
    )


# =============================================================================
# AGGREGATION
# =============================================================================

def _aggregate_bin_detail(
    bin_detail,
):
    metrics = [
        "median_log1p_total_normalized_coverage",

        "raw_theta_F_median",
        "global_theta_F_median",
        "binned_theta_F_median",

        "raw_theta_0_median",
        "global_theta_0_median",
        "binned_theta_0_median",

        "global_raw_p_lt_0_01_fraction",
        "binned_raw_p_lt_0_01_fraction",

        "full_alpha0",
        "full_beta0",
        "full_prior_mean_theta",

        "null_alpha0",
        "null_beta0",
        "null_prior_mean_theta",
    ]

    rows = []

    for b, d in bin_detail.groupby(
        "coverage_bin",
        sort=True,
    ):
        row = {
            "coverage_bin":
                int(
                    b
                ),

            "n_splits":
                int(
                    d[
                        "replicate"
                    ]
                    .nunique()
                ),
        }

        for col in metrics:
            row[
                f"{col}__median"
            ] = _med(
                d[
                    col
                ]
            )

            row[
                f"{col}__q25"
            ] = _q(
                d[
                    col
                ],
                0.25,
            )

            row[
                f"{col}__q75"
            ] = _q(
                d[
                    col
                ],
                0.75,
            )

        rows.append(
            row
        )

    return (
        pd.DataFrame(
            rows
        )
        .sort_values(
            "coverage_bin",
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )


def _overall_summary(
    split_summary,
):
    return pd.DataFrame(
        [
            {
                "n_splits":
                    int(
                        len(
                            split_summary
                        )
                    ),

                "global_mean_p_lt_0_01":
                    float(
                        split_summary[
                            "global_fraction_p_lt_0_01_finite"
                        ]
                        .mean()
                    ),

                "binned_mean_p_lt_0_01":
                    float(
                        split_summary[
                            "binned_fraction_p_lt_0_01_finite"
                        ]
                        .mean()
                    ),

                "global_median_p_lt_0_01":
                    float(
                        split_summary[
                            "global_fraction_p_lt_0_01_finite"
                        ]
                        .median()
                    ),

                "binned_median_p_lt_0_01":
                    float(
                        split_summary[
                            "binned_fraction_p_lt_0_01_finite"
                        ]
                        .median()
                    ),

                "global_mean_BH05_discoveries":
                    float(
                        split_summary[
                            "global_BH05_discoveries"
                        ]
                        .mean()
                    ),

                "binned_mean_BH05_discoveries":
                    float(
                        split_summary[
                            "binned_BH05_discoveries"
                        ]
                        .mean()
                    ),

                "global_complete_null_BH_FDR":
                    float(
                        split_summary[
                            "global_any_BH05"
                        ]
                        .mean()
                    ),

                "binned_complete_null_BH_FDR":
                    float(
                        split_summary[
                            "binned_any_BH05"
                        ]
                        .mean()
                    ),

                "n_splits_raw_p01_improved":
                    int(
                        np.sum(
                            split_summary[
                                "binned_fraction_p_lt_0_01_finite"
                            ]
                            < split_summary[
                                "global_fraction_p_lt_0_01_finite"
                            ]
                        )
                    ),

                "n_splits_raw_p01_worsened":
                    int(
                        np.sum(
                            split_summary[
                                "binned_fraction_p_lt_0_01_finite"
                            ]
                            > split_summary[
                                "global_fraction_p_lt_0_01_finite"
                            ]
                        )
                    ),
            }
        ]
    )


# =============================================================================
# PLOTS
# =============================================================================

def _plot_band(
    ax,
    x,
    agg,
    metric,
    label,
    *,
    linestyle="-",
    marker="o",
):
    ax.plot(
        x,
        agg[
            f"{metric}__median"
        ],
        linewidth=2.3,
        linestyle=linestyle,
        marker=marker,
        label=label,
    )

    ax.fill_between(
        x,
        agg[
            f"{metric}__q25"
        ],
        agg[
            f"{metric}__q75"
        ],
        alpha=0.14,
    )


def _finish(
    ax,
    title,
    ylabel,
    *,
    hline=None,
    hline_label=None,
    log_y=False,
):
    if log_y:
        ax.set_yscale(
            "log"
        )

    if hline is not None:
        ax.axhline(
            hline,
            linestyle=":",
            linewidth=1.5,
            label=hline_label,
        )

    ax.set_xlabel(
        "log1p(total normalized gene coverage)",
        fontsize=14,
    )

    ax.set_ylabel(
        ylabel,
        fontsize=14,
    )

    ax.set_title(
        title,
        fontsize=18,
        pad=16,
    )

    ax.grid(
        alpha=0.20,
        which="both",
    )

    ax.legend(
        fontsize=11,
    )

    plt.tight_layout()
    plt.show()


def _show_plots(
    split_summary,
    agg,
):
    x = (
        agg[
            "median_log1p_total_normalized_coverage__median"
        ]
        .to_numpy(
            dtype=float
        )
    )

    # -------------------------------------------------------------
    # 1. Raw p<.01 by coverage.
    # -------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    _plot_band(
        ax,
        x,
        agg,
        "global_raw_p_lt_0_01_fraction",
        "Global DEB prior",
    )

    _plot_band(
        ax,
        x,
        agg,
        "binned_raw_p_lt_0_01_fraction",
        "12-bin DEB priors",
        linestyle="--",
        marker="s",
    )

    _finish(
        ax,
        "Pickrell: Raw-p calibration by coverage — global vs 12-bin DEB",
        "Fraction with raw p < .01",
        hline=RAW_P_THRESHOLD,
        hline_label="Nominal 1%",
    )

    # -------------------------------------------------------------
    # 2. Full theta: raw vs global DEB vs binned DEB.
    # -------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    _plot_band(
        ax,
        x,
        agg,
        "raw_theta_F_median",
        r"Raw $\widetilde\theta_F=a_F/R_F$",
    )

    _plot_band(
        ax,
        x,
        agg,
        "global_theta_F_median",
        r"Global-DEB $\widehat\theta_F$",
        linestyle="--",
        marker="s",
    )

    _plot_band(
        ax,
        x,
        agg,
        "binned_theta_F_median",
        r"12-bin-DEB $\widehat\theta_F$",
        linestyle="-.",
        marker="^",
    )

    _finish(
        ax,
        "Pickrell: Full precision by coverage",
        r"$\theta_F$",
        log_y=True,
    )

    # -------------------------------------------------------------
    # 3. Null theta: raw vs global DEB vs binned DEB.
    # -------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    _plot_band(
        ax,
        x,
        agg,
        "raw_theta_0_median",
        r"Raw $\widetilde\theta_0=a_0/R_0^\dagger$",
    )

    _plot_band(
        ax,
        x,
        agg,
        "global_theta_0_median",
        r"Global-DEB $\widehat\theta_0$",
        linestyle="--",
        marker="s",
    )

    _plot_band(
        ax,
        x,
        agg,
        "binned_theta_0_median",
        r"12-bin-DEB $\widehat\theta_0$",
        linestyle="-.",
        marker="^",
    )

    _finish(
        ax,
        "Pickrell: Null precision by coverage",
        r"$\theta_0$",
        log_y=True,
    )

    # -------------------------------------------------------------
    # 4. Full DEB prior mean theta by coverage.
    # -------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    _plot_band(
        ax,
        x,
        agg,
        "full_prior_mean_theta",
        r"Full-bin prior mean $\bar\theta_F$",
    )

    _finish(
        ax,
        "Pickrell: Full-model DEB prior mean by coverage bin",
        r"Prior mean $\theta_F$",
        log_y=True,
    )

    # -------------------------------------------------------------
    # 5. Null DEB prior mean theta by coverage.
    # -------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    _plot_band(
        ax,
        x,
        agg,
        "null_prior_mean_theta",
        r"Null-bin prior mean $\bar\theta_0$",
    )

    _finish(
        ax,
        "Pickrell: Null-model DEB prior mean by coverage bin",
        r"Prior mean $\theta_0$",
        log_y=True,
    )

    # -------------------------------------------------------------
    # 6. DEB alpha0 by coverage.
    # -------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    _plot_band(
        ax,
        x,
        agg,
        "full_alpha0",
        r"Full DEB $\alpha_0$",
    )

    _plot_band(
        ax,
        x,
        agg,
        "null_alpha0",
        r"Null DEB $\alpha_0$",
        linestyle="--",
        marker="s",
    )

    _finish(
        ax,
        "Pickrell: Coverage-bin DEB prior concentration",
        r"Gamma prior shape $\alpha_0$",
        log_y=True,
    )

    # -------------------------------------------------------------
    # 7. Paired split-level raw-p calibration.
    # -------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    xg = (
        split_summary[
            "global_fraction_p_lt_0_01_finite"
        ]
        .to_numpy(
            dtype=float
        )
    )

    yb = (
        split_summary[
            "binned_fraction_p_lt_0_01_finite"
        ]
        .to_numpy(
            dtype=float
        )
    )

    ax.scatter(
        xg,
        yb,
        s=65,
        alpha=0.85,
    )

    vmax = float(
        np.nanmax(
            np.concatenate(
                [
                    xg,
                    yb,
                ]
            )
        )
    )

    ax.plot(
        [
            0,
            vmax
            * 1.05,
        ],
        [
            0,
            vmax
            * 1.05,
        ],
        linestyle="--",
        linewidth=1.5,
        label="No change",
    )

    ax.set_xlabel(
        "Global-DEB fraction p < .01",
        fontsize=14,
    )

    ax.set_ylabel(
        "12-bin-DEB fraction p < .01",
        fontsize=14,
    )

    ax.set_title(
        "Pickrell: Split-level calibration — global vs 12-bin DEB",
        fontsize=18,
        pad=16,
    )

    ax.grid(
        alpha=0.20,
    )

    ax.legend(
        fontsize=11,
    )

    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------------
    # 8. BH discovery boxplot.
    # -------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    ax.boxplot(
        [
            split_summary[
                "global_BH05_discoveries"
            ],
            split_summary[
                "binned_BH05_discoveries"
            ],
        ],
        tick_labels=[
            "Global DEB",
            "12-bin DEB",
        ],
        showfliers=True,
    )

    ax.set_ylabel(
        "BH(.05) discoveries",
        fontsize=14,
    )

    ax.set_title(
        "Pickrell complete null: BH(.05) discoveries",
        fontsize=18,
        pad=16,
    )

    ax.grid(
        axis="y",
        alpha=0.20,
    )

    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------------
    # 9. Complete-null BH FDR.
    # -------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    values = [
        float(
            split_summary[
                "global_any_BH05"
            ]
            .mean()
        ),
        float(
            split_summary[
                "binned_any_BH05"
            ]
            .mean()
        ),
    ]

    ax.bar(
        [
            0,
            1,
        ],
        values,
    )

    ax.axhline(
        FDR_CUTOFF,
        linestyle="--",
        linewidth=1.5,
        label="Nominal 5%",
    )

    ax.set_xticks(
        [
            0,
            1,
        ]
    )

    ax.set_xticklabels(
        [
            "Global DEB",
            "12-bin DEB",
        ]
    )

    ax.set_ylabel(
        "P(any BH(.05) rejection)",
        fontsize=14,
    )

    ax.set_title(
        "Pickrell complete null: BH FDR = P(R > 0)",
        fontsize=18,
        pad=16,
    )

    ax.grid(
        axis="y",
        alpha=0.20,
    )

    ax.legend(
        fontsize=11,
    )

    plt.tight_layout()
    plt.show()


# =============================================================================
# TOP-LEVEL
# =============================================================================

def run_pickrell_12bin_deb_counterfactual():
    started = (
        datetime.now()
        .astimezone()
    )

    print(
        f"[start] {_now()}",
        flush=True,
    )

    try:
        reps = (
            _discover_completed_replicates()
        )

        if len(
            reps
        ) != 30:
            print(
                f"[warning] expected 30 completed splits; found {len(reps)}.",
                flush=True,
            )

        OUTPUT_DIR.mkdir(
            parents=True,
            exist_ok=True,
        )

        SPLIT_CACHE_DIR.mkdir(
            parents=True,
            exist_ok=True,
        )

        if not SENTINEL.exists():
            SENTINEL.write_text(
                json.dumps(
                    {
                        "initialized_local_time":
                            _now(),

                        "source_namespace":
                            CURRENT_M03_NAMESPACE,

                        "counterfactual_namespace":
                            COUNTERFACTUAL_NAMESPACE,

                        "coverage_bins":
                            int(
                                N_COVERAGE_BINS
                            ),

                        "global_geometry_frozen":
                            True,

                        "mandatory_exact_reused":
                            True,

                        "resume_enabled":
                            bool(
                                RESUME
                            ),
                    },
                    indent=2,
                    sort_keys=True,
                ),
                encoding="utf-8",
            )

        print(
            "\nPICKRELL 12-BIN DEB COUNTERFACTUAL\n"
            f"source:          {GLOBAL_RUN_DIR}\n"
            f"output:          {OUTPUT_DIR}\n"
            f"completed splits:{len(reps)}\n"
            f"coverage bins:   {N_COVERAGE_BINS}\n"
            "global kappa_c:  FROZEN\n"
            "global S0dag:    FROZEN\n"
            "changed piece:   DEB hyperparameters only\n",
            flush=True,
        )

        payloads = []
        failures = []
        reused = 0

        for j, rep in enumerate(
            reps,
            start=1,
        ):
            try:
                payload, cached = (
                    _run_one_replicate(
                        rep
                    )
                )

                payloads.append(
                    payload
                )

                reused += int(
                    cached
                )

            except Exception as exc:
                failures.append(
                    {
                        "replicate":
                            int(
                                rep
                            ),

                        "error_type":
                            type(
                                exc
                            ).__name__,

                        "error_message":
                            str(
                                exc
                            ),

                        "traceback":
                            traceback.format_exc(),
                    }
                )

                print(
                    f"[FAILED] {_now()} | "
                    f"rep={rep:02d} | "
                    f"{type(exc).__name__}: {exc}",
                    flush=True,
                )

            print(
                f"[progress] "
                f"{j:02d}/{len(reps):02d}",
                flush=True,
            )

        if not payloads:
            raise RuntimeError(
                "No split counterfactuals completed successfully."
            )

        split_summary = pd.DataFrame(
            [
                p[
                    "summary"
                ]
                for p in payloads
            ]
        )

        bin_detail = pd.concat(
            [
                p[
                    "bin_detail"
                ]
                for p in payloads
            ],
            ignore_index=True,
        )

        hyper_detail = pd.concat(
            [
                p[
                    "hyper_detail"
                ]
                for p in payloads
            ],
            ignore_index=True,
        )

        failures_df = pd.DataFrame(
            failures
        )

        split_summary = (
            split_summary
            .sort_values(
                "replicate",
                kind="stable",
            )
            .reset_index(
                drop=True
            )
        )

        bin_detail = (
            bin_detail
            .sort_values(
                [
                    "replicate",
                    "coverage_bin",
                ],
                kind="stable",
            )
            .reset_index(
                drop=True
            )
        )

        aggregate = (
            _aggregate_bin_detail(
                bin_detail
            )
        )

        overall = (
            _overall_summary(
                split_summary
            )
        )

        # Numerical outputs/checkpoints belong under repo ./data.
        _atomic_csv(
            split_summary,
            OUTPUT_DIR
            / "split_summary.csv",
        )

        _atomic_csv(
            bin_detail,
            OUTPUT_DIR
            / "coverage_bin_detail.csv",
        )

        _atomic_csv(
            hyper_detail,
            OUTPUT_DIR
            / "coverage_bin_hyperparameters.csv",
        )

        _atomic_csv(
            aggregate,
            OUTPUT_DIR
            / "coverage_bin_aggregate.csv",
        )

        _atomic_csv(
            overall,
            OUTPUT_DIR
            / "overall_summary.csv",
        )

        _atomic_csv(
            failures_df,
            OUTPUT_DIR
            / "failures.csv",
        )

        print(
            "\nOVERALL GLOBAL-DEB VS 12-BIN-DEB SUMMARY\n",
            flush=True,
        )

        display(
            overall
        )

        print(
            "\nCOVERAGE-BIN MEDIANS ACROSS SPLITS\n",
            flush=True,
        )

        display(
            aggregate[
                [
                    "coverage_bin",
                    "n_splits",
                    "median_log1p_total_normalized_coverage__median",

                    "raw_theta_F_median__median",
                    "global_theta_F_median__median",
                    "binned_theta_F_median__median",

                    "raw_theta_0_median__median",
                    "global_theta_0_median__median",
                    "binned_theta_0_median__median",

                    "full_prior_mean_theta__median",
                    "null_prior_mean_theta__median",

                    "full_alpha0__median",
                    "null_alpha0__median",

                    "global_raw_p_lt_0_01_fraction__median",
                    "binned_raw_p_lt_0_01_fraction__median",
                ]
            ]
        )

        if SHOW_PLOTS:
            _show_plots(
                split_summary,
                aggregate,
            )

        ended = (
            datetime.now()
            .astimezone()
        )

        print(
            "\nSTATUS\n"
            f"successful splits: {len(split_summary)}/{len(reps)}\n"
            f"reused caches:     {reused}\n"
            f"failures:          {len(failures_df)}\n"
            f"output:            {OUTPUT_DIR}\n"
            f"[end] {_now()} | elapsed={ended-started}",
            flush=True,
        )

        return {
            "split_summary":
                split_summary,

            "bin_detail":
                bin_detail,

            "hyper_detail":
                hyper_detail,

            "aggregate":
                aggregate,

            "overall":
                overall,

            "failures":
                failures_df,
        }

    finally:
        # If an exception exits before normal status printing, still print end.
        pass


# =============================================================================
# EXECUTION
# =============================================================================

pickrell_12bin_deb_counterfactual = (
    run_pickrell_12bin_deb_counterfactual()
)


In [ ]:
from __future__ import annotations

"""
Pickrell weighted-reference-CDF contextual DB-geometry diagnostic.

Sean's proposed idea
--------------------
The effective-binomial bridge for NB uses

    p = mu / (mu + r)
    m_eff = y + r

and the ordinary NB half-deviance is exactly the corresponding effective-
binomial half-deviance.

This diagnostic keeps the fitted means and r_g fixed, but changes the
*variation context* of p.

For a weighted reference CDF F_w(p),

    u_epsilon(p)
        = epsilon + (1 - 2*epsilon) * F_w(p)

so the reference-context percentiles are spread over

    [epsilon, 1-epsilon].

The weighted reference CDF uses a gene-level fitted effective proportion

    p_ref,g
        = mean_i(mu_F,gi) / [mean_i(mu_F,gi) + r_g]

with information weights

    W_g(gamma)
        = I_g ** gamma

    I_g
        = sum_i mu_F,gi * r_g / (mu_F,gi + r_g).

Thus:
    gamma = 0   -> ordinary unweighted reference CDF
    gamma = 1   -> NB-information-weighted reference CDF
    larger gamma -> more emphasis on high-information genes.

The CDF is CROSS-FITTED by Pickrell split:
the held-out split's own reference histogram is removed before transforming
that split.

Contextual effective-binomial deviance
--------------------------------------
For each observation,

    p_obs = y / (y + r)
    p_fit = mu / (mu + r)

map BOTH observed and fitted proportions through the same context:

    u_obs = T(p_obs)
    u_fit = T(p_fit)

and calculate

    s_ctx
      = m_eff * KL_Bernoulli(u_obs || u_fit).

This preserves s_ctx=0 when observed=fitted and is the direct warped-coordinate
analogue of the existing effective-binomial/NB deviance.

IMPORTANT
---------
This is a GEOMETRY DIAGNOSTIC, not yet a replacement M03 likelihood.

The current DNB normalizer C(theta) is tied to the original NB deviance.
Once the warped geometry looks promising, the next step is to derive/audit
the corresponding contextual normalizer before using it as a production test.

Therefore parameter selection below uses ONLY residual-geometry flatness.
It NEVER uses p-values or BH results to choose epsilon/gamma.

Goal
----
Find a universal weighted context that makes

    S_F_ctx / a_F
and
    S_0_ctx / a_0

as coverage-invariant as possible while preserving nested geometry reasonably
well.

Outputs
-------
DISPLAY ONLY. Nothing is saved.

Plots:
    1. epsilon x weight-power geometry objective
    2. current vs contextual S_F/a_F
    3. current vs contextual S_0/a_0
    4. contextual nested-difference diagnostics
    5. weighted-CDF p -> u mapping
    6. original vs contextual curvature p(1-p) and u(1-u)
    7. residual-geometry curve overlaid with the EXISTING raw-p<.01 hump
       (p-values are shown only after the geometry-optimal context is selected)
"""

from datetime import datetime
from pathlib import Path
from scipy.special import expit, logit, xlogy

import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


# =============================================================================
# SETTINGS
# =============================================================================

CURRENT_M03_NAMESPACE = "pickrell_m03_seam_safe_20M_kappa_coverage_v1"

N_COVERAGE_BINS = 12

# Universal-context grid.
EPSILON_GRID = (
    1e-4,
    0.001,
    0.005,
    0.01,
    0.025,
    0.05,
    0.10,
    0.15,
    0.20,
)

# gamma controls weighting of the universal reference CDF by NB information.
WEIGHT_POWER_GRID = (
    0.0,
    0.5,
    1.0,
    1.5,
    2.0,
)

# Weighted-CDF numerical grid on logit(p).
CDF_LOGIT_MIN = -25.0
CDF_LOGIT_MAX = 25.0
CDF_N_BINS = 4096

# Grid search uses a fixed deterministic subset for speed.
# The winning context is then reevaluated on ALL genes.
GRID_SEARCH_MAX_GENES = 20_000
GRID_SEARCH_SEED = 20261003

# Nested-geometry penalty in the context-selection criterion.
NESTED_NEGATIVE_PENALTY = 5.0

# Existing raw-p result is shown only after selection; it never enters objective.
RAW_P_THRESHOLD = 0.01

FIGSIZE = (16, 9)


# =============================================================================
# PATHS / TIME
# =============================================================================

def _now() -> str:
    return datetime.now().astimezone().strftime(
        "%Y-%m-%d %I:%M:%S %p %Z"
    )


CWD = Path.cwd().resolve()

if CWD.name.lower() in {"notebook", "notebooks"}:
    PROJECT_ROOT = CWD.parent
elif (CWD / "data").is_dir():
    PROJECT_ROOT = CWD
else:
    raise RuntimeError(
        "Could not resolve repository root. Run from <repo>/notebook, "
        "<repo>/notebooks, or the repository root containing ./data. "
        f"Current working directory: {CWD}"
    )

RUN_DIR = (
    PROJECT_ROOT
    / "data"
    / CURRENT_M03_NAMESPACE
)

CELL_DIR = RUN_DIR / "cells"
CHECKPOINT_ROOT = RUN_DIR / "m03_checkpoints"


# =============================================================================
# CHECKPOINT HELPERS
# =============================================================================

def _load_pickle(path: Path):
    with open(path, "rb") as f:
        return pickle.load(f)


def _checkpoint_dir(replicate: int) -> Path:
    return (
        CHECKPOINT_ROOT
        / f"pickrell_rep{int(replicate):02d}"
    )


def _load_success_stage(
    replicate: int,
    stage_name: str,
):
    path = (
        _checkpoint_dir(replicate)
        / "stages"
        / f"{stage_name}.pkl"
    )

    if not path.exists():
        raise FileNotFoundError(
            f"Missing checkpoint: {path}"
        )

    payload = _load_pickle(path)

    if (
        not isinstance(payload, dict)
        or payload.get("status") != "success"
    ):
        raise RuntimeError(
            f"Replicate {replicate}: "
            f"{stage_name} is not a successful checkpoint."
        )

    return payload["data"]


def _discover_completed_replicates():
    reps = []

    for path in sorted(
        CELL_DIR.glob("pickrell_rep*.csv")
    ):
        try:
            x = pd.read_csv(path)
        except Exception:
            continue

        if len(x) != 1:
            continue

        if str(
            x.loc[0, "status"]
        ).strip().lower() != "success":
            continue

        reps.append(
            int(
                x.loc[0, "replicate"]
            )
        )

    reps = sorted(set(reps))

    if not reps:
        raise RuntimeError(
            "No completed Pickrell replicates were found."
        )

    return reps


# =============================================================================
# BASIC MATH
# =============================================================================

def _half_nb_deviance(
    y,
    mu,
    r,
):
    y = np.asarray(y, dtype=float)
    mu = np.maximum(
        np.asarray(mu, dtype=float),
        1e-300,
    )
    r = np.maximum(
        np.asarray(r, dtype=float),
        1e-300,
    )

    return np.maximum(
        xlogy(
            y,
            y / mu,
        )
        - (
            y + r
        )
        * np.log(
            (
                y + r
            )
            / (
                mu + r
            )
        ),
        0.0,
    )


def _bernoulli_kl(
    q,
    p,
):
    """
    Boundary-safe Bernoulli KL.

    IMPORTANT:
    Do NOT clip q away from 0 or 1.  In the effective-binomial identity,
    q = y/(y+r) can be exactly 0 when y=0.  Artificially replacing q=0
    by 1e-15 creates a tiny per-observation error that can become several
    micro-units after multiplication by a very large m_eff=y+r.

    xlogy evaluates 0*log(0) correctly as 0, so the exact boundary values
    can be retained.
    """

    q = np.asarray(
        q,
        dtype=float,
    )

    p = np.asarray(
        p,
        dtype=float,
    )

    # q is an observed/fitted Bernoulli coordinate and should already be in
    # [0,1].  Retain exact endpoints.  Only protect p against literal floating
    # underflow/rounding to 0 or 1 before taking logs.
    q = np.clip(
        q,
        0.0,
        1.0,
    )

    tiny = np.finfo(float).tiny
    one_minus = 1.0 - np.finfo(float).eps

    p = np.clip(
        p,
        tiny,
        one_minus,
    )

    return (
        xlogy(
            q,
            q,
        )
        - xlogy(
            q,
            p,
        )
        + xlogy(
            1.0 - q,
            1.0 - q,
        )
        - xlogy(
            1.0 - q,
            1.0 - p,
        )
    )


def _contextual_half_deviance(
    y,
    r,
    F_obs,
    F_fit,
    epsilon,
):
    """
    Warped-coordinate effective-binomial half-deviance.

        u = eps + (1-2eps) F_w(p)
        m_eff = y + r
        s = m_eff * KL_Bernoulli(u_obs || u_fit)
    """

    y = np.asarray(
        y,
        dtype=float,
    )

    r = np.asarray(
        r,
        dtype=float,
    )

    u_obs = (
        float(epsilon)
        + (
            1.0
            - 2.0
            * float(epsilon)
        )
        * F_obs
    )

    u_fit = (
        float(epsilon)
        + (
            1.0
            - 2.0
            * float(epsilon)
        )
        * F_fit
    )

    return (
        y + r
    ) * _bernoulli_kl(
        u_obs,
        u_fit,
    )


def _finite_log_sd(values):
    """
    Scale-free flatness measure across coverage bins.
    Smaller = flatter.
    """

    x = np.asarray(
        values,
        dtype=float,
    )

    x = x[
        np.isfinite(x)
        & (
            x > 0
        )
    ]

    if len(x) < 3:
        return np.inf

    return float(
        np.std(
            np.log(x),
            ddof=0,
        )
    )


# =============================================================================
# WEIGHTED REFERENCE CDF
# =============================================================================

CDF_EDGES = np.linspace(
    CDF_LOGIT_MIN,
    CDF_LOGIT_MAX,
    CDF_N_BINS + 1,
)

CDF_CENTERS = (
    CDF_EDGES[:-1]
    + CDF_EDGES[1:]
) / 2.0


def _to_logit(
    p,
):
    p = np.asarray(
        p,
        dtype=float,
    )

    p = np.clip(
        p,
        expit(
            CDF_LOGIT_MIN
        ),
        expit(
            CDF_LOGIT_MAX
        ),
    )

    return np.clip(
        logit(
            p
        ),
        CDF_LOGIT_MIN,
        CDF_LOGIT_MAX,
    )


def _weighted_histogram(
    p,
    weights,
):
    z = _to_logit(
        p
    )

    hist, _ = np.histogram(
        z,
        bins=CDF_EDGES,
        weights=np.asarray(
            weights,
            dtype=float,
        ),
    )

    return hist.astype(
        float
    )


def _cdf_from_hist(
    z,
    hist,
):
    hist = np.asarray(
        hist,
        dtype=float,
    )

    total = float(
        np.sum(hist)
    )

    if (
        not np.isfinite(total)
        or total <= 0
    ):
        raise RuntimeError(
            "Weighted reference histogram has no positive mass."
        )

    # Weighted mid-CDF in each logit bin.
    cumulative_before = (
        np.cumsum(hist)
        - hist
    )

    F_mid = (
        cumulative_before
        + 0.5 * hist
    ) / total

    return np.interp(
        np.asarray(
            z,
            dtype=float,
        ),
        CDF_CENTERS,
        F_mid,
        left=float(
            F_mid[0]
        ),
        right=float(
            F_mid[-1]
        ),
    )


# =============================================================================
# LOAD CURRENT PICKRELL SPLITS
# =============================================================================

def _load_rep_data(
    replicate: int,
):
    core = _load_success_stage(
        replicate,
        "01_core",
    )

    base = _load_success_stage(
        replicate,
        "05_base_m03_result",
    )

    y = np.asarray(
        core["y"],
        dtype=float,
    )

    exposure = np.asarray(
        core["exposure"],
        dtype=float,
    )

    mu_F = np.asarray(
        core["mu_F"],
        dtype=float,
    )

    mu_0 = np.asarray(
        core["mu_0"],
        dtype=float,
    )

    r = np.asarray(
        core["r_hat"],
        dtype=float,
    )

    S_F = np.asarray(
        core["S_F"],
        dtype=float,
    )

    S_0 = np.asarray(
        core["S_0"],
        dtype=float,
    )

    a_F = float(
        core["a_F"]
    )

    a_0 = float(
        core["a_0"]
    )

    p_value = np.asarray(
        base["p_value_m03"],
        dtype=float,
    )

    n_genes = int(
        y.shape[0]
    )

    for name, arr in {
        "mu_F": mu_F,
        "mu_0": mu_0,
    }.items():
        if arr.shape != y.shape:
            raise RuntimeError(
                f"Replicate {replicate}: "
                f"{name} shape={arr.shape} != y shape={y.shape}"
            )

    for name, arr in {
        "r": r,
        "S_F": S_F,
        "S_0": S_0,
        "p_value": p_value,
    }.items():
        if len(arr) != n_genes:
            raise RuntimeError(
                f"Replicate {replicate}: "
                f"{name} length={len(arr)} != n_genes={n_genes}"
            )

    total_normalized_coverage = np.sum(
        y
        / exposure[
            None,
            :
        ],
        axis=1,
    )

    log_coverage = np.log1p(
        total_normalized_coverage
    )

    coverage_bin = (
        np.asarray(
            pd.qcut(
                pd.Series(
                    log_coverage
                ),
                q=N_COVERAGE_BINS,
                labels=False,
                duplicates="drop",
            ),
            dtype=int,
        )
        + 1
    )

    # -------------------------------------------------------------
    # Gene-level universal-reference coordinate.
    # -------------------------------------------------------------
    mu_bar = np.mean(
        mu_F,
        axis=1,
    )

    p_ref_gene = (
        mu_bar
        / (
            mu_bar
            + r
        )
    )

    # NB Fisher-information-like coordinate already used in this project:
    #
    #     I_gi = mu*r/(mu+r) = mu/(1+mu/r)
    #
    # Sum over samples to define a gene's reference-CDF weight.
    info_obs = (
        mu_F
        * r[
            :,
            None,
        ]
        / (
            mu_F
            + r[
                :,
                None,
            ]
        )
    )

    info_gene = np.sum(
        info_obs,
        axis=1,
    )

    return {
        "replicate":
            int(
                replicate
            ),

        "y":
            y,

        "mu_F":
            mu_F,

        "mu_0":
            mu_0,

        "r":
            r,

        "S_F":
            S_F,

        "S_0":
            S_0,

        "a_F":
            a_F,

        "a_0":
            a_0,

        "p_value":
            p_value,

        "log_coverage":
            log_coverage,

        "coverage_bin":
            coverage_bin,

        "p_ref_gene":
            p_ref_gene,

        "info_gene":
            info_gene,
    }


# =============================================================================
# IDENTITY AUDIT
# =============================================================================

def _audit_effective_binomial_identity(
    rep,
):
    y = rep[
        "y"
    ]

    mu_F = rep[
        "mu_F"
    ]

    r = rep[
        "r"
    ][
        :,
        None
    ]

    p_obs = (
        y
        / (
            y + r
        )
    )

    p_fit = (
        mu_F
        / (
            mu_F
            + r
        )
    )

    m_eff = (
        y + r
    )

    binom_half = (
        m_eff
        * _bernoulli_kl(
            p_obs,
            p_fit,
        )
    )

    nb_half = _half_nb_deviance(
        y,
        mu_F,
        r,
    )

    err = np.abs(
        binom_half
        - nb_half
    )

    finite = np.isfinite(
        err
    )

    if not np.any(
        finite
    ):
        raise RuntimeError(
            "Effective-binomial identity audit produced no finite comparisons."
        )

    return {
        "max_abs_error":
            float(
                np.max(
                    err[
                        finite
                    ]
                )
            ),

        "median_abs_error":
            float(
                np.median(
                    err[
                        finite
                    ]
                )
            ),
    }


# =============================================================================
# BUILD CROSS-FIT REFERENCE HISTOGRAMS
# =============================================================================

def _build_reference_histograms(
    rep_data,
):
    """
    Histograms are built from gene-level fitted effective proportions.
    One histogram per (gamma, split), allowing exact leave-one-split-out
    subtraction from the pooled histogram.
    """

    out = {}

    for gamma in WEIGHT_POWER_GRID:
        by_rep = {}

        for rep in rep_data:
            p = np.asarray(
                rep[
                    "p_ref_gene"
                ],
                dtype=float,
            )

            info = np.asarray(
                rep[
                    "info_gene"
                ],
                dtype=float,
            )

            ok = (
                np.isfinite(
                    p
                )
                & np.isfinite(
                    info
                )
                & (
                    info
                    > 0
                )
            )

            if float(
                gamma
            ) == 0.0:
                weights = np.ones(
                    int(
                        np.sum(
                            ok
                        )
                    ),
                    dtype=float,
                )
            else:
                weights = (
                    info[
                        ok
                    ]
                    ** float(
                        gamma
                    )
                )

                # Scale weights numerically; CDF is invariant to this.
                positive = weights[
                    np.isfinite(
                        weights
                    )
                    & (
                        weights
                        > 0
                    )
                ]

                if len(
                    positive
                ):
                    scale = float(
                        np.median(
                            positive
                        )
                    )

                    if (
                        np.isfinite(
                            scale
                        )
                        and scale
                        > 0
                    ):
                        weights = (
                            weights
                            / scale
                        )

            by_rep[
                rep[
                    "replicate"
                ]
            ] = _weighted_histogram(
                p[
                    ok
                ],
                weights,
            )

        total = np.sum(
            np.stack(
                list(
                    by_rep.values()
                ),
                axis=0,
            ),
            axis=0,
        )

        out[
            float(
                gamma
            )
        ] = {
            "by_rep":
                by_rep,

            "total":
                total,
        }

    return out


# =============================================================================
# CONTEXTUAL DEVIANCE FOR A HELD-OUT SPLIT
# =============================================================================

def _prepare_crossfit_cdf_values(
    rep,
    hist,
    gene_indices,
):
    """
    For a held-out split and a fixed reference CDF, calculate F_w(p) for
    p_obs, p_F, and p_0 ONCE. epsilon only rescales these CDF values.
    """

    idx = np.asarray(
        gene_indices,
        dtype=int,
    )

    y = rep[
        "y"
    ][
        idx,
        :
    ]

    mu_F = rep[
        "mu_F"
    ][
        idx,
        :
    ]

    mu_0 = rep[
        "mu_0"
    ][
        idx,
        :
    ]

    r = rep[
        "r"
    ][
        idx
    ][
        :,
        None
    ]

    p_obs = (
        y
        / (
            y + r
        )
    )

    p_F = (
        mu_F
        / (
            mu_F
            + r
        )
    )

    p_0 = (
        mu_0
        / (
            mu_0
            + r
        )
    )

    F_obs = _cdf_from_hist(
        _to_logit(
            p_obs
        ),
        hist,
    )

    F_F = _cdf_from_hist(
        _to_logit(
            p_F
        ),
        hist,
    )

    F_0 = _cdf_from_hist(
        _to_logit(
            p_0
        ),
        hist,
    )

    return {
        "y":
            y,

        "r":
            r,

        "F_obs":
            F_obs,

        "F_F":
            F_F,

        "F_0":
            F_0,
    }


def _evaluate_context_on_subset(
    rep,
    prepared,
    gene_indices,
    epsilon,
):
    idx = np.asarray(
        gene_indices,
        dtype=int,
    )

    s_F = _contextual_half_deviance(
        prepared[
            "y"
        ],
        prepared[
            "r"
        ],
        prepared[
            "F_obs"
        ],
        prepared[
            "F_F"
        ],
        epsilon,
    )

    s_0 = _contextual_half_deviance(
        prepared[
            "y"
        ],
        prepared[
            "r"
        ],
        prepared[
            "F_obs"
        ],
        prepared[
            "F_0"
        ],
        epsilon,
    )

    S_F_ctx = np.sum(
        s_F,
        axis=1,
    )

    S_0_ctx = np.sum(
        s_0,
        axis=1,
    )

    coverage_bin = rep[
        "coverage_bin"
    ][
        idx
    ]

    full_curve = []
    null_curve = []

    for b in sorted(
        np.unique(
            coverage_bin
        )
    ):
        use = (
            coverage_bin
            == b
        )

        full_curve.append(
            float(
                np.median(
                    S_F_ctx[
                        use
                    ]
                    / rep[
                        "a_F"
                    ]
                )
            )
        )

        null_curve.append(
            float(
                np.median(
                    S_0_ctx[
                        use
                    ]
                    / rep[
                        "a_0"
                    ]
                )
            )
        )

    nested_tolerance = (
        1e-10
        * np.maximum(
            1.0,
            np.abs(
                S_F_ctx
            )
        )
    )

    negative_nested = (
        S_0_ctx
        < S_F_ctx
        - nested_tolerance
    )

    return {
        "S_F_ctx":
            S_F_ctx,

        "S_0_ctx":
            S_0_ctx,

        "full_curve":
            np.asarray(
                full_curve,
                dtype=float,
            ),

        "null_curve":
            np.asarray(
                null_curve,
                dtype=float,
            ),

        "flatness_F":
            _finite_log_sd(
                full_curve
            ),

        "flatness_0":
            _finite_log_sd(
                null_curve
            ),

        "nested_negative_fraction":
            float(
                np.mean(
                    negative_nested
                )
            ),
    }


# =============================================================================
# CURRENT-GEOMETRY BASELINE
# =============================================================================

def _current_geometry_flatness(
    rep,
    gene_indices,
):
    idx = np.asarray(
        gene_indices,
        dtype=int,
    )

    bins = rep[
        "coverage_bin"
    ][
        idx
    ]

    SF = (
        rep[
            "S_F"
        ][
            idx
        ]
        / rep[
            "a_F"
        ]
    )

    S0 = (
        rep[
            "S_0"
        ][
            idx
        ]
        / rep[
            "a_0"
        ]
    )

    fcurve = []
    ncurve = []

    for b in sorted(
        np.unique(
            bins
        )
    ):
        use = (
            bins
            == b
        )

        fcurve.append(
            float(
                np.median(
                    SF[
                        use
                    ]
                )
            )
        )

        ncurve.append(
            float(
                np.median(
                    S0[
                        use
                    ]
                )
            )
        )

    return {
        "flatness_F":
            _finite_log_sd(
                fcurve
            ),

        "flatness_0":
            _finite_log_sd(
                ncurve
            ),

        "full_curve":
            np.asarray(
                fcurve,
                dtype=float,
            ),

        "null_curve":
            np.asarray(
                ncurve,
                dtype=float,
            ),
    }


# =============================================================================
# GRID SEARCH — GEOMETRY ONLY
# =============================================================================

def _make_grid_subset(
    n_genes,
    replicate,
):
    """
    Deterministic split-specific grid-search subset.

    Pickrell M03 keeps a split-specific gene universe after zero-count
    filtering, so different frozen 5v5 splits can have different numbers
    of genes.  A single index vector created from split 1 is therefore
    invalid for the other splits.

    Use all genes when n_genes <= GRID_SEARCH_MAX_GENES; otherwise draw a
    deterministic split-specific subset.
    """

    n_genes = int(
        n_genes
    )

    replicate = int(
        replicate
    )

    if (
        GRID_SEARCH_MAX_GENES is None
        or int(
            GRID_SEARCH_MAX_GENES
        )
        >= n_genes
    ):
        return np.arange(
            n_genes,
            dtype=int,
        )

    rng = np.random.default_rng(
        int(
            GRID_SEARCH_SEED
        )
        + replicate
    )

    return np.sort(
        rng.choice(
            n_genes,
            size=int(
                GRID_SEARCH_MAX_GENES
            ),
            replace=False,
        )
    )


def _grid_search_context(
    rep_data,
    reference_histograms,
):
    # Pickrell's split-specific zero-count filtering means the completed
    # M03 fits do NOT all have the same number of genes.  Build a valid,
    # deterministic subset separately for every frozen split.
    subset_by_rep = {}

    subset_sizes = []
    gene_sizes = []

    for rep in rep_data:
        rep_id = int(
            rep[
                "replicate"
            ]
        )

        n_genes = int(
            rep[
                "y"
            ].shape[
                0
            ]
        )

        subset = _make_grid_subset(
            n_genes,
            rep_id,
        )

        subset_by_rep[
            rep_id
        ] = subset

        subset_sizes.append(
            len(
                subset
            )
        )

        gene_sizes.append(
            n_genes
        )

    print(
        "[grid] split-specific gene universes: "
        f"genes min/median/max="
        f"{min(gene_sizes):,}/"
        f"{int(np.median(gene_sizes)):,}/"
        f"{max(gene_sizes):,}; "
        f"grid subset min/median/max="
        f"{min(subset_sizes):,}/"
        f"{int(np.median(subset_sizes)):,}/"
        f"{max(subset_sizes):,}",
        flush=True,
    )

    # Baseline current raw-deviance flatness.
    baseline_rows = []

    for rep in rep_data:
        rep_id = int(
            rep[
                "replicate"
            ]
        )

        subset = subset_by_rep[
            rep_id
        ]

        cur = _current_geometry_flatness(
            rep,
            subset,
        )

        baseline_rows.append(
            {
                "replicate":
                    rep[
                        "replicate"
                    ],

                "flatness_F":
                    cur[
                        "flatness_F"
                    ],

                "flatness_0":
                    cur[
                        "flatness_0"
                    ],
            }
        )

    baseline = pd.DataFrame(
        baseline_rows
    )

    print(
        "\nCURRENT RAW-DEVIANCE COVERAGE FLATNESS\n",
        flush=True,
    )

    display(
        pd.DataFrame(
            [
                {
                    "median_flatness_F":
                        float(
                            baseline[
                                "flatness_F"
                            ]
                            .median()
                        ),

                    "median_flatness_0":
                        float(
                            baseline[
                                "flatness_0"
                            ]
                            .median()
                        ),

                    "combined":
                        float(
                            (
                                baseline[
                                    "flatness_F"
                                ]
                                + baseline[
                                    "flatness_0"
                                ]
                            )
                            .median()
                        ),
                }
            ]
        )
    )

    rows = []

    for gamma in WEIGHT_POWER_GRID:
        gamma = float(
            gamma
        )

        hist_total = (
            reference_histograms[
                gamma
            ][
                "total"
            ]
        )

        by_rep = (
            reference_histograms[
                gamma
            ][
                "by_rep"
            ]
        )

        print(
            f"[grid] weight power gamma={gamma:g}",
            flush=True,
        )

        # Prepare cross-fit CDF values once per split/gamma.
        prepared_by_rep = {}

        for rep in rep_data:
            rep_id = rep[
                "replicate"
            ]

            hist_crossfit = (
                hist_total
                - by_rep[
                    rep_id
                ]
            )

            subset = subset_by_rep[
                int(
                    rep_id
                )
            ]

            prepared_by_rep[
                rep_id
            ] = _prepare_crossfit_cdf_values(
                rep,
                hist_crossfit,
                subset,
            )

        for epsilon in EPSILON_GRID:
            split_rows = []

            for rep in rep_data:
                rep_id = rep[
                    "replicate"
                ]

                subset = subset_by_rep[
                    int(
                        rep_id
                    )
                ]

                fit = _evaluate_context_on_subset(
                    rep,
                    prepared_by_rep[
                        rep_id
                    ],
                    subset,
                    epsilon,
                )

                objective = (
                    fit[
                        "flatness_F"
                    ]
                    + fit[
                        "flatness_0"
                    ]
                    + float(
                        NESTED_NEGATIVE_PENALTY
                    )
                    * fit[
                        "nested_negative_fraction"
                    ]
                )

                split_rows.append(
                    {
                        "replicate":
                            rep_id,

                        "flatness_F":
                            fit[
                                "flatness_F"
                            ],

                        "flatness_0":
                            fit[
                                "flatness_0"
                            ],

                        "nested_negative_fraction":
                            fit[
                                "nested_negative_fraction"
                            ],

                        "objective":
                            objective,
                    }
                )

            sx = pd.DataFrame(
                split_rows
            )

            rows.append(
                {
                    "weight_power":
                        gamma,

                    "epsilon":
                        float(
                            epsilon
                        ),

                    "median_flatness_F":
                        float(
                            sx[
                                "flatness_F"
                            ]
                            .median()
                        ),

                    "median_flatness_0":
                        float(
                            sx[
                                "flatness_0"
                            ]
                            .median()
                        ),

                    "median_nested_negative_fraction":
                        float(
                            sx[
                                "nested_negative_fraction"
                            ]
                            .median()
                        ),

                    "mean_nested_negative_fraction":
                        float(
                            sx[
                                "nested_negative_fraction"
                            ]
                            .mean()
                        ),

                    "median_objective":
                        float(
                            sx[
                                "objective"
                            ]
                            .median()
                        ),

                    "q75_objective":
                        float(
                            sx[
                                "objective"
                            ]
                            .quantile(
                                0.75
                            )
                        ),
                }
            )

    results = (
        pd.DataFrame(
            rows
        )
        .sort_values(
            [
                "median_objective",
                "q75_objective",
            ],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    print(
        "\nCONTEXT GRID — BEST GEOMETRY CANDIDATES\n",
        flush=True,
    )

    display(
        results.head(
            20
        )
    )

    return (
        results,
        subset_by_rep,
        baseline,
    )


# =============================================================================
# FULL-GENE EVALUATION OF WINNING CONTEXT
# =============================================================================

def _evaluate_best_context_all_genes(
    rep_data,
    reference_histograms,
    best_gamma,
    best_epsilon,
):
    detail_rows = []

    pooled_current_full = []
    pooled_current_null = []
    pooled_context_full = []
    pooled_context_null = []

    gamma_info = (
        reference_histograms[
            float(
                best_gamma
            )
        ]
    )

    for j, rep in enumerate(
        rep_data,
        start=1,
    ):
        rep_id = rep[
            "replicate"
        ]

        hist = (
            gamma_info[
                "total"
            ]
            - gamma_info[
                "by_rep"
            ][
                rep_id
            ]
        )

        idx = np.arange(
            rep[
                "y"
            ].shape[
                0
            ],
            dtype=int,
        )

        prepared = _prepare_crossfit_cdf_values(
            rep,
            hist,
            idx,
        )

        fit = _evaluate_context_on_subset(
            rep,
            prepared,
            idx,
            best_epsilon,
        )

        # ---------------------------------------------------------
        # Since a global multiplicative scale can later be absorbed by theta,
        # scale BOTH full and null contextual deviances by ONE split-level
        # scalar so the overall full-model median matches current S_F/a_F.
        # This makes the shape comparison visually meaningful without altering
        # coverage dependence or the nested full-vs-null relationship.
        # ---------------------------------------------------------
        current_full = (
            rep[
                "S_F"
            ]
            / rep[
                "a_F"
            ]
        )

        current_null = (
            rep[
                "S_0"
            ]
            / rep[
                "a_0"
            ]
        )

        context_full_raw = (
            fit[
                "S_F_ctx"
            ]
            / rep[
                "a_F"
            ]
        )

        context_null_raw = (
            fit[
                "S_0_ctx"
            ]
            / rep[
                "a_0"
            ]
        )

        med_current = float(
            np.nanmedian(
                current_full
            )
        )

        med_context = float(
            np.nanmedian(
                context_full_raw
            )
        )

        scale = (
            med_current
            / med_context
            if (
                np.isfinite(
                    med_current
                )
                and np.isfinite(
                    med_context
                )
                and med_context
                > 0
            )
            else 1.0
        )

        context_full = (
            scale
            * context_full_raw
        )

        context_null = (
            scale
            * context_null_raw
        )

        pooled_current_full.append(
            current_full
        )

        pooled_current_null.append(
            current_null
        )

        pooled_context_full.append(
            context_full
        )

        pooled_context_null.append(
            context_null
        )

        for b in sorted(
            np.unique(
                rep[
                    "coverage_bin"
                ]
            )
        ):
            use = (
                rep[
                    "coverage_bin"
                ]
                == b
            )

            detail_rows.append(
                {
                    "replicate":
                        rep_id,

                    "coverage_bin":
                        int(
                            b
                        ),

                    "median_log1p_total_normalized_coverage":
                        float(
                            np.median(
                                rep[
                                    "log_coverage"
                                ][
                                    use
                                ]
                            )
                        ),

                    "current_SF_over_aF":
                        float(
                            np.median(
                                current_full[
                                    use
                                ]
                            )
                        ),

                    "context_SF_over_aF":
                        float(
                            np.median(
                                context_full[
                                    use
                                ]
                            )
                        ),

                    "current_S0_over_a0":
                        float(
                            np.median(
                                current_null[
                                    use
                                ]
                            )
                        ),

                    "context_S0_over_a0":
                        float(
                            np.median(
                                context_null[
                                    use
                                ]
                            )
                        ),

                    "current_raw_p_lt_0_01_fraction":
                        float(
                            np.mean(
                                np.isfinite(
                                    rep[
                                        "p_value"
                                    ][
                                        use
                                    ]
                                )
                                & (
                                    rep[
                                        "p_value"
                                    ][
                                        use
                                    ]
                                    < RAW_P_THRESHOLD
                                )
                            )
                        ),
                }
            )

        print(
            f"[best-context] {j:02d}/{len(rep_data):02d} "
            f"| rep={rep_id:02d} "
            f"| scale={scale:.5g} "
            f"| nested-neg={fit['nested_negative_fraction']:.4f}",
            flush=True,
        )

    detail = pd.DataFrame(
        detail_rows
    )

    agg_rows = []

    for b, d in detail.groupby(
        "coverage_bin",
        sort=True,
    ):
        row = {
            "coverage_bin":
                int(
                    b
                ),

            "n_splits":
                int(
                    d[
                        "replicate"
                    ]
                    .nunique()
                ),
        }

        for col in (
            "median_log1p_total_normalized_coverage",
            "current_SF_over_aF",
            "context_SF_over_aF",
            "current_S0_over_a0",
            "context_S0_over_a0",
            "current_raw_p_lt_0_01_fraction",
        ):
            row[
                f"{col}__median"
            ] = float(
                d[
                    col
                ]
                .median()
            )

            row[
                f"{col}__q25"
            ] = float(
                d[
                    col
                ]
                .quantile(
                    0.25
                )
            )

            row[
                f"{col}__q75"
            ] = float(
                d[
                    col
                ]
                .quantile(
                    0.75
                )
            )

        agg_rows.append(
            row
        )

    aggregate = pd.DataFrame(
        agg_rows
    )

    return (
        detail,
        aggregate,
    )


# =============================================================================
# PLOTS
# =============================================================================

def _plot_grid_objective(
    grid_results,
):
    pivot = (
        grid_results
        .pivot(
            index="weight_power",
            columns="epsilon",
            values="median_objective",
        )
        .sort_index()
    )

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    image = ax.imshow(
        pivot.to_numpy(
            dtype=float
        ),
        aspect="auto",
        origin="lower",
    )

    ax.set_xticks(
        np.arange(
            len(
                pivot.columns
            )
        )
    )

    ax.set_xticklabels(
        [
            f"{x:g}"
            for x in pivot.columns
        ],
        rotation=45,
        ha="right",
    )

    ax.set_yticks(
        np.arange(
            len(
                pivot.index
            )
        )
    )

    ax.set_yticklabels(
        [
            f"{x:g}"
            for x in pivot.index
        ]
    )

    ax.set_xlabel(
        r"$\epsilon$",
        fontsize=14,
    )

    ax.set_ylabel(
        r"Information weight power $\gamma$",
        fontsize=14,
    )

    ax.set_title(
        "Pickrell: Weighted-context residual-geometry objective\n"
        "(lower is flatter; p-values are NOT used in selection)",
        fontsize=18,
        pad=16,
    )

    plt.colorbar(
        image,
        ax=ax,
        label="Median geometry objective",
    )

    plt.tight_layout()
    plt.show()


def _plot_band(
    ax,
    x,
    agg,
    metric,
    label,
    *,
    linestyle="-",
    marker="o",
):
    ax.plot(
        x,
        agg[
            f"{metric}__median"
        ],
        marker=marker,
        linewidth=2.3,
        linestyle=linestyle,
        label=label,
    )

    ax.fill_between(
        x,
        agg[
            f"{metric}__q25"
        ],
        agg[
            f"{metric}__q75"
        ],
        alpha=0.14,
    )


def _plot_best_curves(
    aggregate,
    best_gamma,
    best_epsilon,
):
    x = aggregate[
        "median_log1p_total_normalized_coverage__median"
    ].to_numpy(
        dtype=float
    )

    # Full.
    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    _plot_band(
        ax,
        x,
        aggregate,
        "current_SF_over_aF",
        r"Current $S_F/a_F$",
    )

    _plot_band(
        ax,
        x,
        aggregate,
        "context_SF_over_aF",
        r"Contextual $S_F/a_F$",
        linestyle="--",
        marker="s",
    )

    ax.set_xlabel(
        "log1p(total normalized gene coverage)",
        fontsize=14,
    )

    ax.set_ylabel(
        r"Median $S_F/a_F$",
        fontsize=14,
    )

    ax.set_title(
        rf"Pickrell: Full residual geometry — current vs weighted context "
        rf"($\gamma={best_gamma:g}$, $\epsilon={best_epsilon:g}$)",
        fontsize=18,
        pad=16,
    )

    ax.grid(
        alpha=0.20,
    )

    ax.legend(
        fontsize=11,
    )

    plt.tight_layout()
    plt.show()

    # Null.
    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    _plot_band(
        ax,
        x,
        aggregate,
        "current_S0_over_a0",
        r"Current $S_0/a_0$",
    )

    _plot_band(
        ax,
        x,
        aggregate,
        "context_S0_over_a0",
        r"Contextual $S_0/a_0$",
        linestyle="--",
        marker="s",
    )

    ax.set_xlabel(
        "log1p(total normalized gene coverage)",
        fontsize=14,
    )

    ax.set_ylabel(
        r"Median $S_0/a_0$",
        fontsize=14,
    )

    ax.set_title(
        rf"Pickrell: Null residual geometry — current vs weighted context "
        rf"($\gamma={best_gamma:g}$, $\epsilon={best_epsilon:g}$)",
        fontsize=18,
        pad=16,
    )

    ax.grid(
        alpha=0.20,
    )

    ax.legend(
        fontsize=11,
    )

    plt.tight_layout()
    plt.show()

    # Overlay with existing raw-p hump AFTER geometry selection.
    fig, ax1 = plt.subplots(
        figsize=FIGSIZE
    )

    ax2 = ax1.twinx()

    ax1.plot(
        x,
        aggregate[
            "context_SF_over_aF__median"
        ],
        marker="s",
        linewidth=2.3,
        linestyle="--",
        label=r"Contextual $S_F/a_F$",
    )

    ax1.plot(
        x,
        aggregate[
            "context_S0_over_a0__median"
        ],
        marker="^",
        linewidth=2.3,
        linestyle="-.",
        label=r"Contextual $S_0/a_0$",
    )

    ax2.plot(
        x,
        aggregate[
            "current_raw_p_lt_0_01_fraction__median"
        ],
        marker="o",
        linewidth=2.2,
        label="Existing M03 raw p<.01",
    )

    ax2.axhline(
        RAW_P_THRESHOLD,
        linestyle=":",
        linewidth=1.4,
    )

    ax1.set_xlabel(
        "log1p(total normalized gene coverage)",
        fontsize=14,
    )

    ax1.set_ylabel(
        "Contextual residual geometry",
        fontsize=14,
    )

    ax2.set_ylabel(
        "Existing fraction raw p < .01",
        fontsize=14,
    )

    ax1.set_title(
        "Pickrell: Geometry-optimal contextual residual curve vs existing FPR hump",
        fontsize=18,
        pad=16,
    )

    ax1.grid(
        alpha=0.20,
    )

    lines1, labels1 = (
        ax1.get_legend_handles_labels()
    )

    lines2, labels2 = (
        ax2.get_legend_handles_labels()
    )

    ax1.legend(
        lines1 + lines2,
        labels1 + labels2,
        fontsize=11,
        loc="best",
    )

    plt.tight_layout()
    plt.show()


def _plot_context_mapping(
    reference_histograms,
    best_gamma,
    best_epsilon,
):
    # For visualization only, use pooled all-split reference CDF.
    hist = reference_histograms[
        float(
            best_gamma
        )
    ][
        "total"
    ]

    p_grid = expit(
        np.linspace(
            -12,
            12,
            500,
        )
    )

    F = _cdf_from_hist(
        _to_logit(
            p_grid
        ),
        hist,
    )

    u = (
        float(
            best_epsilon
        )
        + (
            1.0
            - 2.0
            * float(
                best_epsilon
            )
        )
        * F
    )

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    ax.plot(
        p_grid,
        u,
        linewidth=2.5,
        label=r"$u_\epsilon(p)$",
    )

    ax.plot(
        p_grid,
        p_grid,
        linestyle="--",
        linewidth=1.5,
        label="Identity",
    )

    ax.set_xscale(
        "logit"
        if False
        else "linear"
    )

    ax.set_xlabel(
        r"Original effective proportion $p=\mu/(\mu+r)$",
        fontsize=14,
    )

    ax.set_ylabel(
        r"Contextual proportion coordinate $u_\epsilon(p)$",
        fontsize=14,
    )

    ax.set_title(
        rf"Weighted universal variation context "
        rf"($\gamma={best_gamma:g}$, $\epsilon={best_epsilon:g}$)",
        fontsize=18,
        pad=16,
    )

    ax.grid(
        alpha=0.20,
    )

    ax.legend(
        fontsize=11,
    )

    plt.tight_layout()
    plt.show()

    # Curvature.
    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    ax.plot(
        p_grid,
        p_grid
        * (
            1.0
            - p_grid
        ),
        linewidth=2.3,
        label=r"Original $p(1-p)$",
    )

    ax.plot(
        p_grid,
        u
        * (
            1.0
            - u
        ),
        linewidth=2.3,
        label=r"Context $u(1-u)$",
    )

    ax.set_xlabel(
        r"Original effective proportion $p$",
        fontsize=14,
    )

    ax.set_ylabel(
        "Variation curvature",
        fontsize=14,
    )

    ax.set_title(
        "Original vs weighted-context binomial curvature",
        fontsize=18,
        pad=16,
    )

    ax.grid(
        alpha=0.20,
    )

    ax.legend(
        fontsize=11,
    )

    plt.tight_layout()
    plt.show()


# =============================================================================
# TOP-LEVEL
# =============================================================================

def run_pickrell_weighted_context_diagnostic():
    started = (
        datetime.now()
        .astimezone()
    )

    print(
        f"[start] {_now()}",
        flush=True,
    )

    try:
        reps = (
            _discover_completed_replicates()
        )

        print(
            "\nPICKRELL WEIGHTED UNIVERSAL VARIATION-CONTEXT DIAGNOSTIC\n"
            f"namespace:        {CURRENT_M03_NAMESPACE}\n"
            f"completed splits: {len(reps)}\n"
            f"eps grid:         {EPSILON_GRID}\n"
            f"weight powers:    {WEIGHT_POWER_GRID}\n"
            f"CDF grid bins:    {CDF_N_BINS}\n"
            "selection target: residual-geometry flatness ONLY\n",
            flush=True,
        )

        # ---------------------------------------------------------
        # Load all splits.
        # ---------------------------------------------------------
        rep_data = []

        for j, rep_id in enumerate(
            reps,
            start=1,
        ):
            rep = _load_rep_data(
                rep_id
            )

            rep_data.append(
                rep
            )

            print(
                f"[load] {j:02d}/{len(reps):02d} | rep={rep_id:02d}",
                flush=True,
            )

        # ---------------------------------------------------------
        # Verify the effective-binomial bridge numerically.
        # ---------------------------------------------------------
        identity = _audit_effective_binomial_identity(
            rep_data[
                0
            ]
        )

        print(
            "\nEFFECTIVE-BINOMIAL / NB HALF-DEVIANCE IDENTITY AUDIT\n",
            flush=True,
        )

        display(
            pd.DataFrame(
                [
                    identity
                ]
            )
        )

        # The identity is algebraic.  After the boundary-safe KL fix above,
        # remaining differences should be ordinary floating-point roundoff.
        # Keep a strict absolute gate, but no longer fail on the artificial
        # endpoint-clipping error from v1.
        if (
            not np.isfinite(
                identity[
                    "max_abs_error"
                ]
            )
            or identity[
                "max_abs_error"
            ]
            > 1e-7
        ):
            raise RuntimeError(
                "Effective-binomial/NB deviance identity check failed."
            )

        # ---------------------------------------------------------
        # Build weighted reference CDFs.
        # ---------------------------------------------------------
        print(
            "\n[reference] building weighted cross-fit reference CDF histograms...",
            flush=True,
        )

        reference_histograms = (
            _build_reference_histograms(
                rep_data
            )
        )

        # ---------------------------------------------------------
        # Select epsilon/gamma using geometry only.
        # ---------------------------------------------------------
        (
            grid_results,
            subset_by_rep,
            baseline,
        ) = _grid_search_context(
            rep_data,
            reference_histograms,
        )

        best = (
            grid_results
            .iloc[
                0
            ]
        )

        best_gamma = float(
            best[
                "weight_power"
            ]
        )

        best_epsilon = float(
            best[
                "epsilon"
            ]
        )

        print(
            "\nSELECTED CONTEXT — GEOMETRY ONLY\n",
            flush=True,
        )

        display(
            pd.DataFrame(
                [
                    best.to_dict()
                ]
            )
        )

        if (
            best_epsilon
            == min(
                EPSILON_GRID
            )
            or best_epsilon
            == max(
                EPSILON_GRID
            )
        ):
            print(
                "[note] best epsilon is on the edge of the supplied grid; "
                "expand the grid before treating it as an optimum.",
                flush=True,
            )

        if (
            best_gamma
            == min(
                WEIGHT_POWER_GRID
            )
            or best_gamma
            == max(
                WEIGHT_POWER_GRID
            )
        ):
            print(
                "[note] best information-weight power is on the edge of the supplied grid; "
                "expand the grid before treating it as an optimum.",
                flush=True,
            )

        # ---------------------------------------------------------
        # Full-gene reevaluation of geometry-optimal candidate.
        # ---------------------------------------------------------
        (
            best_detail,
            best_aggregate,
        ) = (
            _evaluate_best_context_all_genes(
                rep_data,
                reference_histograms,
                best_gamma,
                best_epsilon,
            )
        )

        print(
            "\nBEST CONTEXT — FULL-GENE COVERAGE CURVES\n",
            flush=True,
        )

        display(
            best_aggregate[
                [
                    "coverage_bin",
                    "n_splits",
                    "median_log1p_total_normalized_coverage__median",
                    "current_SF_over_aF__median",
                    "context_SF_over_aF__median",
                    "current_S0_over_a0__median",
                    "context_S0_over_a0__median",
                    "current_raw_p_lt_0_01_fraction__median",
                ]
            ]
        )

        # ---------------------------------------------------------
        # Plots.
        # ---------------------------------------------------------
        _plot_grid_objective(
            grid_results
        )

        _plot_best_curves(
            best_aggregate,
            best_gamma,
            best_epsilon,
        )

        _plot_context_mapping(
            reference_histograms,
            best_gamma,
            best_epsilon,
        )

        ended = (
            datetime.now()
            .astimezone()
        )

        print(
            "\nSTATUS\n"
            f"selected gamma:   {best_gamma:g}\n"
            f"selected epsilon: {best_epsilon:g}\n"
            f"[end] {_now()} | elapsed={ended-started}",
            flush=True,
        )

        return {
            "grid_results":
                grid_results,

            "best_gamma":
                best_gamma,

            "best_epsilon":
                best_epsilon,

            "best_detail":
                best_detail,

            "best_aggregate":
                best_aggregate,

            "reference_histograms":
                reference_histograms,

            "rep_data":
                rep_data,
        }

    finally:
        pass


# =============================================================================
# EXECUTION
# =============================================================================

pickrell_weighted_context_diagnostic = (
    run_pickrell_weighted_context_diagnostic()
)


In [ ]:
from __future__ import annotations

"""
Pickrell logit-affine weighted-context diagnostic.

This is the second implementation of Sean's "universal variation context" idea.

WHY THIS VERSION
----------------
The first weighted-CDF warp used

    u(p) = eps + (1 - 2 eps) F_w(p)

directly inside the Bernoulli KL.  Locally that inserts F_w'(p)^2 into the
metric, so dense regions of the empirical p-distribution can dominate the
deviance.  That implementation produced pathological geometry.

This version uses the weighted reference distribution ONLY TO DEFINE ANCHORS.

For a cross-fitted information-weighted reference distribution, choose

    p_L = weighted tau quantile
    p_U = weighted (1-tau) quantile

and construct a smooth logit-affine map

    logit u(p) = A + B logit p

such that

    u(p_L) = epsilon
    u(p_U) = 1 - epsilon.

Thus the weighted reference distribution tells us what the useful lower and
upper ends of the proportion scale are, but its local density does NOT enter
the metric.

REFERENCE DISTRIBUTION
----------------------
Current NB effective-binomial fitted proportion:

    p_gi = mu_gi / (mu_gi + r_g)

NB information weight per observation:

    I_gi = mu_gi * r_g / (mu_gi + r_g)
         = mu_gi / (1 + mu_gi/r_g)

Reference-CDF weights:

    w_gi = I_gi ** gamma

where

    gamma = 0   -> unweighted reference distribution
    gamma = 1   -> directly information-weighted reference distribution.

The reference distribution is CROSS-FITTED BY PICKRELL SPLIT:
the held-out split's own p-values/weights are removed before its anchors are
computed.

CONTEXTUAL DEVIANCE
-------------------
For each observation,

    p_obs = y / (y + r)
    p_fit = mu / (mu + r)
    m_eff = y + r

transform both coordinates with the SAME held-out affine context:

    u_obs = logistic(A + B logit(p_obs))
    u_fit = logistic(A + B logit(p_fit))

and calculate

    s_ctx = m_eff * KL_Bernoulli(u_obs || u_fit).

This preserves s_ctx=0 when observed=fitted.

SELECTION
---------
Grid over:

    gamma   : information-weight power
    tau     : weighted reference quantile used as contextual endpoints
    epsilon : target contextual endpoint

Choose parameters using RESIDUAL GEOMETRY ONLY:

    flatness of median S_F_ctx/a_F over coverage
  + flatness of median S_0_ctx/a_0 over coverage
  + penalty for S_0_ctx < S_F_ctx

Existing M03 p-values/FPR are NOT used to select gamma/tau/epsilon.

This is a geometry diagnostic, not yet a production M03 test.  A contextual
DNB normalizer would still need to be derived/audited before replacing the
current likelihood machinery.

OUTPUT
------
DISPLAY ONLY. Nothing is saved.
"""

from datetime import datetime
from pathlib import Path
from scipy.special import expit, logit, xlogy

import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


# =============================================================================
# SETTINGS
# =============================================================================

CURRENT_M03_NAMESPACE = "pickrell_m03_seam_safe_20M_kappa_coverage_v1"

N_COVERAGE_BINS = 12
RAW_P_THRESHOLD = 0.01

# Proposed parameter grids.
WEIGHT_POWER_GRID = (
    0.0,
    0.5,
    1.0,
    1.5,
    2.0,
)

TAU_GRID = (
    0.005,
    0.01,
    0.025,
    0.05,
    0.10,
)

EPSILON_GRID = (
    0.005,
    0.01,
    0.025,
    0.05,
    0.10,
)

# Weighted reference histogram lives on logit(p), where the effective
# proportions are much easier to resolve near zero.
REFERENCE_LOGIT_MIN = -30.0
REFERENCE_LOGIT_MAX = 30.0
REFERENCE_N_BINS = 16_384

# Grid-search subset.  It is deterministic AND stratified by coverage bin.
GRID_SEARCH_MAX_GENES_PER_SPLIT = 6_000
GRID_SEARCH_SEED = 20261003

# Objective.
NESTED_NEGATIVE_PENALTY = 5.0

FIGSIZE = (16, 9)


# =============================================================================
# PATHS / TIME
# =============================================================================

def _now() -> str:
    return datetime.now().astimezone().strftime(
        "%Y-%m-%d %I:%M:%S %p %Z"
    )


CWD = Path.cwd().resolve()

if CWD.name.lower() in {"notebook", "notebooks"}:
    PROJECT_ROOT = CWD.parent
elif (CWD / "data").is_dir():
    PROJECT_ROOT = CWD
else:
    raise RuntimeError(
        "Could not resolve repository root. Run from <repo>/notebook, "
        "<repo>/notebooks, or the repository root containing ./data. "
        f"Current working directory: {CWD}"
    )

RUN_DIR = (
    PROJECT_ROOT
    / "data"
    / CURRENT_M03_NAMESPACE
)

CELL_DIR = RUN_DIR / "cells"
CHECKPOINT_ROOT = RUN_DIR / "m03_checkpoints"


# =============================================================================
# CHECKPOINT HELPERS
# =============================================================================

def _load_pickle(path: Path):
    with open(path, "rb") as f:
        return pickle.load(f)


def _checkpoint_dir(replicate: int) -> Path:
    return (
        CHECKPOINT_ROOT
        / f"pickrell_rep{int(replicate):02d}"
    )


def _load_success_stage(
    replicate: int,
    stage_name: str,
):
    path = (
        _checkpoint_dir(replicate)
        / "stages"
        / f"{stage_name}.pkl"
    )

    if not path.exists():
        raise FileNotFoundError(
            f"Missing checkpoint: {path}"
        )

    payload = _load_pickle(path)

    if (
        not isinstance(payload, dict)
        or payload.get("status") != "success"
    ):
        raise RuntimeError(
            f"Replicate {replicate}: "
            f"{stage_name} is not a successful checkpoint."
        )

    return payload["data"]


def _discover_completed_replicates():
    reps = []

    for path in sorted(
        CELL_DIR.glob("pickrell_rep*.csv")
    ):
        try:
            x = pd.read_csv(path)
        except Exception:
            continue

        if len(x) != 1:
            continue

        if str(
            x.loc[0, "status"]
        ).strip().lower() != "success":
            continue

        reps.append(
            int(
                x.loc[0, "replicate"]
            )
        )

    reps = sorted(set(reps))

    if not reps:
        raise RuntimeError(
            "No completed Pickrell replicates were found."
        )

    return reps


# =============================================================================
# BASIC MATH
# =============================================================================

def _bernoulli_kl(
    q,
    p,
):
    """
    Boundary-safe Bernoulli KL.

    q may be exactly 0 or 1.
    """

    q = np.asarray(q, dtype=float)
    p = np.asarray(p, dtype=float)

    q = np.clip(
        q,
        0.0,
        1.0,
    )

    tiny = np.finfo(float).tiny
    one_minus = 1.0 - np.finfo(float).eps

    p = np.clip(
        p,
        tiny,
        one_minus,
    )

    return (
        xlogy(q, q)
        - xlogy(q, p)
        + xlogy(1.0 - q, 1.0 - q)
        - xlogy(1.0 - q, 1.0 - p)
    )


def _half_nb_deviance(
    y,
    mu,
    r,
):
    y = np.asarray(y, dtype=float)
    mu = np.maximum(
        np.asarray(mu, dtype=float),
        1e-300,
    )
    r = np.maximum(
        np.asarray(r, dtype=float),
        1e-300,
    )

    return np.maximum(
        xlogy(
            y,
            y / mu,
        )
        - (
            y + r
        )
        * np.log(
            (
                y + r
            )
            / (
                mu + r
            )
        ),
        0.0,
    )


def _safe_logit(
    p,
):
    p = np.asarray(
        p,
        dtype=float,
    )

    p = np.clip(
        p,
        expit(
            REFERENCE_LOGIT_MIN
        ),
        expit(
            REFERENCE_LOGIT_MAX
        ),
    )

    return np.clip(
        logit(
            p
        ),
        REFERENCE_LOGIT_MIN,
        REFERENCE_LOGIT_MAX,
    )


def _finite_log_sd(values):
    x = np.asarray(
        values,
        dtype=float,
    )

    x = x[
        np.isfinite(x)
        & (x > 0)
    ]

    if len(x) < 3:
        return np.inf

    return float(
        np.std(
            np.log(x),
            ddof=0,
        )
    )


# =============================================================================
# WEIGHTED REFERENCE HISTOGRAM / QUANTILES
# =============================================================================

REFERENCE_EDGES = np.linspace(
    REFERENCE_LOGIT_MIN,
    REFERENCE_LOGIT_MAX,
    REFERENCE_N_BINS + 1,
)


def _weighted_histogram(
    p,
    weights,
):
    z = _safe_logit(
        p
    )

    hist, _ = np.histogram(
        z,
        bins=REFERENCE_EDGES,
        weights=np.asarray(
            weights,
            dtype=float,
        ),
    )

    return hist.astype(
        float
    )


def _weighted_quantile_from_hist(
    hist,
    probability,
):
    """
    Approximate weighted p-quantile using a fine histogram on logit(p).
    Linear interpolation is performed within the selected logit bin.
    """

    hist = np.asarray(
        hist,
        dtype=float,
    )

    probability = float(
        probability
    )

    if not (
        0.0
        < probability
        < 1.0
    ):
        raise ValueError(
            "probability must lie strictly between 0 and 1."
        )

    total = float(
        np.sum(
            hist
        )
    )

    if (
        not np.isfinite(total)
        or total <= 0
    ):
        raise RuntimeError(
            "Weighted reference histogram contains no positive mass."
        )

    target = (
        probability
        * total
    )

    cumulative = np.cumsum(
        hist
    )

    j = int(
        np.searchsorted(
            cumulative,
            target,
            side="left",
        )
    )

    j = min(
        max(
            j,
            0,
        ),
        len(hist) - 1,
    )

    before = (
        float(
            cumulative[
                j - 1
            ]
        )
        if j > 0
        else 0.0
    )

    mass = float(
        hist[
            j
        ]
    )

    if mass > 0:
        frac = (
            target
            - before
        ) / mass
    else:
        frac = 0.5

    frac = float(
        np.clip(
            frac,
            0.0,
            1.0,
        )
    )

    z = (
        REFERENCE_EDGES[
            j
        ]
        + frac
        * (
            REFERENCE_EDGES[
                j + 1
            ]
            - REFERENCE_EDGES[
                j
            ]
        )
    )

    return float(
        expit(
            z
        )
    )


# =============================================================================
# LOAD CURRENT PICKRELL SPLITS
# =============================================================================

def _load_rep_data(
    replicate: int,
):
    core = _load_success_stage(
        replicate,
        "01_core",
    )

    base = _load_success_stage(
        replicate,
        "05_base_m03_result",
    )

    y = np.asarray(
        core["y"],
        dtype=float,
    )

    exposure = np.asarray(
        core["exposure"],
        dtype=float,
    )

    mu_F = np.asarray(
        core["mu_F"],
        dtype=float,
    )

    mu_0 = np.asarray(
        core["mu_0"],
        dtype=float,
    )

    r = np.asarray(
        core["r_hat"],
        dtype=float,
    )

    S_F = np.asarray(
        core["S_F"],
        dtype=float,
    )

    S_0 = np.asarray(
        core["S_0"],
        dtype=float,
    )

    a_F = float(
        core["a_F"]
    )

    a_0 = float(
        core["a_0"]
    )

    p_value = np.asarray(
        base["p_value_m03"],
        dtype=float,
    )

    n_genes = int(
        y.shape[0]
    )

    for name, arr in {
        "mu_F": mu_F,
        "mu_0": mu_0,
    }.items():
        if arr.shape != y.shape:
            raise RuntimeError(
                f"Replicate {replicate}: "
                f"{name} shape={arr.shape} != y shape={y.shape}"
            )

    for name, arr in {
        "r": r,
        "S_F": S_F,
        "S_0": S_0,
        "p_value": p_value,
    }.items():
        if len(arr) != n_genes:
            raise RuntimeError(
                f"Replicate {replicate}: "
                f"{name} length={len(arr)} != n_genes={n_genes}"
            )

    total_normalized_coverage = np.sum(
        y
        / exposure[
            None,
            :
        ],
        axis=1,
    )

    log_coverage = np.log1p(
        total_normalized_coverage
    )

    coverage_bin = (
        np.asarray(
            pd.qcut(
                pd.Series(
                    log_coverage
                ),
                q=N_COVERAGE_BINS,
                labels=False,
                duplicates="drop",
            ),
            dtype=int,
        )
        + 1
    )

    r_matrix = (
        r[
            :,
            None
        ]
    )

    p_F = (
        mu_F
        / (
            mu_F
            + r_matrix
        )
    )

    # Observation-level NB information.
    information = (
        mu_F
        * r_matrix
        / (
            mu_F
            + r_matrix
        )
    )

    return {
        "replicate":
            int(
                replicate
            ),

        "y":
            y,

        "mu_F":
            mu_F,

        "mu_0":
            mu_0,

        "r":
            r,

        "S_F":
            S_F,

        "S_0":
            S_0,

        "a_F":
            a_F,

        "a_0":
            a_0,

        "p_value":
            p_value,

        "log_coverage":
            log_coverage,

        "coverage_bin":
            coverage_bin,

        "p_reference_obs":
            p_F.reshape(
                -1
            ),

        "information_reference_obs":
            information.reshape(
                -1
            ),
    }


# =============================================================================
# EFFECTIVE-BINOMIAL IDENTITY AUDIT
# =============================================================================

def _audit_effective_binomial_identity(
    rep,
):
    y = rep[
        "y"
    ]

    mu = rep[
        "mu_F"
    ]

    r = rep[
        "r"
    ][
        :,
        None
    ]

    p_obs = (
        y
        / (
            y + r
        )
    )

    p_fit = (
        mu
        / (
            mu + r
        )
    )

    m_eff = (
        y + r
    )

    binom_half = (
        m_eff
        * _bernoulli_kl(
            p_obs,
            p_fit,
        )
    )

    nb_half = _half_nb_deviance(
        y,
        mu,
        r,
    )

    error = np.abs(
        binom_half
        - nb_half
    )

    finite = np.isfinite(
        error
    )

    return {
        "max_abs_error":
            float(
                np.max(
                    error[
                        finite
                    ]
                )
            ),

        "median_abs_error":
            float(
                np.median(
                    error[
                        finite
                    ]
                )
            ),
    }


# =============================================================================
# CROSS-FITTED REFERENCE HISTOGRAMS
# =============================================================================

def _build_reference_histograms(
    rep_data,
):
    """
    One weighted histogram per gamma and Pickrell split, plus pooled total.
    Held-out split histogram is subtracted exactly before anchor estimation.
    """

    output = {}

    for gamma in WEIGHT_POWER_GRID:
        gamma = float(
            gamma
        )

        by_rep = {}

        for rep in rep_data:
            p = np.asarray(
                rep[
                    "p_reference_obs"
                ],
                dtype=float,
            )

            information = np.asarray(
                rep[
                    "information_reference_obs"
                ],
                dtype=float,
            )

            ok = (
                np.isfinite(
                    p
                )
                & np.isfinite(
                    information
                )
                & (
                    information
                    > 0
                )
            )

            if gamma == 0.0:
                weights = np.ones(
                    int(
                        np.sum(
                            ok
                        )
                    ),
                    dtype=float,
                )
            else:
                weights = (
                    information[
                        ok
                    ]
                    ** gamma
                )

                # Purely numerical rescaling. Quantiles are invariant to it.
                positive = weights[
                    np.isfinite(
                        weights
                    )
                    & (
                        weights
                        > 0
                    )
                ]

                if len(
                    positive
                ):
                    scale = float(
                        np.median(
                            positive
                        )
                    )

                    if (
                        np.isfinite(
                            scale
                        )
                        and scale
                        > 0
                    ):
                        weights = (
                            weights
                            / scale
                        )

            by_rep[
                rep[
                    "replicate"
                ]
            ] = _weighted_histogram(
                p[
                    ok
                ],
                weights,
            )

        total = np.sum(
            np.stack(
                list(
                    by_rep.values()
                ),
                axis=0,
            ),
            axis=0,
        )

        output[
            gamma
        ] = {
            "by_rep":
                by_rep,

            "total":
                total,
        }

    return output


# =============================================================================
# LOGIT-AFFINE CONTEXT
# =============================================================================

def _context_parameters_from_anchors(
    p_L,
    p_U,
    epsilon,
):
    p_L = float(
        p_L
    )

    p_U = float(
        p_U
    )

    epsilon = float(
        epsilon
    )

    if not (
        0.0
        < p_L
        < p_U
        < 1.0
    ):
        raise RuntimeError(
            f"Invalid reference anchors p_L={p_L}, p_U={p_U}."
        )

    if not (
        0.0
        < epsilon
        < 0.5
    ):
        raise RuntimeError(
            f"Invalid epsilon={epsilon}."
        )

    z_L = float(
        _safe_logit(
            p_L
        )
    )

    z_U = float(
        _safe_logit(
            p_U
        )
    )

    t_L = float(
        logit(
            epsilon
        )
    )

    t_U = float(
        logit(
            1.0
            - epsilon
        )
    )

    denominator = (
        z_U
        - z_L
    )

    if (
        not np.isfinite(
            denominator
        )
        or denominator
        <= 1e-10
    ):
        raise RuntimeError(
            "Reference anchors are too close on the logit scale."
        )

    B = (
        t_U
        - t_L
    ) / denominator

    A = (
        t_L
        - B
        * z_L
    )

    if (
        not np.isfinite(
            A
        )
        or not np.isfinite(
            B
        )
        or B <= 0
    ):
        raise RuntimeError(
            f"Invalid affine context A={A}, B={B}."
        )

    return (
        float(
            A
        ),
        float(
            B
        ),
    )


def _transform_p(
    p,
    A,
    B,
):
    return expit(
        float(
            A
        )
        + float(
            B
        )
        * _safe_logit(
            p
        )
    )


def _contextual_half_deviance_from_logits(
    y,
    r,
    z_obs,
    z_fit,
    A,
    B,
):
    u_obs = expit(
        float(
            A
        )
        + float(
            B
        )
        * z_obs
    )

    u_fit = expit(
        float(
            A
        )
        + float(
            B
        )
        * z_fit
    )

    return (
        y + r
    ) * _bernoulli_kl(
        u_obs,
        u_fit,
    )


# =============================================================================
# STRATIFIED GRID-SEARCH SUBSETS
# =============================================================================

def _make_stratified_subset(
    rep,
):
    n_genes = int(
        rep[
            "y"
        ].shape[
            0
        ]
    )

    max_genes = int(
        GRID_SEARCH_MAX_GENES_PER_SPLIT
    )

    if max_genes >= n_genes:
        return np.arange(
            n_genes,
            dtype=int,
        )

    bins = np.asarray(
        rep[
            "coverage_bin"
        ],
        dtype=int,
    )

    unique_bins = sorted(
        np.unique(
            bins
        )
    )

    base = (
        max_genes
        // len(
            unique_bins
        )
    )

    remainder = (
        max_genes
        - base
        * len(
            unique_bins
        )
    )

    chosen = []

    for j, b in enumerate(
        unique_bins
    ):
        candidates = np.flatnonzero(
            bins == b
        )

        target = (
            base
            + (
                1
                if j
                < remainder
                else 0
            )
        )

        target = min(
            target,
            len(
                candidates
            ),
        )

        rng = np.random.default_rng(
            int(
                GRID_SEARCH_SEED
            )
            + int(
                rep[
                    "replicate"
                ]
            )
            * 100
            + int(
                b
            )
        )

        if target >= len(
            candidates
        ):
            picked = candidates
        else:
            picked = np.sort(
                rng.choice(
                    candidates,
                    size=target,
                    replace=False,
                )
            )

        chosen.append(
            picked
        )

    out = np.sort(
        np.concatenate(
            chosen
        )
    )

    return out.astype(
        int
    )


# =============================================================================
# PREPARE SPLIT-SPECIFIC LOGIT COORDINATES ON A SUBSET
# =============================================================================

def _prepare_subset(
    rep,
    gene_indices,
):
    idx = np.asarray(
        gene_indices,
        dtype=int,
    )

    y = rep[
        "y"
    ][
        idx,
        :
    ]

    mu_F = rep[
        "mu_F"
    ][
        idx,
        :
    ]

    mu_0 = rep[
        "mu_0"
    ][
        idx,
        :
    ]

    r = rep[
        "r"
    ][
        idx
    ][
        :,
        None
    ]

    p_obs = (
        y
        / (
            y + r
        )
    )

    p_F = (
        mu_F
        / (
            mu_F
            + r
        )
    )

    p_0 = (
        mu_0
        / (
            mu_0
            + r
        )
    )

    return {
        "idx":
            idx,

        "y":
            y,

        "r":
            r,

        "z_obs":
            _safe_logit(
                p_obs
            ),

        "z_F":
            _safe_logit(
                p_F
            ),

        "z_0":
            _safe_logit(
                p_0
            ),
    }


# =============================================================================
# EVALUATE ONE CONTEXT
# =============================================================================

def _evaluate_context(
    rep,
    prepared,
    A,
    B,
):
    s_F = _contextual_half_deviance_from_logits(
        prepared[
            "y"
        ],
        prepared[
            "r"
        ],
        prepared[
            "z_obs"
        ],
        prepared[
            "z_F"
        ],
        A,
        B,
    )

    s_0 = _contextual_half_deviance_from_logits(
        prepared[
            "y"
        ],
        prepared[
            "r"
        ],
        prepared[
            "z_obs"
        ],
        prepared[
            "z_0"
        ],
        A,
        B,
    )

    S_F_ctx = np.sum(
        s_F,
        axis=1,
    )

    S_0_ctx = np.sum(
        s_0,
        axis=1,
    )

    idx = prepared[
        "idx"
    ]

    bins = rep[
        "coverage_bin"
    ][
        idx
    ]

    full_curve = []
    null_curve = []

    for b in sorted(
        np.unique(
            bins
        )
    ):
        use = (
            bins
            == b
        )

        full_curve.append(
            float(
                np.median(
                    S_F_ctx[
                        use
                    ]
                    / rep[
                        "a_F"
                    ]
                )
            )
        )

        null_curve.append(
            float(
                np.median(
                    S_0_ctx[
                        use
                    ]
                    / rep[
                        "a_0"
                    ]
                )
            )
        )

    nested_tolerance = (
        1e-10
        * np.maximum(
            1.0,
            np.abs(
                S_F_ctx
            )
        )
    )

    negative = (
        S_0_ctx
        < S_F_ctx
        - nested_tolerance
    )

    return {
        "S_F_ctx":
            S_F_ctx,

        "S_0_ctx":
            S_0_ctx,

        "full_curve":
            np.asarray(
                full_curve,
                dtype=float,
            ),

        "null_curve":
            np.asarray(
                null_curve,
                dtype=float,
            ),

        "flatness_F":
            _finite_log_sd(
                full_curve
            ),

        "flatness_0":
            _finite_log_sd(
                null_curve
            ),

        "nested_negative_fraction":
            float(
                np.mean(
                    negative
                )
            ),
    }


# =============================================================================
# CURRENT GEOMETRY BASELINE
# =============================================================================

def _current_geometry(
    rep,
    gene_indices,
):
    idx = np.asarray(
        gene_indices,
        dtype=int,
    )

    bins = rep[
        "coverage_bin"
    ][
        idx
    ]

    current_F = (
        rep[
            "S_F"
        ][
            idx
        ]
        / rep[
            "a_F"
        ]
    )

    current_0 = (
        rep[
            "S_0"
        ][
            idx
        ]
        / rep[
            "a_0"
        ]
    )

    fcurve = []
    ncurve = []

    for b in sorted(
        np.unique(
            bins
        )
    ):
        use = (
            bins
            == b
        )

        fcurve.append(
            float(
                np.median(
                    current_F[
                        use
                    ]
                )
            )
        )

        ncurve.append(
            float(
                np.median(
                    current_0[
                        use
                    ]
                )
            )
        )

    return {
        "full_curve":
            np.asarray(
                fcurve,
                dtype=float,
            ),

        "null_curve":
            np.asarray(
                ncurve,
                dtype=float,
            ),

        "flatness_F":
            _finite_log_sd(
                fcurve
            ),

        "flatness_0":
            _finite_log_sd(
                ncurve
            ),
    }


# =============================================================================
# GRID SEARCH
# =============================================================================

def _grid_search(
    rep_data,
    reference_histograms,
):
    subset_by_rep = {
        int(
            rep[
                "replicate"
            ]
        ):
            _make_stratified_subset(
                rep
            )
        for rep in rep_data
    }

    sizes = [
        len(
            subset_by_rep[
                int(
                    rep[
                        "replicate"
                    ]
                )
            ]
        )
        for rep in rep_data
    ]

    print(
        "[grid] stratified subset sizes "
        f"min/median/max="
        f"{min(sizes):,}/"
        f"{int(np.median(sizes)):,}/"
        f"{max(sizes):,}",
        flush=True,
    )

    baseline_rows = []

    for rep in rep_data:
        rep_id = int(
            rep[
                "replicate"
            ]
        )

        cur = _current_geometry(
            rep,
            subset_by_rep[
                rep_id
            ],
        )

        baseline_rows.append(
            {
                "replicate":
                    rep_id,

                "flatness_F":
                    cur[
                        "flatness_F"
                    ],

                "flatness_0":
                    cur[
                        "flatness_0"
                    ],

                "combined":
                    cur[
                        "flatness_F"
                    ]
                    + cur[
                        "flatness_0"
                    ],
            }
        )

    baseline = pd.DataFrame(
        baseline_rows
    )

    print(
        "\nCURRENT RAW-DEVIANCE COVERAGE FLATNESS\n",
        flush=True,
    )

    display(
        pd.DataFrame(
            [
                {
                    "median_flatness_F":
                        float(
                            baseline[
                                "flatness_F"
                            ]
                            .median()
                        ),

                    "median_flatness_0":
                        float(
                            baseline[
                                "flatness_0"
                            ]
                            .median()
                        ),

                    "median_combined":
                        float(
                            baseline[
                                "combined"
                            ]
                            .median()
                        ),
                }
            ]
        )
    )

    # Precompute p logit coordinates once per split.
    prepared_by_rep = {}

    for rep in rep_data:
        rep_id = int(
            rep[
                "replicate"
            ]
        )

        prepared_by_rep[
            rep_id
        ] = _prepare_subset(
            rep,
            subset_by_rep[
                rep_id
            ],
        )

    rows = []

    for gamma in WEIGHT_POWER_GRID:
        gamma = float(
            gamma
        )

        print(
            f"[grid] gamma={gamma:g}",
            flush=True,
        )

        pooled = reference_histograms[
            gamma
        ][
            "total"
        ]

        by_rep_hist = reference_histograms[
            gamma
        ][
            "by_rep"
        ]

        # Precompute held-out anchor quantiles for each tau and split.
        anchors = {}

        for rep in rep_data:
            rep_id = int(
                rep[
                    "replicate"
                ]
            )

            hist_crossfit = (
                pooled
                - by_rep_hist[
                    rep_id
                ]
            )

            anchors[
                rep_id
            ] = {}

            for tau in TAU_GRID:
                p_L = _weighted_quantile_from_hist(
                    hist_crossfit,
                    tau,
                )

                p_U = _weighted_quantile_from_hist(
                    hist_crossfit,
                    1.0 - tau,
                )

                anchors[
                    rep_id
                ][
                    float(
                        tau
                    )
                ] = (
                    p_L,
                    p_U,
                )

        for tau in TAU_GRID:
            tau = float(
                tau
            )

            for epsilon in EPSILON_GRID:
                epsilon = float(
                    epsilon
                )

                split_rows = []

                for rep in rep_data:
                    rep_id = int(
                        rep[
                            "replicate"
                        ]
                    )

                    p_L, p_U = (
                        anchors[
                            rep_id
                        ][
                            tau
                        ]
                    )

                    try:
                        A, B = _context_parameters_from_anchors(
                            p_L,
                            p_U,
                            epsilon,
                        )
                    except Exception:
                        split_rows.append(
                            {
                                "replicate":
                                    rep_id,

                                "flatness_F":
                                    np.inf,

                                "flatness_0":
                                    np.inf,

                                "nested_negative_fraction":
                                    1.0,

                                "objective":
                                    np.inf,

                                "p_L":
                                    p_L,

                                "p_U":
                                    p_U,

                                "A":
                                    np.nan,

                                "B":
                                    np.nan,
                            }
                        )

                        continue

                    fit = _evaluate_context(
                        rep,
                        prepared_by_rep[
                            rep_id
                        ],
                        A,
                        B,
                    )

                    objective = (
                        fit[
                            "flatness_F"
                        ]
                        + fit[
                            "flatness_0"
                        ]
                        + float(
                            NESTED_NEGATIVE_PENALTY
                        )
                        * fit[
                            "nested_negative_fraction"
                        ]
                    )

                    split_rows.append(
                        {
                            "replicate":
                                rep_id,

                            "flatness_F":
                                fit[
                                    "flatness_F"
                                ],

                            "flatness_0":
                                fit[
                                    "flatness_0"
                                ],

                            "nested_negative_fraction":
                                fit[
                                    "nested_negative_fraction"
                                ],

                            "objective":
                                objective,

                            "p_L":
                                p_L,

                            "p_U":
                                p_U,

                            "A":
                                A,

                            "B":
                                B,
                        }
                    )

                sx = pd.DataFrame(
                    split_rows
                )

                finite_obj = np.isfinite(
                    sx[
                        "objective"
                    ]
                )

                if not np.any(
                    finite_obj
                ):
                    continue

                sx = sx.loc[
                    finite_obj
                ].copy()

                rows.append(
                    {
                        "weight_power":
                            gamma,

                        "tau":
                            tau,

                        "epsilon":
                            epsilon,

                        "median_flatness_F":
                            float(
                                sx[
                                    "flatness_F"
                                ]
                                .median()
                            ),

                        "median_flatness_0":
                            float(
                                sx[
                                    "flatness_0"
                                ]
                                .median()
                            ),

                        "median_nested_negative_fraction":
                            float(
                                sx[
                                    "nested_negative_fraction"
                                ]
                                .median()
                            ),

                        "mean_nested_negative_fraction":
                            float(
                                sx[
                                    "nested_negative_fraction"
                                ]
                                .mean()
                            ),

                        "median_objective":
                            float(
                                sx[
                                    "objective"
                                ]
                                .median()
                            ),

                        "q75_objective":
                            float(
                                sx[
                                    "objective"
                                ]
                                .quantile(
                                    0.75
                                )
                            ),

                        "median_p_L":
                            float(
                                sx[
                                    "p_L"
                                ]
                                .median()
                            ),

                        "median_p_U":
                            float(
                                sx[
                                    "p_U"
                                ]
                                .median()
                            ),

                        "median_A":
                            float(
                                sx[
                                    "A"
                                ]
                                .median()
                            ),

                        "median_B":
                            float(
                                sx[
                                    "B"
                                ]
                                .median()
                            ),
                    }
                )

    results = (
        pd.DataFrame(
            rows
        )
        .sort_values(
            [
                "median_objective",
                "q75_objective",
            ],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    if results.empty:
        raise RuntimeError(
            "No valid weighted-anchor context candidates were produced."
        )

    print(
        "\nBEST WEIGHTED-ANCHOR CONTEXT CANDIDATES\n",
        flush=True,
    )

    display(
        results.head(
            25
        )
    )

    return (
        results,
        subset_by_rep,
        baseline,
    )


# =============================================================================
# FULL-GENE EVALUATION OF WINNER
# =============================================================================

def _evaluate_winner_all_genes(
    rep_data,
    reference_histograms,
    best_gamma,
    best_tau,
    best_epsilon,
):
    gamma_info = reference_histograms[
        float(
            best_gamma
        )
    ]

    pooled = gamma_info[
        "total"
    ]

    by_rep_hist = gamma_info[
        "by_rep"
    ]

    detail_rows = []
    anchor_rows = []

    for j, rep in enumerate(
        rep_data,
        start=1,
    ):
        rep_id = int(
            rep[
                "replicate"
            ]
        )

        hist_crossfit = (
            pooled
            - by_rep_hist[
                rep_id
            ]
        )

        p_L = _weighted_quantile_from_hist(
            hist_crossfit,
            best_tau,
        )

        p_U = _weighted_quantile_from_hist(
            hist_crossfit,
            1.0 - best_tau,
        )

        A, B = _context_parameters_from_anchors(
            p_L,
            p_U,
            best_epsilon,
        )

        idx = np.arange(
            rep[
                "y"
            ].shape[
                0
            ],
            dtype=int,
        )

        prepared = _prepare_subset(
            rep,
            idx,
        )

        fit = _evaluate_context(
            rep,
            prepared,
            A,
            B,
        )

        current_full = (
            rep[
                "S_F"
            ]
            / rep[
                "a_F"
            ]
        )

        current_null = (
            rep[
                "S_0"
            ]
            / rep[
                "a_0"
            ]
        )

        context_full_raw = (
            fit[
                "S_F_ctx"
            ]
            / rep[
                "a_F"
            ]
        )

        context_null_raw = (
            fit[
                "S_0_ctx"
            ]
            / rep[
                "a_0"
            ]
        )

        # ONE common split-level scalar only for plotting/comparison.
        # The flatness objective itself is scale invariant.
        current_median = float(
            np.nanmedian(
                current_full
            )
        )

        context_median = float(
            np.nanmedian(
                context_full_raw
            )
        )

        scale = (
            current_median
            / context_median
            if (
                np.isfinite(
                    current_median
                )
                and np.isfinite(
                    context_median
                )
                and context_median
                > 0
            )
            else 1.0
        )

        context_full = (
            scale
            * context_full_raw
        )

        context_null = (
            scale
            * context_null_raw
        )

        anchor_rows.append(
            {
                "replicate":
                    rep_id,

                "p_L":
                    p_L,

                "p_U":
                    p_U,

                "A":
                    A,

                "B":
                    B,

                "plot_scale":
                    scale,

                "nested_negative_fraction":
                    fit[
                        "nested_negative_fraction"
                    ],
            }
        )

        for b in sorted(
            np.unique(
                rep[
                    "coverage_bin"
                ]
            )
        ):
            use = (
                rep[
                    "coverage_bin"
                ]
                == b
            )

            detail_rows.append(
                {
                    "replicate":
                        rep_id,

                    "coverage_bin":
                        int(
                            b
                        ),

                    "median_log1p_total_normalized_coverage":
                        float(
                            np.median(
                                rep[
                                    "log_coverage"
                                ][
                                    use
                                ]
                            )
                        ),

                    "current_SF_over_aF":
                        float(
                            np.median(
                                current_full[
                                    use
                                ]
                            )
                        ),

                    "context_SF_over_aF":
                        float(
                            np.median(
                                context_full[
                                    use
                                ]
                            )
                        ),

                    "current_S0_over_a0":
                        float(
                            np.median(
                                current_null[
                                    use
                                ]
                            )
                        ),

                    "context_S0_over_a0":
                        float(
                            np.median(
                                context_null[
                                    use
                                ]
                            )
                        ),

                    "existing_raw_p_lt_0_01_fraction":
                        float(
                            np.mean(
                                np.isfinite(
                                    rep[
                                        "p_value"
                                    ][
                                        use
                                    ]
                                )
                                & (
                                    rep[
                                        "p_value"
                                    ][
                                        use
                                    ]
                                    < RAW_P_THRESHOLD
                                )
                            )
                        ),
                }
            )

        print(
            f"[winner] {j:02d}/{len(rep_data):02d} "
            f"| rep={rep_id:02d} "
            f"| pL={p_L:.4g} pU={p_U:.4g} "
            f"| B={B:.4g} "
            f"| nested-neg={fit['nested_negative_fraction']:.4f}",
            flush=True,
        )

    detail = pd.DataFrame(
        detail_rows
    )

    anchors = pd.DataFrame(
        anchor_rows
    )

    aggregate_rows = []

    for b, d in detail.groupby(
        "coverage_bin",
        sort=True,
    ):
        row = {
            "coverage_bin":
                int(
                    b
                ),

            "n_splits":
                int(
                    d[
                        "replicate"
                    ]
                    .nunique()
                ),
        }

        for col in (
            "median_log1p_total_normalized_coverage",
            "current_SF_over_aF",
            "context_SF_over_aF",
            "current_S0_over_a0",
            "context_S0_over_a0",
            "existing_raw_p_lt_0_01_fraction",
        ):
            row[
                f"{col}__median"
            ] = float(
                d[
                    col
                ]
                .median()
            )

            row[
                f"{col}__q25"
            ] = float(
                d[
                    col
                ]
                .quantile(
                    0.25
                )
            )

            row[
                f"{col}__q75"
            ] = float(
                d[
                    col
                ]
                .quantile(
                    0.75
                )
            )

        aggregate_rows.append(
            row
        )

    aggregate = pd.DataFrame(
        aggregate_rows
    )

    return (
        detail,
        aggregate,
        anchors,
    )


# =============================================================================
# PLOTS
# =============================================================================

def _plot_grid(
    grid_results,
    best_gamma,
):
    # Heatmap tau x epsilon for best gamma.
    x = grid_results.loc[
        grid_results[
            "weight_power"
        ]
        == float(
            best_gamma
        )
    ].copy()

    pivot = (
        x.pivot(
            index="tau",
            columns="epsilon",
            values="median_objective",
        )
        .sort_index()
    )

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    image = ax.imshow(
        pivot.to_numpy(
            dtype=float
        ),
        aspect="auto",
        origin="lower",
    )

    ax.set_xticks(
        np.arange(
            len(
                pivot.columns
            )
        )
    )

    ax.set_xticklabels(
        [
            f"{v:g}"
            for v in pivot.columns
        ]
    )

    ax.set_yticks(
        np.arange(
            len(
                pivot.index
            )
        )
    )

    ax.set_yticklabels(
        [
            f"{v:g}"
            for v in pivot.index
        ]
    )

    ax.set_xlabel(
        r"$\epsilon$",
        fontsize=14,
    )

    ax.set_ylabel(
        r"Anchor quantile $\tau$",
        fontsize=14,
    )

    ax.set_title(
        rf"Pickrell weighted-anchor geometry objective at best "
        rf"$\gamma={best_gamma:g}$"
        "\n(lower is flatter; p-values not used)",
        fontsize=18,
        pad=16,
    )

    plt.colorbar(
        image,
        ax=ax,
        label="Median geometry objective",
    )

    plt.tight_layout()
    plt.show()

    # Best objective by gamma.
    gamma_best = (
        grid_results
        .sort_values(
            "median_objective",
            kind="stable",
        )
        .groupby(
            "weight_power",
            as_index=False,
            sort=True,
        )
        .first()
    )

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    ax.plot(
        gamma_best[
            "weight_power"
        ],
        gamma_best[
            "median_objective"
        ],
        marker="o",
        linewidth=2.4,
    )

    ax.set_xlabel(
        r"Information weight power $\gamma$",
        fontsize=14,
    )

    ax.set_ylabel(
        "Best median geometry objective",
        fontsize=14,
    )

    ax.set_title(
        "Pickrell: Best logit-affine context at each information-weight power",
        fontsize=18,
        pad=16,
    )

    ax.grid(
        alpha=0.20,
    )

    plt.tight_layout()
    plt.show()


def _plot_band(
    ax,
    x,
    aggregate,
    metric,
    label,
    *,
    linestyle="-",
    marker="o",
):
    ax.plot(
        x,
        aggregate[
            f"{metric}__median"
        ],
        marker=marker,
        linewidth=2.4,
        linestyle=linestyle,
        label=label,
    )

    ax.fill_between(
        x,
        aggregate[
            f"{metric}__q25"
        ],
        aggregate[
            f"{metric}__q75"
        ],
        alpha=0.14,
    )


def _plot_winner_curves(
    aggregate,
    best_gamma,
    best_tau,
    best_epsilon,
):
    x = aggregate[
        "median_log1p_total_normalized_coverage__median"
    ].to_numpy(
        dtype=float
    )

    # Full.
    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    _plot_band(
        ax,
        x,
        aggregate,
        "current_SF_over_aF",
        r"Current $S_F/a_F$",
    )

    _plot_band(
        ax,
        x,
        aggregate,
        "context_SF_over_aF",
        r"Logit-affine context $S_F/a_F$",
        linestyle="--",
        marker="s",
    )

    ax.set_xlabel(
        "log1p(total normalized gene coverage)",
        fontsize=14,
    )

    ax.set_ylabel(
        r"Median $S_F/a_F$",
        fontsize=14,
    )

    ax.set_title(
        rf"Pickrell full geometry — current vs weighted anchors "
        rf"($\gamma={best_gamma:g}$, $\tau={best_tau:g}$, "
        rf"$\epsilon={best_epsilon:g}$)",
        fontsize=18,
        pad=16,
    )

    ax.grid(
        alpha=0.20,
    )

    ax.legend(
        fontsize=11,
    )

    plt.tight_layout()
    plt.show()

    # Null.
    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    _plot_band(
        ax,
        x,
        aggregate,
        "current_S0_over_a0",
        r"Current $S_0/a_0$",
    )

    _plot_band(
        ax,
        x,
        aggregate,
        "context_S0_over_a0",
        r"Logit-affine context $S_0/a_0$",
        linestyle="--",
        marker="s",
    )

    ax.set_xlabel(
        "log1p(total normalized gene coverage)",
        fontsize=14,
    )

    ax.set_ylabel(
        r"Median $S_0/a_0$",
        fontsize=14,
    )

    ax.set_title(
        rf"Pickrell null geometry — current vs weighted anchors "
        rf"($\gamma={best_gamma:g}$, $\tau={best_tau:g}$, "
        rf"$\epsilon={best_epsilon:g}$)",
        fontsize=18,
        pad=16,
    )

    ax.grid(
        alpha=0.20,
    )

    ax.legend(
        fontsize=11,
    )

    plt.tight_layout()
    plt.show()

    # Overlay existing M03 FPR hump AFTER selection.
    fig, ax1 = plt.subplots(
        figsize=FIGSIZE
    )

    ax2 = ax1.twinx()

    ax1.plot(
        x,
        aggregate[
            "context_SF_over_aF__median"
        ],
        marker="s",
        linewidth=2.3,
        linestyle="--",
        label=r"Context $S_F/a_F$",
    )

    ax1.plot(
        x,
        aggregate[
            "context_S0_over_a0__median"
        ],
        marker="^",
        linewidth=2.3,
        linestyle="-.",
        label=r"Context $S_0/a_0$",
    )

    ax2.plot(
        x,
        aggregate[
            "existing_raw_p_lt_0_01_fraction__median"
        ],
        marker="o",
        linewidth=2.2,
        label="Existing M03 raw p<.01",
    )

    ax2.axhline(
        RAW_P_THRESHOLD,
        linestyle=":",
        linewidth=1.4,
    )

    ax1.set_xlabel(
        "log1p(total normalized gene coverage)",
        fontsize=14,
    )

    ax1.set_ylabel(
        "Geometry-optimal contextual deviance / shape",
        fontsize=14,
    )

    ax2.set_ylabel(
        "Existing fraction raw p < .01",
        fontsize=14,
    )

    ax1.set_title(
        "Pickrell: Geometry-optimal logit-affine context vs existing FPR hump",
        fontsize=18,
        pad=16,
    )

    ax1.grid(
        alpha=0.20,
    )

    lines1, labels1 = (
        ax1.get_legend_handles_labels()
    )

    lines2, labels2 = (
        ax2.get_legend_handles_labels()
    )

    ax1.legend(
        lines1 + lines2,
        labels1 + labels2,
        fontsize=11,
        loc="best",
    )

    plt.tight_layout()
    plt.show()


def _plot_mapping(
    anchors,
    best_gamma,
    best_tau,
    best_epsilon,
):
    # Median cross-fit mapping across splits.
    p_L = float(
        anchors[
            "p_L"
        ]
        .median()
    )

    p_U = float(
        anchors[
            "p_U"
        ]
        .median()
    )

    A = float(
        anchors[
            "A"
        ]
        .median()
    )

    B = float(
        anchors[
            "B"
        ]
        .median()
    )

    z_grid = np.linspace(
        REFERENCE_LOGIT_MIN,
        REFERENCE_LOGIT_MAX,
        1000,
    )

    p_grid = expit(
        z_grid
    )

    u_grid = expit(
        A
        + B
        * z_grid
    )

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    ax.plot(
        p_grid,
        u_grid,
        linewidth=2.5,
        label=r"$u(p)$",
    )

    ax.plot(
        p_grid,
        p_grid,
        linestyle="--",
        linewidth=1.5,
        label="Identity",
    )

    ax.scatter(
        [
            p_L,
            p_U,
        ],
        [
            best_epsilon,
            1.0
            - best_epsilon,
        ],
        s=80,
        label="Weighted reference anchors",
    )

    ax.set_xscale(
        "log"
    )

    ax.set_xlabel(
        r"Original effective proportion $p=\mu/(\mu+r)$ (log scale)",
        fontsize=14,
    )

    ax.set_ylabel(
        r"Contextual coordinate $u(p)$",
        fontsize=14,
    )

    ax.set_title(
        rf"Median held-out logit-affine context "
        rf"($\gamma={best_gamma:g}$, $\tau={best_tau:g}$, "
        rf"$\epsilon={best_epsilon:g}$)",
        fontsize=18,
        pad=16,
    )

    ax.grid(
        alpha=0.20,
        which="both",
    )

    ax.legend(
        fontsize=11,
    )

    plt.tight_layout()
    plt.show()

    # Curvature profile.
    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    ax.plot(
        p_grid,
        p_grid
        * (
            1.0
            - p_grid
        ),
        linewidth=2.4,
        label=r"Original $p(1-p)$",
    )

    ax.plot(
        p_grid,
        u_grid
        * (
            1.0
            - u_grid
        ),
        linewidth=2.4,
        label=r"Context $u(1-u)$",
    )

    ax.set_xscale(
        "log"
    )

    ax.set_xlabel(
        r"Original effective proportion $p$ (log scale)",
        fontsize=14,
    )

    ax.set_ylabel(
        "Variation curvature",
        fontsize=14,
    )

    ax.set_title(
        "Original vs logit-affine contextual binomial curvature",
        fontsize=18,
        pad=16,
    )

    ax.grid(
        alpha=0.20,
        which="both",
    )

    ax.legend(
        fontsize=11,
    )

    plt.tight_layout()
    plt.show()


# =============================================================================
# TOP-LEVEL
# =============================================================================

def run_pickrell_logit_affine_weighted_context():
    started = (
        datetime.now()
        .astimezone()
    )

    print(
        f"[start] {_now()}",
        flush=True,
    )

    try:
        reps = (
            _discover_completed_replicates()
        )

        print(
            "\nPICKRELL LOGIT-AFFINE WEIGHTED VARIATION-CONTEXT DIAGNOSTIC\n"
            f"namespace:        {CURRENT_M03_NAMESPACE}\n"
            f"completed splits: {len(reps)}\n"
            f"gamma grid:       {WEIGHT_POWER_GRID}\n"
            f"tau grid:         {TAU_GRID}\n"
            f"epsilon grid:     {EPSILON_GRID}\n"
            f"reference bins:   {REFERENCE_N_BINS:,}\n"
            f"grid max genes:   {GRID_SEARCH_MAX_GENES_PER_SPLIT:,}/split\n"
            "selection:        residual geometry ONLY\n",
            flush=True,
        )

        # ---------------------------------------------------------
        # Load all frozen splits.
        # ---------------------------------------------------------
        rep_data = []

        for j, rep_id in enumerate(
            reps,
            start=1,
        ):
            rep = _load_rep_data(
                rep_id
            )

            rep_data.append(
                rep
            )

            print(
                f"[load] {j:02d}/{len(reps):02d} "
                f"| rep={rep_id:02d} "
                f"| genes={rep['y'].shape[0]:,}",
                flush=True,
            )

        # ---------------------------------------------------------
        # Algebraic bridge check.
        # ---------------------------------------------------------
        identity = _audit_effective_binomial_identity(
            rep_data[
                0
            ]
        )

        print(
            "\nEFFECTIVE-BINOMIAL / NB HALF-DEVIANCE IDENTITY AUDIT\n",
            flush=True,
        )

        display(
            pd.DataFrame(
                [
                    identity
                ]
            )
        )

        if (
            not np.isfinite(
                identity[
                    "max_abs_error"
                ]
            )
            or identity[
                "max_abs_error"
            ]
            > 1e-7
        ):
            raise RuntimeError(
                "Effective-binomial/NB deviance identity check failed."
            )

        # ---------------------------------------------------------
        # Weighted cross-fit reference distributions.
        # ---------------------------------------------------------
        print(
            "\n[reference] building observation-level weighted reference histograms...",
            flush=True,
        )

        reference_histograms = (
            _build_reference_histograms(
                rep_data
            )
        )

        # ---------------------------------------------------------
        # Geometry-only parameter search.
        # ---------------------------------------------------------
        (
            grid_results,
            subset_by_rep,
            baseline,
        ) = _grid_search(
            rep_data,
            reference_histograms,
        )

        best = (
            grid_results
            .iloc[
                0
            ]
        )

        best_gamma = float(
            best[
                "weight_power"
            ]
        )

        best_tau = float(
            best[
                "tau"
            ]
        )

        best_epsilon = float(
            best[
                "epsilon"
            ]
        )

        print(
            "\nSELECTED CONTEXT — GEOMETRY ONLY\n",
            flush=True,
        )

        display(
            pd.DataFrame(
                [
                    best.to_dict()
                ]
            )
        )

        if best_gamma in {
            min(
                WEIGHT_POWER_GRID
            ),
            max(
                WEIGHT_POWER_GRID
            ),
        }:
            print(
                "[note] best gamma is at a supplied-grid boundary.",
                flush=True,
            )

        if best_tau in {
            min(
                TAU_GRID
            ),
            max(
                TAU_GRID
            ),
        }:
            print(
                "[note] best tau is at a supplied-grid boundary.",
                flush=True,
            )

        if best_epsilon in {
            min(
                EPSILON_GRID
            ),
            max(
                EPSILON_GRID
            ),
        }:
            print(
                "[note] best epsilon is at a supplied-grid boundary.",
                flush=True,
            )

        # ---------------------------------------------------------
        # Full-gene reevaluation of winner.
        # ---------------------------------------------------------
        (
            winner_detail,
            winner_aggregate,
            winner_anchors,
        ) = _evaluate_winner_all_genes(
            rep_data,
            reference_histograms,
            best_gamma,
            best_tau,
            best_epsilon,
        )

        print(
            "\nWINNER — HELD-OUT ANCHORS BY SPLIT\n",
            flush=True,
        )

        display(
            winner_anchors
        )

        print(
            "\nWINNER — FULL-GENE COVERAGE CURVES\n",
            flush=True,
        )

        display(
            winner_aggregate[
                [
                    "coverage_bin",
                    "n_splits",
                    "median_log1p_total_normalized_coverage__median",
                    "current_SF_over_aF__median",
                    "context_SF_over_aF__median",
                    "current_S0_over_a0__median",
                    "context_S0_over_a0__median",
                    "existing_raw_p_lt_0_01_fraction__median",
                ]
            ]
        )

        # ---------------------------------------------------------
        # Display plots only.
        # ---------------------------------------------------------
        _plot_grid(
            grid_results,
            best_gamma,
        )

        _plot_winner_curves(
            winner_aggregate,
            best_gamma,
            best_tau,
            best_epsilon,
        )

        _plot_mapping(
            winner_anchors,
            best_gamma,
            best_tau,
            best_epsilon,
        )

        ended = (
            datetime.now()
            .astimezone()
        )

        print(
            "\nSTATUS\n"
            f"selected gamma:   {best_gamma:g}\n"
            f"selected tau:     {best_tau:g}\n"
            f"selected epsilon: {best_epsilon:g}\n"
            f"[end] {_now()} | elapsed={ended-started}",
            flush=True,
        )

        return {
            "grid_results":
                grid_results,

            "baseline":
                baseline,

            "best_gamma":
                best_gamma,

            "best_tau":
                best_tau,

            "best_epsilon":
                best_epsilon,

            "winner_detail":
                winner_detail,

            "winner_aggregate":
                winner_aggregate,

            "winner_anchors":
                winner_anchors,

            "reference_histograms":
                reference_histograms,
        }

    finally:
        pass


# =============================================================================
# EXECUTION
# =============================================================================

pickrell_logit_affine_weighted_context = (
    run_pickrell_logit_affine_weighted_context()
)


In [ ]:
from __future__ import annotations

"""
Pickrell universal DEB-shrinkage-strength diagnostic.

Core idea
---------
The raw first-order precision estimate already contains strong coverage
curvature:

    theta_raw = a / R.

Current DEB uses a Gamma prior with fitted hyperparameters alpha0, beta0:

    theta_DEB = (a + alpha0) / (R + beta0),

where the prior mean is

    m = alpha0 / beta0.

Write

    s_current = alpha0 / (a + alpha0).

Then, before numerical clipping,

    1/theta_DEB
      = (1 - s_current) * 1/theta_raw
        + s_current * 1/m.

So s is literally the shrinkage fraction on the reciprocal-precision scale.

This diagnostic replaces the fitted near-1 shrinkage fraction with a UNIVERSAL
user-controlled value s in [0,1):

    1/theta_s
      = (1-s) * 1/theta_raw
        + s * 1/m,

or equivalently

    nu_s = a*s/(1-s)

    theta_s
      = (a + nu_s) / (R + nu_s/m).

The prior TARGET m is kept exactly as fitted by the current DEB model.
Only prior STRENGTH is changed.

What is frozen
--------------
Everything upstream remains current M03:

    - r_g / alpha_g
    - fitted means
    - C_F and C_0
    - global kappa_c / lambda_c
    - S0dag_final
    - support-floor correction
    - mandatory exact branch

Only theta_F and theta_0 for ordinary first-order genes are recomputed at each
universal shrinkage strength s.

For mandatory exact genes (R_F <= 0), the exact normalized-DNB profile is
independent of DEB shrinkage, so their canonical current M03 p-values are reused.

This is a diagnostic development experiment, not final tuning.

Outputs
-------
Numerical results/checkpoints are saved under repo ./data so the run is
resumable. Figures are DISPLAYED ONLY and are not saved.

Main diagnostics:
    1. current fitted DEB shrinkage fractions s_F and s_0 by split
    2. raw p<.01 mean/median vs universal shrinkage s
    3. complete-null BH FDR = P(any rejection) vs s
    4. mean BH(.05) discoveries vs s
    5. raw-p calibration at .001, .005, .01, .025, .05 vs s
    6. coverage-bin x shrinkage heatmap for raw p<.01
    7. raw p<.01 by coverage for selected s values
    8. theta_F by coverage for selected s values
    9. theta_0 by coverage for selected s values
   10. diagnostic best s based on raw-p calibration, clearly labeled
       development-only / same-data.

Nothing reruns M03 itself.
"""

from datetime import datetime
from pathlib import Path
from scipy.stats import chi2

import json
import os
import pickle
import tempfile
import time
import traceback

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


# =============================================================================
# SETTINGS
# =============================================================================

CURRENT_M03_NAMESPACE = "pickrell_m03_seam_safe_20M_kappa_coverage_v1"
OUTPUT_NAMESPACE = "pickrell_m03_tempered_deb_shrinkage_v1"

N_COVERAGE_BINS = 12

SHRINKAGE_GRID = (
    0.00,
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
    0.925,
    0.95,
    0.975,
    0.99,
    0.995,
    0.999,
)

# Curves shown on the coverage plots.
SELECTED_S_FOR_PLOTS = (
    0.00,
    0.25,
    0.50,
    0.75,
    0.90,
    0.99,
    0.999,
)

CALIBRATION_THRESHOLDS = (
    0.001,
    0.005,
    0.01,
    0.025,
    0.05,
)

RAW_P_THRESHOLD = 0.01
FDR_CUTOFF = 0.05

RESUME = True
SHOW_PLOTS = True

FIGSIZE = (16, 9)


# =============================================================================
# PATHS / TIME
# =============================================================================

def _now() -> str:
    return datetime.now().astimezone().strftime(
        "%Y-%m-%d %I:%M:%S %p %Z"
    )


CWD = Path.cwd().resolve()

if CWD.name.lower() in {"notebook", "notebooks"}:
    PROJECT_ROOT = CWD.parent
elif (CWD / "data").is_dir():
    PROJECT_ROOT = CWD
else:
    raise RuntimeError(
        "Could not resolve repository root. Run from <repo>/notebook, "
        "<repo>/notebooks, or the repository root containing ./data. "
        f"Current working directory: {CWD}"
    )

PROJECT_DATA_DIR = PROJECT_ROOT / "data"

GLOBAL_RUN_DIR = (
    PROJECT_DATA_DIR
    / CURRENT_M03_NAMESPACE
)

GLOBAL_CELL_DIR = (
    GLOBAL_RUN_DIR
    / "cells"
)

M03_CHECKPOINT_ROOT = (
    GLOBAL_RUN_DIR
    / "m03_checkpoints"
)

OUTPUT_DIR = (
    PROJECT_DATA_DIR
    / OUTPUT_NAMESPACE
)

SPLIT_CACHE_DIR = (
    OUTPUT_DIR
    / "per_split"
)

SENTINEL = (
    OUTPUT_DIR
    / "_tempered_deb_shrinkage_v1_initialized.json"
)


# =============================================================================
# FILE HELPERS
# =============================================================================

def _load_pickle(path: Path):
    with open(path, "rb") as f:
        return pickle.load(f)


def _atomic_pickle(obj, path: Path):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    fd, tmp_name = tempfile.mkstemp(
        prefix=f".{path.name}.",
        dir=str(path.parent),
    )
    os.close(fd)

    tmp = Path(tmp_name)

    try:
        with open(tmp, "wb") as f:
            pickle.dump(
                obj,
                f,
                protocol=pickle.HIGHEST_PROTOCOL,
            )
            f.flush()

            try:
                os.fsync(f.fileno())
            except OSError:
                pass

        os.replace(tmp, path)

    finally:
        if tmp.exists():
            try:
                tmp.unlink()
            except OSError:
                pass


def _atomic_csv(df: pd.DataFrame, path: Path):
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    fd, tmp_name = tempfile.mkstemp(
        prefix=f".{path.name}.",
        dir=str(path.parent),
    )
    os.close(fd)

    tmp = Path(tmp_name)

    try:
        df.to_csv(
            tmp,
            index=False,
        )

        os.replace(
            tmp,
            path,
        )

    finally:
        if tmp.exists():
            try:
                tmp.unlink()
            except OSError:
                pass


def _checkpoint_dir(replicate: int) -> Path:
    return (
        M03_CHECKPOINT_ROOT
        / f"pickrell_rep{int(replicate):02d}"
    )


def _load_success_stage(
    replicate: int,
    stage_name: str,
):
    path = (
        _checkpoint_dir(replicate)
        / "stages"
        / f"{stage_name}.pkl"
    )

    if not path.exists():
        raise FileNotFoundError(
            f"Missing checkpoint: {path}"
        )

    payload = _load_pickle(path)

    if (
        not isinstance(payload, dict)
        or payload.get("status") != "success"
    ):
        raise RuntimeError(
            f"Replicate {replicate}: "
            f"{stage_name} is not a successful checkpoint."
        )

    return payload["data"]


def _load_manifest(replicate: int):
    path = (
        _checkpoint_dir(replicate)
        / "m03_manifest.pkl"
    )

    if not path.exists():
        raise FileNotFoundError(
            f"Missing manifest: {path}"
        )

    return _load_pickle(path)


def _discover_completed_replicates():
    reps = []

    for path in sorted(
        GLOBAL_CELL_DIR.glob("pickrell_rep*.csv")
    ):
        try:
            x = pd.read_csv(path)
        except Exception:
            continue

        if len(x) != 1:
            continue

        if str(
            x.loc[0, "status"]
        ).strip().lower() != "success":
            continue

        reps.append(
            int(
                x.loc[0, "replicate"]
            )
        )

    reps = sorted(set(reps))

    if not reps:
        raise RuntimeError(
            "No completed Pickrell replicates were found."
        )

    return reps


# =============================================================================
# NUMERIC HELPERS
# =============================================================================

def _bh_adjust(p):
    p = np.asarray(
        p,
        dtype=float,
    )

    q = np.full_like(
        p,
        np.nan,
    )

    ok = (
        np.isfinite(p)
        & (p >= 0)
        & (p <= 1)
    )

    if not np.any(ok):
        return q

    p0 = p[ok]

    order = np.argsort(
        p0,
        kind="stable",
    )

    ranked = p0[order]

    n = len(ranked)

    adjusted = (
        ranked
        * n
        / np.arange(
            1,
            n + 1,
        )
    )

    adjusted = np.minimum.accumulate(
        adjusted[::-1]
    )[::-1]

    adjusted = np.minimum(
        adjusted,
        1.0,
    )

    restored = np.empty_like(
        adjusted
    )

    restored[order] = adjusted

    q[ok] = restored

    return q


def _med(x):
    x = pd.to_numeric(
        x,
        errors="coerce",
    ).to_numpy(
        dtype=float
    )

    x = x[
        np.isfinite(x)
    ]

    return (
        float(np.median(x))
        if len(x)
        else np.nan
    )


def _q(x, q):
    x = pd.to_numeric(
        x,
        errors="coerce",
    ).to_numpy(
        dtype=float
    )

    x = x[
        np.isfinite(x)
    ]

    return (
        float(
            np.quantile(
                x,
                q,
            )
        )
        if len(x)
        else np.nan
    )


def _raw_theta(a, R):
    R = np.asarray(
        R,
        dtype=float,
    )

    out = np.full(
        len(R),
        np.nan,
        dtype=float,
    )

    ok = (
        np.isfinite(R)
        & (R > 0)
    )

    out[ok] = (
        float(a)
        / R[ok]
    )

    return out


def _tempered_theta(
    raw_theta,
    prior_mean,
    s,
    theta_floor,
    theta_cap,
):
    """
    Universal shrinkage fraction s on reciprocal-precision scale:

        1/theta_s
          = (1-s)/theta_raw + s/prior_mean.

    Equivalent to Gamma prior strength:

        nu = a*s/(1-s).

    Since raw_theta=a/R, 'a' cancels from this direct implementation.
    """

    raw_theta = np.asarray(
        raw_theta,
        dtype=float,
    )

    prior_mean = float(
        prior_mean
    )

    s = float(
        s
    )

    out = np.full(
        len(raw_theta),
        np.nan,
        dtype=float,
    )

    if not (
        0.0
        <= s
        < 1.0
    ):
        raise ValueError(
            f"s must be in [0,1); got {s}."
        )

    ok = (
        np.isfinite(raw_theta)
        & (raw_theta > 0)
        & np.isfinite(prior_mean)
        & (prior_mean > 0)
    )

    if s == 0.0:
        theta = raw_theta[ok]
    else:
        reciprocal = (
            (1.0 - s)
            / raw_theta[ok]
            + s
            / prior_mean
        )

        theta = (
            1.0
            / reciprocal
        )

    out[ok] = np.clip(
        theta,
        float(theta_floor),
        float(theta_cap),
    )

    return out


def _normalized_statistic(
    S_F,
    S_0_used,
    C_F,
    C_0,
    theta_F,
    theta_0,
    n,
    settings,
):
    corrected_score_tolerance = float(
        settings[
            "corrected_score_tolerance"
        ]
    )

    R_F = (
        S_F
        - C_F
    )

    R_0 = (
        S_0_used
        - C_0
    )

    ok = (
        np.isfinite(R_F)
        & np.isfinite(R_0)
        & np.isfinite(theta_F)
        & np.isfinite(theta_0)
        & (R_F > corrected_score_tolerance)
        & (R_0 > corrected_score_tolerance)
        & (theta_F > 0)
        & (theta_0 > 0)
    )

    T_raw = np.full(
        len(S_F),
        np.nan,
    )

    T_raw[ok] = (
        2.0
        * (
            theta_0[ok]
            * R_0[ok]
            - theta_F[ok]
            * R_F[ok]
        )
        + float(n)
        * np.log(
            theta_F[ok]
            / theta_0[ok]
        )
        + 2.0
        * (
            C_0[ok]
            - C_F[ok]
        )
    )

    T = np.full(
        len(S_F),
        np.nan,
    )

    T[ok] = np.maximum(
        T_raw[ok],
        0.0,
    )

    p = np.full(
        len(S_F),
        np.nan,
    )

    p[ok] = chi2.sf(
        T[ok],
        1,
    )

    return {
        "valid_mask":
            ok,

        "T_raw":
            T_raw,

        "T":
            T,

        "p":
            p,

        "R_F":
            R_F,

        "R_0":
            R_0,
    }


# =============================================================================
# ONE SPLIT
# =============================================================================

def _run_one_replicate(
    replicate: int,
):
    cache_path = (
        SPLIT_CACHE_DIR
        / f"pickrell_rep{int(replicate):02d}.pkl"
    )

    if (
        RESUME
        and cache_path.exists()
    ):
        try:
            payload = _load_pickle(
                cache_path
            )

            if (
                isinstance(payload, dict)
                and payload.get("version")
                == "tempered-deb-shrinkage-v1"
            ):
                print(
                    f"[reload] {_now()} | rep={replicate:02d}",
                    flush=True,
                )

                return payload, True

        except Exception:
            pass

    t0 = time.perf_counter()

    core = _load_success_stage(
        replicate,
        "01_core",
    )

    full = _load_success_stage(
        replicate,
        "02_full_firstorder",
    )

    geom = _load_success_stage(
        replicate,
        "03_geometry_support",
    )

    null = _load_success_stage(
        replicate,
        "04_null_deb",
    )

    base = _load_success_stage(
        replicate,
        "05_base_m03_result",
    )

    manifest = _load_manifest(
        replicate
    )

    settings = dict(
        manifest[
            "numerical_settings"
        ]
    )

    theta_floor = float(
        settings[
            "theta_floor"
        ]
    )

    theta_cap = float(
        settings[
            "theta_cap"
        ]
    )

    y = np.asarray(
        core[
            "y"
        ],
        dtype=float,
    )

    exposure = np.asarray(
        core[
            "exposure"
        ],
        dtype=float,
    )

    S_F = np.asarray(
        core[
            "S_F"
        ],
        dtype=float,
    )

    a_F = float(
        core[
            "a_F"
        ]
    )

    a_0 = float(
        core[
            "a_0"
        ]
    )

    n = int(
        core[
            "n"
        ]
    )

    C_F = np.asarray(
        full[
            "C_F"
        ],
        dtype=float,
    )

    C_0 = np.asarray(
        full[
            "C_0"
        ],
        dtype=float,
    )

    R_F = np.asarray(
        full[
            "R_F"
        ],
        dtype=float,
    )

    theta_F_current = np.asarray(
        full[
            "theta_F_first"
        ],
        dtype=float,
    )

    full_theta_fit = dict(
        full[
            "full_theta_fit"
        ]
    )

    mask_Rnonpos = np.asarray(
        full[
            "mask_Rnonpos"
        ],
        dtype=bool,
    )

    S0dag = np.asarray(
        geom[
            "S0dag_final"
        ],
        dtype=float,
    )

    R0dag = np.asarray(
        null[
            "R_0_dagger_first"
        ],
        dtype=float,
    )

    theta_0_current = np.asarray(
        null[
            "theta_0_first"
        ],
        dtype=float,
    )

    null_theta_fit = dict(
        null[
            "null_theta_fit"
        ]
    )

    p_current = np.asarray(
        base[
            "p_value_m03"
        ],
        dtype=float,
    )

    q_current = np.asarray(
        base[
            "q_value_m03"
        ],
        dtype=float,
    )

    n_genes = int(
        y.shape[0]
    )

    # -------------------------------------------------------------
    # Current prior targets and exact current shrinkage fractions.
    # -------------------------------------------------------------

    alphaF = float(
        full_theta_fit[
            "alpha0"
        ]
    )

    betaF = float(
        full_theta_fit[
            "beta0"
        ]
    )

    mF = float(
        full_theta_fit[
            "prior_mean_theta"
        ]
    )

    alpha0 = float(
        null_theta_fit[
            "alpha0"
        ]
    )

    beta0 = float(
        null_theta_fit[
            "beta0"
        ]
    )

    m0 = float(
        null_theta_fit[
            "prior_mean_theta"
        ]
    )

    # Identity: m = alpha/beta.
    if abs(
        alphaF / betaF
        - mF
    ) / max(
        abs(mF),
        1e-15,
    ) > 1e-10:
        raise RuntimeError(
            f"Replicate {replicate}: "
            "full prior mean does not equal alpha0/beta0."
        )

    if abs(
        alpha0 / beta0
        - m0
    ) / max(
        abs(m0),
        1e-15,
    ) > 1e-10:
        raise RuntimeError(
            f"Replicate {replicate}: "
            "null prior mean does not equal alpha0/beta0."
        )

    s_current_F = (
        alphaF
        / (
            a_F
            + alphaF
        )
    )

    s_current_0 = (
        alpha0
        / (
            a_0
            + alpha0
        )
    )

    raw_theta_F = _raw_theta(
        a_F,
        R_F,
    )

    raw_theta_0 = _raw_theta(
        a_0,
        R0dag,
    )

    # -------------------------------------------------------------
    # Verify current DEB posterior equals this reciprocal-scale
    # shrinkage representation for ordinary genes.
    # -------------------------------------------------------------

    current_reconstructed_F = _tempered_theta(
        raw_theta_F,
        mF,
        s_current_F,
        theta_floor,
        theta_cap,
    )

    current_reconstructed_0 = _tempered_theta(
        raw_theta_0,
        m0,
        s_current_0,
        theta_floor,
        theta_cap,
    )

    ordinary_F = (
        np.isfinite(
            theta_F_current
        )
        & np.isfinite(
            current_reconstructed_F
        )
    )

    ordinary_0 = (
        np.isfinite(
            theta_0_current
        )
        & np.isfinite(
            current_reconstructed_0
        )
    )

    if np.any(
        ordinary_F
    ):
        maxerrF = float(
            np.max(
                np.abs(
                    theta_F_current[
                        ordinary_F
                    ]
                    - current_reconstructed_F[
                        ordinary_F
                    ]
                )
            )
        )

        if maxerrF > 1e-10:
            raise RuntimeError(
                f"Replicate {replicate}: "
                f"full DEB shrinkage identity failed; max abs error={maxerrF:.3e}"
            )

    if np.any(
        ordinary_0
    ):
        maxerr0 = float(
            np.max(
                np.abs(
                    theta_0_current[
                        ordinary_0
                    ]
                    - current_reconstructed_0[
                        ordinary_0
                    ]
                )
            )
        )

        if maxerr0 > 1e-10:
            raise RuntimeError(
                f"Replicate {replicate}: "
                f"null DEB shrinkage identity failed; max abs error={maxerr0:.3e}"
            )

    # -------------------------------------------------------------
    # Coverage bins.
    # -------------------------------------------------------------

    total_normalized_coverage = np.sum(
        y
        / exposure[
            None,
            :
        ],
        axis=1,
    )

    log_coverage = np.log1p(
        total_normalized_coverage
    )

    coverage_bin = (
        np.asarray(
            pd.qcut(
                pd.Series(
                    log_coverage
                ),
                q=N_COVERAGE_BINS,
                labels=False,
                duplicates="drop",
            ),
            dtype=int,
        )
        + 1
    )

    actual_bins = sorted(
        int(x)
        for x in np.unique(
            coverage_bin
        )
    )

    # -------------------------------------------------------------
    # Current row first.
    # -------------------------------------------------------------

    split_rows = []
    bin_rows = []
    threshold_rows = []

    def score_one(
        label,
        s_numeric,
        thetaF,
        theta0,
        p_values,
    ):
        q_values = _bh_adjust(
            p_values
        )

        finite = np.isfinite(
            p_values
        )

        split_rows.append(
            {
                "replicate":
                    int(
                        replicate
                    ),

                "method":
                    label,

                "s":
                    s_numeric,

                "n_genes":
                    int(
                        n_genes
                    ),

                "n_finite_p":
                    int(
                        np.sum(
                            finite
                        )
                    ),

                "fraction_p_lt_0_01_finite":
                    float(
                        np.mean(
                            p_values[
                                finite
                            ]
                            < RAW_P_THRESHOLD
                        )
                    )
                    if np.any(
                        finite
                    )
                    else np.nan,

                "BH05_discoveries":
                    int(
                        np.sum(
                            np.isfinite(
                                q_values
                            )
                            & (
                                q_values
                                < FDR_CUTOFF
                            )
                        )
                    ),

                "any_BH05":
                    int(
                        np.any(
                            np.isfinite(
                                q_values
                            )
                            & (
                                q_values
                                < FDR_CUTOFF
                            )
                        )
                    ),

                "median_p":
                    float(
                        np.nanmedian(
                            p_values
                        )
                    ),

                "median_theta_F":
                    float(
                        np.nanmedian(
                            thetaF
                        )
                    ),

                "median_theta_0":
                    float(
                        np.nanmedian(
                            theta0
                        )
                    ),
            }
        )

        for threshold in CALIBRATION_THRESHOLDS:
            threshold_rows.append(
                {
                    "replicate":
                        int(
                            replicate
                        ),

                    "method":
                        label,

                    "s":
                        s_numeric,

                    "threshold":
                        float(
                            threshold
                        ),

                    "fraction_le_threshold":
                        float(
                            np.mean(
                                p_values[
                                    finite
                                ]
                                <= float(
                                    threshold
                                )
                            )
                        )
                        if np.any(
                            finite
                        )
                        else np.nan,
                }
            )

        for b in actual_bins:
            idx = (
                coverage_bin
                == b
            )

            n_bin = int(
                np.sum(
                    idx
                )
            )

            bin_rows.append(
                {
                    "replicate":
                        int(
                            replicate
                        ),

                    "method":
                        label,

                    "s":
                        s_numeric,

                    "coverage_bin":
                        int(
                            b
                        ),

                    "n_genes_bin":
                        n_bin,

                    "median_log1p_total_normalized_coverage":
                        float(
                            np.median(
                                log_coverage[
                                    idx
                                ]
                            )
                        ),

                    "median_theta_F":
                        float(
                            np.nanmedian(
                                thetaF[
                                    idx
                                ]
                            )
                        ),

                    "median_theta_0":
                        float(
                            np.nanmedian(
                                theta0[
                                    idx
                                ]
                            )
                        ),

                    "raw_p_lt_0_01_fraction":
                        float(
                            np.sum(
                                idx
                                & np.isfinite(
                                    p_values
                                )
                                & (
                                    p_values
                                    < RAW_P_THRESHOLD
                                )
                            )
                            / max(
                                n_bin,
                                1,
                            )
                        ),
                }
            )

        return q_values

    score_one(
        "current_DEB",
        np.nan,
        theta_F_current,
        theta_0_current,
        p_current,
    )

    # -------------------------------------------------------------
    # Universal shrinkage-strength grid.
    # -------------------------------------------------------------

    for s in SHRINKAGE_GRID:
        s = float(
            s
        )

        thetaF = _tempered_theta(
            raw_theta_F,
            mF,
            s,
            theta_floor,
            theta_cap,
        )

        theta0 = _tempered_theta(
            raw_theta_0,
            m0,
            s,
            theta_floor,
            theta_cap,
        )

        test = _normalized_statistic(
            S_F,
            S0dag,
            C_F,
            C_0,
            thetaF,
            theta0,
            n,
            settings,
        )

        p = np.asarray(
            test[
                "p"
            ],
            dtype=float,
        ).copy()

        # Mandatory exact profile is independent of DEB shrinkage.
        p[
            mask_Rnonpos
        ] = p_current[
            mask_Rnonpos
        ]

        score_one(
            f"s={s:g}",
            s,
            thetaF,
            theta0,
            p,
        )

    payload = {
        "version":
            "tempered-deb-shrinkage-v1",

        "replicate":
            int(
                replicate
            ),

        "s_current_F":
            float(
                s_current_F
            ),

        "s_current_0":
            float(
                s_current_0
            ),

        "prior_mean_F":
            mF,

        "prior_mean_0":
            m0,

        "alpha0_F":
            alphaF,

        "alpha0_0":
            alpha0,

        "split_results":
            pd.DataFrame(
                split_rows
            ),

        "bin_results":
            pd.DataFrame(
                bin_rows
            ),

        "threshold_results":
            pd.DataFrame(
                threshold_rows
            ),

        "elapsed_seconds":
            float(
                time.perf_counter()
                - t0
            ),
    }

    _atomic_pickle(
        payload,
        cache_path,
    )

    print(
        f"[rep end] {_now()} | "
        f"rep={replicate:02d} | "
        f"s_current_F={s_current_F:.9f} | "
        f"s_current_0={s_current_0:.9f} | "
        f"{payload['elapsed_seconds']:.2f}s",
        flush=True,
    )

    return payload, False


# =============================================================================
# AGGREGATION
# =============================================================================

def _aggregate_split_results(
    split_results,
):
    rows = []

    for method, d in split_results.groupby(
        "method",
        sort=False,
    ):
        s_values = d[
            "s"
        ].dropna()

        s_value = (
            float(
                s_values.iloc[
                    0
                ]
            )
            if len(
                s_values
            )
            else np.nan
        )

        rows.append(
            {
                "method":
                    method,

                "s":
                    s_value,

                "n_splits":
                    int(
                        len(
                            d
                        )
                    ),

                "mean_p_lt_0_01":
                    float(
                        d[
                            "fraction_p_lt_0_01_finite"
                        ]
                        .mean()
                    ),

                "median_p_lt_0_01":
                    float(
                        d[
                            "fraction_p_lt_0_01_finite"
                        ]
                        .median()
                    ),

                "q25_p_lt_0_01":
                    float(
                        d[
                            "fraction_p_lt_0_01_finite"
                        ]
                        .quantile(
                            0.25
                        )
                    ),

                "q75_p_lt_0_01":
                    float(
                        d[
                            "fraction_p_lt_0_01_finite"
                        ]
                        .quantile(
                            0.75
                        )
                    ),

                "mean_BH05_discoveries":
                    float(
                        d[
                            "BH05_discoveries"
                        ]
                        .mean()
                    ),

                "median_BH05_discoveries":
                    float(
                        d[
                            "BH05_discoveries"
                        ]
                        .median()
                    ),

                "complete_null_BH_FDR":
                    float(
                        d[
                            "any_BH05"
                        ]
                        .mean()
                    ),

                "mean_median_p":
                    float(
                        d[
                            "median_p"
                        ]
                        .mean()
                    ),
            }
        )

    return pd.DataFrame(
        rows
    )


def _aggregate_thresholds(
    threshold_results,
):
    rows = []

    for (
        method,
        threshold,
    ), d in threshold_results.groupby(
        [
            "method",
            "threshold",
        ],
        sort=False,
    ):
        s_values = d[
            "s"
        ].dropna()

        s_value = (
            float(
                s_values.iloc[
                    0
                ]
            )
            if len(
                s_values
            )
            else np.nan
        )

        rows.append(
            {
                "method":
                    method,

                "s":
                    s_value,

                "threshold":
                    float(
                        threshold
                    ),

                "mean_fraction_le_threshold":
                    float(
                        d[
                            "fraction_le_threshold"
                        ]
                        .mean()
                    ),

                "median_fraction_le_threshold":
                    float(
                        d[
                            "fraction_le_threshold"
                        ]
                        .median()
                    ),

                "ratio_to_nominal":
                    float(
                        d[
                            "fraction_le_threshold"
                        ]
                        .mean()
                        / float(
                            threshold
                        )
                    ),
            }
        )

    return pd.DataFrame(
        rows
    )


def _aggregate_bins(
    bin_results,
):
    rows = []

    for (
        method,
        b,
    ), d in bin_results.groupby(
        [
            "method",
            "coverage_bin",
        ],
        sort=False,
    ):
        s_values = d[
            "s"
        ].dropna()

        s_value = (
            float(
                s_values.iloc[
                    0
                ]
            )
            if len(
                s_values
            )
            else np.nan
        )

        rows.append(
            {
                "method":
                    method,

                "s":
                    s_value,

                "coverage_bin":
                    int(
                        b
                    ),

                "n_splits":
                    int(
                        d[
                            "replicate"
                        ]
                        .nunique()
                    ),

                "median_log1p_total_normalized_coverage":
                    float(
                        d[
                            "median_log1p_total_normalized_coverage"
                        ]
                        .median()
                    ),

                "median_theta_F":
                    float(
                        d[
                            "median_theta_F"
                        ]
                        .median()
                    ),

                "q25_theta_F":
                    float(
                        d[
                            "median_theta_F"
                        ]
                        .quantile(
                            0.25
                        )
                    ),

                "q75_theta_F":
                    float(
                        d[
                            "median_theta_F"
                        ]
                        .quantile(
                            0.75
                        )
                    ),

                "median_theta_0":
                    float(
                        d[
                            "median_theta_0"
                        ]
                        .median()
                    ),

                "q25_theta_0":
                    float(
                        d[
                            "median_theta_0"
                        ]
                        .quantile(
                            0.25
                        )
                    ),

                "q75_theta_0":
                    float(
                        d[
                            "median_theta_0"
                        ]
                        .quantile(
                            0.75
                        )
                    ),

                "median_raw_p_lt_0_01_fraction":
                    float(
                        d[
                            "raw_p_lt_0_01_fraction"
                        ]
                        .median()
                    ),

                "q25_raw_p_lt_0_01_fraction":
                    float(
                        d[
                            "raw_p_lt_0_01_fraction"
                        ]
                        .quantile(
                            0.25
                        )
                    ),

                "q75_raw_p_lt_0_01_fraction":
                    float(
                        d[
                            "raw_p_lt_0_01_fraction"
                        ]
                        .quantile(
                            0.75
                        )
                    ),
            }
        )

    return pd.DataFrame(
        rows
    )


# =============================================================================
# DEVELOPMENT-ONLY CALIBRATION OBJECTIVE
# =============================================================================

def _diagnostic_calibration_objective(
    threshold_aggregate,
):
    """
    Development-only same-data score.

    Uses several raw-p thresholds to avoid selecting s from p=.01 alone.

    objective(s)
      = RMS over thresholds of
            log( mean_fraction<=t / t ).

    Perfect raw-p calibration gives zero.

    This is NOT a production estimator and must be externally validated.
    """

    x = threshold_aggregate.loc[
        threshold_aggregate[
            "method"
        ]
        != "current_DEB"
    ].copy()

    x = x.loc[
        np.isfinite(
            x[
                "s"
            ]
        )
    ].copy()

    rows = []

    for s, d in x.groupby(
        "s",
        sort=True,
    ):
        ratios = (
            d[
                "mean_fraction_le_threshold"
            ]
            .to_numpy(
                dtype=float
            )
            / d[
                "threshold"
            ]
            .to_numpy(
                dtype=float
            )
        )

        ratios = ratios[
            np.isfinite(
                ratios
            )
            & (
                ratios
                > 0
            )
        ]

        objective = (
            float(
                np.sqrt(
                    np.mean(
                        np.log(
                            ratios
                        )
                        ** 2
                    )
                )
            )
            if len(
                ratios
            )
            else np.inf
        )

        rows.append(
            {
                "s":
                    float(
                        s
                    ),

                "calibration_objective":
                    objective,
            }
        )

    out = (
        pd.DataFrame(
            rows
        )
        .sort_values(
            "calibration_objective",
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    return out


# =============================================================================
# PLOTS
# =============================================================================

def _show_plots(
    method_summary,
    threshold_aggregate,
    bin_aggregate,
    diagnostic_best_s,
):
    numeric = method_summary.loc[
        np.isfinite(
            method_summary[
                "s"
            ]
        )
    ].sort_values(
        "s"
    )

    # -------------------------------------------------------------
    # 1. Mean / median p<.01 vs s.
    # -------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    ax.plot(
        numeric[
            "s"
        ],
        numeric[
            "mean_p_lt_0_01"
        ],
        marker="o",
        linewidth=2.3,
        label="Mean across splits",
    )

    ax.plot(
        numeric[
            "s"
        ],
        numeric[
            "median_p_lt_0_01"
        ],
        marker="s",
        linewidth=2.1,
        linestyle="--",
        label="Median across splits",
    )

    ax.axhline(
        RAW_P_THRESHOLD,
        linestyle=":",
        linewidth=1.5,
        label="Nominal 1%",
    )

    if np.isfinite(
        diagnostic_best_s
    ):
        ax.axvline(
            diagnostic_best_s,
            linestyle="-.",
            linewidth=1.5,
            label=f"Development-only best s={diagnostic_best_s:g}",
        )

    ax.set_xlabel(
        "Universal DEB shrinkage fraction s",
        fontsize=14,
    )

    ax.set_ylabel(
        "Fraction of finite p-values < .01",
        fontsize=14,
    )

    ax.set_title(
        "Pickrell: Raw-p calibration vs DEB shrinkage strength",
        fontsize=18,
        pad=16,
    )

    ax.grid(
        alpha=0.20,
    )

    ax.legend(
        fontsize=11,
    )

    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------------
    # 2. Complete-null BH FDR.
    # -------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    ax.plot(
        numeric[
            "s"
        ],
        numeric[
            "complete_null_BH_FDR"
        ],
        marker="o",
        linewidth=2.3,
    )

    ax.axhline(
        FDR_CUTOFF,
        linestyle=":",
        linewidth=1.5,
        label="Nominal 5%",
    )

    if np.isfinite(
        diagnostic_best_s
    ):
        ax.axvline(
            diagnostic_best_s,
            linestyle="-.",
            linewidth=1.5,
        )

    ax.set_xlabel(
        "Universal DEB shrinkage fraction s",
        fontsize=14,
    )

    ax.set_ylabel(
        "P(any BH(.05) rejection)",
        fontsize=14,
    )

    ax.set_title(
        "Pickrell complete null: BH FDR vs DEB shrinkage strength",
        fontsize=18,
        pad=16,
    )

    ax.grid(
        alpha=0.20,
    )

    ax.legend(
        fontsize=11,
    )

    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------------
    # 3. Mean BH discoveries.
    # -------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    ax.plot(
        numeric[
            "s"
        ],
        numeric[
            "mean_BH05_discoveries"
        ],
        marker="o",
        linewidth=2.3,
    )

    if np.isfinite(
        diagnostic_best_s
    ):
        ax.axvline(
            diagnostic_best_s,
            linestyle="-.",
            linewidth=1.5,
        )

    ax.set_xlabel(
        "Universal DEB shrinkage fraction s",
        fontsize=14,
    )

    ax.set_ylabel(
        "Mean BH(.05) discoveries",
        fontsize=14,
    )

    ax.set_title(
        "Pickrell complete null: BH discoveries vs DEB shrinkage strength",
        fontsize=18,
        pad=16,
    )

    ax.grid(
        alpha=0.20,
    )

    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------------
    # 4. Multi-threshold calibration.
    # -------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    for threshold, d in threshold_aggregate.loc[
        np.isfinite(
            threshold_aggregate[
                "s"
            ]
        )
    ].groupby(
        "threshold",
        sort=True,
    ):
        d = d.sort_values(
            "s"
        )

        ax.plot(
            d[
                "s"
            ],
            d[
                "ratio_to_nominal"
            ],
            marker="o",
            linewidth=2.0,
            label=f"p <= {threshold:g}",
        )

    ax.axhline(
        1.0,
        linestyle=":",
        linewidth=1.5,
        label="Calibrated",
    )

    if np.isfinite(
        diagnostic_best_s
    ):
        ax.axvline(
            diagnostic_best_s,
            linestyle="-.",
            linewidth=1.5,
        )

    ax.set_xlabel(
        "Universal DEB shrinkage fraction s",
        fontsize=14,
    )

    ax.set_ylabel(
        "Observed / nominal tail probability",
        fontsize=14,
    )

    ax.set_title(
        "Pickrell: Multi-threshold raw-p calibration vs shrinkage",
        fontsize=18,
        pad=16,
    )

    ax.grid(
        alpha=0.20,
    )

    ax.legend(
        fontsize=10,
        ncol=2,
    )

    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------------
    # 5. Heatmap: coverage bin x s, p<.01 fraction.
    # -------------------------------------------------------------

    heat = (
        bin_aggregate.loc[
            np.isfinite(
                bin_aggregate[
                    "s"
                ]
            )
        ]
        .pivot(
            index="coverage_bin",
            columns="s",
            values="median_raw_p_lt_0_01_fraction",
        )
        .sort_index()
    )

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    image = ax.imshow(
        heat.to_numpy(
            dtype=float
        ),
        aspect="auto",
        origin="lower",
    )

    ax.set_xticks(
        np.arange(
            len(
                heat.columns
            )
        )
    )

    ax.set_xticklabels(
        [
            f"{x:g}"
            for x in heat.columns
        ],
        rotation=45,
        ha="right",
    )

    ax.set_yticks(
        np.arange(
            len(
                heat.index
            )
        )
    )

    ax.set_yticklabels(
        [
            str(
                x
            )
            for x in heat.index
        ]
    )

    ax.set_xlabel(
        "Universal shrinkage fraction s",
        fontsize=14,
    )

    ax.set_ylabel(
        "Coverage bin",
        fontsize=14,
    )

    ax.set_title(
        "Pickrell: Median raw p<.01 fraction by coverage and shrinkage",
        fontsize=18,
        pad=16,
    )

    plt.colorbar(
        image,
        ax=ax,
        label="Median fraction raw p < .01",
    )

    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------------
    # 6. Selected raw-p coverage curves.
    # -------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    current = (
        bin_aggregate.loc[
            bin_aggregate[
                "method"
            ]
            == "current_DEB"
        ]
        .sort_values(
            "coverage_bin"
        )
    )

    ax.plot(
        current[
            "median_log1p_total_normalized_coverage"
        ],
        current[
            "median_raw_p_lt_0_01_fraction"
        ],
        marker="o",
        linewidth=2.5,
        label="Current DEB",
    )

    for s in SELECTED_S_FOR_PLOTS:
        d = (
            bin_aggregate.loc[
                np.isfinite(
                    bin_aggregate[
                        "s"
                    ]
                )
                & np.isclose(
                    bin_aggregate[
                        "s"
                    ],
                    float(
                        s
                    ),
                )
            ]
            .sort_values(
                "coverage_bin"
            )
        )

        if d.empty:
            continue

        ax.plot(
            d[
                "median_log1p_total_normalized_coverage"
            ],
            d[
                "median_raw_p_lt_0_01_fraction"
            ],
            marker="o",
            linewidth=1.8,
            label=f"s={s:g}",
        )

    ax.axhline(
        RAW_P_THRESHOLD,
        linestyle=":",
        linewidth=1.5,
        label="Nominal 1%",
    )

    ax.set_xlabel(
        "log1p(total normalized gene coverage)",
        fontsize=14,
    )

    ax.set_ylabel(
        "Median fraction raw p < .01",
        fontsize=14,
    )

    ax.set_title(
        "Pickrell: Coverage calibration as DEB shrinkage is relaxed",
        fontsize=18,
        pad=16,
    )

    ax.grid(
        alpha=0.20,
    )

    ax.legend(
        fontsize=9,
        ncol=2,
    )

    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------------
    # 7. theta_F coverage curves.
    # -------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    ax.plot(
        current[
            "median_log1p_total_normalized_coverage"
        ],
        current[
            "median_theta_F"
        ],
        marker="o",
        linewidth=2.5,
        label="Current DEB",
    )

    for s in SELECTED_S_FOR_PLOTS:
        d = (
            bin_aggregate.loc[
                np.isfinite(
                    bin_aggregate[
                        "s"
                    ]
                )
                & np.isclose(
                    bin_aggregate[
                        "s"
                    ],
                    float(
                        s
                    ),
                )
            ]
            .sort_values(
                "coverage_bin"
            )
        )

        if d.empty:
            continue

        ax.plot(
            d[
                "median_log1p_total_normalized_coverage"
            ],
            d[
                "median_theta_F"
            ],
            marker="o",
            linewidth=1.8,
            label=f"s={s:g}",
        )

    ax.set_yscale(
        "log"
    )

    ax.set_xlabel(
        "log1p(total normalized gene coverage)",
        fontsize=14,
    )

    ax.set_ylabel(
        r"Median $\theta_F$",
        fontsize=14,
    )

    ax.set_title(
        "Pickrell: Full precision curvature retained by shrinkage strength",
        fontsize=18,
        pad=16,
    )

    ax.grid(
        alpha=0.20,
        which="both",
    )

    ax.legend(
        fontsize=9,
        ncol=2,
    )

    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------------
    # 8. theta_0 coverage curves.
    # -------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    ax.plot(
        current[
            "median_log1p_total_normalized_coverage"
        ],
        current[
            "median_theta_0"
        ],
        marker="o",
        linewidth=2.5,
        label="Current DEB",
    )

    for s in SELECTED_S_FOR_PLOTS:
        d = (
            bin_aggregate.loc[
                np.isfinite(
                    bin_aggregate[
                        "s"
                    ]
                )
                & np.isclose(
                    bin_aggregate[
                        "s"
                    ],
                    float(
                        s
                    ),
                )
            ]
            .sort_values(
                "coverage_bin"
            )
        )

        if d.empty:
            continue

        ax.plot(
            d[
                "median_log1p_total_normalized_coverage"
            ],
            d[
                "median_theta_0"
            ],
            marker="o",
            linewidth=1.8,
            label=f"s={s:g}",
        )

    ax.set_yscale(
        "log"
    )

    ax.set_xlabel(
        "log1p(total normalized gene coverage)",
        fontsize=14,
    )

    ax.set_ylabel(
        r"Median $\theta_0$",
        fontsize=14,
    )

    ax.set_title(
        "Pickrell: Null precision curvature retained by shrinkage strength",
        fontsize=18,
        pad=16,
    )

    ax.grid(
        alpha=0.20,
        which="both",
    )

    ax.legend(
        fontsize=9,
        ncol=2,
    )

    plt.tight_layout()
    plt.show()


# =============================================================================
# TOP-LEVEL
# =============================================================================

def run_pickrell_tempered_deb_shrinkage():
    started = datetime.now().astimezone()

    print(
        f"[start] {_now()}",
        flush=True,
    )

    try:
        reps = _discover_completed_replicates()

        OUTPUT_DIR.mkdir(
            parents=True,
            exist_ok=True,
        )

        SPLIT_CACHE_DIR.mkdir(
            parents=True,
            exist_ok=True,
        )

        if not SENTINEL.exists():
            SENTINEL.write_text(
                json.dumps(
                    {
                        "initialized_local_time":
                            _now(),

                        "source_namespace":
                            CURRENT_M03_NAMESPACE,

                        "output_namespace":
                            OUTPUT_NAMESPACE,

                        "shrinkage_grid":
                            list(
                                SHRINKAGE_GRID
                            ),

                        "global_geometry_frozen":
                            True,

                        "mandatory_exact_reused":
                            True,

                        "resume_enabled":
                            bool(
                                RESUME
                            ),
                    },
                    indent=2,
                    sort_keys=True,
                ),
                encoding="utf-8",
            )

        print(
            "\nPICKRELL TEMPERED DEB SHRINKAGE DIAGNOSTIC\n"
            f"source:          {GLOBAL_RUN_DIR}\n"
            f"output:          {OUTPUT_DIR}\n"
            f"completed splits:{len(reps)}\n"
            f"s grid:          {SHRINKAGE_GRID}\n"
            "global kappa_c:  FROZEN\n"
            "S0dag geometry:  FROZEN\n"
            "prior means:     FROZEN\n"
            "changed piece:   universal DEB shrinkage strength only\n",
            flush=True,
        )

        payloads = []
        failures = []
        reused = 0

        for j, rep in enumerate(
            reps,
            start=1,
        ):
            try:
                payload, cached = _run_one_replicate(
                    rep
                )

                payloads.append(
                    payload
                )

                reused += int(
                    cached
                )

            except Exception as exc:
                failures.append(
                    {
                        "replicate":
                            int(
                                rep
                            ),

                        "error_type":
                            type(
                                exc
                            ).__name__,

                        "error_message":
                            str(
                                exc
                            ),

                        "traceback":
                            traceback.format_exc(),
                    }
                )

                print(
                    f"[FAILED] {_now()} | "
                    f"rep={rep:02d} | "
                    f"{type(exc).__name__}: {exc}",
                    flush=True,
                )

            print(
                f"[progress] {j:02d}/{len(reps):02d}",
                flush=True,
            )

        if not payloads:
            raise RuntimeError(
                "No split diagnostics completed successfully."
            )

        split_results = pd.concat(
            [
                p[
                    "split_results"
                ]
                for p in payloads
            ],
            ignore_index=True,
        )

        bin_results = pd.concat(
            [
                p[
                    "bin_results"
                ]
                for p in payloads
            ],
            ignore_index=True,
        )

        threshold_results = pd.concat(
            [
                p[
                    "threshold_results"
                ]
                for p in payloads
            ],
            ignore_index=True,
        )

        shrinkage_meta = pd.DataFrame(
            [
                {
                    "replicate":
                        int(
                            p[
                                "replicate"
                            ]
                        ),

                    "s_current_F":
                        float(
                            p[
                                "s_current_F"
                            ]
                        ),

                    "s_current_0":
                        float(
                            p[
                                "s_current_0"
                            ]
                        ),

                    "prior_mean_F":
                        float(
                            p[
                                "prior_mean_F"
                            ]
                        ),

                    "prior_mean_0":
                        float(
                            p[
                                "prior_mean_0"
                            ]
                        ),

                    "alpha0_F":
                        float(
                            p[
                                "alpha0_F"
                            ]
                        ),

                    "alpha0_0":
                        float(
                            p[
                                "alpha0_0"
                            ]
                        ),
                }
                for p in payloads
            ]
        )

        failures_df = pd.DataFrame(
            failures
        )

        method_summary = _aggregate_split_results(
            split_results
        )

        threshold_aggregate = _aggregate_thresholds(
            threshold_results
        )

        bin_aggregate = _aggregate_bins(
            bin_results
        )

        calibration_objective = _diagnostic_calibration_objective(
            threshold_aggregate
        )

        diagnostic_best_s = (
            float(
                calibration_objective.loc[
                    0,
                    "s",
                ]
            )
            if not calibration_objective.empty
            else np.nan
        )

        # ---------------------------------------------------------
        # Save numerical outputs.
        # ---------------------------------------------------------

        _atomic_csv(
            split_results,
            OUTPUT_DIR
            / "split_results.csv",
        )

        _atomic_csv(
            bin_results,
            OUTPUT_DIR
            / "coverage_bin_results.csv",
        )

        _atomic_csv(
            threshold_results,
            OUTPUT_DIR
            / "threshold_results.csv",
        )

        _atomic_csv(
            shrinkage_meta,
            OUTPUT_DIR
            / "current_shrinkage_meta.csv",
        )

        _atomic_csv(
            method_summary,
            OUTPUT_DIR
            / "method_summary.csv",
        )

        _atomic_csv(
            threshold_aggregate,
            OUTPUT_DIR
            / "threshold_aggregate.csv",
        )

        _atomic_csv(
            bin_aggregate,
            OUTPUT_DIR
            / "coverage_bin_aggregate.csv",
        )

        _atomic_csv(
            calibration_objective,
            OUTPUT_DIR
            / "development_calibration_objective.csv",
        )

        _atomic_csv(
            failures_df,
            OUTPUT_DIR
            / "failures.csv",
        )

        # ---------------------------------------------------------
        # Display numerical summaries.
        # ---------------------------------------------------------

        print(
            "\nCURRENT FITTED DEB SHRINKAGE FRACTIONS\n",
            flush=True,
        )

        display(
            shrinkage_meta[
                [
                    "replicate",
                    "s_current_F",
                    "s_current_0",
                    "prior_mean_F",
                    "prior_mean_0",
                    "alpha0_F",
                    "alpha0_0",
                ]
            ]
        )

        print(
            "\nMETHOD SUMMARY\n",
            flush=True,
        )

        display(
            method_summary
        )

        print(
            "\nMULTI-THRESHOLD CALIBRATION\n",
            flush=True,
        )

        display(
            threshold_aggregate
        )

        print(
            "\nDEVELOPMENT-ONLY CALIBRATION OBJECTIVE\n"
            "(same Pickrell null data; lower is better; external validation required)\n",
            flush=True,
        )

        display(
            calibration_objective.head(
                15
            )
        )

        print(
            f"\nDevelopment-only best s = {diagnostic_best_s:g}\n",
            flush=True,
        )

        if SHOW_PLOTS:
            _show_plots(
                method_summary,
                threshold_aggregate,
                bin_aggregate,
                diagnostic_best_s,
            )

        ended = datetime.now().astimezone()

        print(
            "\nSTATUS\n"
            f"successful splits: {len(shrinkage_meta)}/{len(reps)}\n"
            f"reused caches:     {reused}\n"
            f"failures:          {len(failures_df)}\n"
            f"output:            {OUTPUT_DIR}\n"
            f"[end] {_now()} | elapsed={ended-started}",
            flush=True,
        )

        return {
            "split_results":
                split_results,

            "bin_results":
                bin_results,

            "threshold_results":
                threshold_results,

            "shrinkage_meta":
                shrinkage_meta,

            "method_summary":
                method_summary,

            "threshold_aggregate":
                threshold_aggregate,

            "bin_aggregate":
                bin_aggregate,

            "calibration_objective":
                calibration_objective,

            "diagnostic_best_s":
                diagnostic_best_s,

            "failures":
                failures_df,
        }

    finally:
        pass


# =============================================================================
# EXECUTION
# =============================================================================

pickrell_tempered_deb_shrinkage = (
    run_pickrell_tempered_deb_shrinkage()
)


In [ ]:
from __future__ import annotations

"""
Pickrell extreme-tail audit for the universal DEB-shrinkage experiment.

Purpose
-------
The earlier tempered-DEB diagnostic showed that moderate shrinkage gives much
better calibration from p=.001 through .05, but BH is sensitive to much more
extreme p-values when ~10^4 genes are tested.

This script therefore pushes the same UNIVERSAL shrinkage experiment into the
extreme tail without rerunning M03.

For each completed frozen Pickrell 5v5 split and each universal shrinkage
fraction s, it recomputes the ordinary first-order M03 p-values while freezing:

    - r_g / alpha_g
    - fitted means
    - C_F and C_0
    - global kappa_c / lambda_c
    - S0dag_final
    - support-floor geometry
    - mandatory exact-gene p-values

Only theta_F and theta_0 are changed through

    1/theta_s
      = (1-s)/theta_raw + s/m,

where

    theta_raw = a/R
    m         = the CURRENT fitted DEB prior mean.

Diagnostics
-----------
1. Tail calibration down to:
       2.5e-6, 5e-6, 1e-5, 2.5e-5, 5e-5,
       1e-4, 2.5e-4, 5e-4, 1e-3, 5e-3, 1e-2

2. Distribution across the 30 splits of:
       minimum p-value among ALL genes
       minimum p-value among ordinary first-order genes only

3. The split-specific first BH step:
       alpha / m
   where m is the number of finite p-values.

4. Fraction of splits with:
       min(p) <= alpha/m

5. Complete-null BH FDR:
       P(any BH(.05) rejection)

6. Two clearly labeled SAME-DATA development summaries:
       - best s by extended raw-p tail calibration
       - best s by closeness of complete-null BH FDR to .05

These are diagnostics only.  A value chosen on Pickrell must be frozen and
tested on Li2020 without retuning.

Output
------
DISPLAY ONLY. Nothing is saved.
"""

from datetime import datetime
from pathlib import Path
from scipy.stats import chi2

import pickle

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


# =============================================================================
# SETTINGS
# =============================================================================

CURRENT_M03_NAMESPACE = "pickrell_m03_seam_safe_20M_kappa_coverage_v1"

SHRINKAGE_GRID = (
    0.00,
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.35,
    0.40,
    0.45,
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
    0.925,
    0.95,
    0.975,
    0.99,
    0.995,
    0.999,
)

EXTREME_THRESHOLDS = (
    2.5e-6,
    5.0e-6,
    1.0e-5,
    2.5e-5,
    5.0e-5,
    1.0e-4,
    2.5e-4,
    5.0e-4,
    1.0e-3,
    5.0e-3,
    1.0e-2,
)

# More stable subset used only for the same-data "extended calibration"
# development score.  Ultra-extreme thresholds are still reported/plottted,
# but are too sparse to dominate parameter selection.
OBJECTIVE_THRESHOLDS = (
    1.0e-4,
    2.5e-4,
    5.0e-4,
    1.0e-3,
    5.0e-3,
    1.0e-2,
)

FDR_CUTOFF = 0.05

FIGSIZE = (16, 9)


# =============================================================================
# PATHS / TIME
# =============================================================================

def _now() -> str:
    return datetime.now().astimezone().strftime(
        "%Y-%m-%d %I:%M:%S %p %Z"
    )


CWD = Path.cwd().resolve()

if CWD.name.lower() in {"notebook", "notebooks"}:
    PROJECT_ROOT = CWD.parent
elif (CWD / "data").is_dir():
    PROJECT_ROOT = CWD
else:
    raise RuntimeError(
        "Could not resolve repository root. Run from <repo>/notebook, "
        "<repo>/notebooks, or the repository root containing ./data. "
        f"Current working directory: {CWD}"
    )

RUN_DIR = (
    PROJECT_ROOT
    / "data"
    / CURRENT_M03_NAMESPACE
)

CELL_DIR = RUN_DIR / "cells"
CHECKPOINT_ROOT = RUN_DIR / "m03_checkpoints"


# =============================================================================
# CHECKPOINT HELPERS
# =============================================================================

def _load_pickle(path: Path):
    with open(path, "rb") as f:
        return pickle.load(f)


def _checkpoint_dir(replicate: int) -> Path:
    return (
        CHECKPOINT_ROOT
        / f"pickrell_rep{int(replicate):02d}"
    )


def _load_success_stage(
    replicate: int,
    stage_name: str,
):
    path = (
        _checkpoint_dir(replicate)
        / "stages"
        / f"{stage_name}.pkl"
    )

    if not path.exists():
        raise FileNotFoundError(
            f"Missing checkpoint: {path}"
        )

    payload = _load_pickle(path)

    if (
        not isinstance(payload, dict)
        or payload.get("status") != "success"
    ):
        raise RuntimeError(
            f"Replicate {replicate}: "
            f"{stage_name} is not a successful checkpoint."
        )

    return payload["data"]


def _load_manifest(replicate: int):
    path = (
        _checkpoint_dir(replicate)
        / "m03_manifest.pkl"
    )

    if not path.exists():
        raise FileNotFoundError(
            f"Missing manifest: {path}"
        )

    return _load_pickle(path)


def _discover_completed_replicates():
    reps = []

    for path in sorted(
        CELL_DIR.glob("pickrell_rep*.csv")
    ):
        try:
            x = pd.read_csv(path)
        except Exception:
            continue

        if len(x) != 1:
            continue

        if str(
            x.loc[0, "status"]
        ).strip().lower() != "success":
            continue

        reps.append(
            int(
                x.loc[0, "replicate"]
            )
        )

    reps = sorted(set(reps))

    if not reps:
        raise RuntimeError(
            "No completed Pickrell replicates were found."
        )

    return reps


# =============================================================================
# NUMERIC HELPERS
# =============================================================================

def _bh_adjust(p):
    p = np.asarray(
        p,
        dtype=float,
    )

    q = np.full_like(
        p,
        np.nan,
    )

    ok = (
        np.isfinite(p)
        & (p >= 0)
        & (p <= 1)
    )

    if not np.any(ok):
        return q

    p0 = p[ok]

    order = np.argsort(
        p0,
        kind="stable",
    )

    ranked = p0[order]

    n = len(ranked)

    adjusted = (
        ranked
        * n
        / np.arange(
            1,
            n + 1,
        )
    )

    adjusted = np.minimum.accumulate(
        adjusted[::-1]
    )[::-1]

    adjusted = np.minimum(
        adjusted,
        1.0,
    )

    restored = np.empty_like(
        adjusted
    )

    restored[order] = adjusted

    q[ok] = restored

    return q


def _raw_theta(
    a,
    R,
):
    R = np.asarray(
        R,
        dtype=float,
    )

    out = np.full(
        len(R),
        np.nan,
        dtype=float,
    )

    ok = (
        np.isfinite(R)
        & (R > 0)
    )

    out[ok] = (
        float(a)
        / R[ok]
    )

    return out


def _tempered_theta(
    raw_theta,
    prior_mean,
    s,
    theta_floor,
    theta_cap,
):
    raw_theta = np.asarray(
        raw_theta,
        dtype=float,
    )

    prior_mean = float(
        prior_mean
    )

    s = float(
        s
    )

    if not (
        0.0
        <= s
        < 1.0
    ):
        raise ValueError(
            f"s must be in [0,1); got {s}."
        )

    out = np.full(
        len(raw_theta),
        np.nan,
        dtype=float,
    )

    ok = (
        np.isfinite(raw_theta)
        & (raw_theta > 0)
        & np.isfinite(prior_mean)
        & (prior_mean > 0)
    )

    if s == 0.0:
        theta = raw_theta[ok]
    else:
        reciprocal = (
            (1.0 - s)
            / raw_theta[ok]
            + s
            / prior_mean
        )

        theta = (
            1.0
            / reciprocal
        )

    out[ok] = np.clip(
        theta,
        float(theta_floor),
        float(theta_cap),
    )

    return out


def _normalized_statistic(
    S_F,
    S_0_used,
    C_F,
    C_0,
    theta_F,
    theta_0,
    n,
    settings,
):
    corrected_score_tolerance = float(
        settings[
            "corrected_score_tolerance"
        ]
    )

    R_F = (
        S_F
        - C_F
    )

    R_0 = (
        S_0_used
        - C_0
    )

    ok = (
        np.isfinite(R_F)
        & np.isfinite(R_0)
        & np.isfinite(theta_F)
        & np.isfinite(theta_0)
        & (R_F > corrected_score_tolerance)
        & (R_0 > corrected_score_tolerance)
        & (theta_F > 0)
        & (theta_0 > 0)
    )

    T_raw = np.full(
        len(S_F),
        np.nan,
    )

    T_raw[ok] = (
        2.0
        * (
            theta_0[ok]
            * R_0[ok]
            - theta_F[ok]
            * R_F[ok]
        )
        + float(n)
        * np.log(
            theta_F[ok]
            / theta_0[ok]
        )
        + 2.0
        * (
            C_0[ok]
            - C_F[ok]
        )
    )

    T = np.full(
        len(S_F),
        np.nan,
    )

    T[ok] = np.maximum(
        T_raw[ok],
        0.0,
    )

    p = np.full(
        len(S_F),
        np.nan,
    )

    p[ok] = chi2.sf(
        T[ok],
        1,
    )

    return p


# =============================================================================
# ONE SPLIT
# =============================================================================

def _evaluate_one_replicate(
    replicate: int,
):
    core = _load_success_stage(
        replicate,
        "01_core",
    )

    full = _load_success_stage(
        replicate,
        "02_full_firstorder",
    )

    geom = _load_success_stage(
        replicate,
        "03_geometry_support",
    )

    null = _load_success_stage(
        replicate,
        "04_null_deb",
    )

    base = _load_success_stage(
        replicate,
        "05_base_m03_result",
    )

    manifest = _load_manifest(
        replicate
    )

    settings = dict(
        manifest[
            "numerical_settings"
        ]
    )

    theta_floor = float(
        settings[
            "theta_floor"
        ]
    )

    theta_cap = float(
        settings[
            "theta_cap"
        ]
    )

    S_F = np.asarray(
        core[
            "S_F"
        ],
        dtype=float,
    )

    a_F = float(
        core[
            "a_F"
        ]
    )

    a_0 = float(
        core[
            "a_0"
        ]
    )

    n = int(
        core[
            "n"
        ]
    )

    C_F = np.asarray(
        full[
            "C_F"
        ],
        dtype=float,
    )

    C_0 = np.asarray(
        full[
            "C_0"
        ],
        dtype=float,
    )

    R_F = np.asarray(
        full[
            "R_F"
        ],
        dtype=float,
    )

    mask_Rnonpos = np.asarray(
        full[
            "mask_Rnonpos"
        ],
        dtype=bool,
    )

    full_theta_fit = dict(
        full[
            "full_theta_fit"
        ]
    )

    S0dag = np.asarray(
        geom[
            "S0dag_final"
        ],
        dtype=float,
    )

    R0dag = np.asarray(
        null[
            "R_0_dagger_first"
        ],
        dtype=float,
    )

    null_theta_fit = dict(
        null[
            "null_theta_fit"
        ]
    )

    p_current = np.asarray(
        base[
            "p_value_m03"
        ],
        dtype=float,
    )

    q_current = np.asarray(
        base[
            "q_value_m03"
        ],
        dtype=float,
    )

    raw_theta_F = _raw_theta(
        a_F,
        R_F,
    )

    raw_theta_0 = _raw_theta(
        a_0,
        R0dag,
    )

    mF = float(
        full_theta_fit[
            "prior_mean_theta"
        ]
    )

    m0 = float(
        null_theta_fit[
            "prior_mean_theta"
        ]
    )

    alphaF = float(
        full_theta_fit[
            "alpha0"
        ]
    )

    alpha0 = float(
        null_theta_fit[
            "alpha0"
        ]
    )

    s_current_F = (
        alphaF
        / (
            a_F
            + alphaF
        )
    )

    s_current_0 = (
        alpha0
        / (
            a_0
            + alpha0
        )
    )

    rows = []
    tail_rows = []

    ordinary_mask = (
        ~mask_Rnonpos
    )

    def summarize_method(
        method,
        s,
        p,
    ):
        p = np.asarray(
            p,
            dtype=float,
        )

        finite = np.isfinite(
            p
        )

        q = _bh_adjust(
            p
        )

        n_finite = int(
            np.sum(
                finite
            )
        )

        first_bh_threshold = (
            FDR_CUTOFF
            / n_finite
            if n_finite
            else np.nan
        )

        p_all = p[
            finite
        ]

        ordinary_finite = (
            finite
            & ordinary_mask
        )

        p_ordinary = p[
            ordinary_finite
        ]

        min_all = (
            float(
                np.min(
                    p_all
                )
            )
            if len(
                p_all
            )
            else np.nan
        )

        min_ordinary = (
            float(
                np.min(
                    p_ordinary
                )
            )
            if len(
                p_ordinary
            )
            else np.nan
        )

        rows.append(
            {
                "replicate":
                    int(
                        replicate
                    ),

                "method":
                    method,

                "s":
                    s,

                "n_finite_p":
                    n_finite,

                "n_ordinary_finite_p":
                    int(
                        np.sum(
                            ordinary_finite
                        )
                    ),

                "first_BH_threshold":
                    float(
                        first_bh_threshold
                    ),

                "min_p_all":
                    min_all,

                "min_p_ordinary":
                    min_ordinary,

                "min_p_all_over_first_BH":
                    (
                        float(
                            min_all
                            / first_bh_threshold
                        )
                        if (
                            np.isfinite(
                                min_all
                            )
                            and np.isfinite(
                                first_bh_threshold
                            )
                            and first_bh_threshold
                            > 0
                        )
                        else np.nan
                    ),

                "min_p_ordinary_over_first_BH":
                    (
                        float(
                            min_ordinary
                            / first_bh_threshold
                        )
                        if (
                            np.isfinite(
                                min_ordinary
                            )
                            and np.isfinite(
                                first_bh_threshold
                            )
                            and first_bh_threshold
                            > 0
                        )
                        else np.nan
                    ),

                "min_all_below_first_BH":
                    int(
                        np.isfinite(
                            min_all
                        )
                        and np.isfinite(
                            first_bh_threshold
                        )
                        and min_all
                        <= first_bh_threshold
                    ),

                "min_ordinary_below_first_BH":
                    int(
                        np.isfinite(
                            min_ordinary
                        )
                        and np.isfinite(
                            first_bh_threshold
                        )
                        and min_ordinary
                        <= first_bh_threshold
                    ),

                "BH05_discoveries":
                    int(
                        np.sum(
                            np.isfinite(
                                q
                            )
                            & (
                                q
                                < FDR_CUTOFF
                            )
                        )
                    ),

                "any_BH05":
                    int(
                        np.any(
                            np.isfinite(
                                q
                            )
                            & (
                                q
                                < FDR_CUTOFF
                            )
                        )
                    ),
            }
        )

        for threshold in EXTREME_THRESHOLDS:
            count = int(
                np.sum(
                    finite
                    & (
                        p
                        <= float(
                            threshold
                        )
                    )
                )
            )

            tail_rows.append(
                {
                    "replicate":
                        int(
                            replicate
                        ),

                    "method":
                        method,

                    "s":
                        s,

                    "threshold":
                        float(
                            threshold
                        ),

                    "n_finite_p":
                        n_finite,

                    "count_le_threshold":
                        count,

                    "fraction_le_threshold":
                        (
                            float(
                                count
                                / n_finite
                            )
                            if n_finite
                            else np.nan
                        ),
                }
            )

    # Current canonical M03 reference.
    summarize_method(
        "current_DEB",
        np.nan,
        p_current,
    )

    # Universal-s grid.
    for s in SHRINKAGE_GRID:
        s = float(
            s
        )

        theta_F = _tempered_theta(
            raw_theta_F,
            mF,
            s,
            theta_floor,
            theta_cap,
        )

        theta_0 = _tempered_theta(
            raw_theta_0,
            m0,
            s,
            theta_floor,
            theta_cap,
        )

        p = _normalized_statistic(
            S_F,
            S0dag,
            C_F,
            C_0,
            theta_F,
            theta_0,
            n,
            settings,
        )

        # Exact mandatory branch is independent of DEB shrinkage.
        p[
            mask_Rnonpos
        ] = p_current[
            mask_Rnonpos
        ]

        summarize_method(
            f"s={s:g}",
            s,
            p,
        )

    meta = {
        "replicate":
            int(
                replicate
            ),

        "s_current_F":
            float(
                s_current_F
            ),

        "s_current_0":
            float(
                s_current_0
            ),

        "n_mandatory_exact":
            int(
                np.sum(
                    mask_Rnonpos
                )
            ),
    }

    return (
        pd.DataFrame(
            rows
        ),
        pd.DataFrame(
            tail_rows
        ),
        meta,
    )


# =============================================================================
# AGGREGATION
# =============================================================================

def _aggregate_minima(
    split_results,
):
    rows = []

    for method, d in split_results.groupby(
        "method",
        sort=False,
    ):
        s_values = d[
            "s"
        ].dropna()

        s = (
            float(
                s_values.iloc[
                    0
                ]
            )
            if len(
                s_values
            )
            else np.nan
        )

        rows.append(
            {
                "method":
                    method,

                "s":
                    s,

                "n_splits":
                    int(
                        len(
                            d
                        )
                    ),

                "median_min_p_all":
                    float(
                        d[
                            "min_p_all"
                        ]
                        .median()
                    ),

                "q25_min_p_all":
                    float(
                        d[
                            "min_p_all"
                        ]
                        .quantile(
                            0.25
                        )
                    ),

                "q75_min_p_all":
                    float(
                        d[
                            "min_p_all"
                        ]
                        .quantile(
                            0.75
                        )
                    ),

                "median_min_p_ordinary":
                    float(
                        d[
                            "min_p_ordinary"
                        ]
                        .median()
                    ),

                "q25_min_p_ordinary":
                    float(
                        d[
                            "min_p_ordinary"
                        ]
                        .quantile(
                            0.25
                        )
                    ),

                "q75_min_p_ordinary":
                    float(
                        d[
                            "min_p_ordinary"
                        ]
                        .quantile(
                            0.75
                        )
                    ),

                "median_first_BH_threshold":
                    float(
                        d[
                            "first_BH_threshold"
                        ]
                        .median()
                    ),

                "P_min_all_below_first_BH":
                    float(
                        d[
                            "min_all_below_first_BH"
                        ]
                        .mean()
                    ),

                "P_min_ordinary_below_first_BH":
                    float(
                        d[
                            "min_ordinary_below_first_BH"
                        ]
                        .mean()
                    ),

                "complete_null_BH_FDR":
                    float(
                        d[
                            "any_BH05"
                        ]
                        .mean()
                    ),

                "mean_BH05_discoveries":
                    float(
                        d[
                            "BH05_discoveries"
                        ]
                        .mean()
                    ),
            }
        )

    return pd.DataFrame(
        rows
    )


def _aggregate_tails(
    tail_results,
):
    rows = []

    for (
        method,
        threshold,
    ), d in tail_results.groupby(
        [
            "method",
            "threshold",
        ],
        sort=False,
    ):
        s_values = d[
            "s"
        ].dropna()

        s = (
            float(
                s_values.iloc[
                    0
                ]
            )
            if len(
                s_values
            )
            else np.nan
        )

        pooled_count = int(
            d[
                "count_le_threshold"
            ]
            .sum()
        )

        pooled_n = int(
            d[
                "n_finite_p"
            ]
            .sum()
        )

        pooled_fraction = (
            float(
                pooled_count
                / pooled_n
            )
            if pooled_n
            else np.nan
        )

        rows.append(
            {
                "method":
                    method,

                "s":
                    s,

                "threshold":
                    float(
                        threshold
                    ),

                "mean_split_fraction":
                    float(
                        d[
                            "fraction_le_threshold"
                        ]
                        .mean()
                    ),

                "median_split_fraction":
                    float(
                        d[
                            "fraction_le_threshold"
                        ]
                        .median()
                    ),

                "pooled_count":
                    pooled_count,

                "pooled_n":
                    pooled_n,

                "pooled_fraction":
                    pooled_fraction,

                "mean_ratio_to_nominal":
                    float(
                        d[
                            "fraction_le_threshold"
                        ]
                        .mean()
                        / float(
                            threshold
                        )
                    ),

                "pooled_ratio_to_nominal":
                    (
                        float(
                            pooled_fraction
                            / float(
                                threshold
                            )
                        )
                        if np.isfinite(
                            pooled_fraction
                        )
                        else np.nan
                    ),
            }
        )

    return pd.DataFrame(
        rows
    )


def _development_objectives(
    minima_aggregate,
    tail_aggregate,
):
    numeric_tail = tail_aggregate.loc[
        np.isfinite(
            tail_aggregate[
                "s"
            ]
        )
        & tail_aggregate[
            "threshold"
        ].isin(
            OBJECTIVE_THRESHOLDS
        )
    ].copy()

    rows = []

    for s, d in numeric_tail.groupby(
        "s",
        sort=True,
    ):
        ratios = (
            d[
                "mean_ratio_to_nominal"
            ]
            .to_numpy(
                dtype=float
            )
        )

        ratios = ratios[
            np.isfinite(
                ratios
            )
            & (
                ratios
                > 0
            )
        ]

        tail_objective = (
            float(
                np.sqrt(
                    np.mean(
                        np.log(
                            ratios
                        )
                        ** 2
                    )
                )
            )
            if len(
                ratios
            )
            else np.inf
        )

        m = minima_aggregate.loc[
            np.isclose(
                minima_aggregate[
                    "s"
                ],
                float(
                    s
                ),
            )
        ]

        if m.empty:
            continue

        bh_fdr = float(
            m[
                "complete_null_BH_FDR"
            ]
            .iloc[
                0
            ]
        )

        rows.append(
            {
                "s":
                    float(
                        s
                    ),

                "extended_tail_calibration_objective":
                    tail_objective,

                "complete_null_BH_FDR":
                    bh_fdr,

                "BH_FDR_absolute_error":
                    abs(
                        bh_fdr
                        - FDR_CUTOFF
                    ),
            }
        )

    return (
        pd.DataFrame(
            rows
        )
        .sort_values(
            "s"
        )
        .reset_index(
            drop=True
        )
    )


# =============================================================================
# PLOTS
# =============================================================================

def _show_plots(
    split_results,
    minima_aggregate,
    tail_aggregate,
    objectives,
):
    numeric_min = (
        minima_aggregate.loc[
            np.isfinite(
                minima_aggregate[
                    "s"
                ]
            )
        ]
        .sort_values(
            "s"
        )
    )

    # -------------------------------------------------------------
    # 1. Extreme-tail calibration ratios.
    # -------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    for threshold, d in tail_aggregate.loc[
        np.isfinite(
            tail_aggregate[
                "s"
            ]
        )
    ].groupby(
        "threshold",
        sort=True,
    ):
        d = d.sort_values(
            "s"
        )

        ax.plot(
            d[
                "s"
            ],
            d[
                "mean_ratio_to_nominal"
            ],
            marker="o",
            linewidth=1.8,
            label=f"{threshold:g}",
        )

    ax.axhline(
        1.0,
        linestyle=":",
        linewidth=1.5,
        label="Calibrated",
    )

    ax.set_yscale(
        "log"
    )

    ax.set_xlabel(
        "Universal DEB shrinkage fraction s",
        fontsize=14,
    )

    ax.set_ylabel(
        "Observed / nominal tail probability",
        fontsize=14,
    )

    ax.set_title(
        "Pickrell: Extreme raw-p tail calibration vs DEB shrinkage",
        fontsize=18,
        pad=16,
    )

    ax.grid(
        alpha=0.20,
        which="both",
    )

    ax.legend(
        title="p threshold",
        fontsize=9,
        ncol=3,
    )

    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------------
    # 2. Minimum p distribution across splits.
    # -------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    # Faint per-split trajectories.
    for rep, d in split_results.loc[
        np.isfinite(
            split_results[
                "s"
            ]
        )
    ].groupby(
        "replicate",
        sort=True,
    ):
        d = d.sort_values(
            "s"
        )

        ax.plot(
            d[
                "s"
            ],
            d[
                "min_p_all"
            ],
            linewidth=0.8,
            alpha=0.15,
        )

    ax.plot(
        numeric_min[
            "s"
        ],
        numeric_min[
            "median_min_p_all"
        ],
        marker="o",
        linewidth=2.5,
        label="Median min p: all genes",
    )

    ax.fill_between(
        numeric_min[
            "s"
        ],
        numeric_min[
            "q25_min_p_all"
        ],
        numeric_min[
            "q75_min_p_all"
        ],
        alpha=0.15,
    )

    ax.plot(
        numeric_min[
            "s"
        ],
        numeric_min[
            "median_min_p_ordinary"
        ],
        marker="s",
        linewidth=2.2,
        linestyle="--",
        label="Median min p: ordinary genes",
    )

    ax.fill_between(
        numeric_min[
            "s"
        ],
        numeric_min[
            "q25_min_p_ordinary"
        ],
        numeric_min[
            "q75_min_p_ordinary"
        ],
        alpha=0.12,
    )

    ax.plot(
        numeric_min[
            "s"
        ],
        numeric_min[
            "median_first_BH_threshold"
        ],
        linestyle=":",
        linewidth=2.0,
        label=r"Median first-BH cutoff $0.05/m$",
    )

    ax.set_yscale(
        "log"
    )

    ax.set_xlabel(
        "Universal DEB shrinkage fraction s",
        fontsize=14,
    )

    ax.set_ylabel(
        "Minimum p-value across a split",
        fontsize=14,
    )

    ax.set_title(
        "Pickrell: Distribution of split minimum p-values vs shrinkage",
        fontsize=18,
        pad=16,
    )

    ax.grid(
        alpha=0.20,
        which="both",
    )

    ax.legend(
        fontsize=11,
    )

    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------------
    # 3. P(min p <= first BH threshold).
    # -------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    ax.plot(
        numeric_min[
            "s"
        ],
        numeric_min[
            "P_min_all_below_first_BH"
        ],
        marker="o",
        linewidth=2.4,
        label="All genes",
    )

    ax.plot(
        numeric_min[
            "s"
        ],
        numeric_min[
            "P_min_ordinary_below_first_BH"
        ],
        marker="s",
        linewidth=2.2,
        linestyle="--",
        label="Ordinary first-order genes",
    )

    ax.axhline(
        FDR_CUTOFF,
        linestyle=":",
        linewidth=1.5,
        label="5%",
    )

    ax.set_xlabel(
        "Universal DEB shrinkage fraction s",
        fontsize=14,
    )

    ax.set_ylabel(
        r"Fraction of splits with min p $\leq 0.05/m$",
        fontsize=14,
    )

    ax.set_title(
        "Pickrell: First-rank BH-tail behavior vs shrinkage",
        fontsize=18,
        pad=16,
    )

    ax.grid(
        alpha=0.20,
    )

    ax.legend(
        fontsize=11,
    )

    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------------
    # 4. Complete-null BH FDR.
    # -------------------------------------------------------------

    fig, ax = plt.subplots(
        figsize=FIGSIZE
    )

    ax.plot(
        numeric_min[
            "s"
        ],
        numeric_min[
            "complete_null_BH_FDR"
        ],
        marker="o",
        linewidth=2.4,
        label="P(any BH rejection)",
    )

    ax.axhline(
        FDR_CUTOFF,
        linestyle=":",
        linewidth=1.5,
        label="Nominal 5%",
    )

    ax.set_xlabel(
        "Universal DEB shrinkage fraction s",
        fontsize=14,
    )

    ax.set_ylabel(
        "Complete-null BH FDR",
        fontsize=14,
    )

    ax.set_title(
        "Pickrell: Complete-null BH FDR vs shrinkage",
        fontsize=18,
        pad=16,
    )

    ax.grid(
        alpha=0.20,
    )

    ax.legend(
        fontsize=11,
    )

    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------------
    # 5. Same-data development objectives.
    # -------------------------------------------------------------

    fig, ax1 = plt.subplots(
        figsize=FIGSIZE
    )

    ax2 = ax1.twinx()

    ax1.plot(
        objectives[
            "s"
        ],
        objectives[
            "extended_tail_calibration_objective"
        ],
        marker="o",
        linewidth=2.3,
        label="Extended-tail calibration objective",
    )

    ax2.plot(
        objectives[
            "s"
        ],
        objectives[
            "BH_FDR_absolute_error"
        ],
        marker="s",
        linewidth=2.2,
        linestyle="--",
        label="|BH FDR - .05|",
    )

    ax1.set_xlabel(
        "Universal DEB shrinkage fraction s",
        fontsize=14,
    )

    ax1.set_ylabel(
        "Extended-tail calibration objective",
        fontsize=14,
    )

    ax2.set_ylabel(
        "Absolute BH-FDR calibration error",
        fontsize=14,
    )

    ax1.set_title(
        "Pickrell development-only shrinkage diagnostics\n"
        "(same data; values must be frozen before external validation)",
        fontsize=18,
        pad=16,
    )

    ax1.grid(
        alpha=0.20,
    )

    lines1, labels1 = (
        ax1.get_legend_handles_labels()
    )

    lines2, labels2 = (
        ax2.get_legend_handles_labels()
    )

    ax1.legend(
        lines1 + lines2,
        labels1 + labels2,
        fontsize=11,
    )

    plt.tight_layout()
    plt.show()


# =============================================================================
# TOP-LEVEL
# =============================================================================

def run_pickrell_extreme_tail_shrinkage_audit():
    started = datetime.now().astimezone()

    print(
        f"[start] {_now()}",
        flush=True,
    )

    try:
        reps = _discover_completed_replicates()

        print(
            "\nPICKRELL EXTREME-TAIL DEB-SHRINKAGE AUDIT\n"
            f"namespace:        {CURRENT_M03_NAMESPACE}\n"
            f"completed splits: {len(reps)}\n"
            f"s grid:           {SHRINKAGE_GRID}\n"
            f"tail thresholds:  {EXTREME_THRESHOLDS}\n"
            "output:            DISPLAY ONLY\n",
            flush=True,
        )

        split_frames = []
        tail_frames = []
        meta_rows = []

        for j, rep in enumerate(
            reps,
            start=1,
        ):
            split_df, tail_df, meta = _evaluate_one_replicate(
                rep
            )

            split_frames.append(
                split_df
            )

            tail_frames.append(
                tail_df
            )

            meta_rows.append(
                meta
            )

            print(
                f"[progress] {j:02d}/{len(reps):02d} "
                f"| rep={rep:02d} "
                f"| exact={meta['n_mandatory_exact']} "
                f"| s_current_F={meta['s_current_F']:.9f} "
                f"| s_current_0={meta['s_current_0']:.9f}",
                flush=True,
            )

        split_results = pd.concat(
            split_frames,
            ignore_index=True,
        )

        tail_results = pd.concat(
            tail_frames,
            ignore_index=True,
        )

        meta = pd.DataFrame(
            meta_rows
        )

        minima_aggregate = _aggregate_minima(
            split_results
        )

        tail_aggregate = _aggregate_tails(
            tail_results
        )

        objectives = _development_objectives(
            minima_aggregate,
            tail_aggregate,
        )

        best_tail_s = float(
            objectives.sort_values(
                "extended_tail_calibration_objective",
                kind="stable",
            )
            .iloc[
                0
            ][
                "s"
            ]
        )

        best_bh_s = float(
            objectives.sort_values(
                "BH_FDR_absolute_error",
                kind="stable",
            )
            .iloc[
                0
            ][
                "s"
            ]
        )

        print(
            "\nCURRENT DEB SHRINKAGE FRACTIONS\n",
            flush=True,
        )

        display(
            meta
        )

        print(
            "\nMINIMUM-P / BH SUMMARY\n",
            flush=True,
        )

        display(
            minima_aggregate
        )

        print(
            "\nEXTREME-TAIL CALIBRATION SUMMARY\n",
            flush=True,
        )

        display(
            tail_aggregate
        )

        print(
            "\nSAME-DATA DEVELOPMENT OBJECTIVES\n",
            flush=True,
        )

        display(
            objectives
        )

        print(
            "\nDEVELOPMENT-ONLY DIAGNOSTIC VALUES\n"
            f"best extended-tail calibration s = {best_tail_s:g}\n"
            f"best BH-FDR calibration s         = {best_bh_s:g}\n"
            "These are NOT final values; freeze candidates before Li2020 validation.\n",
            flush=True,
        )

        _show_plots(
            split_results,
            minima_aggregate,
            tail_aggregate,
            objectives,
        )

        ended = datetime.now().astimezone()

        print(
            f"[end] {_now()} | elapsed={ended-started}",
            flush=True,
        )

        return {
            "split_results":
                split_results,

            "tail_results":
                tail_results,

            "meta":
                meta,

            "minima_aggregate":
                minima_aggregate,

            "tail_aggregate":
                tail_aggregate,

            "objectives":
                objectives,

            "best_tail_s":
                best_tail_s,

            "best_bh_s":
                best_bh_s,
        }

    finally:
        pass


# =============================================================================
# EXECUTION
# =============================================================================

pickrell_extreme_tail_shrinkage_audit = (
    run_pickrell_extreme_tail_shrinkage_audit()
)


In [27]:
from __future__ import annotations

"""
Li2020 40v40 FIRST-20-REPLICATE capped-DEB transfer test.

Purpose
-------
Test the three frozen DEB prior-concentration caps identified from Pickrell:

    c_max = 0.25   <=> max shrinkage s = 0.20 when the cap binds
    c_max = 1.00   <=> max shrinkage s = 0.50 when the cap binds
    c_max = 3.00   <=> max shrinkage s = 0.75 when the cap binds

on the FIRST 20 FROZEN 40v40 replicates of EACH Li2020 dataset:

    AdLC reps 1..20
    OC   reps 1..20
    TNBC reps 1..20

This is a transfer test.  The three c_max values are NOT tuned here.

What stays frozen
-----------------
All upstream/current M03 quantities are reused from the completed 20M checkpoints:

    - WEB-DP r_g
    - fitted full/null means
    - C_F and C_0
    - the existing GLOBAL kappa_c / lambda_c
    - the existing support-feasible S0dag_final
    - the current fitted DEB PRIOR MEANS m_F and m_0
    - the mandatory exact normalized-DNB branch

Only the ordinary first-order DEB PRIOR CONCENTRATION is capped.

For a fitted DEB prior

    theta ~ Gamma(alpha0, beta0)
    m = alpha0 / beta0,

the candidate cap is

    alpha0* = min(alpha0, c_max * a)
    beta0*  = alpha0* / m

so the fitted prior mean is unchanged.

Then

    theta* = (a + alpha0*) / (R + beta0*).

When the cap binds,

    s = alpha0*/(a + alpha0*) = c_max/(1+c_max),

which is exactly the reciprocal-precision shrinkage fraction studied in Pickrell.

Mandatory exact genes
---------------------
Genes with R_F <= corrected_score_tolerance already use the exact normalized-DNB
profile.  That exact branch does not depend on the ordinary DEB prior strength, so
their canonical current M03 p-values are reused unchanged.

Li2020 call definition
----------------------
A false-positive call is the benchmark's established rule:

    finite BH q <= 0.05
    AND
    finite |log2FC| >= 1

FPR is:

    number of calls / fixed supplied filtered-gene universe.

This script ALSO reports ordinary complete-null BH behavior without the |log2FC|
filter, raw-p calibration down to 2.5e-6, minimum p-values, and calibration by
coverage.

Storage / display
-----------------
Numerical per-split caches and CSV summaries are written under:

    <repo>/data/web_m03_seam_safe_20M_DESC40to5_FULL_v2/
        audit_40v40_capped_deb_first20_v1/

This makes the audit resumable.

FIGURES ARE DISPLAYED ONLY.  They are not saved.

The script will FAIL EXPLICITLY unless reps 1..20 are present for all three
datasets, so "first 20" never silently means "first 20 that happened to finish."
"""

from datetime import datetime
from pathlib import Path
from scipy.stats import chi2

import json
import os
import pickle
import re
import tempfile
import time
import traceback

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display


# =============================================================================
# SETTINGS
# =============================================================================

AUDIT_VERSION = "2026-10-03-li2020-40v40-first20-capped-deb-v1"

RUN_NAMESPACE = "web_m03_seam_safe_20M_DESC40to5_FULL_v2"

TARGET_DATASETS = (
    "AdLC",
    "OC",
    "TNBC",
)

TARGET_REPLICATES = tuple(
    range(
        1,
        21,
    )
)

N_PER_GROUP = 40

C_MAX_VALUES = (
    0.25,
    1.00,
    3.00,
)

N_COVERAGE_BINS = 12

FDR_CUTOFF = 0.05
ABS_LOG2FC_CUTOFF = 1.0

TAIL_THRESHOLDS = (
    2.5e-6,
    5.0e-6,
    1.0e-5,
    2.5e-5,
    5.0e-5,
    1.0e-4,
    2.5e-4,
    5.0e-4,
    1.0e-3,
    5.0e-3,
    1.0e-2,
)

RESUME_AUDIT = True
SHOW_PLOTS = True

FIGSIZE = (
    16,
    9,
)


# =============================================================================
# PATHS / TIME
# =============================================================================

def _now() -> str:
    return (
        datetime.now()
        .astimezone()
        .strftime(
            "%Y-%m-%d %I:%M:%S %p %Z"
        )
    )


CWD = Path.cwd().resolve()

if CWD.name.lower() in {
    "notebook",
    "notebooks",
}:
    PROJECT_ROOT = CWD.parent

elif (
    CWD
    / "data"
).is_dir():
    PROJECT_ROOT = CWD

else:
    raise RuntimeError(
        "Run this from <repo>/notebook, <repo>/notebooks, "
        "or the repository root containing ./data. "
        f"Current working directory: {CWD}"
    )


PROJECT_DATA_DIR = (
    PROJECT_ROOT
    / "data"
)

BENCHMARK_DIR = (
    PROJECT_DATA_DIR
    / RUN_NAMESPACE
)

CELL_DIR = (
    BENCHMARK_DIR
    / "cells"
)

M03_CHECKPOINT_ROOT = (
    BENCHMARK_DIR
    / "m03_checkpoints"
)

AUDIT_DIR = (
    BENCHMARK_DIR
    / "audit_40v40_capped_deb_first20_v1"
)

PER_SPLIT_DIR = (
    AUDIT_DIR
    / "per_split"
)

SENTINEL = (
    AUDIT_DIR
    / "_audit_initialized.json"
)


# =============================================================================
# FILE HELPERS
# =============================================================================

def _safe(
    x,
) -> str:
    x = re.sub(
        r"[^A-Za-z0-9._-]+",
        "_",
        str(
            x
        ),
    )

    return (
        x.strip(
            "._"
        )
        or "unnamed"
    )


def _load_pickle(
    path: Path,
):
    with open(
        path,
        "rb",
    ) as f:
        return pickle.load(
            f
        )


def _atomic_pickle(
    obj,
    path: Path,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    fd, tmp_name = tempfile.mkstemp(
        prefix=f".{path.name}.",
        dir=str(
            path.parent
        ),
    )

    os.close(
        fd
    )

    tmp = Path(
        tmp_name
    )

    try:
        with open(
            tmp,
            "wb",
        ) as f:
            pickle.dump(
                obj,
                f,
                protocol=pickle.HIGHEST_PROTOCOL,
            )

            f.flush()

            try:
                os.fsync(
                    f.fileno()
                )
            except OSError:
                pass

        os.replace(
            tmp,
            path,
        )

    finally:
        if tmp.exists():
            try:
                tmp.unlink()
            except OSError:
                pass


def _atomic_csv(
    df: pd.DataFrame,
    path: Path,
) -> None:
    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    fd, tmp_name = tempfile.mkstemp(
        prefix=f".{path.name}.",
        dir=str(
            path.parent
        ),
    )

    os.close(
        fd
    )

    tmp = Path(
        tmp_name
    )

    try:
        df.to_csv(
            tmp,
            index=False,
        )

        os.replace(
            tmp,
            path,
        )

    finally:
        if tmp.exists():
            try:
                tmp.unlink()
            except OSError:
                pass


def _checkpoint_dir(
    split_id: str,
) -> Path:
    return (
        M03_CHECKPOINT_ROOT
        / _safe(
            split_id
        )
    )


def _load_success_stage(
    split_id: str,
    stage_name: str,
):
    path = (
        _checkpoint_dir(
            split_id
        )
        / "stages"
        / f"{stage_name}.pkl"
    )

    if not path.exists():
        raise FileNotFoundError(
            f"Missing checkpoint: {path}"
        )

    payload = _load_pickle(
        path
    )

    if (
        not isinstance(
            payload,
            dict,
        )
        or payload.get(
            "status"
        )
        != "success"
    ):
        raise RuntimeError(
            f"{split_id}: {stage_name} is not a successful checkpoint."
        )

    return payload[
        "data"
    ]


def _load_manifest(
    split_id: str,
):
    path = (
        _checkpoint_dir(
            split_id
        )
        / "m03_manifest.pkl"
    )

    if not path.exists():
        raise FileNotFoundError(
            f"Missing manifest: {path}"
        )

    return _load_pickle(
        path
    )


# =============================================================================
# DISCOVER EXACTLY REPS 1..20
# =============================================================================

def _discover_requested_splits() -> pd.DataFrame:
    if not CELL_DIR.exists():
        raise RuntimeError(
            f"Cell directory does not exist: {CELL_DIR}"
        )

    rows = []

    for path in sorted(
        CELL_DIR.glob(
            "*.csv"
        )
    ):
        try:
            x = pd.read_csv(
                path
            )
        except Exception:
            continue

        if len(
            x
        ) != 1:
            continue

        if (
            str(
                x.loc[
                    0,
                    "status",
                ]
            )
            .strip()
            .lower()
            != "success"
        ):
            continue

        if int(
            x.loc[
                0,
                "n_per_group",
            ]
        ) != int(
            N_PER_GROUP
        ):
            continue

        dataset = str(
            x.loc[
                0,
                "dataset",
            ]
        )

        replicate = int(
            x.loc[
                0,
                "replicate",
            ]
        )

        if dataset not in TARGET_DATASETS:
            continue

        if replicate not in TARGET_REPLICATES:
            continue

        row = x.iloc[
            0
        ].to_dict()

        row[
            "_cell_path"
        ] = str(
            path
        )

        rows.append(
            row
        )

    out = pd.DataFrame(
        rows
    )

    if out.empty:
        raise RuntimeError(
            "No requested completed 40v40 cells were found."
        )

    out[
        "dataset"
    ] = out[
        "dataset"
    ].astype(
        str
    )

    out[
        "replicate"
    ] = out[
        "replicate"
    ].astype(
        int
    )

    out[
        "split_id"
    ] = out[
        "split_id"
    ].astype(
        str
    )

    # Explicitly require replicate IDs 1..20 for each dataset.
    missing = []

    duplicate = []

    for dataset in TARGET_DATASETS:
        x = out.loc[
            out[
                "dataset"
            ]
            == dataset
        ]

        for replicate in TARGET_REPLICATES:
            n = int(
                np.sum(
                    x[
                        "replicate"
                    ]
                    == replicate
                )
            )

            if n == 0:
                missing.append(
                    (
                        dataset,
                        replicate,
                    )
                )

            elif n > 1:
                duplicate.append(
                    (
                        dataset,
                        replicate,
                        n,
                    )
                )

    if duplicate:
        raise RuntimeError(
            "Duplicate completed cell summaries found for requested "
            f"dataset/replicate combinations: {duplicate}"
        )

    if missing:
        raise RuntimeError(
            "The requested FIRST 20 replicates are not all complete. "
            f"Missing dataset/replicate pairs: {missing}"
        )

    out = (
        out
        .sort_values(
            [
                "dataset",
                "replicate",
            ],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )

    expected = (
        len(
            TARGET_DATASETS
        )
        * len(
            TARGET_REPLICATES
        )
    )

    if len(
        out
    ) != expected:
        raise RuntimeError(
            f"Expected exactly {expected} requested splits; found {len(out)}."
        )

    return out


# =============================================================================
# NUMERIC HELPERS
# =============================================================================

def _bh_adjust(
    p,
):
    p = np.asarray(
        p,
        dtype=float,
    )

    q = np.full_like(
        p,
        np.nan,
    )

    ok = (
        np.isfinite(
            p
        )
        & (
            p
            >= 0
        )
        & (
            p
            <= 1
        )
    )

    if not np.any(
        ok
    ):
        return q

    p0 = p[
        ok
    ]

    order = np.argsort(
        p0,
        kind="stable",
    )

    ranked = p0[
        order
    ]

    n = len(
        ranked
    )

    adjusted = (
        ranked
        * n
        / np.arange(
            1,
            n + 1,
        )
    )

    adjusted = np.minimum.accumulate(
        adjusted[
            ::-1
        ]
    )[
        ::-1
    ]

    adjusted = np.minimum(
        adjusted,
        1.0,
    )

    restored = np.empty_like(
        adjusted
    )

    restored[
        order
    ] = adjusted

    q[
        ok
    ] = restored

    return q


def _normalized_statistic(
    S_F,
    S_0_used,
    C_F,
    C_0,
    theta_F,
    theta_0,
    n,
    corrected_score_tolerance,
):
    R_F = (
        S_F
        - C_F
    )

    R_0 = (
        S_0_used
        - C_0
    )

    ok = (
        np.isfinite(
            R_F
        )
        & np.isfinite(
            R_0
        )
        & np.isfinite(
            theta_F
        )
        & np.isfinite(
            theta_0
        )
        & (
            R_F
            > float(
                corrected_score_tolerance
            )
        )
        & (
            R_0
            > float(
                corrected_score_tolerance
            )
        )
        & (
            theta_F
            > 0
        )
        & (
            theta_0
            > 0
        )
    )

    T_raw = np.full(
        len(
            S_F
        ),
        np.nan,
        dtype=float,
    )

    T_raw[
        ok
    ] = (
        2.0
        * (
            theta_0[
                ok
            ]
            * R_0[
                ok
            ]
            - theta_F[
                ok
            ]
            * R_F[
                ok
            ]
        )
        + float(
            n
        )
        * np.log(
            theta_F[
                ok
            ]
            / theta_0[
                ok
            ]
        )
        + 2.0
        * (
            C_0[
                ok
            ]
            - C_F[
                ok
            ]
        )
    )

    T = np.full(
        len(
            S_F
        ),
        np.nan,
        dtype=float,
    )

    T[
        ok
    ] = np.maximum(
        T_raw[
            ok
        ],
        0.0,
    )

    p = np.full(
        len(
            S_F
        ),
        np.nan,
        dtype=float,
    )

    p[
        ok
    ] = chi2.sf(
        T[
            ok
        ],
        1,
    )

    return {
        "R_F":
            R_F,

        "R_0":
            R_0,

        "valid_mask":
            ok,

        "T_raw":
            T_raw,

        "T":
            T,

        "p":
            p,
    }


def _reconstruct_log2fc(
    core,
):
    group = np.asarray(
        core[
            "group"
        ]
    ).astype(
        str
    )

    levels = list(
        np.unique(
            group
        )
    )

    if len(
        levels
    ) != 2:
        raise RuntimeError(
            f"Expected exactly two groups; found {levels}."
        )

    q_by_group = core[
        "q_by_group"
    ]

    q_A = np.maximum(
        np.asarray(
            q_by_group[
                str(
                    levels[
                        0
                    ]
                )
            ],
            dtype=float,
        ),
        1e-300,
    )

    q_B = np.maximum(
        np.asarray(
            q_by_group[
                str(
                    levels[
                        1
                    ]
                )
            ],
            dtype=float,
        ),
        1e-300,
    )

    return (
        np.log(
            q_B
        )
        - np.log(
            q_A
        )
    ) / np.log(
        2.0
    )


def _current_shrinkage_fraction(
    alpha0,
    a,
):
    return float(
        alpha0
        / (
            a
            + alpha0
        )
    )


def _capped_deb_theta(
    R,
    a,
    alpha0,
    prior_mean,
    c_max,
    theta_floor,
    theta_cap,
    corrected_score_tolerance,
):
    R = np.asarray(
        R,
        dtype=float,
    )

    a = float(
        a
    )

    alpha0 = float(
        alpha0
    )

    prior_mean = float(
        prior_mean
    )

    c_max = float(
        c_max
    )

    if not (
        np.isfinite(
            prior_mean
        )
        and prior_mean
        > 0
    ):
        raise RuntimeError(
            f"Invalid fitted prior mean: {prior_mean}"
        )

    if not (
        np.isfinite(
            alpha0
        )
        and alpha0
        > 0
    ):
        raise RuntimeError(
            f"Invalid fitted prior concentration alpha0: {alpha0}"
        )

    if not (
        np.isfinite(
            c_max
        )
        and c_max
        > 0
    ):
        raise RuntimeError(
            f"Invalid c_max: {c_max}"
        )

    alpha_star = min(
        alpha0,
        c_max
        * a,
    )

    beta_star = (
        alpha_star
        / prior_mean
    )

    theta = np.full(
        len(
            R
        ),
        np.nan,
        dtype=float,
    )

    valid = (
        np.isfinite(
            R
        )
        & (
            R
            > float(
                corrected_score_tolerance
            )
        )
    )

    theta[
        valid
    ] = (
        a
        + alpha_star
    ) / (
        R[
            valid
        ]
        + beta_star
    )

    theta[
        valid
    ] = np.clip(
        theta[
            valid
        ],
        float(
            theta_floor
        ),
        float(
            theta_cap
        ),
    )

    s_used = float(
        alpha_star
        / (
            a
            + alpha_star
        )
    )

    return {
        "theta":
            theta,

        "alpha_star":
            float(
                alpha_star
            ),

        "beta_star":
            float(
                beta_star
            ),

        "s_used":
            s_used,

        "cap_bound":
            bool(
                alpha_star
                < alpha0
                * (
                    1.0
                    - 1e-14
                )
            ),
    }


# =============================================================================
# ONE SPLIT
# =============================================================================

def _audit_one_split(
    row: pd.Series,
):
    split_id = str(
        row[
            "split_id"
        ]
    )

    dataset = str(
        row[
            "dataset"
        ]
    )

    replicate = int(
        row[
            "replicate"
        ]
    )

    cache_path = (
        PER_SPLIT_DIR
        / f"{_safe(split_id)}.pkl"
    )

    if (
        RESUME_AUDIT
        and cache_path.exists()
    ):
        try:
            payload = _load_pickle(
                cache_path
            )

            if (
                isinstance(
                    payload,
                    dict,
                )
                and payload.get(
                    "audit_version"
                )
                == AUDIT_VERSION
            ):
                return payload, True

        except Exception:
            pass

    t0 = time.perf_counter()

    core = _load_success_stage(
        split_id,
        "01_core",
    )

    full = _load_success_stage(
        split_id,
        "02_full_firstorder",
    )

    geom = _load_success_stage(
        split_id,
        "03_geometry_support",
    )

    null = _load_success_stage(
        split_id,
        "04_null_deb",
    )

    base = _load_success_stage(
        split_id,
        "05_base_m03_result",
    )

    manifest = _load_manifest(
        split_id
    )

    settings = dict(
        manifest[
            "numerical_settings"
        ]
    )

    corrected_score_tolerance = float(
        settings[
            "corrected_score_tolerance"
        ]
    )

    theta_floor = float(
        settings[
            "theta_floor"
        ]
    )

    theta_cap = float(
        settings[
            "theta_cap"
        ]
    )

    y = np.asarray(
        core[
            "y"
        ],
        dtype=float,
    )

    exposure = np.asarray(
        core[
            "exposure"
        ],
        dtype=float,
    )

    S_F = np.asarray(
        core[
            "S_F"
        ],
        dtype=float,
    )

    a_F = float(
        core[
            "a_F"
        ]
    )

    a_0 = float(
        core[
            "a_0"
        ]
    )

    n = int(
        core[
            "n"
        ]
    )

    C_F = np.asarray(
        full[
            "C_F"
        ],
        dtype=float,
    )

    C_0 = np.asarray(
        full[
            "C_0"
        ],
        dtype=float,
    )

    R_F = np.asarray(
        full[
            "R_F"
        ],
        dtype=float,
    )

    theta_F_current = np.asarray(
        full[
            "theta_F_first"
        ],
        dtype=float,
    )

    mask_Rnonpos = np.asarray(
        full[
            "mask_Rnonpos"
        ],
        dtype=bool,
    )

    full_theta_fit = dict(
        full[
            "full_theta_fit"
        ]
    )

    S0dag_final = np.asarray(
        geom[
            "S0dag_final"
        ],
        dtype=float,
    )

    kappa_c = float(
        geom[
            "kappa_c"
        ]
    )

    lambda_c = float(
        geom[
            "lambda_c"
        ]
    )

    R0dag = np.asarray(
        null[
            "R_0_dagger_first"
        ],
        dtype=float,
    )

    theta_0_current = np.asarray(
        null[
            "theta_0_first"
        ],
        dtype=float,
    )

    null_theta_fit = dict(
        null[
            "null_theta_fit"
        ]
    )

    test_first_saved = null[
        "test_first"
    ]

    p_current = np.asarray(
        base[
            "p_value_m03"
        ],
        dtype=float,
    )

    q_current = np.asarray(
        base[
            "q_value_m03"
        ],
        dtype=float,
    )

    n_genes = int(
        y.shape[
            0
        ]
    )

    if len(
        p_current
    ) != n_genes:
        raise RuntimeError(
            f"{split_id}: p-value length mismatch."
        )

    # -----------------------------------------------------------------
    # Validate our first-order statistic reconstruction against the
    # checkpoint before changing DEB concentration.
    # -----------------------------------------------------------------

    reconstructed = _normalized_statistic(
        S_F,
        S0dag_final,
        C_F,
        C_0,
        theta_F_current,
        theta_0_current,
        n,
        corrected_score_tolerance,
    )

    p_first_saved = np.asarray(
        test_first_saved[
            "p"
        ],
        dtype=float,
    )

    compare = (
        np.isfinite(
            reconstructed[
                "p"
            ]
        )
        & np.isfinite(
            p_first_saved
        )
    )

    if np.any(
        compare
    ):
        max_firstorder_p_error = float(
            np.max(
                np.abs(
                    reconstructed[
                        "p"
                    ][
                        compare
                    ]
                    - p_first_saved[
                        compare
                    ]
                )
            )
        )

        if max_firstorder_p_error > 1e-12:
            raise RuntimeError(
                f"{split_id}: first-order p reconstruction failed; "
                f"max abs error={max_firstorder_p_error:.3e}"
            )

    # Canonical M03 should equal first-order p for ordinary genes.
    ordinary_compare = (
        ~mask_Rnonpos
        & np.isfinite(
            p_current
        )
        & np.isfinite(
            p_first_saved
        )
    )

    if np.any(
        ordinary_compare
    ):
        max_canonical_ordinary_error = float(
            np.max(
                np.abs(
                    p_current[
                        ordinary_compare
                    ]
                    - p_first_saved[
                        ordinary_compare
                    ]
                )
            )
        )

        if max_canonical_ordinary_error > 1e-12:
            raise RuntimeError(
                f"{split_id}: canonical p does not match first-order p on "
                "ordinary genes; "
                f"max abs error={max_canonical_ordinary_error:.3e}"
            )

    # -----------------------------------------------------------------
    # Reconstruct log2FC / Li2020 fixed denominator.
    # -----------------------------------------------------------------

    log2fc = _reconstruct_log2fc(
        core
    )

    n_genes_fixed = int(
        row[
            "n_genes_fixed"
        ]
    )

    if n_genes_fixed <= 0:
        raise RuntimeError(
            f"{split_id}: invalid n_genes_fixed={n_genes_fixed}"
        )

    current_call = (
        np.isfinite(
            q_current
        )
        & (
            q_current
            <= FDR_CUTOFF
        )
        & np.isfinite(
            log2fc
        )
        & (
            np.abs(
                log2fc
            )
            >= ABS_LOG2FC_CUTOFF
        )
    )

    current_fpr_reconstructed = float(
        np.sum(
            current_call
        )
        / n_genes_fixed
    )

    current_fpr_saved = float(
        row.get(
            "fpr_li2020",
            np.nan,
        )
    )

    if np.isfinite(
        current_fpr_saved
    ):
        if abs(
            current_fpr_reconstructed
            - current_fpr_saved
        ) > 1e-12:
            raise RuntimeError(
                f"{split_id}: current Li2020 FPR reconstruction mismatch: "
                f"saved={current_fpr_saved:.12g}, "
                f"reconstructed={current_fpr_reconstructed:.12g}"
            )

    # -----------------------------------------------------------------
    # Current DEB hyperparameters.
    # -----------------------------------------------------------------

    alpha_F = float(
        full_theta_fit[
            "alpha0"
        ]
    )

    beta_F = float(
        full_theta_fit[
            "beta0"
        ]
    )

    prior_mean_F = float(
        full_theta_fit[
            "prior_mean_theta"
        ]
    )

    alpha_0 = float(
        null_theta_fit[
            "alpha0"
        ]
    )

    beta_0 = float(
        null_theta_fit[
            "beta0"
        ]
    )

    prior_mean_0 = float(
        null_theta_fit[
            "prior_mean_theta"
        ]
    )

    for (
        alpha,
        beta,
        mean,
        label,
    ) in (
        (
            alpha_F,
            beta_F,
            prior_mean_F,
            "full",
        ),
        (
            alpha_0,
            beta_0,
            prior_mean_0,
            "null",
        ),
    ):
        implied = (
            alpha
            / beta
        )

        rel = abs(
            implied
            - mean
        ) / max(
            abs(
                mean
            ),
            1e-15,
        )

        if rel > 1e-10:
            raise RuntimeError(
                f"{split_id}: {label} prior mean mismatch "
                f"alpha/beta={implied}, saved mean={mean}."
            )

    current_s_F = _current_shrinkage_fraction(
        alpha_F,
        a_F,
    )

    current_s_0 = _current_shrinkage_fraction(
        alpha_0,
        a_0,
    )

    # -----------------------------------------------------------------
    # Coverage bins are frozen within each split.
    # -----------------------------------------------------------------

    total_normalized_coverage = np.sum(
        y
        / exposure[
            None,
            :
        ],
        axis=1,
    )

    log_coverage = np.log1p(
        total_normalized_coverage
    )

    coverage_bin = (
        np.asarray(
            pd.qcut(
                pd.Series(
                    log_coverage
                ),
                q=N_COVERAGE_BINS,
                labels=False,
                duplicates="drop",
            ),
            dtype=int,
        )
        + 1
    )

    actual_bins = sorted(
        int(
            b
        )
        for b in np.unique(
            coverage_bin
        )
    )

    # -----------------------------------------------------------------
    # Scoring helper.
    # -----------------------------------------------------------------

    split_rows = []
    tail_rows = []
    bin_rows = []

    def score_method(
        method,
        c_max,
        theta_F,
        theta_0,
        p,
        s_F_used,
        s_0_used,
        alpha_F_used,
        alpha_0_used,
        cap_bound_F,
        cap_bound_0,
    ):
        p = np.asarray(
            p,
            dtype=float,
        )

        q = _bh_adjust(
            p
        )

        finite = np.isfinite(
            p
        )

        li_call = (
            np.isfinite(
                q
            )
            & (
                q
                <= FDR_CUTOFF
            )
            & np.isfinite(
                log2fc
            )
            & (
                np.abs(
                    log2fc
                )
                >= ABS_LOG2FC_CUTOFF
            )
        )

        bh_call = (
            np.isfinite(
                q
            )
            & (
                q
                <= FDR_CUTOFF
            )
        )

        n_li_calls = int(
            np.sum(
                li_call
            )
        )

        n_bh_calls = int(
            np.sum(
                bh_call
            )
        )

        fpr = float(
            n_li_calls
            / n_genes_fixed
        )

        min_p = (
            float(
                np.min(
                    p[
                        finite
                    ]
                )
            )
            if np.any(
                finite
            )
            else np.nan
        )

        first_bh_threshold = (
            float(
                FDR_CUTOFF
                / int(
                    np.sum(
                        finite
                    )
                )
            )
            if np.any(
                finite
            )
            else np.nan
        )

        split_rows.append(
            {
                "split_id":
                    split_id,

                "dataset":
                    dataset,

                "replicate":
                    replicate,

                "method":
                    method,

                "c_max":
                    c_max,

                "n_genes_analyzed":
                    n_genes,

                "n_genes_fixed":
                    n_genes_fixed,

                "kappa_c_frozen":
                    kappa_c,

                "lambda_c_frozen":
                    lambda_c,

                "current_s_F":
                    current_s_F,

                "current_s_0":
                    current_s_0,

                "s_F_used":
                    s_F_used,

                "s_0_used":
                    s_0_used,

                "alpha_F_original":
                    alpha_F,

                "alpha_0_original":
                    alpha_0,

                "alpha_F_used":
                    alpha_F_used,

                "alpha_0_used":
                    alpha_0_used,

                "cap_bound_F":
                    cap_bound_F,

                "cap_bound_0":
                    cap_bound_0,

                "n_mandatory_exact_reused":
                    int(
                        np.sum(
                            mask_Rnonpos
                        )
                    ),

                "n_finite_p":
                    int(
                        np.sum(
                            finite
                        )
                    ),

                "frac_raw_p_lt_0_01":
                    float(
                        np.mean(
                            p[
                                finite
                            ]
                            < 0.01
                        )
                    )
                    if np.any(
                        finite
                    )
                    else np.nan,

                "li2020_calls":
                    n_li_calls,

                "fpr_li2020":
                    fpr,

                "BH05_discoveries_no_fc_filter":
                    n_bh_calls,

                "any_BH05_no_fc_filter":
                    int(
                        n_bh_calls
                        > 0
                    ),

                "any_Li2020_call":
                    int(
                        n_li_calls
                        > 0
                    ),

                "min_p":
                    min_p,

                "first_BH_threshold":
                    first_bh_threshold,

                "min_p_over_first_BH":
                    (
                        float(
                            min_p
                            / first_bh_threshold
                        )
                        if (
                            np.isfinite(
                                min_p
                            )
                            and np.isfinite(
                                first_bh_threshold
                            )
                            and first_bh_threshold
                            > 0
                        )
                        else np.nan
                    ),
            }
        )

        for threshold in TAIL_THRESHOLDS:
            count = int(
                np.sum(
                    finite
                    & (
                        p
                        <= float(
                            threshold
                        )
                    )
                )
            )

            tail_rows.append(
                {
                    "split_id":
                        split_id,

                    "dataset":
                        dataset,

                    "replicate":
                        replicate,

                    "method":
                        method,

                    "c_max":
                        c_max,

                    "threshold":
                        float(
                            threshold
                        ),

                    "n_finite_p":
                        int(
                            np.sum(
                                finite
                            )
                        ),

                    "count_le_threshold":
                        count,

                    "fraction_le_threshold":
                        (
                            float(
                                count
                                / int(
                                    np.sum(
                                        finite
                                    )
                                )
                            )
                            if np.any(
                                finite
                            )
                            else np.nan
                        ),
                }
            )

        for b in actual_bins:
            idx = (
                coverage_bin
                == b
            )

            n_bin = int(
                np.sum(
                    idx
                )
            )

            finite_bin = (
                idx
                & finite
            )

            bin_rows.append(
                {
                    "split_id":
                        split_id,

                    "dataset":
                        dataset,

                    "replicate":
                        replicate,

                    "method":
                        method,

                    "c_max":
                        c_max,

                    "coverage_bin":
                        int(
                            b
                        ),

                    "n_genes_bin":
                        n_bin,

                    "median_log1p_total_normalized_coverage":
                        float(
                            np.median(
                                log_coverage[
                                    idx
                                ]
                            )
                        ),

                    "median_theta_F":
                        float(
                            np.nanmedian(
                                theta_F[
                                    idx
                                ]
                            )
                        ),

                    "median_theta_0":
                        float(
                            np.nanmedian(
                                theta_0[
                                    idx
                                ]
                            )
                        ),

                    "raw_p_lt_0_01_fraction":
                        (
                            float(
                                np.mean(
                                    p[
                                        finite_bin
                                    ]
                                    < 0.01
                                )
                            )
                            if np.any(
                                finite_bin
                            )
                            else np.nan
                        ),

                    "Li2020_call_fraction_of_bin":
                        float(
                            np.sum(
                                li_call
                                & idx
                            )
                            / max(
                                n_bin,
                                1,
                            )
                        ),
                }
            )

        return q

    # -----------------------------------------------------------------
    # Current canonical M03 reference.
    # -----------------------------------------------------------------

    score_method(
        method="current_DEB",
        c_max=np.nan,
        theta_F=theta_F_current,
        theta_0=theta_0_current,
        p=p_current,
        s_F_used=current_s_F,
        s_0_used=current_s_0,
        alpha_F_used=alpha_F,
        alpha_0_used=alpha_0,
        cap_bound_F=False,
        cap_bound_0=False,
    )

    # -----------------------------------------------------------------
    # Three frozen capped-DEB candidates.
    # -----------------------------------------------------------------

    for c_max in C_MAX_VALUES:
        full_cap = _capped_deb_theta(
            R=R_F,
            a=a_F,
            alpha0=alpha_F,
            prior_mean=prior_mean_F,
            c_max=c_max,
            theta_floor=theta_floor,
            theta_cap=theta_cap,
            corrected_score_tolerance=corrected_score_tolerance,
        )

        null_cap = _capped_deb_theta(
            R=R0dag,
            a=a_0,
            alpha0=alpha_0,
            prior_mean=prior_mean_0,
            c_max=c_max,
            theta_floor=theta_floor,
            theta_cap=theta_cap,
            corrected_score_tolerance=corrected_score_tolerance,
        )

        test = _normalized_statistic(
            S_F=S_F,
            S_0_used=S0dag_final,
            C_F=C_F,
            C_0=C_0,
            theta_F=full_cap[
                "theta"
            ],
            theta_0=null_cap[
                "theta"
            ],
            n=n,
            corrected_score_tolerance=corrected_score_tolerance,
        )

        p_candidate = np.asarray(
            test[
                "p"
            ],
            dtype=float,
        ).copy()

        # Mandatory exact normalized-DNB profile does not depend on
        # ordinary DEB prior concentration. Reuse canonical exact p-values.
        p_candidate[
            mask_Rnonpos
        ] = p_current[
            mask_Rnonpos
        ]

        score_method(
            method=f"cmax={c_max:g}",
            c_max=float(
                c_max
            ),
            theta_F=full_cap[
                "theta"
            ],
            theta_0=null_cap[
                "theta"
            ],
            p=p_candidate,
            s_F_used=full_cap[
                "s_used"
            ],
            s_0_used=null_cap[
                "s_used"
            ],
            alpha_F_used=full_cap[
                "alpha_star"
            ],
            alpha_0_used=null_cap[
                "alpha_star"
            ],
            cap_bound_F=full_cap[
                "cap_bound"
            ],
            cap_bound_0=null_cap[
                "cap_bound"
            ],
        )

    payload = {
        "audit_version":
            AUDIT_VERSION,

        "split_id":
            split_id,

        "dataset":
            dataset,

        "replicate":
            replicate,

        "split_results":
            pd.DataFrame(
                split_rows
            ),

        "tail_results":
            pd.DataFrame(
                tail_rows
            ),

        "coverage_results":
            pd.DataFrame(
                bin_rows
            ),

        "elapsed_seconds":
            float(
                time.perf_counter()
                - t0
            ),
    }

    _atomic_pickle(
        payload,
        cache_path,
    )

    return payload, False


# =============================================================================
# AGGREGATION
# =============================================================================

def _aggregate_method_summary(
    split_results: pd.DataFrame,
) -> pd.DataFrame:
    rows = []

    for (
        dataset,
        method,
    ), d in split_results.groupby(
        [
            "dataset",
            "method",
        ],
        sort=True,
    ):
        c_values = d[
            "c_max"
        ].dropna()

        c_max = (
            float(
                c_values.iloc[
                    0
                ]
            )
            if len(
                c_values
            )
            else np.nan
        )

        rows.append(
            {
                "dataset":
                    dataset,

                "method":
                    method,

                "c_max":
                    c_max,

                "n_splits":
                    int(
                        len(
                            d
                        )
                    ),

                "mean_FPR":
                    float(
                        d[
                            "fpr_li2020"
                        ]
                        .mean()
                    ),

                "median_FPR":
                    float(
                        d[
                            "fpr_li2020"
                        ]
                        .median()
                    ),

                "q25_FPR":
                    float(
                        d[
                            "fpr_li2020"
                        ]
                        .quantile(
                            0.25
                        )
                    ),

                "q75_FPR":
                    float(
                        d[
                            "fpr_li2020"
                        ]
                        .quantile(
                            0.75
                        )
                    ),

                "q975_FPR":
                    float(
                        d[
                            "fpr_li2020"
                        ]
                        .quantile(
                            0.975
                        )
                    ),

                "mean_raw_p01":
                    float(
                        d[
                            "frac_raw_p_lt_0_01"
                        ]
                        .mean()
                    ),

                "median_raw_p01":
                    float(
                        d[
                            "frac_raw_p_lt_0_01"
                        ]
                        .median()
                    ),

                "complete_null_BH_FDR_no_fc_filter":
                    float(
                        d[
                            "any_BH05_no_fc_filter"
                        ]
                        .mean()
                    ),

                "P_any_Li2020_call":
                    float(
                        d[
                            "any_Li2020_call"
                        ]
                        .mean()
                    ),

                "mean_BH05_discoveries_no_fc_filter":
                    float(
                        d[
                            "BH05_discoveries_no_fc_filter"
                        ]
                        .mean()
                    ),

                "mean_Li2020_calls":
                    float(
                        d[
                            "li2020_calls"
                        ]
                        .mean()
                    ),

                "median_min_p":
                    float(
                        d[
                            "min_p"
                        ]
                        .median()
                    ),

                "median_s_F_used":
                    float(
                        d[
                            "s_F_used"
                        ]
                        .median()
                    ),

                "median_s_0_used":
                    float(
                        d[
                            "s_0_used"
                        ]
                        .median()
                    ),

                "fraction_full_cap_bound":
                    float(
                        d[
                            "cap_bound_F"
                        ]
                        .mean()
                    ),

                "fraction_null_cap_bound":
                    float(
                        d[
                            "cap_bound_0"
                        ]
                        .mean()
                    ),

                "mean_exact_reused":
                    float(
                        d[
                            "n_mandatory_exact_reused"
                        ]
                        .mean()
                    ),
            }
        )

    return (
        pd.DataFrame(
            rows
        )
        .sort_values(
            [
                "dataset",
                "method",
            ],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )


def _aggregate_paired_changes(
    split_results: pd.DataFrame,
) -> pd.DataFrame:
    current = split_results.loc[
        split_results[
            "method"
        ]
        == "current_DEB"
    ][
        [
            "split_id",
            "dataset",
            "replicate",
            "fpr_li2020",
            "frac_raw_p_lt_0_01",
            "BH05_discoveries_no_fc_filter",
            "li2020_calls",
            "min_p",
        ]
    ].copy()

    current = current.rename(
        columns={
            "fpr_li2020":
                "current_FPR",

            "frac_raw_p_lt_0_01":
                "current_raw_p01",

            "BH05_discoveries_no_fc_filter":
                "current_BH05",

            "li2020_calls":
                "current_Li_calls",

            "min_p":
                "current_min_p",
        }
    )

    candidates = split_results.loc[
        split_results[
            "method"
        ]
        != "current_DEB"
    ].copy()

    merged = candidates.merge(
        current,
        on=[
            "split_id",
            "dataset",
            "replicate",
        ],
        how="left",
        validate="many_to_one",
    )

    merged[
        "delta_FPR"
    ] = (
        merged[
            "fpr_li2020"
        ]
        - merged[
            "current_FPR"
        ]
    )

    merged[
        "delta_raw_p01"
    ] = (
        merged[
            "frac_raw_p_lt_0_01"
        ]
        - merged[
            "current_raw_p01"
        ]
    )

    merged[
        "delta_BH05"
    ] = (
        merged[
            "BH05_discoveries_no_fc_filter"
        ]
        - merged[
            "current_BH05"
        ]
    )

    merged[
        "delta_Li_calls"
    ] = (
        merged[
            "li2020_calls"
        ]
        - merged[
            "current_Li_calls"
        ]
    )

    rows = []

    for (
        dataset,
        method,
    ), d in merged.groupby(
        [
            "dataset",
            "method",
        ],
        sort=True,
    ):
        rows.append(
            {
                "dataset":
                    dataset,

                "method":
                    method,

                "c_max":
                    float(
                        d[
                            "c_max"
                        ]
                        .iloc[
                            0
                        ]
                    ),

                "n_splits":
                    int(
                        len(
                            d
                        )
                    ),

                "mean_delta_FPR":
                    float(
                        d[
                            "delta_FPR"
                        ]
                        .mean()
                    ),

                "median_delta_FPR":
                    float(
                        d[
                            "delta_FPR"
                        ]
                        .median()
                    ),

                "n_FPR_improved":
                    int(
                        np.sum(
                            d[
                                "delta_FPR"
                            ]
                            < 0
                        )
                    ),

                "n_FPR_equal":
                    int(
                        np.sum(
                            d[
                                "delta_FPR"
                            ]
                            == 0
                        )
                    ),

                "n_FPR_worsened":
                    int(
                        np.sum(
                            d[
                                "delta_FPR"
                            ]
                            > 0
                        )
                    ),

                "mean_delta_raw_p01":
                    float(
                        d[
                            "delta_raw_p01"
                        ]
                        .mean()
                    ),

                "mean_delta_BH05":
                    float(
                        d[
                            "delta_BH05"
                        ]
                        .mean()
                    ),

                "mean_delta_Li2020_calls":
                    float(
                        d[
                            "delta_Li_calls"
                        ]
                        .mean()
                    ),

                "median_candidate_min_p":
                    float(
                        d[
                            "min_p"
                        ]
                        .median()
                    ),

                "median_current_min_p":
                    float(
                        d[
                            "current_min_p"
                        ]
                        .median()
                    ),
            }
        )

    return (
        pd.DataFrame(
            rows
        )
        .sort_values(
            [
                "dataset",
                "c_max",
            ],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )


def _aggregate_tail(
    tail_results: pd.DataFrame,
) -> pd.DataFrame:
    rows = []

    for (
        dataset,
        method,
        threshold,
    ), d in tail_results.groupby(
        [
            "dataset",
            "method",
            "threshold",
        ],
        sort=True,
    ):
        pooled_count = int(
            d[
                "count_le_threshold"
            ]
            .sum()
        )

        pooled_n = int(
            d[
                "n_finite_p"
            ]
            .sum()
        )

        pooled_fraction = (
            float(
                pooled_count
                / pooled_n
            )
            if pooled_n
            else np.nan
        )

        c_values = d[
            "c_max"
        ].dropna()

        c_max = (
            float(
                c_values.iloc[
                    0
                ]
            )
            if len(
                c_values
            )
            else np.nan
        )

        rows.append(
            {
                "dataset":
                    dataset,

                "method":
                    method,

                "c_max":
                    c_max,

                "threshold":
                    float(
                        threshold
                    ),

                "mean_split_fraction":
                    float(
                        d[
                            "fraction_le_threshold"
                        ]
                        .mean()
                    ),

                "pooled_count":
                    pooled_count,

                "pooled_n":
                    pooled_n,

                "pooled_fraction":
                    pooled_fraction,

                "mean_ratio_to_nominal":
                    float(
                        d[
                            "fraction_le_threshold"
                        ]
                        .mean()
                        / float(
                            threshold
                        )
                    ),

                "pooled_ratio_to_nominal":
                    (
                        float(
                            pooled_fraction
                            / float(
                                threshold
                            )
                        )
                        if np.isfinite(
                            pooled_fraction
                        )
                        else np.nan
                    ),
            }
        )

    return (
        pd.DataFrame(
            rows
        )
        .sort_values(
            [
                "dataset",
                "method",
                "threshold",
            ],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )


def _aggregate_coverage(
    coverage_results: pd.DataFrame,
) -> pd.DataFrame:
    rows = []

    for (
        dataset,
        method,
        b,
    ), d in coverage_results.groupby(
        [
            "dataset",
            "method",
            "coverage_bin",
        ],
        sort=True,
    ):
        c_values = d[
            "c_max"
        ].dropna()

        c_max = (
            float(
                c_values.iloc[
                    0
                ]
            )
            if len(
                c_values
            )
            else np.nan
        )

        rows.append(
            {
                "dataset":
                    dataset,

                "method":
                    method,

                "c_max":
                    c_max,

                "coverage_bin":
                    int(
                        b
                    ),

                "n_splits":
                    int(
                        d[
                            "split_id"
                        ]
                        .nunique()
                    ),

                "median_log1p_total_normalized_coverage":
                    float(
                        d[
                            "median_log1p_total_normalized_coverage"
                        ]
                        .median()
                    ),

                "median_theta_F":
                    float(
                        d[
                            "median_theta_F"
                        ]
                        .median()
                    ),

                "q25_theta_F":
                    float(
                        d[
                            "median_theta_F"
                        ]
                        .quantile(
                            0.25
                        )
                    ),

                "q75_theta_F":
                    float(
                        d[
                            "median_theta_F"
                        ]
                        .quantile(
                            0.75
                        )
                    ),

                "median_theta_0":
                    float(
                        d[
                            "median_theta_0"
                        ]
                        .median()
                    ),

                "q25_theta_0":
                    float(
                        d[
                            "median_theta_0"
                        ]
                        .quantile(
                            0.25
                        )
                    ),

                "q75_theta_0":
                    float(
                        d[
                            "median_theta_0"
                        ]
                        .quantile(
                            0.75
                        )
                    ),

                "median_raw_p_lt_0_01_fraction":
                    float(
                        d[
                            "raw_p_lt_0_01_fraction"
                        ]
                        .median()
                    ),

                "q25_raw_p_lt_0_01_fraction":
                    float(
                        d[
                            "raw_p_lt_0_01_fraction"
                        ]
                        .quantile(
                            0.25
                        )
                    ),

                "q75_raw_p_lt_0_01_fraction":
                    float(
                        d[
                            "raw_p_lt_0_01_fraction"
                        ]
                        .quantile(
                            0.75
                        )
                    ),

                "median_Li2020_call_fraction_of_bin":
                    float(
                        d[
                            "Li2020_call_fraction_of_bin"
                        ]
                        .median()
                    ),
            }
        )

    return (
        pd.DataFrame(
            rows
        )
        .sort_values(
            [
                "dataset",
                "method",
                "coverage_bin",
            ],
            kind="stable",
        )
        .reset_index(
            drop=True
        )
    )


# =============================================================================
# PLOTS — DISPLAY ONLY
# =============================================================================

def _method_order():
    return (
        "current_DEB",
        "cmax=0.25",
        "cmax=1",
        "cmax=3",
    )


def _show_plots(
    split_results,
    method_summary,
    tail_aggregate,
    coverage_aggregate,
):
    methods = _method_order()

    for dataset in TARGET_DATASETS:
        # ---------------------------------------------------------
        # 1. FPR boxplot.
        # ---------------------------------------------------------
        fig, ax = plt.subplots(
            figsize=FIGSIZE
        )

        data = []

        labels = []

        for method in methods:
            x = split_results.loc[
                (
                    split_results[
                        "dataset"
                    ]
                    == dataset
                )
                & (
                    split_results[
                        "method"
                    ]
                    == method
                ),
                "fpr_li2020",
            ]

            data.append(
                x.to_numpy(
                    dtype=float
                )
            )

            labels.append(
                method
            )

        ax.boxplot(
            data,
            tick_labels=labels,
            showfliers=True,
        )

        ax.set_ylabel(
            "Li2020 FPR",
            fontsize=14,
        )

        ax.set_title(
            f"{dataset} 40v40 reps 1–20: current vs capped DEB",
            fontsize=18,
            pad=16,
        )

        ax.grid(
            axis="y",
            alpha=0.20,
        )

        plt.tight_layout()
        plt.show()

        # ---------------------------------------------------------
        # 2. Paired FPR current vs each candidate.
        # ---------------------------------------------------------
        current = split_results.loc[
            (
                split_results[
                    "dataset"
                ]
                == dataset
            )
            & (
                split_results[
                    "method"
                ]
                == "current_DEB"
            ),
            [
                "split_id",
                "fpr_li2020",
            ],
        ].rename(
            columns={
                "fpr_li2020":
                    "current_FPR"
            }
        )

        for method in methods[
            1:
        ]:
            cand = split_results.loc[
                (
                    split_results[
                        "dataset"
                    ]
                    == dataset
                )
                & (
                    split_results[
                        "method"
                    ]
                    == method
                ),
                [
                    "split_id",
                    "fpr_li2020",
                ],
            ].rename(
                columns={
                    "fpr_li2020":
                        "candidate_FPR"
                }
            )

            z = current.merge(
                cand,
                on="split_id",
                how="inner",
                validate="one_to_one",
            )

            fig, ax = plt.subplots(
                figsize=FIGSIZE
            )

            ax.scatter(
                z[
                    "current_FPR"
                ],
                z[
                    "candidate_FPR"
                ],
                s=70,
                alpha=0.85,
            )

            vmax = float(
                np.nanmax(
                    np.concatenate(
                        [
                            z[
                                "current_FPR"
                            ]
                            .to_numpy(
                                dtype=float
                            ),
                            z[
                                "candidate_FPR"
                            ]
                            .to_numpy(
                                dtype=float
                            ),
                        ]
                    )
                )
            )

            vmax = max(
                vmax,
                1e-8,
            )

            ax.plot(
                [
                    0,
                    vmax
                    * 1.05,
                ],
                [
                    0,
                    vmax
                    * 1.05,
                ],
                linestyle="--",
                linewidth=1.5,
                label="No change",
            )

            ax.set_xlabel(
                "Current DEB Li2020 FPR",
                fontsize=14,
            )

            ax.set_ylabel(
                f"{method} Li2020 FPR",
                fontsize=14,
            )

            ax.set_title(
                f"{dataset} 40v40 reps 1–20: paired FPR — {method}",
                fontsize=18,
                pad=16,
            )

            ax.grid(
                alpha=0.20,
            )

            ax.legend(
                fontsize=11,
            )

            plt.tight_layout()
            plt.show()

        # ---------------------------------------------------------
        # 3. Raw p<.01 calibration by coverage.
        # ---------------------------------------------------------
        fig, ax = plt.subplots(
            figsize=FIGSIZE
        )

        for method in methods:
            d = (
                coverage_aggregate.loc[
                    (
                        coverage_aggregate[
                            "dataset"
                        ]
                        == dataset
                    )
                    & (
                        coverage_aggregate[
                            "method"
                        ]
                        == method
                    )
                ]
                .sort_values(
                    "coverage_bin"
                )
            )

            ax.plot(
                d[
                    "median_log1p_total_normalized_coverage"
                ],
                d[
                    "median_raw_p_lt_0_01_fraction"
                ],
                marker="o",
                linewidth=2.0,
                label=method,
            )

        ax.axhline(
            0.01,
            linestyle=":",
            linewidth=1.5,
            label="Nominal 1%",
        )

        ax.set_xlabel(
            "log1p(total normalized gene coverage)",
            fontsize=14,
        )

        ax.set_ylabel(
            "Median fraction raw p < .01",
            fontsize=14,
        )

        ax.set_title(
            f"{dataset} 40v40 reps 1–20: calibration by coverage",
            fontsize=18,
            pad=16,
        )

        ax.grid(
            alpha=0.20,
        )

        ax.legend(
            fontsize=10,
        )

        plt.tight_layout()
        plt.show()

        # ---------------------------------------------------------
        # 4. Full theta by coverage.
        # ---------------------------------------------------------
        fig, ax = plt.subplots(
            figsize=FIGSIZE
        )

        for method in methods:
            d = (
                coverage_aggregate.loc[
                    (
                        coverage_aggregate[
                            "dataset"
                        ]
                        == dataset
                    )
                    & (
                        coverage_aggregate[
                            "method"
                        ]
                        == method
                    )
                ]
                .sort_values(
                    "coverage_bin"
                )
            )

            ax.plot(
                d[
                    "median_log1p_total_normalized_coverage"
                ],
                d[
                    "median_theta_F"
                ],
                marker="o",
                linewidth=2.0,
                label=method,
            )

        ax.set_xlabel(
            "log1p(total normalized gene coverage)",
            fontsize=14,
        )

        ax.set_ylabel(
            r"Median $\theta_F$",
            fontsize=14,
        )

        ax.set_title(
            f"{dataset} 40v40 reps 1–20: full precision by coverage",
            fontsize=18,
            pad=16,
        )

        ax.grid(
            alpha=0.20,
        )

        ax.legend(
            fontsize=10,
        )

        plt.tight_layout()
        plt.show()

        # ---------------------------------------------------------
        # 5. Null theta by coverage.
        # ---------------------------------------------------------
        fig, ax = plt.subplots(
            figsize=FIGSIZE
        )

        for method in methods:
            d = (
                coverage_aggregate.loc[
                    (
                        coverage_aggregate[
                            "dataset"
                        ]
                        == dataset
                    )
                    & (
                        coverage_aggregate[
                            "method"
                        ]
                        == method
                    )
                ]
                .sort_values(
                    "coverage_bin"
                )
            )

            ax.plot(
                d[
                    "median_log1p_total_normalized_coverage"
                ],
                d[
                    "median_theta_0"
                ],
                marker="o",
                linewidth=2.0,
                label=method,
            )

        ax.set_xlabel(
            "log1p(total normalized gene coverage)",
            fontsize=14,
        )

        ax.set_ylabel(
            r"Median $\theta_0$",
            fontsize=14,
        )

        ax.set_title(
            f"{dataset} 40v40 reps 1–20: null precision by coverage",
            fontsize=18,
            pad=16,
        )

        ax.grid(
            alpha=0.20,
        )

        ax.legend(
            fontsize=10,
        )

        plt.tight_layout()
        plt.show()

        # ---------------------------------------------------------
        # 6. Extreme-tail pooled calibration ratios.
        # ---------------------------------------------------------
        fig, ax = plt.subplots(
            figsize=FIGSIZE
        )

        for method in methods:
            d = (
                tail_aggregate.loc[
                    (
                        tail_aggregate[
                            "dataset"
                        ]
                        == dataset
                    )
                    & (
                        tail_aggregate[
                            "method"
                        ]
                        == method
                    )
                ]
                .sort_values(
                    "threshold"
                )
            )

            ax.plot(
                d[
                    "threshold"
                ],
                d[
                    "pooled_ratio_to_nominal"
                ],
                marker="o",
                linewidth=2.0,
                label=method,
            )

        ax.axhline(
            1.0,
            linestyle=":",
            linewidth=1.5,
            label="Calibrated",
        )

        ax.set_xscale(
            "log"
        )

        ax.set_yscale(
            "log"
        )

        ax.set_xlabel(
            "Raw-p threshold",
            fontsize=14,
        )

        ax.set_ylabel(
            "Pooled observed / nominal tail probability",
            fontsize=14,
        )

        ax.set_title(
            f"{dataset} 40v40 reps 1–20: extreme-tail calibration",
            fontsize=18,
            pad=16,
        )

        ax.grid(
            alpha=0.20,
            which="both",
        )

        ax.legend(
            fontsize=10,
        )

        plt.tight_layout()
        plt.show()

        # ---------------------------------------------------------
        # 7. Complete-null BH FDR by method.
        # ---------------------------------------------------------
        d = method_summary.loc[
            method_summary[
                "dataset"
            ]
            == dataset
        ].copy()

        # Force display order.
        d[
            "_order"
        ] = d[
            "method"
        ].map(
            {
                method: j
                for j, method
                in enumerate(
                    methods
                )
            }
        )

        d = d.sort_values(
            "_order"
        )

        fig, ax = plt.subplots(
            figsize=FIGSIZE
        )

        ax.bar(
            np.arange(
                len(
                    d
                )
            ),
            d[
                "complete_null_BH_FDR_no_fc_filter"
            ],
        )

        ax.axhline(
            0.05,
            linestyle=":",
            linewidth=1.5,
            label="Nominal 5%",
        )

        ax.set_xticks(
            np.arange(
                len(
                    d
                )
            )
        )

        ax.set_xticklabels(
            d[
                "method"
            ],
            rotation=20,
            ha="right",
        )

        ax.set_ylabel(
            "P(any BH(.05) rejection)",
            fontsize=14,
        )

        ax.set_title(
            f"{dataset} 40v40 reps 1–20: complete-null BH FDR",
            fontsize=18,
            pad=16,
        )

        ax.grid(
            axis="y",
            alpha=0.20,
        )

        ax.legend(
            fontsize=11,
        )

        plt.tight_layout()
        plt.show()


# =============================================================================
# TOP-LEVEL
# =============================================================================

def run_li2020_40v40_first20_capped_deb_audit():
    started = (
        datetime.now()
        .astimezone()
    )

    print(
        f"[start] {_now()}",
        flush=True,
    )

    try:
        selected = _discover_requested_splits()

        AUDIT_DIR.mkdir(
            parents=True,
            exist_ok=True,
        )

        PER_SPLIT_DIR.mkdir(
            parents=True,
            exist_ok=True,
        )

        if not SENTINEL.exists():
            SENTINEL.write_text(
                json.dumps(
                    {
                        "audit_version":
                            AUDIT_VERSION,

                        "initialized_local_time":
                            _now(),

                        "source_run_namespace":
                            RUN_NAMESPACE,

                        "datasets":
                            list(
                                TARGET_DATASETS
                            ),

                        "replicates":
                            list(
                                TARGET_REPLICATES
                            ),

                        "n_per_group":
                            int(
                                N_PER_GROUP
                            ),

                        "c_max_values":
                            list(
                                C_MAX_VALUES
                            ),

                        "global_kappa_frozen":
                            True,

                        "support_geometry_frozen":
                            True,

                        "prior_means_frozen":
                            True,

                        "mandatory_exact_reused":
                            True,

                        "resume_audit":
                            bool(
                                RESUME_AUDIT
                            ),
                    },
                    indent=2,
                    sort_keys=True,
                ),
                encoding="utf-8",
            )

        print(
            "\nLI2020 40v40 FIRST-20 CAPPED-DEB TRANSFER TEST\n"
            f"source namespace: {RUN_NAMESPACE}\n"
            f"requested splits: {len(selected)} "
            f"({len(TARGET_REPLICATES)} per dataset)\n"
            f"datasets:         {TARGET_DATASETS}\n"
            f"replicates:       1..20 exactly\n"
            f"c_max values:     {C_MAX_VALUES}\n"
            f"output:           {AUDIT_DIR}\n"
            "global kappa_c:   FROZEN\n"
            "support geometry: FROZEN\n"
            "prior means:      FROZEN\n"
            "exact branch:     REUSED\n",
            flush=True,
        )

        print(
            "Candidate maximum shrinkage fractions if the cap binds:",
            flush=True,
        )

        display(
            pd.DataFrame(
                {
                    "c_max":
                        C_MAX_VALUES,

                    "s_max_if_bound":
                        [
                            c
                            / (
                                1.0
                                + c
                            )
                            for c in C_MAX_VALUES
                        ],
                }
            )
        )

        payloads = []

        failures = []

        reused = 0

        for i, row in selected.iterrows():
            split_id = str(
                row[
                    "split_id"
                ]
            )

            try:
                payload, cached = _audit_one_split(
                    row
                )

                payloads.append(
                    payload
                )

                reused += int(
                    cached
                )

                x = payload[
                    "split_results"
                ]

                current_fpr = float(
                    x.loc[
                        x[
                            "method"
                        ]
                        == "current_DEB",
                        "fpr_li2020",
                    ]
                    .iloc[
                        0
                    ]
                )

                cap_fprs = {
                    str(
                        r[
                            "method"
                        ]
                    ):
                        float(
                            r[
                                "fpr_li2020"
                            ]
                        )
                    for _, r
                    in x.loc[
                        x[
                            "method"
                        ]
                        != "current_DEB"
                    ].iterrows()
                }

                print(
                    f"[{i+1:02d}/{len(selected):02d}] {_now()} | "
                    f"{split_id} | "
                    f"{'reload' if cached else 'computed'} | "
                    f"current={current_fpr:.6f} | "
                    + " | ".join(
                        f"{k}={v:.6f}"
                        for k, v
                        in cap_fprs.items()
                    ),
                    flush=True,
                )

            except Exception as exc:
                failures.append(
                    {
                        "split_id":
                            split_id,

                        "dataset":
                            str(
                                row[
                                    "dataset"
                                ]
                            ),

                        "replicate":
                            int(
                                row[
                                    "replicate"
                                ]
                            ),

                        "error_type":
                            type(
                                exc
                            ).__name__,

                        "error_message":
                            str(
                                exc
                            ),

                        "traceback":
                            traceback.format_exc(),
                    }
                )

                print(
                    f"[FAILED] {_now()} | "
                    f"{split_id} | "
                    f"{type(exc).__name__}: {exc}",
                    flush=True,
                )

        failures_df = pd.DataFrame(
            failures
        )

        if failures:
            _atomic_csv(
                failures_df,
                AUDIT_DIR
                / "failures.csv",
            )

            raise RuntimeError(
                f"{len(failures)} requested splits failed. "
                "See displayed failure table / audit failures.csv."
            )

        split_results = pd.concat(
            [
                p[
                    "split_results"
                ]
                for p in payloads
            ],
            ignore_index=True,
        )

        tail_results = pd.concat(
            [
                p[
                    "tail_results"
                ]
                for p in payloads
            ],
            ignore_index=True,
        )

        coverage_results = pd.concat(
            [
                p[
                    "coverage_results"
                ]
                for p in payloads
            ],
            ignore_index=True,
        )

        method_summary = _aggregate_method_summary(
            split_results
        )

        paired_summary = _aggregate_paired_changes(
            split_results
        )

        tail_aggregate = _aggregate_tail(
            tail_results
        )

        coverage_aggregate = _aggregate_coverage(
            coverage_results
        )

        # ---------------------------------------------------------
        # Durable numerical outputs.
        # ---------------------------------------------------------

        _atomic_csv(
            selected,
            AUDIT_DIR
            / "selected_first20_40v40_splits.csv",
        )

        _atomic_csv(
            split_results,
            AUDIT_DIR
            / "split_results.csv",
        )

        _atomic_csv(
            method_summary,
            AUDIT_DIR
            / "method_summary.csv",
        )

        _atomic_csv(
            paired_summary,
            AUDIT_DIR
            / "paired_vs_current_summary.csv",
        )

        _atomic_csv(
            tail_results,
            AUDIT_DIR
            / "tail_results.csv",
        )

        _atomic_csv(
            tail_aggregate,
            AUDIT_DIR
            / "tail_aggregate.csv",
        )

        _atomic_csv(
            coverage_results,
            AUDIT_DIR
            / "coverage_results.csv",
        )

        _atomic_csv(
            coverage_aggregate,
            AUDIT_DIR
            / "coverage_aggregate.csv",
        )

        # ---------------------------------------------------------
        # Display summaries.
        # ---------------------------------------------------------

        print(
            "\nDATASET x METHOD SUMMARY — FIRST 20 REPS\n",
            flush=True,
        )

        display(
            method_summary
        )

        print(
            "\nPAIRED CHANGE VERSUS CURRENT DEB\n",
            flush=True,
        )

        display(
            paired_summary
        )

        print(
            "\nEXTREME-TAIL CALIBRATION — POOLED WITHIN DATASET\n",
            flush=True,
        )

        display(
            tail_aggregate
        )

        print(
            "\nCOVERAGE SUMMARY\n",
            flush=True,
        )

        display(
            coverage_aggregate
        )

        # Compact cross-dataset comparison at the most useful thresholds.
        compact_tail = tail_aggregate.loc[
            tail_aggregate[
                "threshold"
            ].isin(
                [
                    5e-6,
                    1e-4,
                    1e-3,
                    1e-2,
                ]
            ),
            [
                "dataset",
                "method",
                "threshold",
                "pooled_ratio_to_nominal",
                "pooled_count",
                "pooled_n",
            ],
        ].copy()

        print(
            "\nCOMPACT EXTREME-TAIL CHECK\n",
            flush=True,
        )

        display(
            compact_tail
        )

        if SHOW_PLOTS:
            _show_plots(
                split_results,
                method_summary,
                tail_aggregate,
                coverage_aggregate,
            )

        ended = (
            datetime.now()
            .astimezone()
        )

        print(
            "\nSTATUS\n"
            f"successful requested splits: {len(payloads)}/{len(selected)}\n"
            f"reused audit caches:         {reused}\n"
            f"source benchmark:            {BENCHMARK_DIR}\n"
            f"audit output:                {AUDIT_DIR}\n"
            f"[end] {_now()} | elapsed={ended-started}",
            flush=True,
        )

        return {
            "selected_splits":
                selected,

            "split_results":
                split_results,

            "method_summary":
                method_summary,

            "paired_summary":
                paired_summary,

            "tail_results":
                tail_results,

            "tail_aggregate":
                tail_aggregate,

            "coverage_results":
                coverage_results,

            "coverage_aggregate":
                coverage_aggregate,

            "failures":
                failures_df,
        }

    finally:
        pass


# =============================================================================
# EXECUTION
# =============================================================================

li2020_40v40_first20_capped_deb_audit = (
    run_li2020_40v40_first20_capped_deb_audit()
)


[start] 2026-10-03 01:43:24 PM Eastern Daylight Time


RuntimeError: The requested FIRST 20 replicates are not all complete. Missing dataset/replicate pairs: [('AdLC', 16), ('AdLC', 17), ('AdLC', 18), ('AdLC', 19), ('AdLC', 20), ('OC', 19), ('OC', 20), ('TNBC', 17), ('TNBC', 18), ('TNBC', 19), ('TNBC', 20)]